1)Imports + fixed project paths

In [ ]:
from pathlib import Path
import re
import warnings

import pandas as pd
import numpy as np
import pdfplumber

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

SITUATION_DIR = PROJECT_ROOT / "data" / "raw" / "situation_reports"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SITUATION_OUTPUT = PROCESSED_DIR / "dmc_situation_report_totals.csv"

print("Project root :", PROJECT_ROOT)
print("Situation dir:", SITUATION_DIR)
print("Processed dir:", PROCESSED_DIR)

2)Check all Situation Report PDFs

In [ ]:
situation_pdfs = sorted(SITUATION_DIR.glob("*.pdf"))

print("Total Situation PDFs found:", len(situation_pdfs))

if situation_pdfs:
    print("\nFirst PDF:")
    print(situation_pdfs[0].name)

    print("\nLast PDF:")
    print(situation_pdfs[-1].name)

3)PDF text extraction

In [ ]:
def extract_pdf_text(pdf_path):
    pages_text = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()

                if text:
                    pages_text.append(text)

        return "\n".join(pages_text)

    except Exception as e:
        print(f"Error reading {pdf_path.name}: {e}")
        return ""

4)Helper functions

In [ ]:
def clean_number(value):
    if value is None:
        return np.nan

    value = str(value).replace(",", "").strip()

    match = re.search(r"\d+", value)

    if match:
        return int(match.group())

    return np.nan


def extract_date_from_filename(filename):
    match = re.search(
        r"(20\d{2})[-_]?(\d{2})[-_]?(\d{2})",
        filename
    )

    if match:
        year, month, day = match.groups()

        try:
            return pd.Timestamp(
                year=int(year),
                month=int(month),
                day=int(day)
            )
        except ValueError:
            return pd.NaT

    return pd.NaT


def normalize_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return text


def get_lines(text):
    text = normalize_text(text)

    return [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

5)Disaster type detection

In [ ]:
def detect_disaster_types(text):
    text = text.lower()

    disaster_types = []

    disaster_keywords = {
        "Flood": [
            "flood",
            "flooding",
        ],

        "Landslide": [
            "landslide",
            "land slide",
        ],

        "Heavy Rain": [
            "heavy rain",
            "heavy rainfall",
        ],

        "Strong Wind": [
            "strong wind",
            "high wind",
        ],

        "Cyclone": [
            "cyclone",
            "cyclonic",
        ],

        "Lightning": [
            "lightning",
            "thunderstorm",
        ],
    }

    for disaster, keywords in disaster_keywords.items():

        if any(keyword in text for keyword in keywords):
            disaster_types.append(disaster)

    if not disaster_types:
        return "Other"

    return ", ".join(disaster_types)

6)TOTAL_FIELDS

In [ ]:
TOTAL_FIELDS = [
    "affected_families",
    "affected_people",
    "deaths",
    "injured",
    "missing",
    "fully_damaged_houses",
    "partially_damaged_houses",
    "enterprise_damage",
    "infrastructure_damage",
    "safety_centres",
    "families_in_safety_centres",
    "people_in_safety_centres",
]

7)extract_grand_total()

In [ ]:
def extract_grand_total(text):

    lines = get_lines(text)

    # =====================================================
    # 1. Normal GRAND TOTAL - legacy 12-column format
    # =====================================================

    for line in lines:

        if "grand total" in line.lower():

            part = re.split(
                r"grand\s+total",
                line,
                flags=re.IGNORECASE
            )[-1]

            numbers = re.findall(
                r"(?<!\d)\d[\d,]*(?!\d)",
                part
            )

            values = [
                int(x.replace(",", ""))
                for x in numbers
            ]

            if len(values) >= 12:
                return values[:12], "grand_total"

    # =====================================================
    # 2. District / Province Total confirms all-zero report
    # =====================================================

    for line in lines:

        line_lower = line.lower()

        if (
            "district total" in line_lower
            or "province total" in line_lower
        ):

            numbers = re.findall(
                r"(?<!\d)\d[\d,]*(?!\d)",
                line
            )

            values = [
                int(x.replace(",", ""))
                for x in numbers
            ]

            if (
                len(values) >= 12
                and all(
                    value == 0
                    for value in values[-12:]
                )
            ):
                return [0] * 12, "confirmed_zero_report"

    # =====================================================
    # 3. Broken GRAND TOTAL (#REF!)
    # Use ONE valid Province Total if available
    # =====================================================

    has_ref_error = any(
        (
            "grand total" in line.lower()
            and "#ref!" in line.lower()
        )
        for line in lines
    )

    if has_ref_error:

        province_candidates = []

        for line in lines:

            if "province total" in line.lower():

                numbers = re.findall(
                    r"(?<!\d)\d[\d,]*(?!\d)",
                    line
                )

                values = [
                    int(x.replace(",", ""))
                    for x in numbers
                ]

                if len(values) >= 12:
                    province_candidates.append(
                        values[-12:]
                    )

        if len(province_candidates) == 1:
            return (
                province_candidates[0],
                "grand_total_ref_fallback"
            )

    # =====================================================
    # 4. Nill/Nil zero-impact report
    # =====================================================

    has_grand_total = any(
        "grand total" in line.lower()
        for line in lines
    )

    nil_count = len(
        re.findall(
            r"\b(?:nil|nill)\b",
            text,
            flags=re.IGNORECASE
        )
    )

    numeric_summary_exists = False

    for line in lines:

        numbers = re.findall(
            r"(?<!\d)\d[\d,]*(?!\d)",
            line
        )

        if len(numbers) >= 5:
            numeric_summary_exists = True
            break

    if (
        has_grand_total
        and nil_count >= 3
        and not numeric_summary_exists
    ):
        return [0] * 12, "confirmed_zero_report"

    return None, "failed"

8)extract_modern_total()

In [ ]:
def extract_modern_total(text):

    lines = get_lines(text)

    # =====================================================
    # English "Total ..." rows
    # =====================================================

    for line in lines:

        if not re.match(
            r"^\s*total\b",
            line,
            flags=re.IGNORECASE
        ):
            continue

        numbers = re.findall(
            r"(?<!\d)\d[\d,]*(?!\d)",
            line
        )

        values = [
            int(x.replace(",", ""))
            for x in numbers
        ]

        # -------------------------------------------------
        # 18-column modern format
        # -------------------------------------------------

        if len(values) == 18:

            return {
                "peak_affected_families": values[0],
                "peak_affected_people": values[1],

                "affected_families": values[2],
                "affected_people": values[3],

                "deaths": values[4],
                "injured": np.nan,
                "missing": values[5],

                "fully_damaged_houses": values[6],
                "partially_damaged_houses": values[7],

                "enterprise_damage": np.nan,
                "infrastructure_damage": np.nan,

                "peak_safety_centres": values[8],
                "peak_families_in_safety_centres": values[9],
                "peak_people_in_safety_centres": values[10],

                "safety_centres": values[11],
                "families_in_safety_centres": values[12],
                "people_in_safety_centres": values[13],

                "families_with_relatives": values[14],
                "people_with_relatives": values[15],

                "idp_families": values[16],
                "idp_people": values[17],

            }, "modern_18_column"

        # -------------------------------------------------
        # 13-column modern format
        # -------------------------------------------------

        if len(values) == 13:

            return {
                "affected_families": values[0],
                "affected_people": values[1],

                "deaths": values[2],
                "injured": np.nan,
                "missing": values[3],

                "fully_damaged_houses": values[4],
                "partially_damaged_houses": values[5],

                "enterprise_damage": np.nan,
                "infrastructure_damage": np.nan,

                "safety_centres": values[6],
                "families_in_safety_centres": values[7],
                "people_in_safety_centres": values[8],

                "families_with_relatives": values[9],
                "people_with_relatives": values[10],

                "idp_families": values[11],
                "idp_people": values[12],

            }, "modern_13_column"

        # -------------------------------------------------
        # 11-column modern format
        # -------------------------------------------------

        if len(values) == 11:

            return {
                "affected_families": values[0],
                "affected_people": values[1],

                "deaths": values[2],
                "injured": np.nan,
                "missing": values[3],

                "fully_damaged_houses": values[4],
                "partially_damaged_houses": values[5],

                "enterprise_damage": np.nan,
                "infrastructure_damage": np.nan,

                "safety_centres": values[6],
                "families_in_safety_centres": values[7],
                "people_in_safety_centres": values[8],

                "families_with_relatives": values[9],
                "people_with_relatives": values[10],

            }, "modern_11_column"

        # -------------------------------------------------
        # 9-column modern format
        # -------------------------------------------------

        if len(values) == 9:

            return {
                "affected_families": values[0],
                "affected_people": values[1],

                "deaths": values[2],
                "injured": np.nan,
                "missing": values[3],

                "fully_damaged_houses": values[4],
                "partially_damaged_houses": values[5],

                "enterprise_damage": np.nan,
                "infrastructure_damage": np.nan,

                "safety_centres": values[6],
                "families_in_safety_centres": values[7],
                "people_in_safety_centres": values[8],

            }, "modern_9_column"

        # -------------------------------------------------
        # 7-column compact Total format
        # -------------------------------------------------

        if len(values) == 7:

                if values[1] >= values[0]:

                    return {
                        "affected_families": values[0],
                        "affected_people": values[1],

                        "deaths": values[2],
                        "injured": np.nan,
                        "missing": values[3],

                        "fully_damaged_houses": np.nan,
                        "partially_damaged_houses": np.nan,

                        "enterprise_damage": np.nan,
                        "infrastructure_damage": np.nan,

                        "safety_centres": values[4],
                        "families_in_safety_centres": values[5],
                        "people_in_safety_centres": values[6],

                    }, "modern_7_column"

    # =====================================================
    # Sinhala / summary-style total rows
    # =====================================================

    total_markers = [
        "එකතුව",
        "එ තුව",
    ]

    for line in lines:

        if not any(
            marker in line
            for marker in total_markers
        ):
            continue

        numbers = re.findall(
            r"(?<!\d)\d[\d,]*(?!\d)",
            line
        )

        values = [
            int(x.replace(",", ""))
            for x in numbers
        ]

        # -------------------------------------------------
        # 13 values:
        # DS count + 12 human-impact values
        # -------------------------------------------------

        if len(values) == 13:

            return {
                "affected_families": values[1],
                "affected_people": values[2],

                "deaths": values[3],
                "injured": values[4],
                "missing": values[5],

                "fully_damaged_houses": values[6],
                "partially_damaged_houses": values[7],

                "enterprise_damage": np.nan,
                "infrastructure_damage": np.nan,

                "families_with_relatives": values[8],
                "people_with_relatives": values[9],

                "safety_centres": values[10],
                "families_in_safety_centres": values[11],
                "people_in_safety_centres": values[12],

            }, "modern_summary_13_column"

        # -------------------------------------------------
        # 9-column Sinhala summary
        # -------------------------------------------------

        if len(values) == 9:

            if values[1] >= values[0]:

                return {
                    "affected_families": values[0],
                    "affected_people": values[1],

                    "deaths": values[2],
                    "injured": np.nan,
                    "missing": values[3],

                    "fully_damaged_houses": values[4],
                    "partially_damaged_houses": values[5],

                    "enterprise_damage": np.nan,
                    "infrastructure_damage": np.nan,

                    "safety_centres": values[6],
                    "families_in_safety_centres": values[7],
                    "people_in_safety_centres": values[8],

                }, "modern_summary_9_column"

        # -------------------------------------------------
        # 8 values = DS count + compact 7-value summary
        # -------------------------------------------------

        if len(values) == 8:

            return {
                "affected_families": values[1],
                "affected_people": values[2],

                "deaths": values[3],
                "injured": np.nan,
                "missing": values[4],

                "fully_damaged_houses": np.nan,
                "partially_damaged_houses": np.nan,

                "enterprise_damage": np.nan,
                "infrastructure_damage": np.nan,

                "safety_centres": values[5],
                "families_in_safety_centres": values[6],
                "people_in_safety_centres": values[7],

            }, "modern_summary_8_column"

            
        # -------------------------------------------------
        # 7-value summary
        # -------------------------------------------------

        if len(values) == 7:

            if values[1] >= values[0]:

                return {
                    "affected_families": values[0],
                    "affected_people": values[1],

                    "deaths": values[2],
                    "injured": np.nan,
                    "missing": values[3],

                    "fully_damaged_houses": np.nan,
                    "partially_damaged_houses": np.nan,

                    "enterprise_damage": np.nan,
                    "infrastructure_damage": np.nan,

                    "safety_centres": values[4],
                    "families_in_safety_centres": values[5],
                    "people_in_safety_centres": values[6],

                }, "modern_summary_7_column"

    return None, None

9)is_non_comparable_incident_report

In [ ]:
def is_non_comparable_incident_report(text):

    text_lower = normalize_text(text).lower()

    return (
        "daily disaster incident report" in text_lower
        and "situation report" not in text_lower
    )

10)is_non_situation_report

In [ ]:
def is_non_situation_report(text):

    text_lower = normalize_text(text).lower()

    # Weather forecast documents
    weather_markers = [
        "weather forecast",
        "rainfall amount",
        "condition of rain",
        "state of sea",
        "showers or thundershowers",
        "department of meteorology",
        "wind speed",
        "sea areas",
    ]

    weather_score = sum(
        marker in text_lower
        for marker in weather_markers
    )

    if weather_score >= 2:
        return True

    # River/water-level documents
    river_markers = [
        "minor flood",
        "major flood",
        "river level",
        "water level",
        "rising",
        "falling",
    ]

    river_score = sum(
        marker in text_lower
        for marker in river_markers
    )

    if river_score >= 2:
        return True

    return False

11)Extract Situation Report

In [ ]:
def extract_situation_report(pdf_path):

    text = extract_pdf_text(pdf_path)

    record = {
        "file_name": pdf_path.name,
        "report_date": extract_date_from_filename(
            pdf_path.name
        ),
        "disaster_types": detect_disaster_types(text),
        "text_length": len(text),
    }

    # Optional fields used in newer layouts
    record["families_with_relatives"] = np.nan
    record["people_with_relatives"] = np.nan
    record["peak_affected_families"] = np.nan
    record["peak_affected_people"] = np.nan

    record["peak_safety_centres"] = np.nan
    record["peak_families_in_safety_centres"] = np.nan
    record["peak_people_in_safety_centres"] = np.nan

    record["idp_families"] = np.nan
    record["idp_people"] = np.nan

    # =====================================================
    # 1. Legacy DMC Situation Report layout
    # =====================================================

    totals, extraction_status = extract_grand_total(text)

    if totals is not None:

        for field, value in zip(
            TOTAL_FIELDS,
            totals
        ):
            record[field] = value

        record["grand_total_extracted"] = True
        record["extraction_status"] = extraction_status

        return record

    # =====================================================
    # 2. Modern DMC Situation Report layouts
    # =====================================================

    modern_values, modern_status = extract_modern_total(text)

    if modern_values is not None:

        for field, value in modern_values.items():
            record[field] = value

        record["grand_total_extracted"] = True
        record["extraction_status"] = modern_status

        return record
    # -----------------------------------------------------
    # Non-comparable daily incident report
    # -----------------------------------------------------

    if is_non_comparable_incident_report(text):

        for field in TOTAL_FIELDS:
            record[field] = np.nan

        record["grand_total_extracted"] = False
        record["extraction_status"] = (
            "non_comparable_incident_report"
        )

        return record
    # =====================================================
    # 3. Non-situation documents accidentally in archive
    # =====================================================

    if is_non_situation_report(text):

        for field in TOTAL_FIELDS:
            record[field] = np.nan

        record["grand_total_extracted"] = False
        record["extraction_status"] = "non_situation_report"

        return record

    # =====================================================
    # 4. Still unresolved
    # =====================================================

    for field in TOTAL_FIELDS:
        record[field] = np.nan

    record["grand_total_extracted"] = False

    if len(text.strip()) < 500:
        record["extraction_status"] = "insufficient_text"
    else:
        record["extraction_status"] = "failed"

    return record

12)Full extraction

In [ ]:
all_records = []

for i, pdf_path in enumerate(situation_pdfs, start=1):

    record = extract_situation_report(pdf_path)
    all_records.append(record)

    if i % 50 == 0 or i == len(situation_pdfs):
        print(
            f"Processed {i}/{len(situation_pdfs)} PDFs"
        )

situation_df = pd.DataFrame(all_records)

print("\nExtraction completed.")
print("Total extracted reports:", len(situation_df))

13)Convert to manual review

In [ ]:
mask = (
    situation_df["file_name"]
    == "situation_20260723_1000_677402eb.pdf"
)

situation_df.loc[
    mask,
    "extraction_status"
] = "manual_review_unsupported_layout"

In [ ]:
print(
    situation_df["extraction_status"]
    .value_counts()
)

print(
    "\nMissing affected_people:",
    situation_df["affected_people"].isna().sum()
)

14)extraction quality check

In [ ]:
print("Total reports:", len(situation_df))

print("\nExtraction status:")
print(
    situation_df["extraction_status"]
    .value_counts(dropna=False)
)

print("\nGrand total extracted:")
print(
    situation_df["grand_total_extracted"]
    .value_counts(dropna=False)
)

print("\nDate range:")
print("Oldest:", situation_df["report_date"].min())
print("Newest:", situation_df["report_date"].max())

print("\nMissing affected_people:")
print(
    situation_df["affected_people"]
    .isna()
    .sum()
)

print("\nMissing affected_families:")
print(
    situation_df["affected_families"]
    .isna()
    .sum()
)

15)Processed CSV

In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    PROCESSED_DIR
    / "situation_reports_cleaned.csv"
)

situation_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(situation_df))

# 2. Weather Report Preprocessing

In [ ]:
WEATHER_DIR = Path("data/raw/weather_reports")

weather_pdfs = sorted(
    WEATHER_DIR.glob("*.pdf")
)

print(
    "Total Weather PDFs found:",
    len(weather_pdfs)
)

In [ ]:
# =========================================================
# Inspect sample Weather PDFs
# =========================================================

sample_indexes = [
    0,
    len(weather_pdfs) // 4,
    len(weather_pdfs) // 2,
    (3 * len(weather_pdfs)) // 4,
    len(weather_pdfs) - 1
]

for idx in sample_indexes:

    pdf_path = weather_pdfs[idx]
    text = extract_pdf_text(pdf_path)

    print("\n" + "=" * 100)
    print("PDF:", pdf_path.name)
    print("Text length:", len(text))
    print("=" * 100)

    print(text[:3000])

In [ ]:
import re
from collections import Counter

weather_types = []

for pdf_path in weather_pdfs:

    match = re.match(
        r"weather_\d{8}_\d{4}_(.+)_[0-9a-f]{8}\.pdf$",
        pdf_path.name
    )

    if match:
        weather_types.append(
            match.group(1)
        )
    else:
        weather_types.append(
            "unknown"
        )

weather_type_counts = Counter(weather_types)

for report_type, count in weather_type_counts.most_common():
    print(f"{report_type}: {count}")

In [ ]:
# =========================================================
# Inspect one Weather PDF from each report type
# =========================================================

report_types_to_check = [
    "thunderstorm",
    "heavy_rain",
    "strong_wind",
    "depression",
    "low_pressure",
    "cyclone",
    "rain_related",
    "unknown",
]

for report_type in report_types_to_check:

    matching_files = [
        p for p, t in zip(weather_pdfs, weather_types)
        if t == report_type
    ]

    print("\n" + "=" * 100)
    print("REPORT TYPE:", report_type)
    print("COUNT:", len(matching_files))
    print("=" * 100)

    if not matching_files:
        print("No files found")
        continue

    pdf_path = matching_files[0]
    text = extract_pdf_text(pdf_path)

    print("PDF:", pdf_path.name)
    print("Text length:", len(text))
    print("-" * 100)

    print(text[:3500])

In [ ]:
# =========================================================
# Common Text Helper Functions
# =========================================================

def normalize_text(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace("\u00a0", " ")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def get_lines(text):

    if text is None:
        return []

    return [
        normalize_text(line)
        for line in str(text).splitlines()
        if normalize_text(line)
    ]

In [ ]:
# =========================================================
# Weather Report Parser
# =========================================================

WEATHER_FILENAME_PATTERN = re.compile(
    r"^weather_(\d{8})_(\d{4}|unknown)_(.+)_([0-9a-f]{8})\.pdf$",
    flags=re.IGNORECASE
)


def parse_weather_filename(file_name):

    match = WEATHER_FILENAME_PATTERN.match(file_name)

    if not match:
        return {
            "report_date": pd.NaT,
            "issue_time": np.nan,
            "report_type": "unknown",
        }

    date_text, time_text, report_type, _ = match.groups()

    return {
        "report_date": pd.to_datetime(
            date_text,
            format="%Y%m%d",
            errors="coerce"
        ),

        "issue_time": (
            np.nan
            if time_text.lower() == "unknown"
            else time_text
        ),

        "report_type": report_type.lower(),
    }


def remove_weather_legends(text):

    cleaned_lines = []

    for line in get_lines(text):

        line_lower = normalize_text(line).lower()

        # Do NOT treat rainfall classification legends
        # as actual forecast rainfall.
        if "rainfall amount (mm)" in line_lower:
            continue

        if "spatial rainfall distribution" in line_lower:
            continue

        if "precipitation rate categories" in line_lower:
            continue

        cleaned_lines.append(line)

    return " ".join(cleaned_lines)

def extract_land_weather_text(text):

    text_lower = text.lower()

    # If the bulletin has an explicit land section,
    # start from there.
    land_markers = [
        "for land:",
        "for land areas",
        "for land area",
    ]

    start_index = 0

    for marker in land_markers:
        idx = text_lower.find(marker)

        if idx != -1:
            start_index = idx
            break

    land_text = text[start_index:]
    land_lower = land_text.lower()

    # Stop before marine / sea forecast section
    marine_markers = [
        "weather forecast for sea areas around the island",
        "for the sea areas around the island",
        "for sea areas around the island",
        "for multi-day boats",
        "for multi day boats",
    ]

    end_positions = []

    for marker in marine_markers:

        idx = land_lower.find(marker)

        if idx != -1:
            end_positions.append(idx)

    if end_positions:
        land_text = land_text[
            :min(end_positions)
        ]

    return land_text

def extract_rainfall_threshold_mm(text):

    cleaned_text = remove_weather_legends(text)

    patterns = [
        r"(?:above|over|about|around|approximately|more than)\s*"
        r"(\d{2,3}(?:\.\d+)?)\s*mm",

        r"(\d{2,3}(?:\.\d+)?)\s*mm\s*"
        r"(?:or more|and above)",
    ]

    rainfall_values = []

    for pattern in patterns:

        matches = re.findall(
            pattern,
            cleaned_text,
            flags=re.IGNORECASE
        )

        rainfall_values.extend(
            float(value)
            for value in matches
        )

    if rainfall_values:
        return max(rainfall_values)

    return np.nan


def extract_max_wind_kmph(text):

    cleaned_text = remove_weather_legends(text)

    wind_values = []

    # Example: (50-60) kmph
    ranges = re.findall(
        r"\(?\s*(\d{1,3})\s*[-–]\s*(\d{1,3})"
        r"\s*\)?\s*kmph",
        cleaned_text,
        flags=re.IGNORECASE
    )

    for low, high in ranges:
        wind_values.append(
            max(float(low), float(high))
        )

    # Example: up to 70 kmph
    singles = re.findall(
        r"(?:up to|about|around)\s*"
        r"\(?\s*(\d{1,3})\s*\)?\s*kmph",
        cleaned_text,
        flags=re.IGNORECASE
    )

    wind_values.extend(
        float(value)
        for value in singles
    )

    if wind_values:
        return max(wind_values)

    return np.nan


def extract_warning_color(text):

    match = re.search(
        r"\bCOLOR\s*:\s*(Red|Amber|Green)\b",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).title()

    return np.nan


def extract_forecast_horizon_hours(text):

    # Hours
    match = re.search(
        r"(?:next|for next)\s*(\d{1,3})\s*hours?",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    # Days
    match = re.search(
        r"(?:next|for next)\s*(\d{1,2})\s*days?",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1)) * 24

    return np.nan

def extract_weather_report(pdf_path):

    text = extract_pdf_text(pdf_path)

    metadata = parse_weather_filename(
        pdf_path.name
    )

    text_lower = normalize_text(text).lower()

    report_type = metadata["report_type"]

    # -----------------------------------------------------
    # Detect land / marine context
    # -----------------------------------------------------

    has_land_context = (
        "for land" in text_lower
        or "land areas" in text_lower
        or "weather forecast for sri lanka" in text_lower
        or "general weather forecast for sri lanka" in text_lower
    )

    has_marine_context = (
        "multi-day boats" in text_lower
        or "multi day boats" in text_lower
        or "sea areas" in text_lower
        or "fishing and naval" in text_lower
    )

    marine_only_flag = (
        has_marine_context
        and not has_land_context
    )

    # -----------------------------------------------------
    # Extract land-weather section only
    # -----------------------------------------------------

    land_text = extract_land_weather_text(text)

    rainfall_mm = extract_rainfall_threshold_mm(
        land_text
    )

    max_wind = extract_max_wind_kmph(
        land_text
    )

    # Marine-only reports must not supply
    # land rainfall/wind values
    if marine_only_flag:
        rainfall_mm = np.nan
        max_wind = np.nan

    # -----------------------------------------------------
    # Hazard indicators
    # -----------------------------------------------------

    heavy_rain_flag = (
        not marine_only_flag
        and (
            report_type == "heavy_rain"
            or (
                not pd.isna(rainfall_mm)
                and rainfall_mm >= 50
            )
        )
    )

    thunderstorm_flag = (
        not marine_only_flag
        and (
            report_type == "thunderstorm"
            or "thundershower" in text_lower
            or "thunderstorm" in text_lower
            or "thunder shower" in text_lower
        )
    )
    strong_wind_flag = (
    not marine_only_flag
    and (
        report_type == "strong_wind"
        or (
            not pd.isna(max_wind)
            and max_wind >= 40
        )
    )
    )

    low_pressure_flag = (
        report_type == "low_pressure"
        or "low pressure area" in text_lower
    )

    depression_flag = (
        report_type == "depression"
        or "depression" in text_lower
    )

    cyclone_flag = (
        report_type == "cyclone"
        or "cyclonic storm" in text_lower
        or "cyclone" in text_lower
    )

    experimental_flag = (
        "experimental" in text_lower
    )

    # -----------------------------------------------------
    # Final record
    # -----------------------------------------------------

    record = {
        "file_name": pdf_path.name,

        "report_date": metadata["report_date"],
        "issue_time": metadata["issue_time"],
        "report_type": report_type,

        "warning_color": extract_warning_color(
            text
        ),

        "forecast_horizon_hours":
            extract_forecast_horizon_hours(text),

        "rainfall_threshold_mm":
            rainfall_mm,

        "max_wind_kmph":
            max_wind,

        "heavy_rain_flag":
            heavy_rain_flag,

        "thunderstorm_flag":
            thunderstorm_flag,

        "strong_wind_flag":
            strong_wind_flag,

        "low_pressure_flag":
            low_pressure_flag,

        "depression_flag":
            depression_flag,

        "cyclone_flag":
            cyclone_flag,

        "experimental_flag":
            experimental_flag,

        "marine_only_flag":
            marine_only_flag,

        "text_length":
            len(text),

        "extraction_status": (
            "ok"
            if len(text.strip()) >= 100
            else "insufficient_text"
        ),
    }

    return record


full 4644 extraction

In [ ]:
weather_records = []

for i, pdf_path in enumerate(
    weather_pdfs,
    start=1
):

    record = extract_weather_report(
        pdf_path
    )

    weather_records.append(record)

    if (
        i % 100 == 0
        or i == len(weather_pdfs)
    ):
        print(
            f"Processed {i}/{len(weather_pdfs)} PDFs"
        )

weather_df = pd.DataFrame(
    weather_records
)

print("\nWeather extraction completed.")
print(
    "Total reports:",
    len(weather_df)
)

In [ ]:
# =========================================================
# Weather Extraction Quality Check
# =========================================================

print("Total reports:", len(weather_df))

print("\nExtraction status:")
print(
    weather_df["extraction_status"]
    .value_counts(dropna=False)
)

print("\nReport types:")
print(
    weather_df["report_type"]
    .value_counts(dropna=False)
)

print("\nWarning colors:")
print(
    weather_df["warning_color"]
    .value_counts(dropna=False)
)

print("\nMissing rainfall threshold:")
print(
    weather_df["rainfall_threshold_mm"]
    .isna()
    .sum()
)

print("\nMissing max wind:")
print(
    weather_df["max_wind_kmph"]
    .isna()
    .sum()
)

print("\nHeavy rain flag:")
print(
    weather_df["heavy_rain_flag"]
    .value_counts(dropna=False)
)

print("\nStrong wind flag:")
print(
    weather_df["strong_wind_flag"]
    .value_counts(dropna=False)
)

print("\nMarine only:")
print(
    weather_df["marine_only_flag"]
    .value_counts(dropna=False)
)

print("\nExperimental:")
print(
    weather_df["experimental_flag"]
    .value_counts(dropna=False)
)

print("\nDate range:")
print("Oldest:", weather_df["report_date"].min())
print("Newest:", weather_df["report_date"].max())

print("\nRainfall threshold range:")
print(
    weather_df["rainfall_threshold_mm"].min(),
    "to",
    weather_df["rainfall_threshold_mm"].max()
)

print("\nWind speed range:")
print(
    weather_df["max_wind_kmph"].min(),
    "to",
    weather_df["max_wind_kmph"].max()
)

CSV Ptocess


In [ ]:
WEATHER_OUTPUT = (
    PROCESSED_DIR
    / "weather_reports_cleaned.csv"
)

weather_df.to_csv(
    WEATHER_OUTPUT,
    index=False
)

print("Saved:", WEATHER_OUTPUT)
print("Rows:", len(weather_df))

# 3. River Report Preprocessing

In [ ]:
RIVER_DIR = Path("data/raw/river_reports")

river_pdfs = sorted(
    RIVER_DIR.glob("*.pdf")
)

print(
    "Total River PDFs found:",
    len(river_pdfs)
)

In [ ]:
# =========================================================
# Count River Report Types
# =========================================================

import re
from collections import Counter

river_types = []

for pdf_path in river_pdfs:

    match = re.match(
        r"^river_\d{8}_(?:\d{4}|unknown)_(.+)_[0-9a-f]{8}\.pdf$",
        pdf_path.name,
        flags=re.IGNORECASE
    )

    if match:
        river_types.append(
            match.group(1)
        )
    else:
        river_types.append(
            "unknown"
        )

river_type_counts = Counter(
    river_types
)

for report_type, count in river_type_counts.most_common():
    print(
        f"{report_type}: {count}"
    )

In [ ]:
# =========================================================
# Inspect One Sample from Each River Report Type
# =========================================================

river_types_to_check = [
    "water_level",
    "water_level_rainfall",
    "flood_warning",
    "flood_advisory",
]

for report_type in river_types_to_check:

    matching_files = [
        p for p, t in zip(river_pdfs, river_types)
        if t == report_type
    ]

    print("\n" + "=" * 100)
    print("REPORT TYPE:", report_type)
    print("COUNT:", len(matching_files))
    print("=" * 100)

    if not matching_files:
        continue

    pdf_path = matching_files[0]

    text = extract_pdf_text(pdf_path)

    print("PDF:", pdf_path.name)
    print("Text length:", len(text))
    print("-" * 100)

    print(text[:4000])

In [ ]:
# =========================================================
# River Report Parser
# =========================================================

RIVER_FILENAME_PATTERN = re.compile(
    r"^river_(\d{8})_(\d{4}|unknown)_(.+)_([0-9a-f]{8})\.pdf$",
    flags=re.IGNORECASE
)


def parse_river_filename(file_name):

    match = RIVER_FILENAME_PATTERN.match(file_name)

    if not match:
        return {
            "report_date": pd.NaT,
            "issue_time": np.nan,
            "report_type": "unknown",
        }

    date_text, time_text, report_type, _ = match.groups()

    return {
        "report_date": pd.to_datetime(
            date_text,
            format="%Y%m%d",
            errors="coerce"
        ),

        "issue_time": (
            np.nan
            if time_text.lower() == "unknown"
            else time_text
        ),

        "report_type": report_type.lower(),
    }


RIVER_ROW_PATTERN = re.compile(
    r"^(?P<label>.+?)\s+"
    r"(?P<unit>m|ft)\s+"
    r"(?P<alert>-?\d+(?:\.\d+)?)\s+"
    r"(?P<minor>-?\d+(?:\.\d+)?)\s+"
    r"(?P<major>-?\d+(?:\.\d+)?)\s+"
    r"(?P<previous>-?\d+(?:\.\d+)?)\s+"
    r"(?P<current>-?\d+(?:\.\d+)?)\s+"
    r"(?P<status>"
    r"Normal|Alert|Minor Flood|Major Flood"
    r")"
    r"(?:\s+(?P<trend>Rising|Falling))?\s+"
    r"(?P<rain>-|NA|-?\d+(?:\.\d+)?)$",
    flags=re.IGNORECASE
)


def extract_river_station_rows(
    text,
    pdf_path,
    metadata
):

    rows = []

    for line in get_lines(text):

        match = RIVER_ROW_PATTERN.match(
            line.strip()
        )

        if not match:
            continue

        values = match.groupdict()

        previous_level = float(
            values["previous"]
        )

        current_level = float(
            values["current"]
        )

        level_change = (
            current_level
            - previous_level
        )

        if level_change > 0:
            derived_trend = "Rising"
        elif level_change < 0:
            derived_trend = "Falling"
        else:
            derived_trend = "Stable"

        rainfall_text = values["rain"]

        rainfall_24h = (
            np.nan
            if rainfall_text.upper() in ["-", "NA"]
            else float(rainfall_text)
        )

        alert_level = float(
            values["alert"]
        )

        alert_ratio = (
            current_level / alert_level
            if alert_level > 0
            else np.nan
        )

        row = {
            "file_name": pdf_path.name,
            "report_date": metadata["report_date"],
            "issue_time": metadata["issue_time"],
            "report_type": metadata["report_type"],

            # Keep complete station label safely.
            # We will standardize station/basin names later.
            "station_label_raw":
                normalize_text(values["label"]),

            "unit":
                values["unit"].lower(),

            "alert_level":
                alert_level,

            "minor_flood_level":
                float(values["minor"]),

            "major_flood_level":
                float(values["major"]),

            "water_level_previous":
                previous_level,

            "water_level_current":
                current_level,

            "level_change":
                level_change,

            "status":
                values["status"].title(),

            "reported_trend": (
                values["trend"].title()
                if values["trend"]
                else np.nan
            ),

            "derived_trend":
                derived_trend,

            "rainfall_24h_mm":
                rainfall_24h,

            "alert_ratio":
                alert_ratio,
        }

        rows.append(row)

    return rows

def is_river_warning_document(text):

    text_lower = normalize_text(text).lower()

    # Normalized compact version helps with broken PDF text
    # such as "Irrig ation"
    compact_text = re.sub(
        r"\s+",
        "",
        text_lower
    )

    warning_markers = [
        "flood warning message",
        "flood warning",
        "major flood situation",
        "high flood situation",
        "early warning",
        "warning is issued",
        "warning issued",
    ]

    standard_warning = any(
        marker in text_lower
        for marker in warning_markers
    )

    # DMC forwarding an Irrigation Department warning
    dmc_forwarded_warning = (
        "disastermanagementcentre"
        in compact_text
        and (
            "dmc/08/irrigation"
            in compact_text
            or "irrigation"
            in compact_text
        )
    )

    return (
        standard_warning
        or dmc_forwarded_warning
    )


def extract_river_report(pdf_path):

    text = extract_pdf_text(pdf_path)

    metadata = parse_river_filename(
        pdf_path.name
    )

    text_lower = normalize_text(text).lower()

    station_rows = extract_river_station_rows(
        text,
        pdf_path,
        metadata
    )

    station_df_temp = pd.DataFrame(
        station_rows
    )

    report_type = metadata["report_type"]

    # -----------------------------------------------------
    # Warning withdrawal detection
    # -----------------------------------------------------

    # withdrawal_markers = [
    #     "hereby withdrawn",
    #     "withdrawal",
    #     "withdrawn",
    #     "removal",
    #     "threat is over",
    # ]

    # warning_withdrawn_flag = any(
    #     marker in text_lower
    #     for marker in withdrawal_markers
    # )
    # warning_document = (
    # report_type in [
    #     "flood_warning",
    #     "flood_advisory",
    # ]
    # or is_river_warning_document(text)
    # )
    # active_warning_flag = (
    #     warning_document
    #     and not warning_withdrawn_flag
    # )

    # -----------------------------------------------------
    # Warning / Advisory classification
    # -----------------------------------------------------

    warning_document = (
        report_type in [
            "flood_warning",
            "flood_advisory",
        ]
        or (
            len(station_rows) == 0
            and is_river_warning_document(text)
        )
    )

    # Warning withdrawal markers
    withdrawal_markers = [
        "hereby withdrawn",
        "withdrawal",
        "withdrawn",
        "removal",
        "threat is over",
    ]

    warning_withdrawn_flag = (
        warning_document
        and any(
            marker in text_lower
            for marker in withdrawal_markers
        )
    )

    # Active status cannot be determined
    # when the warning PDF has no extractable text.
    if (
        warning_document
        and len(text.strip()) < 100
    ):
        active_warning_flag = np.nan

    else:
        active_warning_flag = (
            warning_document
            and not warning_withdrawn_flag
        )
    # -----------------------------------------------------
    # Station summaries
    # -----------------------------------------------------

    if not station_df_temp.empty:

        status_lower = (
            station_df_temp["status"]
            .str.lower()
        )

        normal_count = int(
            (status_lower == "normal").sum()
        )

        alert_count = int(
            (status_lower == "alert").sum()
        )

        minor_flood_count = int(
            (status_lower == "minor flood").sum()
        )

        major_flood_count = int(
            (status_lower == "major flood").sum()
        )

        rising_count = int(
            (
                station_df_temp[
                    "derived_trend"
                ] == "Rising"
            ).sum()
        )

        falling_count = int(
            (
                station_df_temp[
                    "derived_trend"
                ] == "Falling"
            ).sum()
        )

        max_rainfall = (
            station_df_temp[
                "rainfall_24h_mm"
            ].max()
        )

        mean_rainfall = (
            station_df_temp[
                "rainfall_24h_mm"
            ].mean()
        )

        max_alert_ratio = (
            station_df_temp[
                "alert_ratio"
            ].max()
        )

    else:

        normal_count = 0
        alert_count = 0
        minor_flood_count = 0
        major_flood_count = 0
        rising_count = 0
        falling_count = 0

        max_rainfall = np.nan
        mean_rainfall = np.nan
        max_alert_ratio = np.nan

    # -----------------------------------------------------
    # Extraction status
    # -----------------------------------------------------

    if len(text.strip()) < 100:

        extraction_status = (
            "insufficient_text"
        )

    elif (
        report_type in [
            "water_level",
         "water_level_rainfall",
        ]
        and len(station_rows) == 0
        and is_river_warning_document(text)
    ):

         extraction_status = (
        "warning_document_no_station_table"
        )

    elif (
        report_type in [
            "water_level",
            "water_level_rainfall",
        ]
        and len(station_rows) == 0
    ):

        extraction_status = (
            "no_station_rows"
        )

    else:

        extraction_status = "ok"
   

    return {
        "file_name":
            pdf_path.name,

        "report_date":
            metadata["report_date"],

        "issue_time":
            metadata["issue_time"],

        "report_type":
            report_type,

        "station_rows_extracted":
            len(station_rows),

        "normal_station_count":
            normal_count,

        "alert_station_count":
            alert_count,

        "minor_flood_station_count":
            minor_flood_count,

        "major_flood_station_count":
            major_flood_count,

        "rising_station_count":
            rising_count,

        "falling_station_count":
            falling_count,

        "max_rainfall_24h_mm":
            max_rainfall,

        "mean_rainfall_24h_mm":
            mean_rainfall,

        "max_alert_ratio":
            max_alert_ratio,

        "warning_withdrawn_flag":
            warning_withdrawn_flag,

        "active_warning_flag":
            active_warning_flag,

        "text_length":
            len(text),

        "extraction_status":
            extraction_status,
    }, station_rows

In [ ]:
# =========================================================
# Extract All River Reports
# =========================================================

river_report_records = []
river_station_records = []

for i, pdf_path in enumerate(
    river_pdfs,
    start=1
):

    report_record, station_rows = (
        extract_river_report(pdf_path)
    )

    river_report_records.append(
        report_record
    )

    river_station_records.extend(
        station_rows
    )

    if (
        i % 50 == 0
        or i == len(river_pdfs)
    ):
        print(
            f"Processed {i}/{len(river_pdfs)} PDFs"
        )


river_df = pd.DataFrame(
    river_report_records
)

river_station_df = pd.DataFrame(
    river_station_records
)

print("\nRiver extraction completed.")

print(
    "Total River reports:",
    len(river_df)
)

print(
    "Total station observations:",
    len(river_station_df)
)

In [ ]:
# =========================================================
# River Extraction Quality Check
# =========================================================

print("Total River reports:", len(river_df))

print("\nExtraction status:")
print(
    river_df["extraction_status"]
    .value_counts(dropna=False)
)

print("\nReport types:")
print(
    river_df["report_type"]
    .value_counts(dropna=False)
)

print("\nStation rows extracted per report:")
print(
    river_df["station_rows_extracted"]
    .describe()
)

print("\nTotal station observations:")
print(
    len(river_station_df)
)

print("\nFlood / Alert station totals:")
print(
    "Alert:",
    river_df["alert_station_count"].sum()
)

print(
    "Minor Flood:",
    river_df["minor_flood_station_count"].sum()
)

print(
    "Major Flood:",
    river_df["major_flood_station_count"].sum()
)

print("\nTrend totals:")
print(
    "Rising:",
    river_df["rising_station_count"].sum()
)

print(
    "Falling:",
    river_df["falling_station_count"].sum()
)

print("\nWarning withdrawal:")
print(
    river_df["warning_withdrawn_flag"]
    .value_counts(dropna=False)
)

print("\nActive flood warnings:")
print(
    river_df["active_warning_flag"]
    .value_counts(dropna=False)
)

print("\nDate range:")
print(
    "Oldest:",
    river_df["report_date"].min()
)

print(
    "Newest:",
    river_df["report_date"].max()
)

print("\nRainfall range:")
print(
    river_station_df["rainfall_24h_mm"].min(),
    "to",
    river_station_df["rainfall_24h_mm"].max()
)

print("\nWater-level status:")
print(
    river_station_df["status"]
    .value_counts(dropna=False)
)

In [ ]:
# =========================================================
# Save Cleaned River Data
# =========================================================

RIVER_REPORT_OUTPUT = (
    PROCESSED_DIR / "river_reports_cleaned.csv"
)

RIVER_STATION_OUTPUT = (
    PROCESSED_DIR / "river_station_observations.csv"
)

river_df.to_csv(
    RIVER_REPORT_OUTPUT,
    index=False
)

river_station_df.to_csv(
    RIVER_STATION_OUTPUT,
    index=False
)

print("Saved:", RIVER_REPORT_OUTPUT)
print("Report rows:", len(river_df))

print("\nSaved:", RIVER_STATION_OUTPUT)
print("Station rows:", len(river_station_df))

In [ ]:
# =========================================================
# Final River Validation
# =========================================================

print("Reports needing review:")

print(
    river_df.loc[
        river_df["extraction_status"] != "ok",
        [
            "file_name",
            "report_date",
            "report_type",
            "station_rows_extracted",
            "text_length",
            "extraction_status"
        ]
    ].to_string(index=False)
)


print("\nTop 10 highest 24-hour rainfall observations:")

print(
    river_station_df.nlargest(
        10,
        "rainfall_24h_mm"
    )[
        [
            "file_name",
            "report_date",
            "station_label_raw",
            "rainfall_24h_mm",
            "status",
            "water_level_current"
        ]
    ].to_string(index=False)
)

In [ ]:
review_files = river_df.loc[
    river_df["extraction_status"] == "no_station_rows",
    "file_name"
]

for file_name in review_files:

    pdf_path = RIVER_DIR / file_name
    text = extract_pdf_text(pdf_path)

    print("\n" + "=" * 100)
    print("PDF:", file_name)
    print("=" * 100)

    print(text[:3500])

In [ ]:
# =========================================================
# Check Active Warning Classification
# =========================================================

print("Active warning by report type:")
print(
    pd.crosstab(
        river_df["report_type"],
        river_df["active_warning_flag"]
    )
)

print("\nActive warnings outside flood_warning/flood_advisory:")

suspicious_active = river_df[
    (river_df["active_warning_flag"] == True)
    &
    (~river_df["report_type"].isin(
        ["flood_warning", "flood_advisory"]
    ))
]

print("Count:", len(suspicious_active))

print(
    suspicious_active[
        [
            "file_name",
            "report_type",
            "station_rows_extracted",
            "extraction_status"
        ]
    ].to_string(index=False)
)

# 4. Landslide Report Preprocessing

In [ ]:
# =========================================================
# Load Landslide PDFs
# =========================================================

LANDSLIDE_DIR = Path(
    "data/raw/landslide_reports"
)

landslide_pdfs = sorted(
    LANDSLIDE_DIR.glob("*.pdf")
)

print(
    "Total Landslide PDFs found:",
    len(landslide_pdfs)
)

In [ ]:
# =========================================================
# Count Landslide Report Types
# =========================================================

LANDSLIDE_FILENAME_PATTERN = re.compile(
    r"^landslide_\d{8}_(?:\d{4}|unknown)_(.+)_[0-9a-f]{8}\.pdf$",
    flags=re.IGNORECASE
)

landslide_types = []

for pdf_path in landslide_pdfs:

    match = LANDSLIDE_FILENAME_PATTERN.match(
        pdf_path.name
    )

    if match:
        landslide_types.append(
            match.group(1).lower()
        )
    else:
        landslide_types.append(
            "unknown"
        )

landslide_type_counts = Counter(
    landslide_types
)

for report_type, count in (
    landslide_type_counts.most_common()
):
    print(
        f"{report_type}: {count}"
    )

In [ ]:
# =========================================================
# Inspect Oldest & Newest Sample of Each Landslide Type
# =========================================================

landslide_types_to_check = [
    "early_warning",
    "warning",
    "landslide_related",
]

for report_type in landslide_types_to_check:

    matching_files = [
        p for p, t in zip(
            landslide_pdfs,
            landslide_types
        )
        if t == report_type
    ]

    print("\n" + "=" * 100)
    print("REPORT TYPE:", report_type)
    print("COUNT:", len(matching_files))
    print("=" * 100)

    if not matching_files:
        continue

    samples = [
        matching_files[0],
        matching_files[-1]
    ]

    # Avoid duplicate if only one file exists
    samples = list(dict.fromkeys(samples))

    for pdf_path in samples:

        text = extract_pdf_text(pdf_path)

        print("\nPDF:", pdf_path.name)
        print("Text length:", len(text))
        print("-" * 100)

        print(text[:3500])

In [ ]:
# =========================================================
# Test Landslide Table Extraction
# =========================================================

import pdfplumber

sample_names = [
    "landslide_20180818_1700_early_warning_3b05de13.pdf",
    "landslide_20260809_0930_early_warning_1ca23480.pdf",
    "landslide_20260613_1600_warning_15dcc1f3.pdf",
]

for file_name in sample_names:

    pdf_path = LANDSLIDE_DIR / file_name

    print("\n" + "=" * 100)
    print("PDF:", file_name)
    print("=" * 100)

    with pdfplumber.open(pdf_path) as pdf:

        for page_no, page in enumerate(pdf.pages, start=1):

            tables = page.extract_tables()

            print(
                f"\nPage {page_no} - Tables found:",
                len(tables)
            )

            for table_no, table in enumerate(
                tables,
                start=1
            ):

                print(
                    f"\nTABLE {table_no}"
                )

                for row in table[:15]:
                    print(row)

In [ ]:
# =========================================================
# Test Legacy Landslide Reports with Layout-Preserved Text
# =========================================================

legacy_samples = [
    "landslide_20180818_1700_early_warning_3b05de13.pdf",
    "landslide_20180520_1900_warning_8833b61d.pdf",
    "landslide_20180520_2000_landslide_related_3d546d49.pdf",
]

for file_name in legacy_samples:

    pdf_path = LANDSLIDE_DIR / file_name

    print("\n" + "=" * 110)
    print("PDF:", file_name)
    print("=" * 110)

    with pdfplumber.open(pdf_path) as pdf:

        for page_no, page in enumerate(
            pdf.pages,
            start=1
        ):

            text = page.extract_text(
                layout=True,
                x_tolerance=2,
                y_tolerance=3
            )

            if not text:
                continue

            # Show pages containing the risk-location section
            if (
                "Locations" in text
                or "Divisional Secretariat" in text
                or "District" in text
            ):

                print(
                    f"\n--- PAGE {page_no} ---\n"
                )

                print(text[:6000])

In [ ]:
# =========================================================
# Final Landslide Parser
# Modern tables + Legacy layout fallback
# =========================================================

LANDSLIDE_FILENAME_PATTERN = re.compile(
    r"^landslide_(\d{8})_(\d{4}|unknown)_(.+)_([0-9a-f]{8})\.pdf$",
    flags=re.IGNORECASE
)


DISTRICT_VARIANTS = {
    "ampara": "Ampara",
    "anuradhapura": "Anuradhapura",
    "badulla": "Badulla",
    "batticaloa": "Batticaloa",
    "colombo": "Colombo",
    "galle": "Galle",
    "gampaha": "Gampaha",
    "hambantota": "Hambantota",
    "jaffna": "Jaffna",
    "kalutara": "Kalutara",
    "kandy": "Kandy",
    "kegalle": "Kegalle",
    "kilinochchi": "Kilinochchi",
    "kurunegala": "Kurunegala",
    "mannar": "Mannar",
    "matale": "Matale",
    "matara": "Matara",
    "monaragala": "Monaragala",
    "moneragala": "Monaragala",
    "mullaitivu": "Mullaitivu",
    "nuwara eliya": "Nuwara Eliya",
    "polonnaruwa": "Polonnaruwa",
    "puttalam": "Puttalam",
    "ratnapura": "Ratnapura",
    "rathnapura": "Ratnapura",
    "trincomalee": "Trincomalee",
    "vavuniya": "Vavuniya",
}


LEVEL_INFO = {
    1: {
        "color": "Yellow",
        "action": "Watch"
    },
    2: {
        "color": "Amber",
        "action": "Alert"
    },
    3: {
        "color": "Red",
        "action": "Evacuation"
    }
}


def parse_landslide_filename(file_name):

    match = LANDSLIDE_FILENAME_PATTERN.match(
        file_name
    )

    if not match:
        return {
            "report_date": pd.NaT,
            "issue_time": np.nan,
            "report_type": "unknown",
        }

    date_text, time_text, report_type, _ = (
        match.groups()
    )

    return {
        "report_date": pd.to_datetime(
            date_text,
            format="%Y%m%d",
            errors="coerce"
        ),

        "issue_time": (
            np.nan
            if time_text.lower() == "unknown"
            else time_text
        ),

        "report_type": report_type.lower(),
    }


def normalize_district(value):

    if value is None:
        return None

    value = normalize_text(value).lower()

    # Exact match first
    if value in DISTRICT_VARIANTS:
        return DISTRICT_VARIANTS[value]

    # Handle minor surrounding text
    for variant, canonical in DISTRICT_VARIANTS.items():

        if re.search(
            rf"\b{re.escape(variant)}\b",
            value
        ):
            return canonical

    return None


def detect_warning_level_from_line(line):

    line_lower = normalize_text(
        line
    ).lower()

    # Explicit colours are the safest signal
    if "yellow" in line_lower:
        if (
            "level" in line_lower
            or "color" in line_lower
            or "colour" in line_lower
        ):
            return 1

    if "amber" in line_lower:
        if (
            "level" in line_lower
            or "color" in line_lower
            or "colour" in line_lower
            or "alert" in line_lower
        ):
            return 2

    if "red" in line_lower:
        if (
            "level" in line_lower
            or "color" in line_lower
            or "colour" in line_lower
            or "evacuation" in line_lower
        ):
            return 3

    return None


def extract_message_id(text):

    match = re.search(
        r"Message\s*ID\s*:?\s*"
        r"(LEWM-[A-Za-z0-9\-]+)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1)

    return np.nan


def extract_update_number(text):

    match = re.search(
        r"Update\s*:?\s*(\d+)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return np.nan


def extract_validity_period(text):

    # Works mainly for modern NBRO format
    pattern = re.compile(
        r"From\s+"
        r"(\d{1,2}:\d{2})\s*hrs\s*on\s*"
        r"(\d{1,2}[./-]\d{1,2}[./-]\d{4})"
        r"\s+To\s+"
        r"(\d{1,2}:\d{2})\s*hrs\s*on\s*"
        r"(\d{1,2}[./-]\d{1,2}[./-]\d{4})",
        flags=re.IGNORECASE
    )

    match = pattern.search(
        normalize_text(text)
    )

    if not match:
        return pd.NaT, pd.NaT

    start_time, start_date, end_time, end_date = (
        match.groups()
    )

    start_dt = pd.to_datetime(
        f"{start_date} {start_time}",
        dayfirst=True,
        errors="coerce"
    )

    end_dt = pd.to_datetime(
        f"{end_date} {end_time}",
        dayfirst=True,
        errors="coerce"
    )

    return start_dt, end_dt


# =========================================================
# Modern table extraction
# =========================================================

def extract_modern_landslide_areas(
    pdf_path,
    metadata
):

    rows = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_no, page in enumerate(
            pdf.pages,
            start=1
        ):

            tables = page.extract_tables()

            for table in tables:

                for row in table:

                    if not row or len(row) < 4:
                        continue

                    district = normalize_district(
                        row[0]
                    )

                    if district is None:
                        continue

                    # Column positions:
                    # 1 = Yellow
                    # 2 = Amber
                    # 3 = Red
                    for column_index, level in [
                        (1, 1),
                        (2, 2),
                        (3, 3),
                    ]:

                        cell_text = normalize_text(
                            row[column_index]
                            if row[column_index]
                            else ""
                        )

                        if not cell_text:
                            continue

                        rows.append({
                            "file_name":
                                pdf_path.name,

                            "report_date":
                                metadata["report_date"],

                            "issue_time":
                                metadata["issue_time"],

                            "report_type":
                                metadata["report_type"],

                            "district":
                                district,

                            "warning_level":
                                level,

                            "warning_color":
                                LEVEL_INFO[level]["color"],

                            "warning_action":
                                LEVEL_INFO[level]["action"],

                            "area_text_raw":
                                cell_text,

                            "page_number":
                                page_no,

                            "extraction_method":
                                "modern_table",
                        })

    return rows

# =========================================================
# Build DSD -> District Mapping
# =========================================================

def extract_dsd_names(area_text):

    if area_text is None or pd.isna(area_text):
        return []

    text = str(area_text)

    text = (
        text.replace("↑", "")
            .replace("↓", "")
            .replace("*", "")
    )

    text = re.split(
        r"Divisional Secretariat",
        text,
        maxsplit=1,
        flags=re.IGNORECASE
    )[0]

    text = re.sub(
        r"\s+\band\b\s+",
        ",",
        text,
        flags=re.IGNORECASE
    )

    parts = text.split(",")

    cleaned = []

    for part in parts:

        part = normalize_text(part)

        part = re.sub(
            r"[^A-Za-z\s\-]",
            "",
            part
        ).strip()

        if len(part) >= 3:
            cleaned.append(part)

    return cleaned


def build_dsd_to_district():

    mapping = {}

    print("Building DSD -> District mapping...")

    for pdf_path in landslide_pdfs:

        metadata = parse_landslide_filename(
            pdf_path.name
        )

        # Modern tables are used as reliable training/reference
        rows = extract_modern_landslide_areas(
            pdf_path,
            metadata
        )

        for row in rows:

            district = row["district"]

            dsd_names = extract_dsd_names(
                row["area_text_raw"]
            )

            for dsd in dsd_names:

                key = normalize_text(
                    dsd
                ).lower()

                mapping[key] = district

    # Important aliases from older NBRO reports
    mapping.update({

        "kothmale": "Nuwara Eliya",
        "kotmale": "Nuwara Eliya",
        "ambagamuwa": "Nuwara Eliya",

        "kalawana": "Ratnapura",
        "nivithigala": "Ratnapura",
        "kuruwita": "Ratnapura",
        "pelmadulla": "Ratnapura",
        "eheliyagoda": "Ratnapura",

        "deraniyagala": "Kegalle",
        "dehiowita": "Kegalle",
        "yatiyanthota": "Kegalle",
        "aranayaka": "Kegalle",

        "agalawatta": "Kalutara",
        "bulathsinhala": "Kalutara",
        "walallawita": "Kalutara",
        "ingiriya": "Kalutara",

        "thawalama": "Galle",
        "nagoda": "Galle",

        "pasbage korale": "Kandy",
    })

    return mapping


dsd_to_district = build_dsd_to_district()

print(
    "DSD -> District mappings:",
    len(dsd_to_district)
)


# =========================================================
# Legacy Landslide Area Extraction - UPDATED
# =========================================================

def extract_legacy_landslide_areas(
    pdf_path,
    metadata
):

    rows = []

    current_level = None
    location_section_started = False

    with pdfplumber.open(pdf_path) as pdf:

        for page_no, page in enumerate(
            pdf.pages,
            start=1
        ):

            text = page.extract_text(
                layout=True,
                x_tolerance=2,
                y_tolerance=3
            )

            if not text:
                continue

            for line in text.splitlines():

                clean_line = normalize_text(line)

                if not clean_line:
                    continue

                line_lower = clean_line.lower()

                # -----------------------------------------
                # Detect warning level
                # -----------------------------------------

                detected_level = (
                    detect_warning_level_from_line(
                        clean_line
                    )
                )

                if detected_level is not None:
                    current_level = detected_level

                # -----------------------------------------
                # Detect risk-location section
                # Tolerant to legacy OCR errors
                # -----------------------------------------

                if (
                    (
                        "location" in line_lower
                        and "risk" in line_lower
                    )
                    or
                    (
                        "locrt" in line_lower
                        and (
                            "risk" in line_lower
                            or "ristr" in line_lower
                        )
                    )
                    or
                    (
                        "divisional secretariat"
                        in line_lower
                    )
                ):
                    location_section_started = True

                if not location_section_started:
                    continue

                if current_level is None:
                    continue

                matched_districts = set()

                # -----------------------------------------
                # Method 1 - direct district name
                # -----------------------------------------

                for variant, canonical in (
                    DISTRICT_VARIANTS.items()
                ):

                    if re.search(
                        rf"\b{re.escape(variant)}\b",
                        line_lower
                    ):

                        # Avoid NBRO address false positive
                        if (
                            canonical == "Colombo"
                            and (
                                "jawatta" in line_lower
                                or "colombo 05" in line_lower
                            )
                        ):
                            continue

                        matched_districts.add(
                            canonical
                        )

                # -----------------------------------------
                # Method 2 - infer district from DSD
                # -----------------------------------------

                english_line = re.sub(
                    r"[^a-z\s\-]",
                    " ",
                    line_lower
                )

                english_line = normalize_text(
                    english_line
                )

                for dsd_name, district in (
                    dsd_to_district.items()
                ):

                    if len(dsd_name) < 5:
                        continue

                    if re.search(
                        rf"\b{re.escape(dsd_name)}\b",
                        english_line
                    ):
                        matched_districts.add(
                            district
                        )

                # -----------------------------------------
                # Save extracted observations
                # -----------------------------------------

                for district in matched_districts:

                    rows.append({

                        "file_name":
                            pdf_path.name,

                        "report_date":
                            metadata["report_date"],

                        "issue_time":
                            metadata["issue_time"],

                        "report_type":
                            metadata["report_type"],

                        "district":
                            district,

                        "warning_level":
                            current_level,

                        "warning_color":
                            LEVEL_INFO[
                                current_level
                            ]["color"],

                        "warning_action":
                            LEVEL_INFO[
                                current_level
                            ]["action"],

                        "area_text_raw":
                            clean_line,

                        "page_number":
                            page_no,

                        "extraction_method":
                            "legacy_layout",
                    })

    return rows

# =========================================================
# One Landslide PDF
# =========================================================

def extract_landslide_report(pdf_path):

    metadata = parse_landslide_filename(
        pdf_path.name
    )

    text = extract_pdf_text(
        pdf_path
    )

    message_id = extract_message_id(
        text
    )

    update_number = extract_update_number(
        text
    )

    validity_start, validity_end = (
        extract_validity_period(text)
    )

    # Try modern structured tables first
    area_rows = extract_modern_landslide_areas(
        pdf_path,
        metadata
    )

    extraction_method = "modern_table"

    # Fall back to legacy layout
    if len(area_rows) == 0:

        area_rows = extract_legacy_landslide_areas(
            pdf_path,
            metadata
        )

        extraction_method = "legacy_layout"

    # Remove duplicate district-level observations
    if area_rows:

        temp_df = pd.DataFrame(
            area_rows
        )

        temp_df = temp_df.drop_duplicates(
            subset=[
                "file_name",
                "district",
                "warning_level"
            ]
        )

        area_rows = temp_df.to_dict(
            "records"
        )

    levels = [
        row["warning_level"]
        for row in area_rows
    ]

    districts = {
        row["district"]
        for row in area_rows
    }

    if levels:
        max_level = max(levels)
        max_color = LEVEL_INFO[
            max_level
        ]["color"]
    else:
        max_level = np.nan
        max_color = np.nan

    level_1_count = sum(
        level == 1
        for level in levels
    )

    level_2_count = sum(
        level == 2
        for level in levels
    )

    level_3_count = sum(
        level == 3
        for level in levels
    )

    if len(text.strip()) < 100:

        extraction_status = (
            "insufficient_text"
        )

    elif len(area_rows) == 0:

        extraction_status = (
            "no_warning_areas"
        )

    else:

        extraction_status = "ok"

    report_record = {

        "file_name":
            pdf_path.name,

        "report_date":
            metadata["report_date"],

        "issue_time":
            metadata["issue_time"],

        "report_type":
            metadata["report_type"],

        "message_id":
            message_id,

        "update_number":
            update_number,

        "validity_start":
            validity_start,

        "validity_end":
            validity_end,

        "districts_extracted":
            len(districts),

        "area_rows_extracted":
            len(area_rows),

        "level_1_districts":
            level_1_count,

        "level_2_districts":
            level_2_count,

        "level_3_districts":
            level_3_count,

        "max_warning_level":
            max_level,

        "max_warning_color":
            max_color,

        "extraction_method":
            extraction_method,

        "text_length":
            len(text),

        "extraction_status":
            extraction_status,
    }

    return report_record, area_rows

In [ ]:
# =========================================================
# Extract All Landslide Reports
# =========================================================

landslide_report_records = []
landslide_area_records = []

for i, pdf_path in enumerate(
    landslide_pdfs,
    start=1
):

    report_record, area_rows = (
        extract_landslide_report(
            pdf_path
        )
    )

    landslide_report_records.append(
        report_record
    )

    landslide_area_records.extend(
        area_rows
    )

    if (
        i % 50 == 0
        or i == len(landslide_pdfs)
    ):
        print(
            f"Processed {i}/{len(landslide_pdfs)} PDFs"
        )


landslide_df = pd.DataFrame(
    landslide_report_records
)

landslide_area_df = pd.DataFrame(
    landslide_area_records
)

print("\nLandslide extraction completed.")

print(
    "Total Landslide reports:",
    len(landslide_df)
)

print(
    "Total district-warning observations:",
    len(landslide_area_df)
)

In [ ]:
# =========================================================
# Landslide Extraction Quality Check
# =========================================================

print("Total Landslide reports:", len(landslide_df))

print("\nExtraction status:")
print(
    landslide_df["extraction_status"]
    .value_counts(dropna=False)
)

print("\nReport types:")
print(
    landslide_df["report_type"]
    .value_counts(dropna=False)
)

print("\nExtraction methods:")
print(
    landslide_df["extraction_method"]
    .value_counts(dropna=False)
)

print("\nTotal district-warning observations:")
print(
    len(landslide_area_df)
)

print("\nWarning levels:")
print(
    landslide_area_df["warning_level"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nWarning colours:")
print(
    landslide_area_df["warning_color"]
    .value_counts(dropna=False)
)

print("\nTop districts by warning observations:")
print(
    landslide_area_df["district"]
    .value_counts(dropna=False)
    .head(15)
)

print("\nMaximum warning level per report:")
print(
    landslide_df["max_warning_level"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDate range:")
print(
    "Oldest:",
    landslide_df["report_date"].min()
)

print(
    "Newest:",
    landslide_df["report_date"].max()
)

print("\nDistricts extracted per report:")
print(
    landslide_df["districts_extracted"]
    .describe()
)

print("\nReports with no extracted warning areas:")
print(
    landslide_df.loc[
        landslide_df["extraction_status"] != "ok",
        [
            "file_name",
            "report_date",
            "report_type",
            "extraction_method",
            "text_length",
            "extraction_status"
        ]
    ].to_string(index=False)
)

In [ ]:
# =========================================================
# Final Landslide Review Classification
# =========================================================

def classify_unresolved_landslide_report(row):

    # Keep already successful records unchanged
    if row["extraction_status"] == "ok":
        return "ok"

    if row["extraction_status"] == "insufficient_text":
        return "insufficient_text"

    pdf_path = LANDSLIDE_DIR / row["file_name"]

    text = extract_pdf_text(pdf_path)
    text_lower = normalize_text(text).lower()

    # Misfiled river/water-level reports
    river_markers = (
        "islandwide water level & rainfall situation in major rivers"
        in text_lower
        or (
            "gauging station" in text_lower
            and "river basin" in text_lower
            and "water level" in text_lower
        )
    )

    if river_markers:
        return "non_landslide_report"

    # Genuine landslide report but legacy layout/OCR
    # prevents reliable district-level severity extraction
    return "manual_review_legacy_layout"


landslide_df["extraction_status"] = (
    landslide_df.apply(
        classify_unresolved_landslide_report,
        axis=1
    )
)

print("Final extraction status:")
print(
    landslide_df["extraction_status"]
    .value_counts(dropna=False)
)

In [ ]:
# =========================================================
# Save Cleaned Landslide Data
# =========================================================

LANDSLIDE_REPORT_OUTPUT = (
    PROCESSED_DIR / "landslide_reports_cleaned.csv"
)

LANDSLIDE_AREA_OUTPUT = (
    PROCESSED_DIR / "landslide_warning_areas.csv"
)

landslide_df.to_csv(
    LANDSLIDE_REPORT_OUTPUT,
    index=False
)

landslide_area_df.to_csv(
    LANDSLIDE_AREA_OUTPUT,
    index=False
)

print("Saved:", LANDSLIDE_REPORT_OUTPUT)
print("Report rows:", len(landslide_df))

print("\nSaved:", LANDSLIDE_AREA_OUTPUT)
print(
    "District-warning rows:",
    len(landslide_area_df)
)

# 5. Multi-Source Data Integration

In [ ]:
# =========================================================
# Load All Cleaned Datasets
# =========================================================

SITUATION_FILE = (
    PROCESSED_DIR / "situation_reports_cleaned.csv"
)

WEATHER_FILE = (
    PROCESSED_DIR / "weather_reports_cleaned.csv"
)

RIVER_REPORT_FILE = (
    PROCESSED_DIR / "river_reports_cleaned.csv"
)

RIVER_STATION_FILE = (
    PROCESSED_DIR / "river_station_observations.csv"
)

LANDSLIDE_REPORT_FILE = (
    PROCESSED_DIR / "landslide_reports_cleaned.csv"
)

LANDSLIDE_AREA_FILE = (
    PROCESSED_DIR / "landslide_warning_areas.csv"
)


situation_clean = pd.read_csv(
    SITUATION_FILE
)

weather_clean = pd.read_csv(
    WEATHER_FILE
)

river_report_clean = pd.read_csv(
    RIVER_REPORT_FILE
)

river_station_clean = pd.read_csv(
    RIVER_STATION_FILE
)

landslide_report_clean = pd.read_csv(
    LANDSLIDE_REPORT_FILE
)

landslide_area_clean = pd.read_csv(
    LANDSLIDE_AREA_FILE
)


print("Situation:", situation_clean.shape)
print("Weather:", weather_clean.shape)

print(
    "River reports:",
    river_report_clean.shape
)

print(
    "River stations:",
    river_station_clean.shape
)

print(
    "Landslide reports:",
    landslide_report_clean.shape
)

print(
    "Landslide areas:",
    landslide_area_clean.shape
)

In [ ]:
# =========================================================
# Inspect Dataset Schemas
# =========================================================

datasets = {
    "SITUATION": situation_clean,
    "WEATHER": weather_clean,
    "RIVER REPORT": river_report_clean,
    "RIVER STATION": river_station_clean,
    "LANDSLIDE REPORT": landslide_report_clean,
    "LANDSLIDE AREA": landslide_area_clean,
}

for name, df in datasets.items():

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    print("Rows:", len(df))

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst row:")
    print(
        df.head(1).to_string(index=False)
    )

In [ ]:
# =========================================================
# Standardize Report Date-Time Across All Sources
# =========================================================

def hhmm_to_time_string(value):

    if pd.isna(value):
        return "0000"

    # Handles values such as 530.0, 900, "0930"
    try:
        value = str(int(float(value)))
    except:
        value = str(value).strip()

    return value.zfill(4)


def create_report_datetime(
    df,
    date_col="report_date",
    time_col=None,
    filename_time=False
):

    result = df.copy()

    result[date_col] = pd.to_datetime(
        result[date_col],
        errors="coerce"
    )

    # -----------------------------------------
    # Situation reports:
    # extract HHMM directly from filename
    # -----------------------------------------
    if filename_time:

        extracted_time = (
            result["file_name"]
            .str.extract(
                r"_(\d{8})_(\d{4})_"
            )[1]
        )

        result["issue_time_standard"] = (
            extracted_time
            .fillna("0000")
        )

    # -----------------------------------------
    # Other datasets already have issue_time
    # -----------------------------------------
    elif time_col is not None:

        result["issue_time_standard"] = (
            result[time_col]
            .apply(hhmm_to_time_string)
        )

    else:

        result["issue_time_standard"] = "0000"

    result["report_datetime"] = pd.to_datetime(
        result[date_col].dt.strftime("%Y-%m-%d")
        + " "
        + result["issue_time_standard"].str[:2]
        + ":"
        + result["issue_time_standard"].str[2:],
        errors="coerce"
    )

    return result


# =========================================================
# Apply to all datasets
# =========================================================

situation_int = create_report_datetime(
    situation_clean,
    filename_time=True
)

weather_int = create_report_datetime(
    weather_clean,
    time_col="issue_time"
)

river_report_int = create_report_datetime(
    river_report_clean,
    time_col="issue_time"
)

river_station_int = create_report_datetime(
    river_station_clean,
    time_col="issue_time"
)

landslide_report_int = create_report_datetime(
    landslide_report_clean,
    time_col="issue_time"
)

landslide_area_int = create_report_datetime(
    landslide_area_clean,
    time_col="issue_time"
)


# =========================================================
# Validation
# =========================================================

integration_sets = {
    "Situation": situation_int,
    "Weather": weather_int,
    "River Report": river_report_int,
    "River Station": river_station_int,
    "Landslide Report": landslide_report_int,
    "Landslide Area": landslide_area_int,
}

for name, df in integration_sets.items():

    print("\n", name)

    print(
        "Rows:",
        len(df)
    )

    print(
        "Missing datetime:",
        df["report_datetime"].isna().sum()
    )

    print(
        "Start:",
        df["report_datetime"].min()
    )

    print(
        "End:",
        df["report_datetime"].max()
    )

In [ ]:
# =========================================================
# Integrate Weather Features into Situation Timeline
# Uses ONLY weather information available before each
# Situation report -> prevents future data leakage
# =========================================================

weather_work = weather_int.copy()

# Convert booleans safely
weather_bool_cols = [
    "heavy_rain_flag",
    "thunderstorm_flag",
    "strong_wind_flag",
    "low_pressure_flag",
    "depression_flag",
    "cyclone_flag",
]

for col in weather_bool_cols:
    weather_work[col] = (
        weather_work[col]
        .fillna(False)
        .astype(bool)
    )


# Warning colour -> severity
weather_color_map = {
    "Green": 1,
    "Amber": 2,
    "Red": 3
}

weather_work["weather_warning_level"] = (
    weather_work["warning_color"]
    .map(weather_color_map)
    .fillna(0)
)


# Sort once
weather_work = weather_work.sort_values(
    "report_datetime"
).reset_index(drop=True)

situation_with_weather = situation_int.sort_values(
    "report_datetime"
).reset_index(drop=True)


weather_times = weather_work[
    "report_datetime"
].values


weather_feature_rows = []

for situation_time in situation_with_weather[
    "report_datetime"
]:

    # Only reports at or BEFORE Situation timestamp
    right = weather_work[
        "report_datetime"
    ].searchsorted(
        situation_time,
        side="right"
    )

    # Previous 24 hours
    window_start = (
        situation_time
        - pd.Timedelta(hours=24)
    )

    left = weather_work[
        "report_datetime"
    ].searchsorted(
        window_start,
        side="left"
    )

    recent = weather_work.iloc[
        left:right
    ]

    # Latest available weather report
    previous_weather = weather_work.iloc[
        :right
    ]

    latest = (
        previous_weather.iloc[-1]
        if len(previous_weather) > 0
        else None
    )

    # -----------------------------------------
    # No recent Weather reports
    # -----------------------------------------

    if recent.empty:

        feature_row = {
            "weather_reports_24h": 0,

            "weather_max_rainfall_threshold_24h":
                np.nan,

            "weather_max_wind_24h":
                np.nan,

            "weather_max_warning_level_24h":
                0,

            "weather_heavy_rain_24h":
                0,

            "weather_thunderstorm_24h":
                0,

            "weather_strong_wind_24h":
                0,

            "weather_low_pressure_24h":
                0,

            "weather_depression_24h":
                0,

            "weather_cyclone_24h":
                0,
        }

    else:

        feature_row = {

            "weather_reports_24h":
                len(recent),

            "weather_max_rainfall_threshold_24h":
                recent[
                    "rainfall_threshold_mm"
                ].max(),

            "weather_max_wind_24h":
                recent[
                    "max_wind_kmph"
                ].max(),

            "weather_max_warning_level_24h":
                recent[
                    "weather_warning_level"
                ].max(),

            "weather_heavy_rain_24h":
                int(
                    recent[
                        "heavy_rain_flag"
                    ].any()
                ),

            "weather_thunderstorm_24h":
                int(
                    recent[
                        "thunderstorm_flag"
                    ].any()
                ),

            "weather_strong_wind_24h":
                int(
                    recent[
                        "strong_wind_flag"
                    ].any()
                ),

            "weather_low_pressure_24h":
                int(
                    recent[
                        "low_pressure_flag"
                    ].any()
                ),

            "weather_depression_24h":
                int(
                    recent[
                        "depression_flag"
                    ].any()
                ),

            "weather_cyclone_24h":
                int(
                    recent[
                        "cyclone_flag"
                    ].any()
                ),
        }

    # -----------------------------------------
    # Latest available weather report metadata
    # -----------------------------------------

    if latest is not None:

        age_hours = (
            situation_time
            - latest["report_datetime"]
        ).total_seconds() / 3600

        feature_row[
            "latest_weather_age_hours"
        ] = age_hours

        feature_row[
            "latest_weather_report_type"
        ] = latest["report_type"]

    else:

        feature_row[
            "latest_weather_age_hours"
        ] = np.nan

        feature_row[
            "latest_weather_report_type"
        ] = np.nan

    weather_feature_rows.append(
        feature_row
    )


weather_features_df = pd.DataFrame(
    weather_feature_rows
)

situation_weather_int = pd.concat(
    [
        situation_with_weather,
        weather_features_df
    ],
    axis=1
)


print(
    "Integrated rows:",
    len(situation_weather_int)
)

print(
    "Original Situation rows:",
    len(situation_int)
)

print("\nWeather feature columns:")

print(
    weather_features_df.columns.tolist()
)

print("\n24h Weather report coverage:")

print(
    (
        situation_weather_int[
            "weather_reports_24h"
        ] > 0
    ).value_counts()
)

print("\nMaximum weather values:")

print(
    situation_weather_int[
        [
            "weather_max_rainfall_threshold_24h",
            "weather_max_wind_24h",
            "weather_max_warning_level_24h"
        ]
    ].max()
)

In [ ]:
# =========================================================
# Integrate River Features into Situation + Weather Timeline
# =========================================================

river_work = river_report_int.copy()

# ---------------------------------------------------------
# Clean boolean columns safely
# ---------------------------------------------------------

def clean_bool_value(value):

    if pd.isna(value):
        return np.nan

    if value in [True, 1, "True", "true", "1"]:
        return True

    if value in [False, 0, "False", "false", "0"]:
        return False

    return np.nan


river_work["active_warning_bool"] = (
    river_work["active_warning_flag"]
    .apply(clean_bool_value)
)

river_work["warning_withdrawn_bool"] = (
    river_work["warning_withdrawn_flag"]
    .apply(clean_bool_value)
)


# ---------------------------------------------------------
# Sort by time
# ---------------------------------------------------------

river_work = (
    river_work
    .sort_values("report_datetime")
    .reset_index(drop=True)
)

situation_river_base = (
    situation_weather_int
    .sort_values("report_datetime")
    .reset_index(drop=True)
)

river_start_time = (
    river_work["report_datetime"].min()
)

river_feature_rows = []


for situation_time in (
    situation_river_base["report_datetime"]
):

    # -----------------------------------------------------
    # River archive not yet available
    # -----------------------------------------------------

    if situation_time < river_start_time:

        river_feature_rows.append({

            "river_data_available": 0,

            "river_reports_24h": 0,

            "river_max_alert_stations_24h":
                np.nan,

            "river_max_minor_flood_stations_24h":
                np.nan,

            "river_max_major_flood_stations_24h":
                np.nan,

            "river_max_rising_stations_24h":
                np.nan,

            "river_max_rainfall_24h_mm":
                np.nan,

            "river_max_alert_ratio_24h":
                np.nan,

            "river_active_warning_24h":
                np.nan,

            "river_warning_withdrawn_24h":
                np.nan,

            "latest_river_age_hours":
                np.nan,

            "latest_river_report_type":
                np.nan,
        })

        continue


    # -----------------------------------------------------
    # Reports available only BEFORE Situation timestamp
    # -----------------------------------------------------

    right = (
        river_work["report_datetime"]
        .searchsorted(
            situation_time,
            side="right"
        )
    )

    window_start = (
        situation_time
        - pd.Timedelta(hours=24)
    )

    left = (
        river_work["report_datetime"]
        .searchsorted(
            window_start,
            side="left"
        )
    )

    recent = river_work.iloc[
        left:right
    ]

    previous_river = river_work.iloc[
        :right
    ]


    # -----------------------------------------------------
    # Latest available river report
    # -----------------------------------------------------

    if len(previous_river) > 0:

        latest = previous_river.iloc[-1]

        latest_age = (
            situation_time
            - latest["report_datetime"]
        ).total_seconds() / 3600

        latest_type = (
            latest["report_type"]
        )

    else:

        latest_age = np.nan
        latest_type = np.nan


    # -----------------------------------------------------
    # No River report during previous 24 hours
    # -----------------------------------------------------

    if recent.empty:

        feature_row = {

            "river_data_available": 1,

            "river_reports_24h": 0,

            "river_max_alert_stations_24h":
                np.nan,

            "river_max_minor_flood_stations_24h":
                np.nan,

            "river_max_major_flood_stations_24h":
                np.nan,

            "river_max_rising_stations_24h":
                np.nan,

            "river_max_rainfall_24h_mm":
                np.nan,

            "river_max_alert_ratio_24h":
                np.nan,

            "river_active_warning_24h":
                np.nan,

            "river_warning_withdrawn_24h":
                np.nan,

            "latest_river_age_hours":
                latest_age,

            "latest_river_report_type":
                latest_type,
        }


    # -----------------------------------------------------
    # River reports exist during previous 24 hours
    # -----------------------------------------------------

    else:

        feature_row = {

            "river_data_available": 1,

            "river_reports_24h":
                len(recent),

            # MAX is used instead of SUM because
            # stations may appear repeatedly in updates

            "river_max_alert_stations_24h":
                recent[
                    "alert_station_count"
                ].max(),

            "river_max_minor_flood_stations_24h":
                recent[
                    "minor_flood_station_count"
                ].max(),

            "river_max_major_flood_stations_24h":
                recent[
                    "major_flood_station_count"
                ].max(),

            "river_max_rising_stations_24h":
                recent[
                    "rising_station_count"
                ].max(),

            "river_max_rainfall_24h_mm":
                recent[
                    "max_rainfall_24h_mm"
                ].max(),

            "river_max_alert_ratio_24h":
                recent[
                    "max_alert_ratio"
                ].max(),

            "river_active_warning_24h":
                int(
                    recent[
                        "active_warning_bool"
                    ]
                    .fillna(False)
                    .any()
                ),

            "river_warning_withdrawn_24h":
                int(
                    recent[
                        "warning_withdrawn_bool"
                    ]
                    .fillna(False)
                    .any()
                ),

            "latest_river_age_hours":
                latest_age,

            "latest_river_report_type":
                latest_type,
        }


    river_feature_rows.append(
        feature_row
    )


# =========================================================
# Attach River features
# =========================================================

river_features_df = pd.DataFrame(
    river_feature_rows
)

situation_weather_river_int = pd.concat(
    [
        situation_river_base,
        river_features_df
    ],
    axis=1
)


# =========================================================
# Validation
# =========================================================

print(
    "Integrated rows:",
    len(situation_weather_river_int)
)

print(
    "Original Situation rows:",
    len(situation_int)
)

print("\nRiver data availability:")

print(
    situation_weather_river_int[
        "river_data_available"
    ].value_counts()
)

print("\n24h River coverage:")

print(
    (
        situation_weather_river_int[
            "river_reports_24h"
        ] > 0
    ).value_counts()
)

print("\nMaximum River values:")

print(
    situation_weather_river_int[
        [
            "river_max_alert_stations_24h",
            "river_max_minor_flood_stations_24h",
            "river_max_major_flood_stations_24h",
            "river_max_rainfall_24h_mm",
            "river_max_alert_ratio_24h"
        ]
    ].max()
)

In [ ]:
# =========================================================
# Integrate Landslide Features into Situation Timeline
# =========================================================

landslide_work = landslide_report_int.copy()

# ---------------------------------------------------------
# Convert validity columns back to datetime
# ---------------------------------------------------------

landslide_work["validity_start"] = pd.to_datetime(
    landslide_work["validity_start"],
    errors="coerce"
)

landslide_work["validity_end"] = pd.to_datetime(
    landslide_work["validity_end"],
    errors="coerce"
)

# Use only reliably extracted landslide reports
landslide_work = landslide_work[
    landslide_work["extraction_status"] == "ok"
].copy()

landslide_work = (
    landslide_work
    .sort_values("report_datetime")
    .reset_index(drop=True)
)

integration_base = (
    situation_weather_river_int
    .sort_values("report_datetime")
    .reset_index(drop=True)
)

landslide_feature_rows = []


for situation_time in integration_base["report_datetime"]:

    # -----------------------------------------------------
    # Only reports available at/before Situation timestamp
    # -----------------------------------------------------

    right = (
        landslide_work["report_datetime"]
        .searchsorted(
            situation_time,
            side="right"
        )
    )

    previous_reports = landslide_work.iloc[:right]

    # No previous landslide report
    if previous_reports.empty:

        landslide_feature_rows.append({

            "landslide_data_available": 0,
            "landslide_active_warning": 0,

            "landslide_max_warning_level": 0,

            "landslide_level1_districts": 0,
            "landslide_level2_districts": 0,
            "landslide_level3_districts": 0,

            "landslide_districts_at_risk": 0,

            "latest_landslide_age_hours":
                np.nan,

            "latest_landslide_report_type":
                np.nan,
        })

        continue


    # -----------------------------------------------------
    # Latest available landslide bulletin
    # -----------------------------------------------------

    latest = previous_reports.iloc[-1]

    age_hours = (
        situation_time
        - latest["report_datetime"]
    ).total_seconds() / 3600


    # -----------------------------------------------------
    # Determine whether latest warning is still active
    # -----------------------------------------------------

    validity_start = latest["validity_start"]
    validity_end = latest["validity_end"]

    if pd.notna(validity_end):

        start_ok = (
            pd.isna(validity_start)
            or validity_start <= situation_time
        )

        end_ok = (
            validity_end >= situation_time
        )

        active_warning = (
            start_ok and end_ok
        )

    else:

        # Legacy/fallback rule:
        # warning considered recent for 24 hours
        active_warning = (
            age_hours <= 24
        )


    # -----------------------------------------------------
    # Active warning features
    # -----------------------------------------------------

    if active_warning:

        max_level = (
            latest["max_warning_level"]
            if pd.notna(
                latest["max_warning_level"]
            )
            else 0
        )

        level1 = int(
            latest["level_1_districts"]
        )

        level2 = int(
            latest["level_2_districts"]
        )

        level3 = int(
            latest["level_3_districts"]
        )

        districts_at_risk = int(
            latest["districts_extracted"]
        )

    else:

        max_level = 0
        level1 = 0
        level2 = 0
        level3 = 0
        districts_at_risk = 0


    landslide_feature_rows.append({

        "landslide_data_available": 1,

        "landslide_active_warning":
            int(active_warning),

        "landslide_max_warning_level":
            max_level,

        "landslide_level1_districts":
            level1,

        "landslide_level2_districts":
            level2,

        "landslide_level3_districts":
            level3,

        "landslide_districts_at_risk":
            districts_at_risk,

        "latest_landslide_age_hours":
            age_hours,

        "latest_landslide_report_type":
            latest["report_type"],
    })


# =========================================================
# Attach Landslide features
# =========================================================

landslide_features_df = pd.DataFrame(
    landslide_feature_rows
)

integrated_df = pd.concat(
    [
        integration_base,
        landslide_features_df
    ],
    axis=1
)


# =========================================================
# Validation
# =========================================================

print(
    "Final integrated rows:",
    len(integrated_df)
)

print(
    "Original Situation rows:",
    len(situation_int)
)

print("\nLandslide data availability:")

print(
    integrated_df[
        "landslide_data_available"
    ].value_counts()
)

print("\nActive landslide warning:")

print(
    integrated_df[
        "landslide_active_warning"
    ].value_counts()
)

print("\nMaximum Landslide warning level:")

print(
    integrated_df[
        "landslide_max_warning_level"
    ].max()
)

print("\nMaximum districts at risk:")

print(
    integrated_df[
        "landslide_districts_at_risk"
    ].max()
)

In [ ]:
# =========================================================
# Final Integrated Dataset Quality Check
# =========================================================

print("Total integrated rows:", len(integrated_df))

print("\nDate range:")
print(
    integrated_df["report_datetime"].min(),
    "to",
    integrated_df["report_datetime"].max()
)

print("\nDuplicate Situation reports:")
print(
    integrated_df["file_name"].duplicated().sum()
)

print("\nMissing target values:")

target_cols = [
    "affected_people",
    "affected_families",
    "safety_centres",
    "families_in_safety_centres",
    "people_in_safety_centres"
]

print(
    integrated_df[
        target_cols
    ].isna().sum()
)

print("\nWeather coverage:")
print(
    (integrated_df["weather_reports_24h"] > 0)
    .value_counts()
)

print("\nRiver availability:")
print(
    integrated_df["river_data_available"]
    .value_counts()
)

print("\nActive landslide warnings:")
print(
    integrated_df["landslide_active_warning"]
    .value_counts()
)

print("\nFinal dataset shape:")
print(
    integrated_df.shape
)

In [ ]:
# =========================================================
# Save Final Integrated Dataset
# =========================================================

INTEGRATED_OUTPUT = (
    PROCESSED_DIR /
    "integrated_disaster_dataset.csv"
)

integrated_df.to_csv(
    INTEGRATED_OUTPUT,
    index=False
)

print("Saved:", INTEGRATED_OUTPUT)
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

# 6. Exploratory Data Analysis (EDA)

In [ ]:
# =========================================================
# EDA Setup + Target Variable Summary
# =========================================================

eda_df = integrated_df.copy()

eda_df["report_datetime"] = pd.to_datetime(
    eda_df["report_datetime"],
    errors="coerce"
)

eda_df = (
    eda_df
    .sort_values("report_datetime")
    .reset_index(drop=True)
)

TARGET_COLS = [
    "affected_people",
    "affected_families",
    "safety_centres",
    "families_in_safety_centres",
    "people_in_safety_centres"
]


print("EDA rows:", len(eda_df))

print(
    "Date range:",
    eda_df["report_datetime"].min(),
    "to",
    eda_df["report_datetime"].max()
)


print("\n" + "=" * 80)
print("TARGET DESCRIPTIVE STATISTICS")
print("=" * 80)

print(
    eda_df[TARGET_COLS]
    .describe()
    .T
)


print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

print(
    eda_df[TARGET_COLS]
    .isna()
    .sum()
)


print("\n" + "=" * 80)
print("ZERO vs POSITIVE VALUES")
print("=" * 80)

for col in TARGET_COLS:

    valid = eda_df[col].dropna()

    print(f"\n{col}")

    print(
        "Zero:",
        int((valid == 0).sum())
    )

    print(
        "Positive:",
        int((valid > 0).sum())
    )

    print(
        "Maximum:",
        valid.max()
    )

In [ ]:
# =========================================================
# EDA - Target Distribution Analysis
# =========================================================

import matplotlib.pyplot as plt
import numpy as np


for col in TARGET_COLS:

    valid = eda_df[col].dropna()

    # -----------------------------------------------------
    # Raw distribution
    # -----------------------------------------------------

    plt.figure(figsize=(9, 5))

    plt.hist(
        valid,
        bins=50
    )

    plt.title(
        f"Distribution of {col}"
    )

    plt.xlabel(col)
    plt.ylabel("Number of Reports")

    plt.tight_layout()
    plt.show()


    # -----------------------------------------------------
    # Log-transformed distribution
    # log1p allows zero values
    # -----------------------------------------------------

    log_values = np.log1p(valid)

    plt.figure(figsize=(9, 5))

    plt.hist(
        log_values,
        bins=50
    )

    plt.title(
        f"Log1p Distribution of {col}"
    )

    plt.xlabel(
        f"log1p({col})"
    )

    plt.ylabel(
        "Number of Reports"
    )

    plt.tight_layout()
    plt.show()


# =========================================================
# Skewness Comparison
# =========================================================

print("\nSkewness comparison:")

for col in TARGET_COLS:

    valid = eda_df[col].dropna()

    print(
        f"\n{col}"
    )

    print(
        "Raw skewness:",
        round(valid.skew(), 3)
    )

    print(
        "Log1p skewness:",
        round(
            np.log1p(valid).skew(),
            3
        )
    )

In [ ]:
# =========================================================
# EDA - Time Series Trends and Major Impact Peaks
# =========================================================

# ---------------------------------------------------------
# 1. Affected People Over Time
# ---------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.plot(
    eda_df["report_datetime"],
    eda_df["affected_people"]
)

plt.title(
    "Affected People Over Time"
)

plt.xlabel("Report Date")
plt.ylabel("Affected People")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 2. People in Safety Centres Over Time
# ---------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.plot(
    eda_df["report_datetime"],
    eda_df["people_in_safety_centres"]
)

plt.title(
    "People in Safety Centres Over Time"
)

plt.xlabel("Report Date")
plt.ylabel("People in Safety Centres")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 3. Log-transformed Affected People Trend
# ---------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.plot(
    eda_df["report_datetime"],
    np.log1p(
        eda_df["affected_people"]
    )
)

plt.title(
    "Log1p Affected People Over Time"
)

plt.xlabel("Report Date")
plt.ylabel("log1p(Affected People)")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 4. Top 15 Highest-Impact Situation Reports
# ---------------------------------------------------------

top_impacts = (
    eda_df[
        [
            "report_datetime",
            "disaster_types",
            "affected_people",
            "affected_families",
            "safety_centres",
            "people_in_safety_centres"
        ]
    ]
    .dropna(
        subset=["affected_people"]
    )
    .nlargest(
        15,
        "affected_people"
    )
)

print("\nTop 15 highest-impact reports:")

print(
    top_impacts.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# 5. Disaster Type Distribution
# ---------------------------------------------------------

print(
    "\nMost common disaster-type labels:"
)

print(
    eda_df["disaster_types"]
    .value_counts(dropna=False)
    .head(20)
)

In [ ]:
# =========================================================
# EDA - Inspect Major 2025 December Disaster Episode
# =========================================================

major_episode = eda_df[
    (
        eda_df["report_datetime"] >=
        "2025-11-25"
    )
    &
    (
        eda_df["report_datetime"] <=
        "2025-12-15 23:59:59"
    )
].copy()


cols_to_show = [
    "report_datetime",
    "file_name",
    "disaster_types",

    "affected_people",
    "affected_families",

    "safety_centres",
    "people_in_safety_centres",

    "weather_heavy_rain_24h",
    "weather_max_rainfall_threshold_24h",
    "weather_max_warning_level_24h",

    "river_max_alert_stations_24h",
    "river_max_minor_flood_stations_24h",
    "river_max_major_flood_stations_24h",
    "river_active_warning_24h",

    "landslide_active_warning",
    "landslide_max_warning_level"
]


print(
    major_episode[
        cols_to_show
    ].to_string(index=False)
)


print("\nNumber of reports in episode:")
print(len(major_episode))


print("\nDisaster labels:")
print(
    major_episode["disaster_types"]
    .value_counts(dropna=False)
)


print("\nRepeated affected_people values:")

print(
    major_episode[
        "affected_people"
    ]
    .value_counts()
    .head(15)
)

In [ ]:
# =========================================================
# EDA - Audit Duplicate / Conflicting Situation Snapshots
# =========================================================

snapshot_cols = [
    "affected_people",
    "affected_families",
    "safety_centres",
    "families_in_safety_centres",
    "people_in_safety_centres"
]


# ---------------------------------------------------------
# 1. How many timestamps contain multiple Situation PDFs?
# ---------------------------------------------------------

timestamp_counts = (
    eda_df.groupby("report_datetime")
    .size()
    .sort_values(ascending=False)
)

multiple_timestamp_counts = (
    timestamp_counts[
        timestamp_counts > 1
    ]
)

print("Total Situation rows:", len(eda_df))

print(
    "Unique report timestamps:",
    eda_df["report_datetime"].nunique()
)

print(
    "Timestamps with multiple reports:",
    len(multiple_timestamp_counts)
)

print(
    "Rows belonging to duplicated timestamps:",
    int(
        multiple_timestamp_counts.sum()
    )
)


# ---------------------------------------------------------
# 2. Exact same timestamp + exact same target values
# ---------------------------------------------------------

exact_snapshot_duplicate = (
    eda_df.duplicated(
        subset=[
            "report_datetime",
            *snapshot_cols
        ],
        keep=False
    )
)

print(
    "\nRows with same timestamp "
    "and same target snapshot:",
    exact_snapshot_duplicate.sum()
)


# ---------------------------------------------------------
# 3. Find conflicting snapshots at same timestamp
# ---------------------------------------------------------

conflicting_times = []

for report_time, group in eda_df.groupby(
    "report_datetime"
):

    if len(group) <= 1:
        continue

    conflict = False

    for col in snapshot_cols:

        values = (
            group[col]
            .dropna()
            .unique()
        )

        if len(values) > 1:
            conflict = True
            break

    if conflict:
        conflicting_times.append(
            report_time
        )


print(
    "\nTimestamps with conflicting target values:",
    len(conflicting_times)
)


# ---------------------------------------------------------
# 4. Display conflicting reports
# ---------------------------------------------------------

conflicting_df = (
    eda_df[
        eda_df["report_datetime"].isin(
            conflicting_times
        )
    ][
        [
            "report_datetime",
            "file_name",
            "disaster_types",
            *snapshot_cols,
            "extraction_status"
        ]
    ]
    .sort_values(
        [
            "report_datetime",
            "affected_people"
        ]
    )
)


print("\nConflicting reports:")

print(
    conflicting_df.head(50)
    .to_string(index=False)
)

In [ ]:
# =========================================================
# EDA - Concurrent Situation Report Stream Audit
# =========================================================

# 1. Find target snapshots that repeat across many dates
snapshot_frequency = (
    eda_df
    .dropna(
        subset=[
            "affected_people",
            "affected_families"
        ]
    )
    .groupby(
        [
            "affected_people",
            "affected_families"
        ]
    )
    .agg(
        report_count=("file_name", "count"),
        unique_dates=("report_datetime", lambda x: x.dt.date.nunique()),
        first_seen=("report_datetime", "min"),
        last_seen=("report_datetime", "max"),
        disaster_labels=(
            "disaster_types",
            lambda x: " | ".join(
                sorted(
                    set(
                        x.dropna().astype(str)
                    )
                )
            )
        )
    )
    .reset_index()
    .sort_values(
        ["unique_dates", "report_count"],
        ascending=False
    )
)

print(
    "Most frequently repeated target snapshots:"
)

print(
    snapshot_frequency
    .head(25)
    .to_string(index=False)
)


# =========================================================
# 2. Reports at timestamps with conflicts
# =========================================================

conflict_only = eda_df[
    eda_df["report_datetime"].isin(
        conflicting_times
    )
].copy()


print("\nConflict-period disaster labels:")

print(
    conflict_only[
        "disaster_types"
    ]
    .value_counts(dropna=False)
    .head(20)
)


# =========================================================
# 3. Check the suspicious 26,238 affected stream
# =========================================================

suspected_stream = eda_df[
    (
        eda_df["affected_people"] == 26238
    )
    &
    (
        eda_df["affected_families"] == 8357
    )
][
    [
        "report_datetime",
        "file_name",
        "disaster_types",
        "affected_people",
        "affected_families",
        "safety_centres",
        "people_in_safety_centres"
    ]
].sort_values(
    "report_datetime"
)


print(
    "\nReports with 26,238 affected / 8,357 families:"
)

print(
    suspected_stream.to_string(index=False)
)


print(
    "\nNumber of these reports:",
    len(suspected_stream)
)

if len(suspected_stream) > 0:

    print(
        "Period:",
        suspected_stream["report_datetime"].min(),
        "to",
        suspected_stream["report_datetime"].max()
    )

In [ ]:
# =========================================================
# EDA - Inspect Situation Report Headings for Event Identity
# =========================================================

from pathlib import Path

SITUATION_RAW_DIR = Path(
    "data/raw/situation_reports"
)


def find_situation_pdf(file_name):

    matches = list(
        SITUATION_RAW_DIR.rglob(file_name)
    )

    if matches:
        return matches[0]

    return None


# ---------------------------------------------------------
# Select representative reports
# ---------------------------------------------------------

sample_files = []


# 26,238 stream:
# beginning, middle and end
if len(suspected_stream) > 0:

    sample_files.extend([
        suspected_stream.iloc[0]["file_name"],
        suspected_stream.iloc[
            len(suspected_stream) // 2
        ]["file_name"],
        suspected_stream.iloc[-1]["file_name"]
    ])


# Add reports from one conflicting timestamp
example_conflict_time = pd.Timestamp(
    "2024-08-25 09:00:00"
)

conflict_samples = eda_df[
    eda_df["report_datetime"] ==
    example_conflict_time
]["file_name"].tolist()

sample_files.extend(
    conflict_samples
)


# Add one major December 2025 report
december_sample = eda_df[
    eda_df["report_datetime"] ==
    pd.Timestamp("2025-11-28 09:00:00")
]["file_name"].tolist()

sample_files.extend(
    december_sample
)


# Remove duplicate filenames
sample_files = list(
    dict.fromkeys(sample_files)
)


# ---------------------------------------------------------
# Print first 30 non-empty lines from each PDF
# ---------------------------------------------------------

for file_name in sample_files:

    print("\n" + "=" * 100)

    print(
        "FILE:",
        file_name
    )

    pdf_path = find_situation_pdf(
        file_name
    )

    if pdf_path is None:

        print(
            "PDF NOT FOUND"
        )

        continue


    text = extract_pdf_text(
        pdf_path
    )

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]


    print(
        "\nFIRST 30 NON-EMPTY LINES:\n"
    )

    for line in lines[:30]:

        print(line)

In [ ]:
# =========================================================
# EDA - Classify Situation Report Type from PDF Heading
# =========================================================

import re


def classify_situation_report_heading(pdf_path):

    text = extract_pdf_text(pdf_path)

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # Only inspect the beginning of the report.
    # This avoids classifying based on words appearing later in tables.
    header_text = " ".join(lines[:20])

    header_lower = header_text.lower()


    # -----------------------------------------------------
    # 1. Drought-specific Situation Reports
    # -----------------------------------------------------

    if (
        "drought situation report" in header_lower
        or "drought" in header_lower
        or "නියඟ" in header_text
    ):
        return "drought_report"


    # -----------------------------------------------------
    # 2. Special Situation Reports
    # -----------------------------------------------------

    if (
        "special situation report" in header_lower
        or "විශේෂ" in header_text
    ):
        return "special_situation_report"


    # -----------------------------------------------------
    # 3. Standard Situation Reports
    # -----------------------------------------------------

    if (
        "situation report" in header_lower
        or "තත්ත්ව වාර්තාව" in header_text
    ):
        return "general_situation_report"


    return "other_heading"


# =========================================================
# Apply classification
# =========================================================

heading_categories = []


for _, row in eda_df.iterrows():

    pdf_path = find_situation_pdf(
        row["file_name"]
    )

    if pdf_path is None:

        category = "pdf_not_found"

    else:

        category = classify_situation_report_heading(
            pdf_path
        )

    heading_categories.append(
        category
    )


eda_df["situation_report_category"] = (
    heading_categories
)


print(
    "Situation report categories:"
)

print(
    eda_df[
        "situation_report_category"
    ]
    .value_counts(dropna=False)
)


# =========================================================
# Check target volume by report category
# =========================================================

print(
    "\nAffected people by report category:"
)

print(
    eda_df.groupby(
        "situation_report_category"
    )["affected_people"]
    .agg(
        [
            "count",
            "median",
            "max"
        ]
    )
)


# =========================================================
# Check duplicate timestamps again after excluding drought
# =========================================================

non_drought_check = eda_df[
    eda_df[
        "situation_report_category"
    ] != "drought_report"
].copy()


timestamp_counts_non_drought = (
    non_drought_check
    .groupby("report_datetime")
    .size()
)


print(
    "\nRows after excluding drought:",
    len(non_drought_check)
)

print(
    "Timestamps with multiple reports after excluding drought:",
    int(
        (
            timestamp_counts_non_drought > 1
        ).sum()
    )
)

In [ ]:
# =========================================================
# EDA - Inspect Remaining Other-Heading Situation Reports
# =========================================================

other_heading_df = eda_df[
    eda_df["situation_report_category"] == "other_heading"
].copy()

print("Total other-heading reports:", len(other_heading_df))


for _, row in other_heading_df.iterrows():

    print("\n" + "=" * 100)

    print("FILE:", row["file_name"])
    print("DATE:", row["report_datetime"])
    print("AFFECTED PEOPLE:", row["affected_people"])
    print("AFFECTED FAMILIES:", row["affected_families"])
    print("EXTRACTION STATUS:", row["extraction_status"])

    pdf_path = find_situation_pdf(
        row["file_name"]
    )

    if pdf_path is None:
        print("PDF NOT FOUND")
        continue

    text = extract_pdf_text(
        pdf_path
    )

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    print("\nFIRST 8 NON-EMPTY LINES:")

    for line in lines[:8]:
        print(line)

In [ ]:
# =========================================================
# EDA - Final Analytical Scope Classification
# =========================================================

eda_df["scope_status"] = "retain"


# ---------------------------------------------------------
# Exclude drought reports
# Project scope: Flood + rainfall-induced landslides
# ---------------------------------------------------------

eda_df.loc[
    eda_df["situation_report_category"] == "drought_report",
    "scope_status"
] = "exclude_drought"


# ---------------------------------------------------------
# Exclude files that are not Situation Reports
# e.g. weather forecast / river report stored in archive
# ---------------------------------------------------------

eda_df.loc[
    eda_df["extraction_status"] == "non_situation_report",
    "scope_status"
] = "exclude_non_situation"


# ---------------------------------------------------------
# Exclude non-comparable incident-report format
# ---------------------------------------------------------

eda_df.loc[
    eda_df["extraction_status"] == "non_comparable_incident_report",
    "scope_status"
] = "exclude_non_comparable"


print("Final scope classification:")

print(
    eda_df["scope_status"]
    .value_counts()
)


# =========================================================
# Create analytical EDA dataset
# =========================================================

eda_scope_df = (
    eda_df[
        eda_df["scope_status"] == "retain"
    ]
    .copy()
    .sort_values("report_datetime")
    .reset_index(drop=True)
)


print(
    "\nOriginal rows:",
    len(eda_df)
)

print(
    "Analytical EDA rows:",
    len(eda_scope_df)
)

print(
    "\nMissing affected_people:",
    eda_scope_df["affected_people"]
    .isna()
    .sum()
)

In [ ]:
# =========================================================
# EDA - Situation Report Timestamp Consistency Audit
# =========================================================

import re
import pandas as pd


MONTH_MAP = {
    "january": 1,
    "february": 2,
    "march": 3,
    "april": 4,
    "may": 5,
    "june": 6,
    "july": 7,
    "august": 8,
    "september": 9,
    "october": 10,
    "november": 11,
    "december": 12
}


def build_timestamp(year, month, day, hour_part, minute_part=None):

    try:
        year = int(year)
        month = int(month)
        day = int(day)

        hour_text = str(hour_part).strip()

        # Example: 0900, 1800
        if (
            minute_part is None
            and len(hour_text) in [3, 4]
        ):
            hour_text = hour_text.zfill(4)

            hour = int(hour_text[:2])
            minute = int(hour_text[2:])

        else:

            hour = int(hour_part)

            minute = (
                int(minute_part)
                if minute_part is not None
                else 0
            )

        return pd.Timestamp(
            year=year,
            month=month,
            day=day,
            hour=hour,
            minute=minute
        )

    except:
        return pd.NaT


def extract_internal_report_datetime(pdf_path):

    text = extract_pdf_text(pdf_path)

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # Header only
    header = " ".join(lines[:25])

    candidates = []


    # -----------------------------------------------------
    # Pattern 1:
    # 2025.11.28 ... 0900
    # 2025.11.28 ... වේලාව - 1800
    # -----------------------------------------------------

    numeric_pattern = re.compile(
        r"(20\d{2})[./-](\d{1,2})[./-](\d{1,2})"
        r".{0,45}?"
        r"(?:වේලාව|TIME|Time|time|at|දින)"
        r"\s*[:\-]?\s*"
        r"(\d{3,4}|\d{1,2}[:.]\d{2})"
    )

    for match in numeric_pattern.finditer(header):

        time_value = match.group(4)

        if ":" in time_value or "." in time_value:

            parts = re.split(
                r"[:.]",
                time_value
            )

            ts = build_timestamp(
                match.group(1),
                match.group(2),
                match.group(3),
                parts[0],
                parts[1]
            )

        else:

            ts = build_timestamp(
                match.group(1),
                match.group(2),
                match.group(3),
                time_value
            )

        if pd.notna(ts):

            candidates.append(
                (
                    match.start(),
                    ts,
                    "numeric_header"
                )
            )


    # -----------------------------------------------------
    # Pattern 2:
    # Date :27th July 2024 at 0900hrs
    # -----------------------------------------------------

    english_pattern = re.compile(
        r"Date\s*:?\s*"
        r"(\d{1,2})"
        r"(?:st|nd|rd|th)?\s+"
        r"([A-Za-z]+)\s+"
        r"(20\d{2})"
        r".{0,20}?"
        r"(?:at\s*)?"
        r"(\d{3,4}|\d{1,2}[:.]\d{2})",
        re.IGNORECASE
    )

    for match in english_pattern.finditer(header):

        month_name = (
            match.group(2)
            .lower()
            .strip()
        )

        month = MONTH_MAP.get(
            month_name
        )

        if month is None:
            continue

        time_value = match.group(4)

        if ":" in time_value or "." in time_value:

            parts = re.split(
                r"[:.]",
                time_value
            )

            ts = build_timestamp(
                match.group(3),
                month,
                match.group(1),
                parts[0],
                parts[1]
            )

        else:

            ts = build_timestamp(
                match.group(3),
                month,
                match.group(1),
                time_value
            )

        if pd.notna(ts):

            candidates.append(
                (
                    match.start(),
                    ts,
                    "english_header"
                )
            )


    # -----------------------------------------------------
    # If a report contains start + end timestamps,
    # use the LAST timestamp in the header.
    # -----------------------------------------------------

    if len(candidates) == 0:

        return (
            pd.NaT,
            "unparsed"
        )

    candidates.sort(
        key=lambda x: x[0]
    )

    _, timestamp, method = candidates[-1]

    return (
        timestamp,
        method
    )


# =========================================================
# Run Audit
# =========================================================

timestamp_audit_rows = []


for _, row in eda_scope_df.iterrows():

    pdf_path = find_situation_pdf(
        row["file_name"]
    )

    if pdf_path is None:

        internal_datetime = pd.NaT
        method = "pdf_not_found"

    else:

        internal_datetime, method = (
            extract_internal_report_datetime(
                pdf_path
            )
        )


    if pd.notna(internal_datetime):

        difference_hours = (
            row["report_datetime"]
            - internal_datetime
        ).total_seconds() / 3600

    else:

        difference_hours = float("nan")


    timestamp_audit_rows.append(
        {
            "file_name": row["file_name"],
            "filename_datetime":
                row["report_datetime"],

            "internal_datetime":
                internal_datetime,

            "difference_hours":
                difference_hours,

            "parse_method":
                method,

            "affected_people":
                row["affected_people"]
        }
    )


timestamp_audit_df = pd.DataFrame(
    timestamp_audit_rows
)


# =========================================================
# Summary
# =========================================================

parsed_mask = (
    timestamp_audit_df[
        "internal_datetime"
    ].notna()
)

mismatch_mask = (
    parsed_mask
    &
    (
        timestamp_audit_df[
            "difference_hours"
        ].abs() > 0.01
    )
)


print(
    "Analytical reports:",
    len(timestamp_audit_df)
)

print(
    "Internal timestamp parsed:",
    int(parsed_mask.sum())
)

print(
    "Internal timestamp not parsed:",
    int((~parsed_mask).sum())
)

print(
    "\nExact timestamp matches:",
    int(
        (
            parsed_mask
            &
            ~mismatch_mask
        ).sum()
    )
)

print(
    "Timestamp mismatches:",
    int(mismatch_mask.sum())
)


# =========================================================
# Display mismatches
# =========================================================

timestamp_mismatches = (
    timestamp_audit_df[
        mismatch_mask
    ]
    .copy()
    .sort_values(
        "difference_hours",
        key=lambda x: x.abs(),
        ascending=False
    )
)


print(
    "\nTimestamp mismatches:"
)

print(
    timestamp_mismatches[
        [
            "file_name",
            "filename_datetime",
            "internal_datetime",
            "difference_hours",
            "affected_people"
        ]
    ]
    .head(50)
    .to_string(index=False)
)

In [ ]:
# =========================================================
# Final Timestamp Resolution for Analytical Dataset
# =========================================================

# Keep the DMC archive/file timestamp as the operational
# availability timestamp used for forecasting.

eda_scope_df["availability_datetime"] = (
    eda_scope_df["report_datetime"]
)


# ---------------------------------------------------------
# Attach audit information
# ---------------------------------------------------------

timestamp_qc = timestamp_audit_df[
    [
        "file_name",
        "internal_datetime",
        "difference_hours",
        "parse_method"
    ]
].copy()


eda_scope_df = eda_scope_df.merge(
    timestamp_qc,
    on="file_name",
    how="left"
)


# ---------------------------------------------------------
# Create QC flag
# ---------------------------------------------------------

def classify_timestamp_qc(row):

    if pd.isna(row["internal_datetime"]):
        return "internal_time_unparsed"

    diff = row["difference_hours"]

    if abs(diff) <= 2:
        return "close_match"

    if diff < 0:
        return "internal_after_archive_or_parse_issue"

    if abs(diff) > 72:
        return "large_difference_review"

    return "reporting_period_difference"


eda_scope_df["timestamp_qc_flag"] = (
    eda_scope_df.apply(
        classify_timestamp_qc,
        axis=1
    )
)


eda_scope_df["timestamp_source"] = (
    "dmc_archive_availability"
)


# ---------------------------------------------------------
# Final check
# ---------------------------------------------------------

print("Analytical rows:", len(eda_scope_df))

print("\nTimestamp QC:")
print(
    eda_scope_df["timestamp_qc_flag"]
    .value_counts()
)

print(
    "\nTimestamp used for analysis:"
)

print(
    eda_scope_df["timestamp_source"]
    .value_counts()
)

print(
    "\nMissing availability timestamps:",
    eda_scope_df[
        "availability_datetime"
    ].isna().sum()
)

## 6.2 Temporal Impact Analysis

In [ ]:
# =========================================================
# 6.2 Temporal Impact Analysis
# =========================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Create month variable
# ---------------------------------------------------------

eda_scope_df["month"] = (
    eda_scope_df["availability_datetime"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


# =========================================================
# 1. Affected People Over Time
# =========================================================

plt.figure(figsize=(13, 5))

plt.scatter(
    eda_scope_df["availability_datetime"],
    eda_scope_df["affected_people"],
    s=15,
    alpha=0.6
)

plt.title("Affected Population Reported Over Time")
plt.xlabel("Date")
plt.ylabel("Affected People")

plt.tight_layout()
plt.show()


# =========================================================
# 2. Log1p Affected People Over Time
# =========================================================

plt.figure(figsize=(13, 5))

plt.scatter(
    eda_scope_df["availability_datetime"],
    np.log1p(
        eda_scope_df["affected_people"]
    ),
    s=15,
    alpha=0.6
)

plt.title("Log-Transformed Affected Population Over Time")
plt.xlabel("Date")
plt.ylabel("log1p(Affected People)")

plt.tight_layout()
plt.show()


# =========================================================
# 3. People in Safety Centres Over Time
# =========================================================

plt.figure(figsize=(13, 5))

plt.scatter(
    eda_scope_df["availability_datetime"],
    eda_scope_df["people_in_safety_centres"],
    s=15,
    alpha=0.6
)

plt.title("People in Safety Centres Over Time")
plt.xlabel("Date")
plt.ylabel("People in Safety Centres")

plt.tight_layout()
plt.show()


# =========================================================
# 4. Monthly Maximum Affected Population
# =========================================================

monthly_max = (
    eda_scope_df
    .groupby("month")["affected_people"]
    .max()
)

plt.figure(figsize=(13, 5))

plt.plot(
    monthly_max.index,
    monthly_max.values,
    marker="o"
)

plt.title("Monthly Maximum Affected Population")
plt.xlabel("Month")
plt.ylabel("Maximum Affected People")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# =========================================================
# 5. Monthly Median Affected Population
# =========================================================

monthly_median = (
    eda_scope_df
    .groupby("month")["affected_people"]
    .median()
)

plt.figure(figsize=(13, 5))

plt.plot(
    monthly_median.index,
    monthly_median.values,
    marker="o"
)

plt.title("Monthly Median Affected Population")
plt.xlabel("Month")
plt.ylabel("Median Affected People")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# =========================================================
# 6. Monthly Maximum Safety-Centre Demand
# =========================================================

monthly_safety = (
    eda_scope_df
    .groupby("month")["people_in_safety_centres"]
    .max()
)

plt.figure(figsize=(13, 5))

plt.plot(
    monthly_safety.index,
    monthly_safety.values,
    marker="o"
)

plt.title("Monthly Maximum Safety-Centre Demand")
plt.xlabel("Month")
plt.ylabel("People in Safety Centres")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# =========================================================
# Monthly Summary
# =========================================================

monthly_summary = (
    eda_scope_df
    .groupby("month")
    .agg(
        reports=("file_name", "count"),

        median_affected_people=(
            "affected_people",
            "median"
        ),

        max_affected_people=(
            "affected_people",
            "max"
        ),

        max_people_in_safety_centres=(
            "people_in_safety_centres",
            "max"
        )
    )
    .reset_index()
)


print("\nTop 10 months by maximum affected population:")

print(
    monthly_summary
    .nlargest(
        10,
        "max_affected_people"
    )
    .to_string(index=False)
)

## 6.3 Weather Conditions and Human Impact

In [ ]:
# =========================================================
# 6.3 Weather Conditions vs Human Impact
# =========================================================

weather_eda = eda_scope_df.copy()

weather_eda["log_affected_people"] = np.log1p(
    weather_eda["affected_people"]
)

weather_eda["log_safety_population"] = np.log1p(
    weather_eda["people_in_safety_centres"]
)


# =========================================================
# 1. Weather Warning Level vs Affected Population
# =========================================================

warning_levels = [0, 1, 2, 3]

warning_data = [
    weather_eda.loc[
        weather_eda["weather_max_warning_level_24h"] == level,
        "log_affected_people"
    ].dropna()
    for level in warning_levels
]

plt.figure(figsize=(9, 5))

plt.boxplot(
    warning_data,
    labels=[
        "No/None",
        "Green",
        "Amber",
        "Red"
    ]
)

plt.title(
    "Human Impact by Maximum Weather Warning Level"
)

plt.xlabel("Maximum Weather Warning Level in Previous 24 Hours")
plt.ylabel("log1p(Affected People)")

plt.tight_layout()
plt.show()


# =========================================================
# 2. Forecast Rainfall Threshold vs Human Impact
# =========================================================

rain_plot = weather_eda.dropna(
    subset=[
        "weather_max_rainfall_threshold_24h",
        "affected_people"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    rain_plot["weather_max_rainfall_threshold_24h"],
    rain_plot["log_affected_people"],
    alpha=0.5,
    s=18
)

plt.title(
    "Forecast Rainfall Threshold vs Human Impact"
)

plt.xlabel(
    "Maximum Forecast Rainfall Threshold in Previous 24h (mm)"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 3. Maximum Wind Speed vs Human Impact
# =========================================================

wind_plot = weather_eda.dropna(
    subset=[
        "weather_max_wind_24h",
        "affected_people"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    wind_plot["weather_max_wind_24h"],
    wind_plot["log_affected_people"],
    alpha=0.5,
    s=18
)

plt.title(
    "Maximum Forecast Wind Speed vs Human Impact"
)

plt.xlabel(
    "Maximum Wind Speed in Previous 24h (km/h)"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 4. Heavy Rain Signal vs Human Impact
# =========================================================

heavy_rain_groups = []

heavy_rain_labels = []


for value, label in [
    (0, "No Heavy Rain Signal"),
    (1, "Heavy Rain Signal")
]:

    group = weather_eda.loc[
        weather_eda["weather_heavy_rain_24h"] == value,
        "log_affected_people"
    ].dropna()

    heavy_rain_groups.append(group)
    heavy_rain_labels.append(label)


plt.figure(figsize=(8, 5))

plt.boxplot(
    heavy_rain_groups,
    labels=heavy_rain_labels
)

plt.title(
    "Human Impact by Heavy-Rain Signal"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 5. Weather Warning Level vs Safety-Centre Demand
# =========================================================

safety_warning_data = [
    weather_eda.loc[
        weather_eda["weather_max_warning_level_24h"] == level,
        "log_safety_population"
    ].dropna()
    for level in warning_levels
]


plt.figure(figsize=(9, 5))

plt.boxplot(
    safety_warning_data,
    labels=[
        "No/None",
        "Green",
        "Amber",
        "Red"
    ]
)

plt.title(
    "Safety-Centre Demand by Weather Warning Level"
)

plt.xlabel(
    "Maximum Weather Warning Level in Previous 24 Hours"
)

plt.ylabel(
    "log1p(People in Safety Centres)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 6. Weather Severity Summary
# =========================================================

weather_warning_summary = (
    weather_eda
    .groupby(
        "weather_max_warning_level_24h"
    )
    .agg(
        reports=(
            "file_name",
            "count"
        ),

        median_affected_people=(
            "affected_people",
            "median"
        ),

        mean_affected_people=(
            "affected_people",
            "mean"
        ),

        max_affected_people=(
            "affected_people",
            "max"
        ),

        median_people_in_safety_centres=(
            "people_in_safety_centres",
            "median"
        ),

        max_people_in_safety_centres=(
            "people_in_safety_centres",
            "max"
        )
    )
    .reset_index()
)


print(
    "\nWeather warning level summary:"
)

print(
    weather_warning_summary
    .to_string(index=False)
)


# =========================================================
# 7. Spearman Association Check
# =========================================================

weather_corr_cols = [
    "affected_people",
    "people_in_safety_centres",
    "weather_max_rainfall_threshold_24h",
    "weather_max_wind_24h",
    "weather_max_warning_level_24h"
]


weather_spearman = (
    weather_eda[
        weather_corr_cols
    ]
    .corr(
        method="spearman"
    )
)


print(
    "\nSpearman correlations with affected_people:"
)

print(
    weather_spearman[
        "affected_people"
    ]
    .sort_values(
        ascending=False
    )
)

## 6.4 River Conditions and Human Impact

In [ ]:
# =========================================================
# 6.4 River Conditions vs Human Impact
# =========================================================

river_eda = eda_scope_df.copy()

river_eda["log_affected_people"] = np.log1p(
    river_eda["affected_people"]
)

river_eda["log_safety_population"] = np.log1p(
    river_eda["people_in_safety_centres"]
)


# ---------------------------------------------------------
# IMPORTANT:
# Analyze river severity only when a river report
# actually exists in the preceding 24 hours.
# Missing river report != no flood.
# ---------------------------------------------------------

river_covered = river_eda[
    river_eda["river_reports_24h"] > 0
].copy()


print(
    "Rows with River report coverage in previous 24h:",
    len(river_covered)
)


# =========================================================
# 1. Major Flood Stations vs Human Impact
# =========================================================

major_plot = river_covered.dropna(
    subset=[
        "river_max_major_flood_stations_24h",
        "affected_people"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    major_plot["river_max_major_flood_stations_24h"],
    major_plot["log_affected_people"],
    alpha=0.6,
    s=22
)

plt.title(
    "Major Flood Stations vs Human Impact"
)

plt.xlabel(
    "Maximum Number of Major-Flood Stations in Previous 24h"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 2. Minor Flood Stations vs Human Impact
# =========================================================

minor_plot = river_covered.dropna(
    subset=[
        "river_max_minor_flood_stations_24h",
        "affected_people"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    minor_plot["river_max_minor_flood_stations_24h"],
    minor_plot["log_affected_people"],
    alpha=0.6,
    s=22
)

plt.title(
    "Minor Flood Stations vs Human Impact"
)

plt.xlabel(
    "Maximum Number of Minor-Flood Stations in Previous 24h"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 3. River Alert Ratio vs Human Impact
# =========================================================

alert_ratio_plot = river_covered.dropna(
    subset=[
        "river_max_alert_ratio_24h",
        "affected_people"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    alert_ratio_plot["river_max_alert_ratio_24h"],
    alert_ratio_plot["log_affected_people"],
    alpha=0.6,
    s=22
)

plt.title(
    "Maximum River Alert Ratio vs Human Impact"
)

plt.xlabel(
    "Maximum River Alert Ratio in Previous 24h"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 4. River Rainfall vs Human Impact
# =========================================================

river_rain_plot = river_covered.dropna(
    subset=[
        "river_max_rainfall_24h_mm",
        "affected_people"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    river_rain_plot["river_max_rainfall_24h_mm"],
    river_rain_plot["log_affected_people"],
    alpha=0.6,
    s=22
)

plt.title(
    "River-Station Rainfall vs Human Impact"
)

plt.xlabel(
    "Maximum 24h Rainfall Reported by River Stations (mm)"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 5. Active River Warning vs Human Impact
# =========================================================

active_warning_groups = []
active_warning_labels = []

for value, label in [
    (0, "No Active Warning"),
    (1, "Active Warning")
]:

    group = river_covered.loc[
        river_covered["river_active_warning_24h"] == value,
        "log_affected_people"
    ].dropna()

    active_warning_groups.append(group)
    active_warning_labels.append(label)


plt.figure(figsize=(8, 5))

plt.boxplot(
    active_warning_groups,
    tick_labels=active_warning_labels
)

plt.title(
    "Human Impact by Active River Warning"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 6. Major Flood Stations vs Safety-Centre Demand
# =========================================================

safety_plot = river_covered.dropna(
    subset=[
        "river_max_major_flood_stations_24h",
        "people_in_safety_centres"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    safety_plot["river_max_major_flood_stations_24h"],
    safety_plot["log_safety_population"],
    alpha=0.6,
    s=22
)

plt.title(
    "Major Flood Stations vs Safety-Centre Demand"
)

plt.xlabel(
    "Maximum Number of Major-Flood Stations in Previous 24h"
)

plt.ylabel(
    "log1p(People in Safety Centres)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 7. Active Warning Summary
# =========================================================

river_warning_summary = (
    river_covered
    .dropna(
        subset=["river_active_warning_24h"]
    )
    .groupby(
        "river_active_warning_24h"
    )
    .agg(
        reports=("file_name", "count"),

        median_affected_people=(
            "affected_people",
            "median"
        ),

        mean_affected_people=(
            "affected_people",
            "mean"
        ),

        max_affected_people=(
            "affected_people",
            "max"
        ),

        median_people_in_safety_centres=(
            "people_in_safety_centres",
            "median"
        ),

        max_people_in_safety_centres=(
            "people_in_safety_centres",
            "max"
        )
    )
    .reset_index()
)


print(
    "\nRiver active-warning summary:"
)

print(
    river_warning_summary.to_string(
        index=False
    )
)


# =========================================================
# 8. Spearman Associations
# =========================================================

river_corr_cols = [
    "affected_people",
    "people_in_safety_centres",
    "river_max_alert_stations_24h",
    "river_max_minor_flood_stations_24h",
    "river_max_major_flood_stations_24h",
    "river_max_rising_stations_24h",
    "river_max_rainfall_24h_mm",
    "river_max_alert_ratio_24h",
    "river_active_warning_24h"
]


river_spearman = (
    river_covered[
        river_corr_cols
    ]
    .corr(
        method="spearman"
    )
)


print(
    "\nSpearman correlations with affected_people:"
)

print(
    river_spearman[
        "affected_people"
    ]
    .sort_values(
        ascending=False
    )
)

## 6.5 Landslide Warning Conditions and Human Impact

In [ ]:
# =========================================================
# 6.5 Landslide Warning Conditions vs Human Impact
# =========================================================

landslide_eda = eda_scope_df.copy()

landslide_eda["log_affected_people"] = np.log1p(
    landslide_eda["affected_people"]
)

landslide_eda["log_safety_population"] = np.log1p(
    landslide_eda["people_in_safety_centres"]
)


print(
    "Landslide analytical rows:",
    len(landslide_eda)
)

print(
    "\nActive landslide warning counts:"
)

print(
    landslide_eda[
        "landslide_active_warning"
    ].value_counts(
        dropna=False
    )
)


# =========================================================
# 1. Active Landslide Warning vs Human Impact
# =========================================================

active_groups = []
active_labels = []

for value, label in [
    (0, "No Active Warning"),
    (1, "Active Warning")
]:

    group = landslide_eda.loc[
        landslide_eda["landslide_active_warning"] == value,
        "log_affected_people"
    ].dropna()

    active_groups.append(group)
    active_labels.append(label)


plt.figure(figsize=(8, 5))

plt.boxplot(
    active_groups,
    tick_labels=active_labels
)

plt.title(
    "Human Impact by Active Landslide Warning"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 2. Landslide Warning Level vs Human Impact
# =========================================================

warning_levels = [0, 1, 2, 3]

level_groups = []

for level in warning_levels:

    group = landslide_eda.loc[
        landslide_eda["landslide_max_warning_level"] == level,
        "log_affected_people"
    ].dropna()

    level_groups.append(group)


plt.figure(figsize=(9, 5))

plt.boxplot(
    level_groups,
    tick_labels=[
        "Inactive / None",
        "Level 1",
        "Level 2",
        "Level 3"
    ]
)

plt.title(
    "Human Impact by Landslide Warning Level"
)

plt.xlabel(
    "Maximum Active Landslide Warning Level"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 3. Districts at Risk vs Human Impact
# =========================================================

district_plot = landslide_eda[
    landslide_eda["landslide_active_warning"] == 1
].dropna(
    subset=[
        "landslide_districts_at_risk",
        "affected_people"
    ]
)


plt.figure(figsize=(9, 5))

plt.scatter(
    district_plot["landslide_districts_at_risk"],
    district_plot["log_affected_people"],
    alpha=0.6,
    s=22
)

plt.title(
    "Landslide Districts at Risk vs Human Impact"
)

plt.xlabel(
    "Number of Districts Under Active Landslide Warning"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 4. Level 3 Districts vs Human Impact
# =========================================================

level3_plot = landslide_eda[
    landslide_eda["landslide_active_warning"] == 1
].dropna(
    subset=[
        "landslide_level3_districts",
        "affected_people"
    ]
)


plt.figure(figsize=(9, 5))

plt.scatter(
    level3_plot["landslide_level3_districts"],
    level3_plot["log_affected_people"],
    alpha=0.6,
    s=22
)

plt.title(
    "Level 3 Landslide Districts vs Human Impact"
)

plt.xlabel(
    "Number of Level 3 Districts"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 5. Warning Level vs Safety-Centre Demand
# =========================================================

safety_level_groups = []

for level in warning_levels:

    group = landslide_eda.loc[
        landslide_eda["landslide_max_warning_level"] == level,
        "log_safety_population"
    ].dropna()

    safety_level_groups.append(group)


plt.figure(figsize=(9, 5))

plt.boxplot(
    safety_level_groups,
    tick_labels=[
        "Inactive / None",
        "Level 1",
        "Level 2",
        "Level 3"
    ]
)

plt.title(
    "Safety-Centre Demand by Landslide Warning Level"
)

plt.xlabel(
    "Maximum Active Landslide Warning Level"
)

plt.ylabel(
    "log1p(People in Safety Centres)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 6. Active Landslide Warning Summary
# =========================================================

landslide_summary = (
    landslide_eda
    .groupby(
        "landslide_active_warning"
    )
    .agg(
        reports=(
            "file_name",
            "count"
        ),

        median_affected_people=(
            "affected_people",
            "median"
        ),

        mean_affected_people=(
            "affected_people",
            "mean"
        ),

        max_affected_people=(
            "affected_people",
            "max"
        ),

        median_people_in_safety_centres=(
            "people_in_safety_centres",
            "median"
        ),

        max_people_in_safety_centres=(
            "people_in_safety_centres",
            "max"
        ),

        median_districts_at_risk=(
            "landslide_districts_at_risk",
            "median"
        ),

        max_districts_at_risk=(
            "landslide_districts_at_risk",
            "max"
        )
    )
    .reset_index()
)


print(
    "\nLandslide active-warning summary:"
)

print(
    landslide_summary.to_string(
        index=False
    )
)


# =========================================================
# 7. Spearman Associations
# =========================================================

landslide_corr_cols = [
    "affected_people",
    "people_in_safety_centres",
    "landslide_active_warning",
    "landslide_max_warning_level",
    "landslide_level1_districts",
    "landslide_level2_districts",
    "landslide_level3_districts",
    "landslide_districts_at_risk"
]


landslide_spearman = (
    landslide_eda[
        landslide_corr_cols
    ]
    .corr(
        method="spearman"
    )
)


print(
    "\nSpearman correlations with affected_people:"
)

print(
    landslide_spearman[
        "affected_people"
    ]
    .sort_values(
        ascending=False
    )
)

## 6.6 Safety-Centre Demand Analysis

In [ ]:
# =========================================================
# 6.6 Safety-Centre Demand Analysis
# =========================================================

safety_eda = eda_scope_df.copy()


# =========================================================
# 1. Create useful EDA-only variables
# =========================================================

safety_eda["safety_centre_use_flag"] = (
    safety_eda["people_in_safety_centres"] > 0
).astype(int)


# Occupancy per active safety centre
safety_eda["people_per_safety_centre"] = np.where(
    safety_eda["safety_centres"] > 0,
    safety_eda["people_in_safety_centres"]
    / safety_eda["safety_centres"],
    np.nan
)


# Share of affected population staying in safety centres
safety_eda["safety_centre_population_ratio"] = np.where(
    safety_eda["affected_people"] > 0,
    safety_eda["people_in_safety_centres"]
    / safety_eda["affected_people"],
    np.nan
)


safety_eda["log_affected_people"] = np.log1p(
    safety_eda["affected_people"]
)

safety_eda["log_safety_population"] = np.log1p(
    safety_eda["people_in_safety_centres"]
)


# =========================================================
# 2. Zero vs Positive Safety-Centre Demand
# =========================================================

print("Safety-centre usage:")

print(
    safety_eda["safety_centre_use_flag"]
    .value_counts()
)


print("\nSafety-centre usage percentage:")

print(
    (
        safety_eda["safety_centre_use_flag"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)


# =========================================================
# 3. Safety-Centre Population Distribution
# =========================================================

plt.figure(figsize=(9, 5))

plt.hist(
    safety_eda[
        "people_in_safety_centres"
    ].dropna(),
    bins=50
)

plt.title(
    "Distribution of People in Safety Centres"
)

plt.xlabel(
    "People in Safety Centres"
)

plt.ylabel(
    "Number of Reports"
)

plt.tight_layout()
plt.show()


# =========================================================
# 4. Log Distribution
# =========================================================

plt.figure(figsize=(9, 5))

plt.hist(
    safety_eda[
        "log_safety_population"
    ].dropna(),
    bins=50
)

plt.title(
    "Log1p Distribution of Safety-Centre Population"
)

plt.xlabel(
    "log1p(People in Safety Centres)"
)

plt.ylabel(
    "Number of Reports"
)

plt.tight_layout()
plt.show()


# =========================================================
# 5. Affected Population vs Safety-Centre Population
# =========================================================

relationship_df = safety_eda.dropna(
    subset=[
        "affected_people",
        "people_in_safety_centres"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    relationship_df["log_affected_people"],
    relationship_df["log_safety_population"],
    alpha=0.5,
    s=20
)

plt.title(
    "Human Impact vs Safety-Centre Demand"
)

plt.xlabel(
    "log1p(Affected People)"
)

plt.ylabel(
    "log1p(People in Safety Centres)"
)

plt.tight_layout()
plt.show()


# =========================================================
# 6. Number of Safety Centres vs Population Sheltered
# =========================================================

centre_relationship = safety_eda[
    safety_eda["safety_centres"] > 0
].dropna(
    subset=[
        "safety_centres",
        "people_in_safety_centres"
    ]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    centre_relationship["safety_centres"],
    centre_relationship["people_in_safety_centres"],
    alpha=0.5,
    s=20
)

plt.title(
    "Number of Safety Centres vs Sheltered Population"
)

plt.xlabel(
    "Number of Active Safety Centres"
)

plt.ylabel(
    "People in Safety Centres"
)

plt.tight_layout()
plt.show()


# =========================================================
# 7. Occupancy per Safety Centre
# =========================================================

occupancy = (
    safety_eda[
        "people_per_safety_centre"
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)


plt.figure(figsize=(9, 5))

plt.hist(
    occupancy,
    bins=50
)

plt.title(
    "Distribution of People per Active Safety Centre"
)

plt.xlabel(
    "People per Safety Centre"
)

plt.ylabel(
    "Number of Reports"
)

plt.tight_layout()
plt.show()


# =========================================================
# 8. Safety-Centre Population Ratio
# =========================================================

ratio_data = (
    safety_eda[
        "safety_centre_population_ratio"
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)


plt.figure(figsize=(9, 5))

plt.hist(
    ratio_data,
    bins=50
)

plt.title(
    "Share of Affected Population in Safety Centres"
)

plt.xlabel(
    "People in Safety Centres / Affected People"
)

plt.ylabel(
    "Number of Reports"
)

plt.tight_layout()
plt.show()


# =========================================================
# 9. Summary Statistics
# =========================================================

print(
    "\nSafety-centre demand summary:"
)

print(
    safety_eda[
        [
            "safety_centres",
            "people_in_safety_centres",
            "people_per_safety_centre",
            "safety_centre_population_ratio"
        ]
    ]
    .describe()
    .T
)


# =========================================================
# 10. Spearman Relationships
# =========================================================

safety_corr_cols = [
    "affected_people",
    "affected_families",
    "safety_centres",
    "families_in_safety_centres",
    "people_in_safety_centres"
]


safety_spearman = (
    safety_eda[
        safety_corr_cols
    ]
    .corr(
        method="spearman"
    )
)


print(
    "\nSpearman correlations with people_in_safety_centres:"
)

print(
    safety_spearman[
        "people_in_safety_centres"
    ]
    .sort_values(
        ascending=False
    )
)

In [ ]:
# =========================================================
# 6.6A Safety-Centre Consistency Audit
# =========================================================

safety_audit = safety_eda.copy()


# ---------------------------------------------------------
# 1. Identify logically unusual rows
# ---------------------------------------------------------

safety_audit["ratio_gt_1"] = (
    safety_audit["safety_centre_population_ratio"] > 1
)

safety_audit["safety_people_gt_affected"] = (
    safety_audit["people_in_safety_centres"]
    >
    safety_audit["affected_people"]
)

safety_audit["safety_families_gt_affected"] = (
    safety_audit["families_in_safety_centres"]
    >
    safety_audit["affected_families"]
)

safety_audit["people_present_but_no_centres"] = (
    (safety_audit["people_in_safety_centres"] > 0)
    &
    (safety_audit["safety_centres"] == 0)
)


# ---------------------------------------------------------
# 2. Summary
# ---------------------------------------------------------

print(
    "Rows with safety-centre population ratio > 1:",
    int(safety_audit["ratio_gt_1"].sum())
)

print(
    "Rows where people in safety centres > affected people:",
    int(safety_audit["safety_people_gt_affected"].sum())
)

print(
    "Rows where families in safety centres > affected families:",
    int(safety_audit["safety_families_gt_affected"].sum())
)

print(
    "Rows with people in safety centres but safety_centres = 0:",
    int(safety_audit["people_present_but_no_centres"].sum())
)


# ---------------------------------------------------------
# 3. Inspect the most extreme ratio cases
# ---------------------------------------------------------

extreme_safety_rows = (
    safety_audit[
        safety_audit["ratio_gt_1"]
    ][
        [
            "availability_datetime",
            "file_name",
            "disaster_types",
            "affected_people",
            "affected_families",
            "safety_centres",
            "families_in_safety_centres",
            "people_in_safety_centres",
            "safety_centre_population_ratio",
            "extraction_status"
        ]
    ]
    .sort_values(
        "safety_centre_population_ratio",
        ascending=False
    )
)


print(
    "\nTop unusual safety-centre ratio rows:"
)

print(
    extreme_safety_rows
    .head(30)
    .to_string(index=False)
)


# ---------------------------------------------------------
# 4. Ratio distribution only within logically consistent rows
# ---------------------------------------------------------

valid_ratio = safety_audit[
    (
        safety_audit["safety_centre_population_ratio"] >= 0
    )
    &
    (
        safety_audit["safety_centre_population_ratio"] <= 1
    )
][
    "safety_centre_population_ratio"
]


print(
    "\nValid ratio rows:",
    len(valid_ratio)
)

print(
    "Median valid ratio:",
    valid_ratio.median()
)

print(
    "75th percentile:",
    valid_ratio.quantile(0.75)
)

print(
    "Maximum valid ratio:",
    valid_ratio.max()
)

In [ ]:
# =========================================================
# 6.6B Safety-Centre Consistency Resolution
# =========================================================

# Keep all original observations.
# Only create a cleaned ratio where the subset assumption
# is logically consistent.

safety_eda["safety_ratio_valid_flag"] = (
    (safety_eda["affected_people"] > 0)
    &
    (safety_eda["people_in_safety_centres"] >= 0)
    &
    (
        safety_eda["people_in_safety_centres"]
        <= safety_eda["affected_people"]
    )
)


safety_eda["safety_centre_population_ratio_clean"] = np.where(
    safety_eda["safety_ratio_valid_flag"],
    
    safety_eda["people_in_safety_centres"]
    / safety_eda["affected_people"],
    
    np.nan
)


print(
    "Valid shelter-ratio rows:",
    safety_eda[
        "safety_ratio_valid_flag"
    ].sum()
)

print(
    "Ratio excluded from interpretation:",
    (
        ~safety_eda[
            "safety_ratio_valid_flag"
        ]
    ).sum()
)

print(
    "\nClean ratio summary:"
)

print(
    safety_eda[
        "safety_centre_population_ratio_clean"
    ].describe()
)

## 6.7 Multi-Source Correlation Analysis

In [ ]:
# =========================================================
# 6.7 Multi-Source Spearman Correlation Heatmap
# =========================================================

correlation_cols = [
    # Human impact
    "affected_people",
    "affected_families",
    "people_in_safety_centres",

    # Weather
    "weather_max_rainfall_threshold_24h",
    "weather_max_wind_24h",
    "weather_max_warning_level_24h",

    # River
    "river_max_alert_stations_24h",
    "river_max_minor_flood_stations_24h",
    "river_max_major_flood_stations_24h",
    "river_max_alert_ratio_24h",

    # Landslide
    "landslide_active_warning",
    "landslide_max_warning_level",
    "landslide_districts_at_risk"
]


corr_df = (
    eda_scope_df[
        correlation_cols
    ]
    .corr(method="spearman")
)


# =========================================================
# Heatmap using Matplotlib
# =========================================================

plt.figure(figsize=(13, 10))

image = plt.imshow(
    corr_df,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(
    image,
    label="Spearman Correlation"
)

plt.xticks(
    range(len(correlation_cols)),
    correlation_cols,
    rotation=90
)

plt.yticks(
    range(len(correlation_cols)),
    correlation_cols
)

plt.title(
    "Multi-Source Spearman Correlation Heatmap"
)

plt.tight_layout()
plt.show()


# =========================================================
# Correlations with Main Human-Impact Target
# =========================================================

affected_corr = (
    corr_df["affected_people"]
    .drop("affected_people")
    .sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
)


print(
    "\nVariables associated with affected_people:"
)

print(
    affected_corr
)


# =========================================================
# Correlations with Safety-Centre Demand
# =========================================================

safety_corr = (
    corr_df["people_in_safety_centres"]
    .drop("people_in_safety_centres")
    .sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
)


print(
    "\nVariables associated with people_in_safety_centres:"
)

print(
    safety_corr
)

### 6.8 Key EDA Findings

1. Human-impact variables are strongly right-skewed. Log1p transformation substantially reduced the skewness of affected population and affected families.

2. Major disaster episodes create repeated cumulative Situation Report snapshots. Therefore, observations cannot be treated as fully independent and random train-test splitting should be avoided.

3. Drought reports, non-Situation files, and a non-comparable incident report were excluded from the analytical scope while the original integrated master dataset was preserved.

4. Weather indicators showed weak positive monotonic associations with human impact when considered individually. This indicates that weather information alone is insufficient for explaining disaster impact.

5. River severity indicators showed stronger associations than weather-only indicators during the period for which river observations were available. Major/minor flood station counts and river alert ratio were positively associated with affected population.

6. Landslide warning indicators showed meaningful positive associations with both human impact and safety-centre demand. Higher warning levels and a larger number of districts at risk were generally associated with larger impacts.

7. Safety-centre demand was strongly associated with overall human impact. However, same-report safety-centre variables must not be used as predictors of the same target because this would introduce target leakage.

8. A small number of reports contained safety-centre populations larger than the corresponding affected-population value. Original observations were retained, but the derived shelter-population ratio was restricted to logically consistent rows and will not be used blindly as a modelling feature.

9. Overall EDA supports a multi-source forecasting approach combining previous human-impact conditions, weather, river, landslide and temporal information rather than relying on a single hazard indicator.

In [26]:
# =========================================================
# SAFE RESUME DIAGNOSTIC
# =========================================================

import pandas as pd
from pathlib import Path

processed_dir = Path("data/processed")

integrated_df = pd.read_csv(
    processed_dir / "integrated_disaster_dataset.csv"
)

print("Dataset shape:", integrated_df.shape)

print("\nALL COLUMNS:")
for i, col in enumerate(integrated_df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nPOSSIBLE DATE / TIME COLUMNS:")

datetime_candidates = [
    col
    for col in integrated_df.columns
    if any(
        word in col.lower()
        for word in [
            "date",
            "time",
            "datetime",
            "timestamp",
            "available",
            "issue"
        ]
    )
]

for col in datetime_candidates:
    print("-", col)

print("\nFirst row of possible date/time columns:")

if datetime_candidates:
    print(
        integrated_df[
            datetime_candidates
        ].head(3).to_string()
    )

Dataset shape: (1456, 62)

ALL COLUMNS:
01. file_name
02. report_date
03. disaster_types
04. text_length
05. families_with_relatives
06. people_with_relatives
07. peak_affected_families
08. peak_affected_people
09. peak_safety_centres
10. peak_families_in_safety_centres
11. peak_people_in_safety_centres
12. idp_families
13. idp_people
14. affected_families
15. affected_people
16. deaths
17. injured
18. missing
19. fully_damaged_houses
20. partially_damaged_houses
21. enterprise_damage
22. infrastructure_damage
23. safety_centres
24. families_in_safety_centres
25. people_in_safety_centres
26. grand_total_extracted
27. extraction_status
28. issue_time_standard
29. report_datetime
30. weather_reports_24h
31. weather_max_rainfall_threshold_24h
32. weather_max_wind_24h
33. weather_max_warning_level_24h
34. weather_heavy_rain_24h
35. weather_thunderstorm_24h
36. weather_strong_wind_24h
37. weather_low_pressure_24h
38. weather_depression_24h
39. weather_cyclone_24h
40. latest_weather_age_hour

In [27]:
# =========================================================
# RESUME FROM INTEGRATED DATASET - SAFE VERSION
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path

processed_dir = Path("data/processed")

integrated_df = pd.read_csv(
    processed_dir / "integrated_disaster_dataset.csv"
)

# Parse saved integrated timestamp
integrated_df["report_datetime"] = pd.to_datetime(
    integrated_df["report_datetime"],
    errors="coerce"
)

# Temporary alias used by later feature-engineering code
integrated_df["availability_datetime"] = (
    integrated_df["report_datetime"]
)

print("=" * 65)
print("RESUME DATA AUDIT")
print("=" * 65)

print(
    "Shape:",
    integrated_df.shape
)

print(
    "Missing report_datetime:",
    integrated_df["report_datetime"].isna().sum()
)

print(
    "Earliest datetime:",
    integrated_df["report_datetime"].min()
)

print(
    "Latest datetime:",
    integrated_df["report_datetime"].max()
)

# Expected final integrated snapshot
expected_latest = pd.Timestamp(
    "2026-08-14 10:00:00"
)

actual_latest = (
    integrated_df[
        "report_datetime"
    ].max()
)

print(
    "Matches final snapshot:",
    actual_latest == expected_latest
)

print("\nExtraction status counts:")

print(
    integrated_df[
        "extraction_status"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nDrought rows:"
)

drought_mask = (
    integrated_df[
        "disaster_types"
    ]
    .fillna("")
    .str.contains(
        "Drought",
        case=False,
        regex=False
    )
)

print(
    drought_mask.sum()
)

print("=" * 65)

RESUME DATA AUDIT
Shape: (1456, 63)
Missing report_datetime: 0
Earliest datetime: 2023-01-02 06:00:00
Latest datetime: 2026-08-14 10:00:00
Matches final snapshot: True

Extraction status counts:
extraction_status
grand_total                         1293
modern_11_column                      53
confirmed_zero_report                 38
modern_9_column                       35
non_situation_report                   9
modern_7_column                        8
modern_18_column                       4
modern_summary_7_column                3
insufficient_text                      3
modern_summary_13_column               2
modern_13_column                       2
grand_total_ref_fallback               2
modern_summary_8_column                1
modern_summary_9_column                1
non_comparable_incident_report         1
manual_review_unsupported_layout       1
Name: count, dtype: int64

Drought rows:
0


In [29]:
# =========================================================
# FIND ORIGINAL EDA SCOPE / DROUGHT FILTER CODE
# WITHOUT RUNNING OLD NOTEBOOK CELLS
# =========================================================

import json
from pathlib import Path

notebook_path = Path(
    "FloodImpact_LK_Final.ipynb"
)

with notebook_path.open(
    "r",
    encoding="utf-8"
) as f:
    notebook_json = json.load(f)


search_terms = [
    "drought",
    "eda_scope_df",
    "non_situation_report",
    "non_comparable",
    "analytical scope"
]


print("=" * 80)
print("SEARCHING NOTEBOOK FOR ORIGINAL SCOPE FILTER")
print("=" * 80)


for cell_number, cell in enumerate(
    notebook_json.get(
        "cells",
        []
    )
):

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    source_lower = source.lower()


    if any(
        term.lower() in source_lower
        for term in search_terms
    ):

        print(
            f"\n{'=' * 80}"
        )

        print(
            f"CELL INDEX: {cell_number}"
        )

        print(
            f"CELL TYPE: {cell.get('cell_type')}"
        )

        print(
            "=" * 80
        )

        print(
            source
        )


print(
    "\nSEARCH COMPLETE"
)

SEARCHING NOTEBOOK FOR ORIGINAL SCOPE FILTER

CELL INDEX: 16
CELL TYPE: markdown
9)is_non_comparable_incident_report

CELL INDEX: 17
CELL TYPE: code
def is_non_comparable_incident_report(text):

    text_lower = normalize_text(text).lower()

    return (
        "daily disaster incident report" in text_lower
        and "situation report" not in text_lower
    )

CELL INDEX: 18
CELL TYPE: markdown
10)is_non_situation_report

CELL INDEX: 19
CELL TYPE: code
def is_non_situation_report(text):

    text_lower = normalize_text(text).lower()

    # Weather forecast documents
    weather_markers = [
        "weather forecast",
        "rainfall amount",
        "condition of rain",
        "state of sea",
        "showers or thundershowers",
        "department of meteorology",
        "wind speed",
        "sea areas",
    ]

    weather_score = sum(
        marker in text_lower
        for marker in weather_markers
    )

    if weather_score >= 2:
        return True

    # River/water-lev

In [30]:
# =========================================================
# FAST RESUME - RECREATE ORIGINAL ANALYTICAL SCOPE
#
# Does NOT rerun full PDF extraction.
# Reads only enough PDF text to obtain the first 20
# non-empty lines used by the original heading classifier.
# Then permanently saves the analytical dataset.
# =========================================================

from pathlib import Path

import fitz
import numpy as np
import pandas as pd


# =========================================================
# 1. LOAD SAVED INTEGRATED DATA
# =========================================================

processed_dir = Path("data/processed")

integrated_df = pd.read_csv(
    processed_dir
    / "integrated_disaster_dataset.csv"
)


integrated_df["report_datetime"] = pd.to_datetime(
    integrated_df["report_datetime"],
    errors="coerce"
)


print(
    "Integrated rows:",
    len(integrated_df)
)


# =========================================================
# 2. LOCATE SITUATION REPORT PDFs
# =========================================================

situation_pdf_dir = Path(
    "data/raw/situation_reports"
)


pdf_paths = list(
    situation_pdf_dir.rglob("*.pdf")
)


pdf_lookup = {
    path.name: path
    for path in pdf_paths
}


print(
    "Situation PDFs located:",
    len(pdf_lookup)
)


# =========================================================
# 3. FAST HEADER READER
#
# Reproduces the important behaviour of the original:
# inspect only the first 20 non-empty text lines.
#
# We stop reading pages once 20 lines are collected.
# =========================================================

def extract_first_20_lines(pdf_path):

    lines = []

    try:

        doc = fitz.open(
            pdf_path
        )

        for page in doc:

            page_text = (
                page.get_text(
                    "text"
                )
                or ""
            )

            page_lines = [
                line.strip()
                for line
                in page_text.splitlines()
                if line.strip()
            ]


            lines.extend(
                page_lines
            )


            # Original classifier only used first 20 lines.
            if len(lines) >= 20:
                break


        doc.close()


        return lines[:20]


    except Exception:

        return []


# =========================================================
# 4. ORIGINAL HEADING CLASSIFICATION LOGIC
# =========================================================

def classify_heading_fast(
    pdf_path
):

    lines = (
        extract_first_20_lines(
            pdf_path
        )
    )


    if len(lines) == 0:

        return "pdf_text_unavailable"


    header_text = " ".join(
        lines
    )


    header_lower = (
        header_text.lower()
    )


    # -----------------------------------------------------
    # Drought-specific Situation Report
    # -----------------------------------------------------

    if (
        "drought situation report"
        in header_lower

        or "drought"
        in header_lower

        or "නියඟ"
        in header_text
    ):

        return "drought_report"


    # -----------------------------------------------------
    # Special Situation Report
    # -----------------------------------------------------

    if (
        "special situation report"
        in header_lower

        or "විශේෂ"
        in header_text
    ):

        return (
            "special_situation_report"
        )


    # -----------------------------------------------------
    # Standard Situation Report
    # -----------------------------------------------------

    if (
        "situation report"
        in header_lower

        or "තත්ත්ව වාර්තාව"
        in header_text
    ):

        return (
            "general_situation_report"
        )


    return "other_heading"


# =========================================================
# 5. CLASSIFY ALL 1456 REPORTS
#
# This is much lighter than full extraction because
# only the heading is read.
# =========================================================

categories = []


for i, row in integrated_df.iterrows():

    file_name = (
        row["file_name"]
    )


    pdf_path = (
        pdf_lookup.get(
            file_name
        )
    )


    if pdf_path is None:

        category = (
            "pdf_not_found"
        )

    else:

        category = (
            classify_heading_fast(
                pdf_path
            )
        )


    categories.append(
        category
    )


    if (
        (i + 1) % 250
        == 0
    ):

        print(
            f"Checked {i + 1}"
            f"/{len(integrated_df)} reports"
        )


integrated_df[
    "situation_report_category"
] = categories


# =========================================================
# 6. ORIGINAL FINAL SCOPE LOGIC
# =========================================================

integrated_df[
    "scope_status"
] = "retain"


# Drought
integrated_df.loc[
    integrated_df[
        "situation_report_category"
    ]
    == "drought_report",

    "scope_status"

] = "exclude_drought"


# Non-situation documents
integrated_df.loc[
    integrated_df[
        "extraction_status"
    ]
    == "non_situation_report",

    "scope_status"

] = "exclude_non_situation"


# Non-comparable incident report
integrated_df.loc[
    integrated_df[
        "extraction_status"
    ]
    == "non_comparable_incident_report",

    "scope_status"

] = "exclude_non_comparable"


# =========================================================
# 7. RECREATE eda_scope_df
# =========================================================

eda_scope_df = (

    integrated_df[
        integrated_df[
            "scope_status"
        ]
        == "retain"
    ]

    .copy()

    .sort_values(
        "report_datetime"
    )

    .reset_index(
        drop=True
    )
)


# Operational forecasting timestamp
eda_scope_df[
    "availability_datetime"
] = (
    eda_scope_df[
        "report_datetime"
    ]
)


# =========================================================
# 8. SAVE PERMANENT RESUME DATASET
# =========================================================

resume_output_path = (
    processed_dir
    / "analytical_scope_dataset.csv"
)


eda_scope_df.to_csv(
    resume_output_path,
    index=False
)


# Also save classification audit
scope_audit_path = (
    processed_dir
    / "situation_scope_classification.csv"
)


integrated_df[
    [
        "file_name",
        "report_datetime",
        "situation_report_category",
        "extraction_status",
        "scope_status"
    ]
].to_csv(
    scope_audit_path,
    index=False
)


# =========================================================
# 9. FINAL QC
# =========================================================

print(
    "\n"
    + "=" * 70
)

print(
    "FAST RESUME SCOPE QC"
)

print(
    "=" * 70
)


print(
    "\nSituation report categories:"
)

print(
    integrated_df[
        "situation_report_category"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nFinal scope classification:"
)

print(
    integrated_df[
        "scope_status"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nOriginal integrated rows:",
    len(
        integrated_df
    )
)


print(
    "Analytical rows:",
    len(
        eda_scope_df
    )
)


print(
    "Missing affected_people:",
    eda_scope_df[
        "affected_people"
    ]
    .isna()
    .sum()
)


print(
    "Latest analytical datetime:",
    eda_scope_df[
        "availability_datetime"
    ].max()
)


print(
    "\nSaved:",
    resume_output_path
)

print(
    "Saved scope audit:",
    scope_audit_path
)


print(
    "\nEXPECTED FROZEN RESULT:"
)

print(
    "exclude_drought = 61"
)

print(
    "exclude_non_situation = 9"
)

print(
    "exclude_non_comparable = 1"
)

print(
    "retain = 1385"
)

print(
    "=" * 70
)

Integrated rows: 1456
Situation PDFs located: 1456
Checked 250/1456 reports
Checked 500/1456 reports
Checked 750/1456 reports
Checked 1000/1456 reports
Checked 1250/1456 reports

FAST RESUME SCOPE QC

Situation report categories:
situation_report_category
other_heading               1362
general_situation_report      94
Name: count, dtype: int64

Final scope classification:
scope_status
retain                    1446
exclude_non_situation        9
exclude_non_comparable       1
Name: count, dtype: int64

Original integrated rows: 1456
Analytical rows: 1446
Missing affected_people: 4
Latest analytical datetime: 2026-08-14 10:00:00

Saved: data\processed\analytical_scope_dataset.csv
Saved scope audit: data\processed\situation_scope_classification.csv

EXPECTED FROZEN RESULT:
exclude_drought = 61
exclude_non_situation = 9
exclude_non_comparable = 1
retain = 1385


In [31]:
# =========================================================
# FIND ORIGINAL PDF TEXT EXTRACTION FUNCTION
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

search_terms = [
    "def extract_pdf_text",
    "extract_pdf_text =",
    "def find_situation_pdf"
]

for i, cell in enumerate(nb["cells"]):

    source = "".join(
        cell.get("source", [])
    )

    if any(
        term in source
        for term in search_terms
    ):
        print("\n" + "=" * 80)
        print("CELL INDEX:", i)
        print("=" * 80)
        print(source)


CELL INDEX: 5
def extract_pdf_text(pdf_path):
    pages_text = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()

                if text:
                    pages_text.append(text)

        return "\n".join(pages_text)

    except Exception as e:
        print(f"Error reading {pdf_path.name}: {e}")
        return ""

CELL INDEX: 82
# =========================================================
# EDA - Inspect Situation Report Headings for Event Identity
# =========================================================

from pathlib import Path

SITUATION_RAW_DIR = Path(
    "data/raw/situation_reports"
)


def find_situation_pdf(file_name):

    matches = list(
        SITUATION_RAW_DIR.rglob(file_name)
    )

    if matches:
        return matches[0]

    return None


# ---------------------------------------------------------
# Select representative reports
# -------------------------------------------

In [32]:
# =========================================================
# FINAL SCOPE RECOVERY
# Reproduce original EDA drought classification
# using pdfplumber
# =========================================================

from pathlib import Path
import pandas as pd
import pdfplumber


# =========================================================
# 1. LOAD INTEGRATED DATA
# =========================================================

processed_dir = Path("data/processed")

integrated_df = pd.read_csv(
    processed_dir / "integrated_disaster_dataset.csv"
)

integrated_df["report_datetime"] = pd.to_datetime(
    integrated_df["report_datetime"],
    errors="coerce"
)

print("Integrated rows:", len(integrated_df))


# =========================================================
# 2. LOCATE PDFs
# =========================================================

pdf_dir = Path(
    "data/raw/situation_reports"
)

pdf_files = list(
    pdf_dir.rglob("*.pdf")
)

pdf_lookup = {
    p.name: p
    for p in pdf_files
}

print(
    "Situation PDFs located:",
    len(pdf_lookup)
)


# =========================================================
# 3. ORIGINAL PDFPLUMBER EXTRACTION BEHAVIOUR
#    Optimized: stop once first 20 lines are available
# =========================================================

def extract_first_20_lines_pdfplumber(
    pdf_path
):

    lines = []

    try:

        with pdfplumber.open(
            pdf_path
        ) as pdf:

            for page in pdf.pages:

                text = (
                    page.extract_text()
                )

                if not text:
                    continue

                page_lines = [
                    line.strip()
                    for line
                    in text.splitlines()
                    if line.strip()
                ]

                lines.extend(
                    page_lines
                )

                # Original classifier only needs
                # first 20 non-empty lines
                if len(lines) >= 20:
                    break

        return lines[:20]

    except Exception as e:

        print(
            "Error:",
            pdf_path.name,
            e
        )

        return []


# =========================================================
# 4. ORIGINAL HEADING CLASSIFIER
# =========================================================

def classify_situation_report_heading_resume(
    pdf_path
):

    lines = (
        extract_first_20_lines_pdfplumber(
            pdf_path
        )
    )

    if len(lines) == 0:

        return "pdf_text_unavailable"


    header_text = " ".join(
        lines
    )

    header_lower = (
        header_text.lower()
    )


    # ------------------------------
    # Drought report
    # ------------------------------

    if (
        "drought situation report"
        in header_lower

        or "drought"
        in header_lower

        or "නියඟ"
        in header_text
    ):

        return "drought_report"


    # ------------------------------
    # Special situation report
    # ------------------------------

    if (
        "special situation report"
        in header_lower

        or "විශේෂ"
        in header_text
    ):

        return (
            "special_situation_report"
        )


    # ------------------------------
    # General situation report
    # ------------------------------

    if (
        "situation report"
        in header_lower

        or "තත්ත්ව වාර්තාව"
        in header_text
    ):

        return (
            "general_situation_report"
        )


    return "other_heading"


# =========================================================
# 5. CLASSIFY
# =========================================================

categories = []


for i, row in integrated_df.iterrows():

    file_name = row["file_name"]

    pdf_path = (
        pdf_lookup.get(
            file_name
        )
    )


    if pdf_path is None:

        category = "pdf_not_found"

    else:

        category = (
            classify_situation_report_heading_resume(
                pdf_path
            )
        )


    categories.append(
        category
    )


    if (
        (i + 1) % 250
        == 0
    ):

        print(
            f"Checked "
            f"{i + 1}/"
            f"{len(integrated_df)}"
        )


recovered_df = (
    integrated_df.copy()
)

recovered_df[
    "situation_report_category"
] = categories


# =========================================================
# 6. ORIGINAL SCOPE LOGIC
# =========================================================

recovered_df[
    "scope_status"
] = "retain"


# Drought
recovered_df.loc[
    recovered_df[
        "situation_report_category"
    ]
    == "drought_report",

    "scope_status"

] = "exclude_drought"


# Non-situation report
recovered_df.loc[
    recovered_df[
        "extraction_status"
    ]
    == "non_situation_report",

    "scope_status"

] = "exclude_non_situation"


# Non-comparable report
recovered_df.loc[
    recovered_df[
        "extraction_status"
    ]
    == "non_comparable_incident_report",

    "scope_status"

] = "exclude_non_comparable"


# =========================================================
# 7. CREATE CANDIDATE ANALYTICAL DATASET
# =========================================================

candidate_scope_df = (

    recovered_df[
        recovered_df[
            "scope_status"
        ]
        == "retain"
    ]

    .copy()

    .sort_values(
        "report_datetime"
    )

    .reset_index(
        drop=True
    )
)


candidate_scope_df[
    "availability_datetime"
] = (
    candidate_scope_df[
        "report_datetime"
    ]
)


# =========================================================
# 8. QC
# =========================================================

print("\n" + "=" * 70)

print(
    "FINAL SCOPE RECOVERY QC"
)

print("=" * 70)


print(
    "\nSituation report categories:"
)

print(
    recovered_df[
        "situation_report_category"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nScope classification:"
)

scope_counts = (
    recovered_df[
        "scope_status"
    ]
    .value_counts()
)

print(
    scope_counts
)


drought_count = int(
    (
        recovered_df[
            "situation_report_category"
        ]
        == "drought_report"
    ).sum()
)


retain_count = int(
    (
        recovered_df[
            "scope_status"
        ]
        == "retain"
    ).sum()
)


non_situation_count = int(
    (
        recovered_df[
            "scope_status"
        ]
        == "exclude_non_situation"
    ).sum()
)


non_comparable_count = int(
    (
        recovered_df[
            "scope_status"
        ]
        == "exclude_non_comparable"
    ).sum()
)


missing_target = int(
    candidate_scope_df[
        "affected_people"
    ]
    .isna()
    .sum()
)


print(
    "\nDrought reports:",
    drought_count
)

print(
    "Retained analytical rows:",
    retain_count
)

print(
    "Non-situation excluded:",
    non_situation_count
)

print(
    "Non-comparable excluded:",
    non_comparable_count
)

print(
    "Missing affected_people:",
    missing_target
)


# =========================================================
# 9. STRICT VALIDATION
# =========================================================

qc_passed = (

    drought_count == 61

    and retain_count == 1385

    and non_situation_count == 9

    and non_comparable_count == 1

    and missing_target == 4
)


# =========================================================
# 10. SAVE ONLY IF EXACT MATCH
# =========================================================

if qc_passed:

    eda_scope_df = (
        candidate_scope_df.copy()
    )

    final_scope_path = (
        processed_dir
        / "analytical_scope_dataset.csv"
    )

    final_audit_path = (
        processed_dir
        / "situation_scope_classification.csv"
    )


    eda_scope_df.to_csv(
        final_scope_path,
        index=False
    )


    recovered_df[
        [
            "file_name",
            "report_datetime",
            "situation_report_category",
            "extraction_status",
            "scope_status"
        ]
    ].to_csv(
        final_audit_path,
        index=False
    )


    print("\n" + "=" * 70)

    print(
        "QC PASSED ✅"
    )

    print(
        "EXACT FROZEN ANALYTICAL DATASET RECOVERED"
    )

    print(
        "Analytical rows:",
        len(eda_scope_df)
    )

    print(
        "Latest datetime:",
        eda_scope_df[
            "availability_datetime"
        ].max()
    )

    print(
        "\nSaved:",
        final_scope_path
    )

    print("=" * 70)


else:

    print("\n" + "=" * 70)

    print(
        "QC FAILED ❌"
    )

    print(
        "DO NOT RUN SECTION 7 YET."
    )

    print(
        "The incorrect analytical_scope_dataset.csv "
        "has NOT been accepted."
    )

    print("=" * 70)

Integrated rows: 1456
Situation PDFs located: 1456
Checked 250/1456
Checked 500/1456
Checked 750/1456
Checked 1000/1456
Checked 1250/1456

FINAL SCOPE RECOVERY QC

Situation report categories:
situation_report_category
general_situation_report    1378
drought_report                61
other_heading                 17
Name: count, dtype: int64

Scope classification:
scope_status
retain                    1385
exclude_drought             61
exclude_non_situation        9
exclude_non_comparable       1
Name: count, dtype: int64

Drought reports: 61
Retained analytical rows: 1385
Non-situation excluded: 9
Non-comparable excluded: 1
Missing affected_people: 4

QC PASSED ✅
EXACT FROZEN ANALYTICAL DATASET RECOVERED
Analytical rows: 1385
Latest datetime: 2026-08-14 10:00:00

Saved: data\processed\analytical_scope_dataset.csv


In [33]:
# =========================================================
# PERMANENT FAST RESUME
# Start from saved analytical dataset
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path

processed_dir = Path("data/processed")

eda_scope_df = pd.read_csv(
    processed_dir / "analytical_scope_dataset.csv"
)

# Restore datetime types
for col in [
    "report_datetime",
    "availability_datetime"
]:
    if col in eda_scope_df.columns:
        eda_scope_df[col] = pd.to_datetime(
            eda_scope_df[col],
            errors="coerce"
        )

print("=" * 60)
print("FAST RESUME READY ✅")
print("=" * 60)

print(
    "Analytical rows:",
    len(eda_scope_df)
)

print(
    "Date range:",
    eda_scope_df["availability_datetime"].min(),
    "->",
    eda_scope_df["availability_datetime"].max()
)

print(
    "Missing affected_people:",
    eda_scope_df["affected_people"].isna().sum()
)

FAST RESUME READY ✅
Analytical rows: 1385
Date range: 2023-01-02 06:00:00 -> 2026-08-14 10:00:00
Missing affected_people: 4


In [28]:
# =========================================================
# FIND HOW DROUGHT REPORTS ARE STORED
# =========================================================

print("=" * 70)
print("DISASTER TYPE AUDIT")
print("=" * 70)

print("\nUnique disaster_types values:")
print(
    integrated_df["disaster_types"]
    .fillna("MISSING")
    .value_counts(dropna=False)
    .head(100)
    .to_string()
)


# ---------------------------------------------------------
# Search "drought" across every text column
# ---------------------------------------------------------

text_columns = integrated_df.select_dtypes(
    include=["object"]
).columns


drought_anywhere_mask = pd.Series(
    False,
    index=integrated_df.index
)


matched_columns = []


for col in text_columns:

    mask = (
        integrated_df[col]
        .fillna("")
        .astype(str)
        .str.contains(
            "drought",
            case=False,
            regex=False
        )
    )

    if mask.any():

        matched_columns.append(
            (col, int(mask.sum()))
        )

        drought_anywhere_mask = (
            drought_anywhere_mask | mask
        )


print("\nColumns containing the word 'drought':")

if matched_columns:

    for col, count in matched_columns:
        print(
            f"- {col}: {count}"
        )

else:

    print(
        "No text column contains 'drought'."
    )


print(
    "\nRows containing drought anywhere:",
    int(
        drought_anywhere_mask.sum()
    )
)


# ---------------------------------------------------------
# Show matching rows
# ---------------------------------------------------------

if drought_anywhere_mask.any():

    display_columns = [
        col
        for col in [
            "file_name",
            "report_date",
            "report_datetime",
            "disaster_types",
            "extraction_status"
        ]
        if col in integrated_df.columns
    ]

    print("\nSample matching rows:")

    display(
        integrated_df.loc[
            drought_anywhere_mask,
            display_columns
        ].head(20)
    )


# ---------------------------------------------------------
# Also inspect extraction-status exclusions
# ---------------------------------------------------------

print("\nRows with known non-analytical statuses:")

known_exclusion_statuses = [
    "non_situation_report",
    "non_comparable_incident_report",
    "insufficient_text",
    "manual_review_unsupported_layout"
]

print(
    integrated_df[
        "extraction_status"
    ]
    .value_counts()
    .reindex(
        known_exclusion_statuses,
        fill_value=0
    )
)

print("=" * 70)

DISASTER TYPE AUDIT

Unique disaster_types values:
disaster_types
Other                                                            449
Strong Wind                                                      174
Flood, Landslide, Heavy Rain, Strong Wind, Lightning             129
Flood, Heavy Rain, Strong Wind, Lightning                         95
Heavy Rain, Strong Wind                                           70
Strong Wind, Lightning                                            54
Landslide                                                         52
Flood, Heavy Rain, Strong Wind                                    44
Heavy Rain, Strong Wind, Lightning                                42
Landslide, Heavy Rain, Strong Wind, Lightning                     39
Flood, Strong Wind                                                36
Lightning                                                         29
Flood                                                             27
Flood, Landslide, Heavy Rain, Strong 

# 7. Feature Engineering for Forecasting

## 7.1 Canonical Forecasting Timeline

In [34]:
# =========================================================
# 7.1 Build Canonical Forecasting Timeline
# =========================================================

timeline_source = eda_scope_df.copy()


# ---------------------------------------------------------
# Preserve original row identity
# ---------------------------------------------------------

timeline_source["original_row_id"] = np.arange(
    len(timeline_source)
)


# ---------------------------------------------------------
# Ranking variables
#
# If multiple valid Situation Reports share the same
# availability timestamp:
#
# 1. Prefer a row with a valid affected_people target
# 2. Prefer the broader / larger affected population snapshot
# 3. Then affected families
# 4. Then safety-centre population
#
# IMPORTANT:
# We DO NOT sum reports at the same timestamp because
# concurrent reports may overlap and summing can double count.
# ---------------------------------------------------------

timeline_source["target_available"] = (
    timeline_source["affected_people"]
    .notna()
    .astype(int)
)


timeline_source = timeline_source.sort_values(
    [
        "availability_datetime",
        "target_available",
        "affected_people",
        "affected_families",
        "people_in_safety_centres",
        "file_name"
    ],
    ascending=[
        True,
        False,
        False,
        False,
        False,
        True
    ]
)


# ---------------------------------------------------------
# Number of candidate Situation Reports at each timestamp
# ---------------------------------------------------------

timestamp_report_counts = (
    timeline_source
    .groupby("availability_datetime")
    .size()
    .rename("reports_at_timestamp")
)


# ---------------------------------------------------------
# Select one canonical national snapshot per timestamp
# ---------------------------------------------------------

forecast_timeline_df = (
    timeline_source
    .drop_duplicates(
        subset=["availability_datetime"],
        keep="first"
    )
    .copy()
)


forecast_timeline_df = (
    forecast_timeline_df
    .merge(
        timestamp_report_counts,
        on="availability_datetime",
        how="left"
    )
    .sort_values("availability_datetime")
    .reset_index(drop=True)
)


# =========================================================
# Basic validation
# =========================================================

print(
    "Analytical source rows:",
    len(eda_scope_df)
)

print(
    "Canonical timeline rows:",
    len(forecast_timeline_df)
)

print(
    "Removed concurrent rows:",
    len(eda_scope_df)
    - len(forecast_timeline_df)
)

print(
    "\nDuplicate timestamps remaining:",
    forecast_timeline_df[
        "availability_datetime"
    ].duplicated().sum()
)

print(
    "Missing affected_people:",
    forecast_timeline_df[
        "affected_people"
    ].isna().sum()
)

print(
    "Maximum reports at one timestamp:",
    forecast_timeline_df[
        "reports_at_timestamp"
    ].max()
)


# =========================================================
# Time gap to previous canonical Situation Report
# =========================================================

forecast_timeline_df["hours_since_previous_report"] = (
    forecast_timeline_df[
        "availability_datetime"
    ]
    .diff()
    .dt.total_seconds()
    / 3600
)


print(
    "\nTime-gap summary (hours):"
)

print(
    forecast_timeline_df[
        "hours_since_previous_report"
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


print(
    "\nLargest 20 gaps between reports:"
)

print(
    forecast_timeline_df[
        [
            "availability_datetime",
            "file_name",
            "affected_people",
            "disaster_types",
            "hours_since_previous_report"
        ]
    ]
    .nlargest(
        20,
        "hours_since_previous_report"
    )
    .to_string(index=False)
)

Analytical source rows: 1385
Canonical timeline rows: 1323
Removed concurrent rows: 62

Duplicate timestamps remaining: 0
Missing affected_people: 4
Maximum reports at one timestamp: 3

Time-gap summary (hours):
count    1322.000000
mean       23.966717
std        15.820293
min         1.000000
25%        16.000000
50%        24.000000
75%        24.000000
90%        24.000000
95%        48.000000
99%        96.000000
max       234.000000
Name: hours_since_previous_report, dtype: float64

Largest 20 gaps between reports:
availability_datetime                            file_name  affected_people                            disaster_types  hours_since_previous_report
  2025-12-27 10:00:00 situation_20251227_1000_5df19cba.pdf         510744.0                                     Other                        234.0
  2023-06-23 18:00:00 situation_20230623_1800_75bd691c.pdf          13444.0                                     Other                        192.0
  2026-08-05 13:00:00 situation_

In [35]:
# =========================================================
# 7.2 Build Leakage-Safe Next-Period Forecasting Targets
# =========================================================

forecast_df = forecast_timeline_df.copy()

forecast_df = (
    forecast_df
    .sort_values("availability_datetime")
    .reset_index(drop=True)
)


# =========================================================
# 1. Time until next Situation Report
# =========================================================

forecast_df["next_report_datetime"] = (
    forecast_df["availability_datetime"]
    .shift(-1)
)

forecast_df["hours_to_next_report"] = (
    (
        forecast_df["next_report_datetime"]
        - forecast_df["availability_datetime"]
    )
    .dt.total_seconds()
    / 3600
)


# =========================================================
# 2. Next-period targets
# =========================================================

forecast_df["target_next_affected_people"] = (
    forecast_df["affected_people"]
    .shift(-1)
)

forecast_df["target_next_affected_families"] = (
    forecast_df["affected_families"]
    .shift(-1)
)

forecast_df["target_next_safety_centres"] = (
    forecast_df["safety_centres"]
    .shift(-1)
)

forecast_df["target_next_people_in_safety_centres"] = (
    forecast_df["people_in_safety_centres"]
    .shift(-1)
)


# =========================================================
# 3. Define valid forecasting pair
#
# Primary horizon:
# next DMC Situation Report within 48 hours
# =========================================================

MAX_FORECAST_GAP_HOURS = 48


forecast_df["valid_next_period_pair"] = (
    forecast_df["hours_to_next_report"]
    .between(
        0,
        MAX_FORECAST_GAP_HOURS,
        inclusive="right"
    )
    &
    forecast_df["affected_people"].notna()
    &
    forecast_df["target_next_affected_people"].notna()
)


# =========================================================
# 4. Main supervised modelling dataset
# =========================================================

model_base_df = (
    forecast_df[
        forecast_df["valid_next_period_pair"]
    ]
    .copy()
    .reset_index(drop=True)
)


# =========================================================
# 5. Validation
# =========================================================

print(
    "Canonical timeline rows:",
    len(forecast_df)
)

print(
    "Valid next-period forecasting pairs:",
    len(model_base_df)
)

print(
    "Pairs excluded:",
    len(forecast_df)
    - len(model_base_df)
)


print(
    "\nForecast-gap distribution:"
)

print(
    model_base_df[
        "hours_to_next_report"
    ].describe()
)


print(
    "\nForecast horizon counts:"
)

print(
    model_base_df[
        "hours_to_next_report"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nMaximum accepted forecast gap:",
    model_base_df[
        "hours_to_next_report"
    ].max()
)


print(
    "\nMissing next targets:"
)

print(
    model_base_df[
        [
            "target_next_affected_people",
            "target_next_affected_families",
            "target_next_safety_centres",
            "target_next_people_in_safety_centres"
        ]
    ]
    .isna()
    .sum()
)

Canonical timeline rows: 1323
Valid next-period forecasting pairs: 1281
Pairs excluded: 42

Forecast-gap distribution:
count    1281.000000
mean       22.028493
std         8.078050
min         1.000000
25%        13.500000
50%        24.000000
75%        24.000000
max        48.000000
Name: hours_to_next_report, dtype: float64

Forecast horizon counts:
hours_to_next_report
1.000000       1
2.500000       1
3.000000       2
4.000000       6
5.000000       1
5.500000       1
6.000000       5
7.000000       1
8.000000       5
9.000000       3
10.500000      1
11.000000      2
11.500000      1
12.000000    288
12.500000      1
13.000000      1
13.500000      1
15.000000      5
16.000000      2
17.000000      1
18.000000      7
20.000000      1
21.000000      2
22.000000      3
22.333333      1
22.500000      1
23.000000      4
23.500000      2
24.000000    845
24.500000      1
25.000000      5
25.500000      1
25.666667      1
26.000000      1
30.000000      1
31.000000      1
32.000000  

## 7.3 Leakage-Safe Lag, Growth and Temporal Features

In [36]:
# =========================================================
# 7.3 Leakage-Safe Temporal and Autoregressive Features
# =========================================================

feature_df = forecast_df.copy()

feature_df = (
    feature_df
    .sort_values("availability_datetime")
    .reset_index(drop=True)
)


# =========================================================
# 1. Current-state log features
# =========================================================

feature_df["log_current_affected_people"] = np.log1p(
    feature_df["affected_people"]
)

feature_df["log_current_affected_families"] = np.log1p(
    feature_df["affected_families"]
)

feature_df["log_current_safety_centres"] = np.log1p(
    feature_df["safety_centres"]
)

feature_df["log_current_people_in_safety_centres"] = np.log1p(
    feature_df["people_in_safety_centres"]
)


# =========================================================
# 2. Previous canonical report
# =========================================================

feature_df["previous_report_gap_hours"] = (
    feature_df["availability_datetime"]
    .diff()
    .dt.total_seconds()
    / 3600
)


# Previous report is considered continuous only when
# it is within 48 hours.
feature_df["previous_context_valid"] = (
    feature_df["previous_report_gap_hours"]
    .between(
        0,
        48,
        inclusive="right"
    )
)


previous_cols = [
    "affected_people",
    "affected_families",
    "safety_centres",
    "people_in_safety_centres"
]


for col in previous_cols:

    prev_col = f"prev_{col}"

    feature_df[prev_col] = (
        feature_df[col]
        .shift(1)
    )

    # Remove previous values when there is a long gap
    feature_df.loc[
        ~feature_df["previous_context_valid"],
        prev_col
    ] = np.nan


# =========================================================
# 3. Recent impact-change features
# =========================================================

feature_df["affected_people_change_from_prev"] = (
    feature_df["affected_people"]
    -
    feature_df["prev_affected_people"]
)

feature_df["affected_families_change_from_prev"] = (
    feature_df["affected_families"]
    -
    feature_df["prev_affected_families"]
)

feature_df["safety_population_change_from_prev"] = (
    feature_df["people_in_safety_centres"]
    -
    feature_df["prev_people_in_safety_centres"]
)


# Log-change is more stable than percentage growth,
# especially when previous values are zero.

feature_df["affected_people_log_change_from_prev"] = (
    np.log1p(feature_df["affected_people"])
    -
    np.log1p(feature_df["prev_affected_people"])
)

feature_df["safety_population_log_change_from_prev"] = (
    np.log1p(feature_df["people_in_safety_centres"])
    -
    np.log1p(feature_df["prev_people_in_safety_centres"])
)


# =========================================================
# 4. Temporal features known at prediction time
# =========================================================

feature_df["report_hour"] = (
    feature_df["availability_datetime"].dt.hour
)

feature_df["report_month"] = (
    feature_df["availability_datetime"].dt.month
)

feature_df["report_dayofyear"] = (
    feature_df["availability_datetime"].dt.dayofyear
)

feature_df["report_dayofweek"] = (
    feature_df["availability_datetime"].dt.dayofweek
)


# Cyclical representations
feature_df["month_sin"] = np.sin(
    2 * np.pi *
    feature_df["report_month"] / 12
)

feature_df["month_cos"] = np.cos(
    2 * np.pi *
    feature_df["report_month"] / 12
)

feature_df["dayofyear_sin"] = np.sin(
    2 * np.pi *
    feature_df["report_dayofyear"] / 365.25
)

feature_df["dayofyear_cos"] = np.cos(
    2 * np.pi *
    feature_df["report_dayofyear"] / 365.25
)


# =========================================================
# 5. Forecasting targets
# =========================================================

feature_df["target_next_log_affected_people"] = np.log1p(
    feature_df["target_next_affected_people"]
)

feature_df["target_next_log_people_in_safety_centres"] = np.log1p(
    feature_df["target_next_people_in_safety_centres"]
)


# Change in human impact at the NEXT report
feature_df["target_affected_people_change"] = (
    feature_df["target_next_affected_people"]
    -
    feature_df["affected_people"]
)

feature_df["target_affected_people_log_change"] = (
    np.log1p(
        feature_df["target_next_affected_people"]
    )
    -
    np.log1p(
        feature_df["affected_people"]
    )
)


# Next-report safety-centre usage classification
feature_df["target_next_safety_centre_use"] = (
    feature_df["target_next_people_in_safety_centres"] > 0
).astype(int)


# =========================================================
# 6. Keep only valid next-period forecasting pairs
# =========================================================

model_feature_df = (
    feature_df[
        feature_df["valid_next_period_pair"]
    ]
    .copy()
    .reset_index(drop=True)
)


# =========================================================
# 7. Validation
# =========================================================

print(
    "Model feature rows:",
    len(model_feature_df)
)

print(
    "\nPrevious-context availability:"
)

print(
    model_feature_df[
        "previous_context_valid"
    ].value_counts()
)


print(
    "\nTarget next affected-population behaviour:"
)

current = model_feature_df["affected_people"]
next_target = model_feature_df[
    "target_next_affected_people"
]

print(
    "Increase:",
    int((next_target > current).sum())
)

print(
    "Unchanged:",
    int((next_target == current).sum())
)

print(
    "Decrease:",
    int((next_target < current).sum())
)


print(
    "\nNext safety-centre usage:"
)

print(
    model_feature_df[
        "target_next_safety_centre_use"
    ].value_counts()
)


print(
    "\nMissing key autoregressive features:"
)

print(
    model_feature_df[
        [
            "prev_affected_people",
            "prev_affected_families",
            "prev_people_in_safety_centres",
            "affected_people_log_change_from_prev",
            "safety_population_log_change_from_prev"
        ]
    ]
    .isna()
    .sum()
)

Model feature rows: 1281

Previous-context availability:
previous_context_valid
True     1247
False      34
Name: count, dtype: int64

Target next affected-population behaviour:
Increase: 545
Unchanged: 320
Decrease: 416

Next safety-centre usage:
target_next_safety_centre_use
0    679
1    602
Name: count, dtype: int64

Missing key autoregressive features:
prev_affected_people                      36
prev_affected_families                    36
prev_people_in_safety_centres             36
affected_people_log_change_from_prev      36
safety_population_log_change_from_prev    36
dtype: int64


## 7.4 Event-Aware Chronological Train-Validation-Test Split

In [39]:
# =========================================================
# 7.4 Event-Aware Chronological Train / Validation / Test Split
# =========================================================

split_df = (
    model_feature_df
    .sort_values("availability_datetime")
    .reset_index(drop=True)
    .copy()
)


# =========================================================
# 1. Find natural episode boundaries
#
# A new reporting episode is assumed when the previous
# canonical Situation Report is more than 48 hours away.
# =========================================================

boundary_positions = split_df.index[
    ~split_df["previous_context_valid"]
].tolist()

# Remove first row because it is naturally the beginning
boundary_positions = [
    i for i in boundary_positions
    if i > 0
]


print(
    "Natural >48h boundaries available:",
    len(boundary_positions)
)


# =========================================================
# 2. Desired chronological proportions
# =========================================================

n = len(split_df)

desired_train_end = int(n * 0.70)
desired_val_end = int(n * 0.85)


def nearest_boundary(
    target_position,
    candidates,
    minimum_position,
    maximum_position
):

    valid_candidates = [
        x for x in candidates
        if minimum_position < x < maximum_position
    ]

    if len(valid_candidates) == 0:
        return target_position

    return min(
        valid_candidates,
        key=lambda x: abs(
            x - target_position
        )
    )


# Training boundary
train_end = nearest_boundary(
    desired_train_end,
    boundary_positions,
    int(n * 0.60),
    int(n * 0.80)
)


# Validation/Test boundary
val_end = nearest_boundary(
    desired_val_end,
    boundary_positions,
    train_end + 20,
    int(n * 0.95)
)


# =========================================================
# 3. Assign partitions
# =========================================================

split_df["dataset_split"] = "train"

split_df.loc[
    train_end:val_end - 1,
    "dataset_split"
] = "validation"

split_df.loc[
    val_end:,
    "dataset_split"
] = "test"


# =========================================================
# 4. Verify chronological separation
# =========================================================

print("\nSplit counts:")

print(
    split_df[
        "dataset_split"
    ].value_counts(
        sort=False
    )
)


print("\nSplit date ranges:")

for split_name in [
    "train",
    "validation",
    "test"
]:

    temp = split_df[
        split_df["dataset_split"]
        == split_name
    ]

    print(
        f"\n{split_name.upper()}"
    )

    print(
        "Rows:",
        len(temp)
    )

    print(
        "From:",
        temp["availability_datetime"].min()
    )

    print(
        "To:",
        temp["availability_datetime"].max()
    )


# =========================================================
# 5. Gap between partitions
# =========================================================

train_last_time = split_df.loc[
    split_df["dataset_split"] == "train",
    "availability_datetime"
].max()

validation_first_time = split_df.loc[
    split_df["dataset_split"] == "validation",
    "availability_datetime"
].min()


validation_last_time = split_df.loc[
    split_df["dataset_split"] == "validation",
    "availability_datetime"
].max()

test_first_time = split_df.loc[
    split_df["dataset_split"] == "test",
    "availability_datetime"
].min()


train_val_gap = (
    validation_first_time
    - train_last_time
).total_seconds() / 3600


val_test_gap = (
    test_first_time
    - validation_last_time
).total_seconds() / 3600


print(
    "\nTrain → Validation gap (hours):",
    train_val_gap
)

print(
    "Validation → Test gap (hours):",
    val_test_gap
)


# =========================================================
# 6. River-data availability by partition
# =========================================================

print(
    "\nRiver data availability by split:"
)

print(
    pd.crosstab(
        split_df["dataset_split"],
        split_df["river_data_available"]
    )
)


# =========================================================
# 7. Target summary by partition
# =========================================================

print(
    "\nAffected-people target summary by split:"
)

print(
    split_df.groupby(
        "dataset_split"
    )[
        "target_next_affected_people"
    ]
    .agg(
        [
            "count",
            "median",
            "mean",
            "max"
        ]
    )
)

Natural >48h boundaries available: 33

Split counts:
dataset_split
train         899
validation    232
test          150
Name: count, dtype: int64

Split date ranges:

TRAIN
Rows: 899
From: 2023-01-02 06:00:00
To: 2025-05-13 09:00:00

VALIDATION
Rows: 232
From: 2025-05-17 09:00:00
To: 2025-12-16 06:00:00

TEST
Rows: 150
From: 2025-12-27 10:00:00
To: 2026-08-13 10:00:00

Train → Validation gap (hours): 96.0
Validation → Test gap (hours): 268.0

River data availability by split:
river_data_available    0    1
dataset_split                 
test                    0  150
train                 801   98
validation              0  232

Affected-people target summary by split:
               count  median           mean        max
dataset_split                                         
test             150  4798.0   81798.040000   499671.0
train            899  1089.0   26246.645161   475225.0
validation       232   357.0  290926.254310  2179138.0


In [ ]:
## 7.5 Impact Escalation Risk Target Definition

In [41]:
# =========================================================
# 7.5 Impact Escalation Risk Target Definition
# Revised - Absolute Human-Impact Escalation
# =========================================================

risk_df = split_df.copy()


# =========================================================
# 1. Absolute change in affected population
# =========================================================

risk_df["absolute_affected_change"] = (
    risk_df["target_next_affected_people"]
    -
    risk_df["affected_people"]
)


# Keep log change only as an analytical variable.
# It is NOT used directly to define the risk class.

risk_df["impact_log_change"] = (
    np.log1p(
        risk_df["target_next_affected_people"]
    )
    -
    np.log1p(
        risk_df["affected_people"]
    )
)


# =========================================================
# 2. Learn escalation thresholds from TRAINING DATA ONLY
# =========================================================

train_positive_increases = risk_df.loc[
    (
        risk_df["dataset_split"] == "train"
    )
    &
    (
        risk_df["absolute_affected_change"] > 0
    ),
    "absolute_affected_change"
].dropna()


MODERATE_HIGH_THRESHOLD = (
    train_positive_increases.quantile(0.50)
)

HIGH_CRITICAL_THRESHOLD = (
    train_positive_increases.quantile(0.85)
)


print(
    "Training positive-escalation cases:",
    len(train_positive_increases)
)

print(
    "Moderate → High threshold:",
    MODERATE_HIGH_THRESHOLD
)

print(
    "High → Critical threshold:",
    HIGH_CRITICAL_THRESHOLD
)


# =========================================================
# 3. Assign research-derived escalation classes
# =========================================================

def assign_escalation_risk(change):

    if pd.isna(change):
        return np.nan

    # No increase or impact reduction
    if change <= 0:
        return "Low"

    # Smaller positive escalation
    if change <= MODERATE_HIGH_THRESHOLD:
        return "Moderate"

    # Substantial escalation
    if change <= HIGH_CRITICAL_THRESHOLD:
        return "High"

    # Largest positive escalations
    return "Critical"


risk_df["impact_escalation_risk"] = (
    risk_df["absolute_affected_change"]
    .apply(assign_escalation_risk)
)


# =========================================================
# 4. Ordinal code
# =========================================================

RISK_MAP = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
    "Critical": 3
}

risk_df["impact_escalation_risk_code"] = (
    risk_df[
        "impact_escalation_risk"
    ].map(RISK_MAP)
)


# =========================================================
# 5. Overall distribution
# =========================================================

print(
    "\nOverall escalation-risk distribution:"
)

print(
    risk_df[
        "impact_escalation_risk"
    ].value_counts()
)


print(
    "\nOverall escalation-risk percentage:"
)

print(
    (
        risk_df[
            "impact_escalation_risk"
        ]
        .value_counts(
            normalize=True
        )
        * 100
    ).round(2)
)


# =========================================================
# 6. Distribution by split
# =========================================================

print(
    "\nRisk classes by dataset split:"
)

print(
    pd.crosstab(
        risk_df["dataset_split"],
        risk_df[
            "impact_escalation_risk"
        ]
    )
)


# =========================================================
# 7. Risk-class interpretation
# =========================================================

risk_change_summary = (
    risk_df
    .groupby(
        "impact_escalation_risk"
    )
    .agg(
        reports=(
            "file_name",
            "count"
        ),

        median_current_affected=(
            "affected_people",
            "median"
        ),

        median_next_affected=(
            "target_next_affected_people",
            "median"
        ),

        median_absolute_change=(
            "absolute_affected_change",
            "median"
        ),

        mean_absolute_change=(
            "absolute_affected_change",
            "mean"
        ),

        max_absolute_change=(
            "absolute_affected_change",
            "max"
        ),

        median_log_change=(
            "impact_log_change",
            "median"
        )
    )
    .reindex(
        [
            "Low",
            "Moderate",
            "High",
            "Critical"
        ]
    )
)


print(
    "\nRisk-class change summary:"
)

print(
    risk_change_summary.to_string()
)


# =========================================================
# 8. Sanity check
# =========================================================

positive_class_medians = (
    risk_change_summary.loc[
        [
            "Moderate",
            "High",
            "Critical"
        ],
        "median_absolute_change"
    ]
)


print(
    "\nMedian positive escalation by severity:"
)

print(
    positive_class_medians
)

print(
    "\nSeverity ordering valid:",
    positive_class_medians.is_monotonic_increasing
)

Training positive-escalation cases: 382
Moderate → High threshold: 142.0
High → Critical threshold: 5184.649999999999

Overall escalation-risk distribution:
impact_escalation_risk
Low         736
Moderate    278
High        176
Critical     91
Name: count, dtype: int64

Overall escalation-risk percentage:
impact_escalation_risk
Low         57.46
Moderate    21.70
High        13.74
Critical     7.10
Name: proportion, dtype: float64

Risk classes by dataset split:
impact_escalation_risk  Critical  High  Low  Moderate
dataset_split                                        
test                           9    12   94        35
train                         58   133  517       191
validation                    24    31  125        52

Risk-class change summary:
                        reports  median_current_affected  median_next_affected  median_absolute_change  mean_absolute_change  max_absolute_change  median_log_change
impact_escalation_risk                                                

## 7.6 Final Feature Selection and Leakage Audit

In [43]:
# =========================================================
# 7.6 Final Feature Selection and Leakage Audit
# =========================================================

modeling_df = risk_df.copy()


# =========================================================
# 1. Current Situation hazard-label flags
# These are supplementary signals only.
# =========================================================

disaster_text = (
    modeling_df["disaster_types"]
    .fillna("")
    .astype(str)
)

modeling_df["current_flood_flag"] = (
    disaster_text.str.contains(
        "Flood",
        case=False,
        regex=False
    ).astype(int)
)

modeling_df["current_landslide_flag"] = (
    disaster_text.str.contains(
        "Landslide",
        case=False,
        regex=False
    ).astype(int)
)

modeling_df["current_heavy_rain_flag"] = (
    disaster_text.str.contains(
        "Heavy Rain",
        case=False,
        regex=False
    ).astype(int)
)


# =========================================================
# 2. Common non-river context features
# =========================================================

CONTEXT_FEATURES = [

    # Weather - previous 24h
    "weather_reports_24h",
    "weather_max_rainfall_threshold_24h",
    "weather_max_wind_24h",
    "weather_max_warning_level_24h",
    "weather_heavy_rain_24h",
    "weather_thunderstorm_24h",
    "weather_strong_wind_24h",
    "weather_low_pressure_24h",
    "weather_depression_24h",
    "weather_cyclone_24h",
    "latest_weather_age_hours",

    # Landslide
    "landslide_active_warning",
    "landslide_max_warning_level",
    "landslide_level1_districts",
    "landslide_level2_districts",
    "landslide_level3_districts",
    "landslide_districts_at_risk",
    "latest_landslide_age_hours",

    # Current Situation labels
    "current_flood_flag",
    "current_landslide_flag",
    "current_heavy_rain_flag",

    # Calendar / seasonality
    "report_hour",
    "report_dayofweek",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",
]


# =========================================================
# 3. Current human-impact state
#
# These values are available at prediction time because
# they belong to the CURRENT Situation Report.
# =========================================================

CURRENT_STATE_FEATURES = [

    "affected_people",
    "affected_families",

    "safety_centres",
    "people_in_safety_centres",

    "log_current_affected_people",
    "log_current_affected_families",
    "log_current_safety_centres",
    "log_current_people_in_safety_centres",
]


# =========================================================
# 4. Previous-report / change features
# =========================================================

TEMPORAL_STATE_FEATURES = [

    "previous_context_valid",
    "previous_report_gap_hours",

    "prev_affected_people",
    "prev_affected_families",
    "prev_safety_centres",
    "prev_people_in_safety_centres",

    "affected_people_change_from_prev",
    "affected_families_change_from_prev",
    "safety_population_change_from_prev",

    "affected_people_log_change_from_prev",
    "safety_population_log_change_from_prev",
]


# =========================================================
# 5. River features
#
# Kept separately because historical River coverage
# is much shorter than Situation/Weather/Landslide coverage.
# =========================================================

RIVER_FEATURES = [

    "river_data_available",
    "river_reports_24h",

    "river_max_alert_stations_24h",
    "river_max_minor_flood_stations_24h",
    "river_max_major_flood_stations_24h",
    "river_max_rising_stations_24h",

    "river_max_rainfall_24h_mm",
    "river_max_alert_ratio_24h",

    "river_active_warning_24h",
    "river_warning_withdrawn_24h",

    "latest_river_age_hours",
]


# =========================================================
# 6. Two experimental feature sets
# =========================================================

CORE_FEATURES = list(
    dict.fromkeys(
        CONTEXT_FEATURES
        + CURRENT_STATE_FEATURES
        + TEMPORAL_STATE_FEATURES
    )
)

FULL_MULTISOURCE_FEATURES = list(
    dict.fromkeys(
        CORE_FEATURES
        + RIVER_FEATURES
    )
)


# =========================================================
# 7. Define forecasting targets
# =========================================================

IMPACT_TARGET = (
    "target_next_affected_people"
)

IMPACT_LOG_TARGET = (
    "target_next_log_affected_people"
)

SAFETY_TARGET = (
    "target_next_people_in_safety_centres"
)

SAFETY_LOG_TARGET = (
    "target_next_log_people_in_safety_centres"
)

SAFETY_USE_TARGET = (
    "target_next_safety_centre_use"
)

RISK_TARGET = (
    "impact_escalation_risk"
)


# =========================================================
# 8. Explicit future / leakage columns
# =========================================================

FORBIDDEN_COLUMNS = {

    # Future report information
    "next_report_datetime",
    "hours_to_next_report",
    "valid_next_period_pair",

    # Future targets
    "target_next_affected_people",
    "target_next_affected_families",
    "target_next_safety_centres",
    "target_next_people_in_safety_centres",
    "target_next_log_affected_people",
    "target_next_log_people_in_safety_centres",
    "target_next_safety_centre_use",

    # Variables calculated using future target
    "absolute_affected_change",
    "impact_log_change",
    "impact_escalation_risk",
    "impact_escalation_risk_code",

    # Dataset assignment
    "dataset_split"
}


# =========================================================
# 9. Leakage audit
# =========================================================

def leakage_audit(feature_list, name):

    selected = set(feature_list)

    leakage = sorted(
        selected.intersection(
            FORBIDDEN_COLUMNS
        )
    )

    print(
        f"\n{name}"
    )

    print(
        "Number of features:",
        len(feature_list)
    )

    print(
        "Forbidden / future features found:",
        leakage
    )

    print(
        "Leakage audit passed:",
        len(leakage) == 0
    )


leakage_audit(
    CORE_FEATURES,
    "CORE FEATURE SET"
)

leakage_audit(
    FULL_MULTISOURCE_FEATURES,
    "FULL MULTI-SOURCE FEATURE SET"
)


# =========================================================
# 10. Missing-value audit
# =========================================================

def missing_feature_summary(
    df,
    features,
    name
):

    result = pd.DataFrame({
        "missing_count":
            df[features].isna().sum(),

        "missing_percent":
            (
                df[features]
                .isna()
                .mean()
                * 100
            )
    })

    result = result[
        result["missing_count"] > 0
    ]

    result = result.sort_values(
        "missing_percent",
        ascending=False
    )

    print(
        f"\n{name} - Missing Features:"
    )

    print(
        result.head(25)
        .to_string()
    )


missing_feature_summary(
    modeling_df,
    CORE_FEATURES,
    "CORE"
)

missing_feature_summary(
    modeling_df,
    FULL_MULTISOURCE_FEATURES,
    "FULL MULTI-SOURCE"
)


# =========================================================
# 11. Constant / zero-variance feature audit
# =========================================================

def find_constant_features(
    df,
    features
):

    return [
        col
        for col in features
        if df[col].nunique(
            dropna=True
        ) <= 1
    ]


core_constant = find_constant_features(
    modeling_df,
    CORE_FEATURES
)

full_constant = find_constant_features(
    modeling_df,
    FULL_MULTISOURCE_FEATURES
)


print(
    "\nConstant CORE features:",
    core_constant
)

print(
    "Constant FULL features:",
    full_constant
)


# =========================================================
# 12. Final chronological modelling partitions
# =========================================================

train_df = modeling_df[
    modeling_df["dataset_split"]
    == "train"
].copy()

validation_df = modeling_df[
    modeling_df["dataset_split"]
    == "validation"
].copy()

test_df = modeling_df[
    modeling_df["dataset_split"]
    == "test"
].copy()


print(
    "\nFinal modelling partitions:"
)

print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(validation_df)
)

print(
    "Test:",
    len(test_df)
)


# =========================================================
# 13. Final feature counts
# =========================================================

print(
    "\nCORE features:",
    len(CORE_FEATURES)
)

print(
    "FULL multi-source features:",
    len(FULL_MULTISOURCE_FEATURES)
)


CORE FEATURE SET
Number of features: 46
Forbidden / future features found: []
Leakage audit passed: True

FULL MULTI-SOURCE FEATURE SET
Number of features: 57
Forbidden / future features found: []
Leakage audit passed: True

CORE - Missing Features:
                                        missing_count  missing_percent
weather_max_wind_24h                              739        57.689305
weather_max_rainfall_threshold_24h                533        41.608119
prev_affected_people                               36         2.810304
prev_affected_families                             36         2.810304
affected_families_change_from_prev                 36         2.810304
prev_safety_centres                                36         2.810304
prev_people_in_safety_centres                      36         2.810304
affected_people_change_from_prev                   36         2.810304
affected_people_log_change_from_prev               36         2.810304
safety_population_change_from_prev     

In [44]:
# =========================================================
# FIND FINAL MODEL TRAINING CELLS
# WITHOUT RUNNING THEM
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "GradientBoostingRegressor",
    "RandomForestClassifier",
    "HistGradientBoostingClassifier",
    "loss=\"huber\"",
    "loss='huber'",
    "loss=\"quantile\"",
    "loss='quantile'",
    "final impact",
    "final safety",
    "final risk",
    "refit"
]

print("=" * 80)
print("FINAL MODEL CELL SEARCH")
print("=" * 80)

for i, cell in enumerate(nb["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    matched = [
        keyword
        for keyword in keywords
        if keyword.lower() in source.lower()
    ]

    if matched:
        first_lines = "\n".join(
            source.splitlines()[:12]
        )

        print("\n" + "=" * 80)
        print("CELL INDEX:", i)
        print("MATCHED:", matched)
        print("-" * 80)
        print(first_lines)

FINAL MODEL CELL SEARCH

CELL INDEX: 123
MATCHED: ['GradientBoostingRegressor', 'RandomForestClassifier', 'HistGradientBoostingClassifier', "loss='huber'", "loss='quantile'", 'final impact', 'final safety', 'final risk', 'refit']
--------------------------------------------------------------------------------
# =========================================================
# FIND FINAL MODEL TRAINING CELLS
# WITHOUT RUNNING THEM
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

CELL INDEX: 127
MATCHED: ['GradientBoostingRegressor']
--------------------------------------------------------------------------------
# =========================================================
# 8.2 CORE Human-Impact Regression Models
#
# Compare:
# 1. Persistence baseline
# 2. Ridge
# 3. Random Forest
# 4. HistGradientBoosting
#
# Two str

In [45]:
# =========================================================
# SHOW EXACT FINAL MODEL LOCK CELL
# =========================================================

import json
from pathlib import Path

notebook_path = Path(
    "FloodImpact_LK_Final.ipynb"
)

with open(
    notebook_path,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)


for cell_index in [144, 145]:

    cell = nb["cells"][cell_index]

    print("\n" + "=" * 90)
    print("CELL INDEX:", cell_index)
    print("=" * 90)

    print(
        "".join(
            cell.get(
                "source",
                []
            )
        )
    )


CELL INDEX: 144
## 8.11 Final Model Lock and Untouched Test Evaluation

CELL INDEX: 145
# =========================================================
# 8.11 FINAL MODEL LOCK + UNTOUCHED TEST EVALUATION
#
# Model selection is COMPLETE.
#
# Final selected approaches:
#
# Human Impact:
# GradientBoosting Huber additive-change model
#
# Safety Demand Stage 1:
# Random Forest classifier
#
# Safety Demand Stage 2:
# GradientBoosting Huber additive-change model
#
# Escalation Risk:
# Original HistGradientBoosting classifier
#
# Models are now refitted using TRAIN + VALIDATION,
# then evaluated ONCE on the untouched TEST set.
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classific

In [47]:
# =========================================================
# RECOVER 46-FEATURE CORE LIST FROM CURRENT MEMORY
# =========================================================

candidate_feature_lists = []

for name, obj in list(globals().items()):

    if isinstance(obj, (list, tuple)):

        if (
            len(obj) == 46
            and all(
                isinstance(x, str)
                for x in obj
            )
        ):

            # Check whether these are actual columns
            # available in the modelling dataframe
            if all(
                col in train_df.columns
                for col in obj
            ):

                candidate_feature_lists.append(
                    (name, list(obj))
                )


print("46-feature candidates found:")

for name, features in candidate_feature_lists:
    print("-", name, ":", len(features))


if len(candidate_feature_lists) == 1:

    original_name, recovered_features = (
        candidate_feature_lists[0]
    )

    CORE_MODEL_FEATURES = (
        recovered_features.copy()
    )

    print("\n✅ CORE_MODEL_FEATURES recovered")
    print("Original variable:", original_name)
    print("Feature count:", len(CORE_MODEL_FEATURES))


elif len(candidate_feature_lists) > 1:

    print(
        "\n⚠️ More than one 46-feature list found."
    )

    print(
        "Do NOT choose automatically yet."
    )


else:

    print(
        "\n❌ No suitable 46-feature list found."
    )

46-feature candidates found:
- CORE_FEATURES : 46

✅ CORE_MODEL_FEATURES recovered
Original variable: CORE_FEATURES
Feature count: 46


In [49]:
# =========================================================
# FINAL DEPLOYMENT MODEL REBUILD + SAVE
#
# Uses ONLY the already-selected final models.
# Skips:
# - Baseline experiments
# - Model comparison
# - Grid search
# - River experiment
# - Validation experiments
#
# Requires:
# train_df
# validation_df
# test_df
# CORE_MODEL_FEATURES
# =========================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)


# =========================================================
# 1. BASIC CHECK
# =========================================================

required_objects = [
    "train_df",
    "validation_df",
    "test_df",
    "CORE_MODEL_FEATURES"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# =========================================================
# 2. TRANSFORMATION FUNCTIONS
# =========================================================

def signed_log1p(values):

    values = np.asarray(
        values,
        dtype=float
    )

    return (
        np.sign(values)
        * np.log1p(
            np.abs(values)
        )
    )


def inverse_signed_log1p(values):

    values = np.asarray(
        values,
        dtype=float
    )

    return (
        np.sign(values)
        * np.expm1(
            np.abs(values)
        )
    )


# =========================================================
# 3. TRAIN + VALIDATION
# =========================================================

development_df = pd.concat(
    [
        train_df,
        validation_df
    ],
    axis=0
).copy()


X_dev = development_df[
    CORE_MODEL_FEATURES
].copy()


for col in CORE_MODEL_FEATURES:

    X_dev[col] = pd.to_numeric(
        X_dev[col],
        errors="coerce"
    )


print("=" * 70)
print("FINAL DEPLOYMENT MODEL TRAINING")
print("=" * 70)

print(
    "Development rows:",
    len(development_df)
)

print(
    "CORE features:",
    len(CORE_MODEL_FEATURES)
)

print(
    "Test rows preserved:",
    len(test_df)
)


# =========================================================
# 4. HUMAN-IMPACT MODEL
#
# Predicts signed-log change:
# next affected - current affected
# =========================================================

impact_delta = (

    development_df[
        "target_next_affected_people"
    ].astype(float)

    -

    development_df[
        "affected_people"
    ].astype(float)
)


impact_target = signed_log1p(
    impact_delta
)


final_impact_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            random_state=42
        )
    )
])


final_impact_model.fit(
    X_dev,
    impact_target
)


print(
    "Human-impact model trained ✅"
)


# =========================================================
# 5. SAFETY-CENTRE USAGE CLASSIFIER
# =========================================================

safety_use_target = (
    development_df[
        "target_next_safety_centre_use"
    ]
    .astype(int)
)


final_safety_classifier = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=3,
            max_features=0.7,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])


final_safety_classifier.fit(
    X_dev,
    safety_use_target
)


print(
    "Safety-usage classifier trained ✅"
)


# =========================================================
# 6. SAFETY-CENTRE AMOUNT MODEL
#
# Train only where next safety-centre population > 0
# =========================================================

positive_mask = (

    development_df[
        "target_next_people_in_safety_centres"
    ]
    > 0
)


X_dev_positive = (
    X_dev.loc[
        positive_mask
    ]
    .copy()
)


next_safety_positive = (

    development_df.loc[
        positive_mask,
        "target_next_people_in_safety_centres"
    ]
    .astype(float)
)


current_safety_positive = (

    development_df.loc[
        positive_mask,
        "people_in_safety_centres"
    ]
    .astype(float)
)


safety_delta = (

    next_safety_positive
    -
    current_safety_positive
)


safety_amount_target = signed_log1p(
    safety_delta
)


final_safety_amount_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=8,
            random_state=42
        )
    )
])


final_safety_amount_model.fit(
    X_dev_positive,
    safety_amount_target
)


print(
    "Safety-demand amount model trained ✅"
)


# =========================================================
# 7. ESCALATION-RISK MODEL
# =========================================================

risk_target = (

    development_df[
        "impact_escalation_risk"
    ]
    .astype(str)
)


final_risk_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=15,
            min_samples_leaf=15,
            l2_regularization=3.0,
            class_weight="balanced",
            random_state=42
        )
    )
])


final_risk_model.fit(
    X_dev,
    risk_target
)


print(
    "Escalation-risk model trained ✅"
)


# =========================================================
# 8. CREATE MODELS FOLDER
# =========================================================

models_dir = Path(
    "models"
)

models_dir.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# 9. SAVE FINAL PIPELINES
#
# Imputer is already inside each Pipeline,
# so separate preprocessor files are NOT needed.
# =========================================================

joblib.dump(
    final_impact_model,
    models_dir
    / "human_impact_model.joblib"
)

joblib.dump(
    final_safety_classifier,
    models_dir
    / "safety_usage_model.joblib"
)

joblib.dump(
    final_safety_amount_model,
    models_dir
    / "safety_demand_model.joblib"
)

joblib.dump(
    final_risk_model,
    models_dir
    / "escalation_risk_model.joblib"
)


# =========================================================
# 10. SAVE DEPLOYMENT METADATA
# =========================================================

risk_order = [
    "Low",
    "Moderate",
    "High",
    "Critical"
]


metadata = {

    "project":
        "FloodImpact-LK",

    "forecast_definition":
        (
            "Next eligible DMC Situation Report "
            "within a maximum 48-hour gap"
        ),

    "feature_set":
        list(
            CORE_MODEL_FEATURES
        ),

    "feature_count":
        len(
            CORE_MODEL_FEATURES
        ),

    "development_rows":
        len(
            development_df
        ),

    "development_start":
        str(
            development_df[
                "availability_datetime"
            ].min()
        ),

    "development_end":
        str(
            development_df[
                "availability_datetime"
            ].max()
        ),

    "test_rows":
        len(
            test_df
        ),

    "human_impact": {
        "model":
            "GradientBoostingRegressor",
        "loss":
            "huber",
        "prediction_target":
            "signed_log1p(next-current affected people)",
        "deployment_output":
            "current + inverse transformed predicted change"
    },

    "safety_usage": {
        "model":
            "RandomForestClassifier",
        "threshold":
            0.5
    },

    "safety_demand": {
        "model":
            "GradientBoostingRegressor",
        "loss":
            "huber",
        "stage":
            "positive-demand amount model",
        "prediction_target":
            (
                "signed_log1p(next-current "
                "safety-centre population)"
            )
    },

    "risk": {
        "model":
            "HistGradientBoostingClassifier",
        "classes":
            risk_order,
        "status":
            (
                "Experimental research-derived "
                "indicator; not an official warning"
            )
    }
}


with open(
    models_dir
    / "model_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )


# =========================================================
# 11. VERIFY SAVED FILES
# =========================================================

expected_files = [
    "human_impact_model.joblib",
    "safety_usage_model.joblib",
    "safety_demand_model.joblib",
    "escalation_risk_model.joblib",
    "model_metadata.json"
]


print("\n" + "=" * 70)
print("MODEL SAVE QC")
print("=" * 70)


all_saved = True


for file_name in expected_files:

    file_path = (
        models_dir
        / file_name
    )

    exists = (
        file_path.exists()
    )

    print(
        f"{file_name}:",
        "✅" if exists else "❌"
    )

    if not exists:
        all_saved = False


print()


if all_saved:

    print(
        "ALL FINAL DEPLOYMENT MODELS SAVED ✅"
    )

else:

    print(
        "MODEL SAVE INCOMPLETE ❌"
    )


print("=" * 70)

FINAL DEPLOYMENT MODEL TRAINING
Development rows: 1131
CORE features: 46
Test rows preserved: 150
Human-impact model trained ✅
Safety-usage classifier trained ✅
Safety-demand amount model trained ✅
Escalation-risk model trained ✅

MODEL SAVE QC
human_impact_model.joblib: ✅
safety_usage_model.joblib: ✅
safety_demand_model.joblib: ✅
escalation_risk_model.joblib: ✅
model_metadata.json: ✅

ALL FINAL DEPLOYMENT MODELS SAVED ✅


In [52]:
# =========================================================
# SAVE DEPLOYMENT 90% UNCERTAINTY MODELS
# Train on ALL labelled historical observations
# only AFTER final held-out test evaluation was completed.
# =========================================================

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor


# =========================================================
# 1. ALL LABELLED HISTORICAL DATA
# =========================================================

all_labelled_df = (
    pd.concat(
        [
            train_df,
            validation_df,
            test_df
        ],
        axis=0
    )
    .sort_values(
        "availability_datetime"
    )
    .reset_index(drop=True)
)


print(
    "All labelled deployment rows:",
    len(all_labelled_df)
)


# =========================================================
# 2. CORE FEATURES
# =========================================================

X_all = (
    all_labelled_df[
        CORE_MODEL_FEATURES
    ]
    .copy()
)


for col in CORE_MODEL_FEATURES:

    X_all[col] = pd.to_numeric(
        X_all[col],
        errors="coerce"
    )


print(
    "CORE feature count:",
    len(CORE_MODEL_FEATURES)
)


# =========================================================
# 3. HUMAN-IMPACT CHANGE TARGET
# =========================================================

all_impact_change = (
    all_labelled_df[
        "target_next_affected_people"
    ].astype(float)
    -
    all_labelled_df[
        "affected_people"
    ].astype(float)
)


all_impact_change_log = signed_log1p(
    all_impact_change
)


# =========================================================
# 4. EXACT ORIGINAL QUANTILE MODEL
# =========================================================

def build_deployment_quantile_model(alpha):

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            GradientBoostingRegressor(
                loss="quantile",
                alpha=alpha,
                n_estimators=250,
                learning_rate=0.03,
                max_depth=2,
                min_samples_leaf=10,
                random_state=42
            )
        )
    ])


# =========================================================
# 5. LOWER 5% + UPPER 95%
# =========================================================

deployment_lower_90_model = (
    build_deployment_quantile_model(
        0.05
    )
)

deployment_upper_90_model = (
    build_deployment_quantile_model(
        0.95
    )
)


deployment_lower_90_model.fit(
    X_all,
    all_impact_change_log
)

deployment_upper_90_model.fit(
    X_all,
    all_impact_change_log
)


print(
    "Lower 5% quantile model trained ✅"
)

print(
    "Upper 95% quantile model trained ✅"
)


# =========================================================
# 6. SAVE
# =========================================================

models_dir = Path("models")

models_dir.mkdir(
    parents=True,
    exist_ok=True
)


lower_path = (
    models_dir
    / "deployment_lower_90_model.joblib"
)

upper_path = (
    models_dir
    / "deployment_upper_90_model.joblib"
)


joblib.dump(
    deployment_lower_90_model,
    lower_path
)

joblib.dump(
    deployment_upper_90_model,
    upper_path
)


# =========================================================
# 7. QC
# =========================================================

print("\n" + "=" * 70)

print(
    "DEPLOYMENT UNCERTAINTY MODEL QC"
)

print("=" * 70)

print(
    "Training rows:",
    len(all_labelled_df)
)

print(
    "Expected rows:",
    1281
)

print(
    "Lower model saved:",
    lower_path.exists()
)

print(
    "Upper model saved:",
    upper_path.exists()
)


if (
    len(all_labelled_df) == 1281
    and lower_path.exists()
    and upper_path.exists()
):

    print(
        "\nDEPLOYMENT 90% INTERVAL MODELS SAVED ✅"
    )

else:

    print(
        "\nQC FAILED ❌"
    )

print("=" * 70)

All labelled deployment rows: 1281
CORE feature count: 46
Lower 5% quantile model trained ✅
Upper 95% quantile model trained ✅

DEPLOYMENT UNCERTAINTY MODEL QC
Training rows: 1281
Expected rows: 1281
Lower model saved: True
Upper model saved: True

DEPLOYMENT 90% INTERVAL MODELS SAVED ✅


In [54]:
# =========================================================
# FINAL OPERATIONAL DEPLOYMENT MODELS
#
# IMPORTANT:
# - Final test evaluation is already locked/completed.
# - These models are refitted on ALL 1281 labelled rows
#   only for dashboard/current deployment use.
# - Evaluation models are NOT overwritten.
# =========================================================

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)


# =========================================================
# 1. HELPER FUNCTIONS
# =========================================================

def signed_log1p(x):

    x = np.asarray(
        x,
        dtype=float
    )

    return (
        np.sign(x)
        *
        np.log1p(
            np.abs(x)
        )
    )


def inverse_signed_log1p(x):

    x = np.asarray(
        x,
        dtype=float
    )

    return (
        np.sign(x)
        *
        np.expm1(
            np.abs(x)
        )
    )


# =========================================================
# 2. ALL LABELLED HISTORICAL DATA
# =========================================================

all_labelled_df = (
    pd.concat(
        [
            train_df,
            validation_df,
            test_df
        ],
        axis=0
    )
    .sort_values(
        "availability_datetime"
    )
    .reset_index(drop=True)
)


print("=" * 70)
print("OPERATIONAL DEPLOYMENT MODEL REFIT")
print("=" * 70)

print(
    "All labelled rows:",
    len(all_labelled_df)
)


# =========================================================
# 3. PREPARE CORE FEATURES
# =========================================================

X_all = (
    all_labelled_df[
        CORE_MODEL_FEATURES
    ]
    .copy()
)


for col in CORE_MODEL_FEATURES:

    X_all[col] = pd.to_numeric(
        X_all[col],
        errors="coerce"
    )


print(
    "CORE feature count:",
    len(CORE_MODEL_FEATURES)
)


# =========================================================
# 4. HUMAN-IMPACT DEPLOYMENT MODEL
# =========================================================

all_impact_delta = (
    all_labelled_df[
        "target_next_affected_people"
    ].astype(float)
    -
    all_labelled_df[
        "affected_people"
    ].astype(float)
)


all_impact_delta_log = signed_log1p(
    all_impact_delta
)


deployment_impact_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            random_state=42
        )
    )
])


deployment_impact_model.fit(
    X_all,
    all_impact_delta_log
)


print(
    "Human-impact deployment model trained ✅"
)


# =========================================================
# 5. SAFETY-USAGE DEPLOYMENT CLASSIFIER
# =========================================================

deployment_safety_classifier = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=3,
            max_features=0.7,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])


deployment_safety_classifier.fit(
    X_all,
    all_labelled_df[
        "target_next_safety_centre_use"
    ].astype(int)
)


print(
    "Safety-usage deployment classifier trained ✅"
)


# =========================================================
# 6. SAFETY-AMOUNT DEPLOYMENT MODEL
# =========================================================

positive_mask = (
    all_labelled_df[
        "target_next_people_in_safety_centres"
    ]
    > 0
)


X_all_positive = (
    X_all.loc[
        positive_mask
    ]
    .copy()
)


all_safety_delta = (
    all_labelled_df.loc[
        positive_mask,
        "target_next_people_in_safety_centres"
    ].astype(float)
    -
    all_labelled_df.loc[
        positive_mask,
        "people_in_safety_centres"
    ].astype(float)
)


all_safety_delta_log = signed_log1p(
    all_safety_delta
)


deployment_safety_amount_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=8,
            random_state=42
        )
    )
])


deployment_safety_amount_model.fit(
    X_all_positive,
    all_safety_delta_log
)


print(
    "Safety-demand deployment model trained ✅"
)


# =========================================================
# 7. ESCALATION-RISK DEPLOYMENT MODEL
# =========================================================

deployment_risk_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=15,
            min_samples_leaf=15,
            l2_regularization=3.0,
            class_weight="balanced",
            random_state=42
        )
    )
])


deployment_risk_model.fit(
    X_all,
    all_labelled_df[
        "impact_escalation_risk"
    ].astype(str)
)


print(
    "Escalation-risk deployment model trained ✅"
)


# =========================================================
# 8. SAVE DEPLOYMENT MODELS
# =========================================================

models_dir = Path("models")

models_dir.mkdir(
    parents=True,
    exist_ok=True
)


deployment_files = {

    "deployment_human_impact_model.joblib":
        deployment_impact_model,

    "deployment_safety_usage_model.joblib":
        deployment_safety_classifier,

    "deployment_safety_demand_model.joblib":
        deployment_safety_amount_model,

    "deployment_escalation_risk_model.joblib":
        deployment_risk_model
}


for file_name, model in deployment_files.items():

    joblib.dump(
        model,
        models_dir / file_name
    )


# =========================================================
# 9. SAVE DEPLOYMENT FEATURE LIST
# =========================================================

joblib.dump(
    list(CORE_MODEL_FEATURES),
    models_dir / "core_model_features.joblib"
)


# =========================================================
# 10. QC
# =========================================================

print("\n" + "=" * 70)
print("OPERATIONAL MODEL SAVE QC")
print("=" * 70)


all_ok = True


for file_name in deployment_files:

    path = models_dir / file_name

    exists = path.exists()

    print(
        f"{file_name}:",
        "✅" if exists else "❌"
    )

    if not exists:
        all_ok = False


feature_path = (
    models_dir
    / "core_model_features.joblib"
)


print(
    "core_model_features.joblib:",
    "✅" if feature_path.exists() else "❌"
)


if not feature_path.exists():
    all_ok = False


print(
    "\nDeployment training rows:",
    len(all_labelled_df)
)

print(
    "Expected:",
    1281
)


if (
    all_ok
    and len(all_labelled_df) == 1281
    and len(CORE_MODEL_FEATURES) == 46
):

    print(
        "\nALL OPERATIONAL DEPLOYMENT MODELS SAVED ✅"
    )

else:

    print(
        "\nDEPLOYMENT QC FAILED ❌"
    )


print("=" * 70)

OPERATIONAL DEPLOYMENT MODEL REFIT
All labelled rows: 1281
CORE feature count: 46
Human-impact deployment model trained ✅
Safety-usage deployment classifier trained ✅
Safety-demand deployment model trained ✅
Escalation-risk deployment model trained ✅

OPERATIONAL MODEL SAVE QC
deployment_human_impact_model.joblib: ✅
deployment_safety_usage_model.joblib: ✅
deployment_safety_demand_model.joblib: ✅
deployment_escalation_risk_model.joblib: ✅
core_model_features.joblib: ✅

Deployment training rows: 1281
Expected: 1281

ALL OPERATIONAL DEPLOYMENT MODELS SAVED ✅


In [55]:
# =========================================================
# LATEST NATIONAL MODEL SNAPSHOT
#
# NO RETRAINING.
# Loads the saved deployment models and predicts the
# latest available canonical Situation Report.
# =========================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


# =========================================================
# 1. PATHS
# =========================================================

models_dir = Path("models")
processed_dir = Path("data/processed")


# =========================================================
# 2. LOAD SAVED DEPLOYMENT ASSETS
# =========================================================

deployment_impact_model = joblib.load(
    models_dir / "deployment_human_impact_model.joblib"
)

deployment_safety_classifier = joblib.load(
    models_dir / "deployment_safety_usage_model.joblib"
)

deployment_safety_amount_model = joblib.load(
    models_dir / "deployment_safety_demand_model.joblib"
)

deployment_risk_model = joblib.load(
    models_dir / "deployment_escalation_risk_model.joblib"
)

deployment_lower_90_model = joblib.load(
    models_dir / "deployment_lower_90_model.joblib"
)

deployment_upper_90_model = joblib.load(
    models_dir / "deployment_upper_90_model.joblib"
)

CORE_MODEL_FEATURES = joblib.load(
    models_dir / "core_model_features.joblib"
)


print("Saved deployment models loaded ✅")
print("CORE feature count:", len(CORE_MODEL_FEATURES))


# =========================================================
# 3. HELPER FUNCTION
# =========================================================

def inverse_signed_log1p(x):

    x = np.asarray(
        x,
        dtype=float
    )

    return (
        np.sign(x)
        *
        np.expm1(
            np.abs(x)
        )
    )


# =========================================================
# 4. GET LATEST CANONICAL REPORT
#
# feature_df includes latest report even when its
# next-period target is not yet available.
# =========================================================

if "feature_df" not in globals():

    raise RuntimeError(
        "feature_df is not available. "
        "Do not retrain models; first restore the "
        "Section 7 feature-engineering state."
    )


latest_current_row = (
    feature_df
    .sort_values(
        "availability_datetime"
    )
    .iloc[[-1]]
    .copy()
)


# =========================================================
# 5. RECREATE CURRENT HAZARD FLAGS
# =========================================================

latest_disaster_text = (
    latest_current_row[
        "disaster_types"
    ]
    .fillna("")
    .astype(str)
    .str.lower()
)


latest_current_row[
    "current_flood_flag"
] = (
    latest_disaster_text
    .str.contains(
        "flood",
        regex=False
    )
    .astype(int)
)


latest_current_row[
    "current_landslide_flag"
] = (
    latest_disaster_text
    .str.contains(
        "landslide",
        regex=False
    )
    .astype(int)
)


latest_current_row[
    "current_heavy_rain_flag"
] = (
    latest_disaster_text
    .str.contains(
        "heavy rain",
        regex=False
    )
    .astype(int)
)


# =========================================================
# 6. FEATURE QC
# =========================================================

missing_latest_features = [
    col
    for col in CORE_MODEL_FEATURES
    if col not in latest_current_row.columns
]


print(
    "Missing latest CORE features:",
    missing_latest_features
)


if missing_latest_features:

    raise RuntimeError(
        "Latest row is missing required features: "
        + str(missing_latest_features)
    )


X_latest = (
    latest_current_row[
        CORE_MODEL_FEATURES
    ]
    .copy()
)


for col in CORE_MODEL_FEATURES:

    X_latest[col] = pd.to_numeric(
        X_latest[col],
        errors="coerce"
    )


latest = latest_current_row.iloc[0]


# =========================================================
# 7. HUMAN-IMPACT POINT FORECAST
# =========================================================

latest_current_affected = float(
    latest[
        "affected_people"
    ]
)


impact_delta_log = (
    deployment_impact_model
    .predict(
        X_latest
    )
)


latest_predicted_delta = float(
    inverse_signed_log1p(
        impact_delta_log
    )[0]
)


latest_predicted_affected = max(
    0.0,
    latest_current_affected
    +
    latest_predicted_delta
)


# =========================================================
# 8. 90% MODEL-BASED PREDICTION INTERVAL
# =========================================================

lower_delta_log = (
    deployment_lower_90_model
    .predict(
        X_latest
    )
)


upper_delta_log = (
    deployment_upper_90_model
    .predict(
        X_latest
    )
)


lower_delta = float(
    inverse_signed_log1p(
        lower_delta_log
    )[0]
)


upper_delta = float(
    inverse_signed_log1p(
        upper_delta_log
    )[0]
)


latest_lower_90 = max(
    0.0,
    latest_current_affected
    +
    min(
        lower_delta,
        upper_delta
    )
)


latest_upper_90 = max(
    0.0,
    latest_current_affected
    +
    max(
        lower_delta,
        upper_delta
    )
)


# =========================================================
# 9. SAFETY-CENTRE USE PROBABILITY
# =========================================================

latest_safety_use_probability = float(
    deployment_safety_classifier
    .predict_proba(
        X_latest
    )[0, 1]
)


latest_safety_use_prediction = int(
    latest_safety_use_probability
    >= 0.5
)


# =========================================================
# 10. SAFETY-CENTRE POPULATION FORECAST
# =========================================================

latest_current_safety_population = float(
    latest[
        "people_in_safety_centres"
    ]
)


safety_delta_log = (
    deployment_safety_amount_model
    .predict(
        X_latest
    )
)


latest_safety_delta = float(
    inverse_signed_log1p(
        safety_delta_log
    )[0]
)


latest_predicted_safety_raw = max(
    0.0,
    latest_current_safety_population
    +
    latest_safety_delta
)


latest_predicted_safety_population = (
    latest_predicted_safety_raw
    if latest_safety_use_prediction == 1
    else 0.0
)


# =========================================================
# 11. EXPERIMENTAL ESCALATION RISK
# =========================================================

latest_risk_prediction = (
    deployment_risk_model
    .predict(
        X_latest
    )[0]
)


risk_probabilities = (
    deployment_risk_model
    .predict_proba(
        X_latest
    )[0]
)


risk_classes = (
    deployment_risk_model
    .named_steps["model"]
    .classes_
)


latest_risk_probability_dict = {
    str(label): float(probability)
    for label, probability
    in zip(
        risk_classes,
        risk_probabilities
    )
}


# =========================================================
# 12. BASIC HAZARD CONTEXT
# =========================================================

reported_hazards = str(
    latest.get(
        "disaster_types",
        ""
    )
)


current_flood_flag = int(
    latest[
        "current_flood_flag"
    ]
)


current_landslide_flag = int(
    latest[
        "current_landslide_flag"
    ]
)


current_heavy_rain_flag = int(
    latest[
        "current_heavy_rain_flag"
    ]
)


# =========================================================
# 13. CREATE CORE DASHBOARD SNAPSHOT
# =========================================================

snapshot = {

    "snapshot_datetime":
        str(
            latest[
                "availability_datetime"
            ]
        ),

    "forecast_definition":
        (
            "Next eligible DMC Situation Report "
            "within a maximum 48-hour gap"
        ),

    "geographic_scope":
        "Sri Lanka - national",

    "reported_hazards":
        reported_hazards,

    "current_affected_people":
        latest_current_affected,

    "predicted_next_affected_people":
        latest_predicted_affected,

    "predicted_affected_change":
        latest_predicted_delta,

    "prediction_lower_90":
        latest_lower_90,

    "prediction_upper_90":
        latest_upper_90,

    "prediction_interval_label":
        "90% model-based prediction interval",

    "current_safety_population":
        latest_current_safety_population,

    "safety_use_probability":
        latest_safety_use_probability,

    "safety_use_prediction":
        latest_safety_use_prediction,

    "predicted_next_safety_population":
        latest_predicted_safety_population,

    "experimental_escalation_risk":
        str(
            latest_risk_prediction
        ),

    "risk_probabilities":
        latest_risk_probability_dict,

    "hazard_flags": {

        "flood":
            current_flood_flag,

        "landslide":
            current_landslide_flag,

        "heavy_rain":
            current_heavy_rain_flag
    },

    "important_note":
        (
            "Human-impact and safety-demand forecasts "
            "are national multi-hazard forecasts. "
            "They must not be interpreted as Flood-only, "
            "Landslide-only, or district-level forecasts."
        )
}


# =========================================================
# 14. SAVE JSON
# =========================================================

snapshot_path = (
    processed_dir
    / "latest_model_snapshot.json"
)


with open(
    snapshot_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        snapshot,
        f,
        indent=4
    )


# =========================================================
# 15. DISPLAY RESULT
# =========================================================

print("\n" + "=" * 72)
print("LATEST FLOODIMPACT-LK MODEL SNAPSHOT")
print("=" * 72)

print(
    "Report time:",
    snapshot[
        "snapshot_datetime"
    ]
)

print(
    "Reported hazards:",
    reported_hazards
)

print(
    "\nCurrent affected people:",
    round(
        latest_current_affected,
        2
    )
)

print(
    "Predicted next affected people:",
    round(
        latest_predicted_affected,
        2
    )
)

print(
    "Predicted change:",
    round(
        latest_predicted_delta,
        2
    )
)

print(
    "90% model-based range:",
    round(
        latest_lower_90,
        2
    ),
    "-",
    round(
        latest_upper_90,
        2
    )
)

print(
    "\nCurrent safety population:",
    round(
        latest_current_safety_population,
        2
    )
)

print(
    "Safety-use probability:",
    f"{latest_safety_use_probability * 100:.2f}%"
)

print(
    "Predicted next safety population:",
    round(
        latest_predicted_safety_population,
        2
    )
)

print(
    "\nExperimental escalation risk:",
    latest_risk_prediction
)

print(
    "Risk probabilities:"
)

for label, probability in (
    latest_risk_probability_dict.items()
):

    print(
        f"  {label}: "
        f"{probability * 100:.2f}%"
    )


print(
    "\nSaved:",
    snapshot_path
)

print(
    "File exists:",
    snapshot_path.exists()
)

print("=" * 72)

Saved deployment models loaded ✅
CORE feature count: 46
Missing latest CORE features: []

LATEST FLOODIMPACT-LK MODEL SNAPSHOT
Report time: 2026-08-14 10:00:00
Reported hazards: Flood, Landslide, Heavy Rain, Strong Wind

Current affected people: 16536.0
Predicted next affected people: 16535.58
Predicted change: -0.42
90% model-based range: 7224.1 - 17781.32

Current safety population: 188.0
Safety-use probability: 96.11%
Predicted next safety population: 185.63

Experimental escalation risk: Moderate
Risk probabilities:
  Critical: 0.32%
  High: 4.10%
  Low: 28.27%
  Moderate: 67.31%

Saved: data\processed\latest_model_snapshot.json
File exists: True


In [56]:
# =========================================================
# FIND ORIGINAL FINAL RAPID-GROWTH / TREND CELL
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

matches = []

for i, cell in enumerate(nb["cells"]):

    source = "".join(
        cell.get("source", [])
    )

    source_lower = source.lower()

    # Skip this diagnostic cell itself
    if (
        "find original final rapid-growth / trend cell"
        in source_lower
    ):
        continue

    if (
        "refined_trend_df" in source
        or
        "rapid_growth_score" in source
        or
        "rapid growth anomaly" in source_lower
    ):

        matches.append(i)

        print("\n" + "=" * 90)
        print("MATCHED CELL:", i)
        print("=" * 90)
        print(source)


print("\nMatching cells:", matches)


MATCHED CELL: 165
# =========================================================
# 12.1 IMPACT TREND + RAPID-GROWTH ANOMALY DETECTION
#
# Purpose:
# Detect unusually rapid increases in affected population.
#
# IMPORTANT:
# - Uses current and PAST reports only.
# - Rolling historical statistics are shifted by one row,
#   so the current observation is never used to define
#   its own anomaly threshold.
# - This is a research-derived monitoring indicator,
#   not an official DMC warning.
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 1. Use the full canonical feature timeline
# =========================================================

trend_df = (
    feature_df
    .copy()
    .sort_values(
        "availability_datetime"
    )
    .reset_index(drop=True)
)


print(
    "Trend-monitoring rows:",
    len(trend_df)
)


# ======================

In [57]:
# =========================================================
# FINAL REFINED RAPID-GROWTH / TREND REBUILD
# Standalone recovery version of Section 12.2
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd


processed_dir = Path("data/processed")


# =========================================================
# 1. BUILD FINAL TREND TIMELINE DIRECTLY FROM feature_df
# =========================================================

refined_trend_df = (
    feature_df
    .copy()
    .sort_values(
        "availability_datetime"
    )
    .reset_index(drop=True)
)


refined_trend_df[
    "impact_absolute_change"
] = pd.to_numeric(
    refined_trend_df[
        "affected_people_change_from_prev"
    ],
    errors="coerce"
)


refined_trend_df[
    "impact_log_change"
] = pd.to_numeric(
    refined_trend_df[
        "affected_people_log_change_from_prev"
    ],
    errors="coerce"
)


# =========================================================
# 2. SIMPLE IMPACT TREND
# =========================================================

def classify_impact_trend(row):

    if row["previous_context_valid"] != True:
        return "Insufficient Previous Context"

    change = row[
        "impact_absolute_change"
    ]

    if pd.isna(change):
        return "Insufficient Previous Context"

    if change > 0:
        return "Increasing"

    elif change < 0:
        return "Decreasing"

    return "Stable"


refined_trend_df[
    "impact_trend"
] = refined_trend_df.apply(
    classify_impact_trend,
    axis=1
)


# =========================================================
# 3. FINAL FROZEN SETTINGS
# =========================================================

HISTORY_WINDOW = 120
MIN_POSITIVE_HISTORY = 20
MIN_MATERIAL_INCREASE = 142.0
EXTREME_PERCENTILE = 0.95


absolute_growth_percentile = []
relative_growth_percentile = []
historical_positive_count = []


# =========================================================
# 4. LEAKAGE-SAFE HISTORICAL PERCENTILES
# =========================================================

for i in range(
    len(refined_trend_df)
):

    current_row = (
        refined_trend_df.iloc[i]
    )

    if (
        current_row[
            "previous_context_valid"
        ] != True
        or pd.isna(
            current_row[
                "impact_absolute_change"
            ]
        )
        or pd.isna(
            current_row[
                "impact_log_change"
            ]
        )
    ):

        absolute_growth_percentile.append(
            np.nan
        )

        relative_growth_percentile.append(
            np.nan
        )

        historical_positive_count.append(
            0
        )

        continue


    # Only PRIOR reports
    start_position = max(
        0,
        i - HISTORY_WINDOW
    )


    historical_window = (
        refined_trend_df
        .iloc[
            start_position:i
        ]
        .copy()
    )


    historical_positive = (
        historical_window[
            (
                historical_window[
                    "previous_context_valid"
                ] == True
            )
            &
            (
                historical_window[
                    "impact_absolute_change"
                ] > 0
            )
            &
            historical_window[
                "impact_absolute_change"
            ].notna()
            &
            historical_window[
                "impact_log_change"
            ].notna()
        ]
    )


    n_history = len(
        historical_positive
    )


    historical_positive_count.append(
        n_history
    )


    if (
        n_history
        < MIN_POSITIVE_HISTORY
    ):

        absolute_growth_percentile.append(
            np.nan
        )

        relative_growth_percentile.append(
            np.nan
        )

        continue


    current_absolute = float(
        current_row[
            "impact_absolute_change"
        ]
    )


    current_relative = float(
        current_row[
            "impact_log_change"
        ]
    )


    abs_percentile = np.mean(
        historical_positive[
            "impact_absolute_change"
        ].to_numpy(
            dtype=float
        )
        <= current_absolute
    )


    rel_percentile = np.mean(
        historical_positive[
            "impact_log_change"
        ].to_numpy(
            dtype=float
        )
        <= current_relative
    )


    absolute_growth_percentile.append(
        abs_percentile
    )

    relative_growth_percentile.append(
        rel_percentile
    )


# =========================================================
# 5. BOUNDED RAPID-GROWTH SCORE 0-100
# =========================================================

refined_trend_df[
    "absolute_growth_percentile"
] = absolute_growth_percentile


refined_trend_df[
    "relative_growth_percentile"
] = relative_growth_percentile


refined_trend_df[
    "historical_positive_count"
] = historical_positive_count


refined_trend_df[
    "rapid_growth_score"
] = (
    np.minimum(
        refined_trend_df[
            "absolute_growth_percentile"
        ],
        refined_trend_df[
            "relative_growth_percentile"
        ]
    )
    * 100
)


# =========================================================
# 6. FINAL ANOMALY RULE
# =========================================================

refined_trend_df[
    "rapid_growth_anomaly"
] = (
    (
        refined_trend_df[
            "previous_context_valid"
        ] == True
    )
    &
    (
        refined_trend_df[
            "impact_absolute_change"
        ] >= MIN_MATERIAL_INCREASE
    )
    &
    (
        refined_trend_df[
            "absolute_growth_percentile"
        ] >= EXTREME_PERCENTILE
    )
    &
    (
        refined_trend_df[
            "relative_growth_percentile"
        ] >= EXTREME_PERCENTILE
    )
)


# =========================================================
# 7. FINAL MONITORING STATUS
# =========================================================

def refined_monitoring_status(row):

    if row[
        "previous_context_valid"
    ] != True:

        return (
            "Gap / No Comparable Previous Report"
        )


    if pd.isna(
        row[
            "rapid_growth_score"
        ]
    ):

        return (
            "Insufficient Historical Baseline"
        )


    if bool(
        row[
            "rapid_growth_anomaly"
        ]
    ):

        return "Rapid Growth Anomaly"


    if row[
        "impact_absolute_change"
    ] > 0:

        return "Normal Increase"


    elif row[
        "impact_absolute_change"
    ] < 0:

        return "Decrease"


    return "Stable"


refined_trend_df[
    "impact_monitoring_status"
] = refined_trend_df.apply(
    refined_monitoring_status,
    axis=1
)


# =========================================================
# 8. SAVE FOR DASHBOARD
# =========================================================

trend_path = (
    processed_dir
    / "trend_history_dashboard.csv"
)


refined_trend_df.to_csv(
    trend_path,
    index=False
)


# =========================================================
# 9. QC
# =========================================================

latest_refined_row = (
    refined_trend_df.iloc[-1]
)


print("=" * 72)
print("FINAL REFINED RAPID-GROWTH MONITORING")
print("=" * 72)

print(
    "\nStatus distribution:"
)

print(
    refined_trend_df[
        "impact_monitoring_status"
    ].value_counts()
)


print(
    "\nRapid-growth anomalies:",
    int(
        refined_trend_df[
            "rapid_growth_anomaly"
        ].sum()
    )
)


print(
    "Percentage flagged:",
    round(
        refined_trend_df[
            "rapid_growth_anomaly"
        ].mean()
        * 100,
        2
    ),
    "%"
)


print(
    "\nLatest date:",
    latest_refined_row[
        "availability_datetime"
    ]
)

print(
    "Latest affected:",
    latest_refined_row[
        "affected_people"
    ]
)

print(
    "Change from previous:",
    latest_refined_row[
        "impact_absolute_change"
    ]
)

print(
    "Trend:",
    latest_refined_row[
        "impact_trend"
    ]
)

print(
    "Rapid-growth score:",
    round(
        latest_refined_row[
            "rapid_growth_score"
        ],
        4
    )
)

print(
    "Monitoring status:",
    latest_refined_row[
        "impact_monitoring_status"
    ]
)


print(
    "\nSaved:",
    trend_path
)

print(
    "File exists:",
    trend_path.exists()
)

print("=" * 72)

FINAL REFINED RAPID-GROWTH MONITORING

Status distribution:
impact_monitoring_status
Normal Increase                        520
Decrease                               396
Stable                                 293
Insufficient Historical Baseline        73
Gap / No Comparable Previous Report     36
Rapid Growth Anomaly                     5
Name: count, dtype: int64

Rapid-growth anomalies: 5
Percentage flagged: 0.38 %

Latest date: 2026-08-14 10:00:00
Latest affected: 16536.0
Change from previous: 16.0
Trend: Increasing
Rapid-growth score: 5.7692
Monitoring status: Normal Increase

Saved: data\processed\trend_history_dashboard.csv
File exists: True


In [58]:
# =========================================================
# FINAL DRPI + SCPI + HAZARD CONTEXT + FRESHNESS
# Lightweight dashboard recovery
#
# NO MODEL TRAINING
# =========================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


processed_dir = Path("data/processed")


# =========================================================
# 1. REQUIRED OBJECT CHECK
# =========================================================

required_objects = [
    "latest",
    "latest_current_affected",
    "latest_predicted_delta",
    "latest_current_safety_population",
    "latest_predicted_safety_population",
    "latest_safety_use_probability",
    "refined_trend_df"
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# Recreate all labelled data if necessary
if "all_labelled_df" not in globals():

    all_labelled_df = (
        pd.concat(
            [
                train_df,
                validation_df,
                test_df
            ],
            axis=0
        )
        .sort_values(
            "availability_datetime"
        )
        .reset_index(drop=True)
    )


# =========================================================
# 2. HELPERS
# =========================================================

def safe_numeric(value):

    return pd.to_numeric(
        pd.Series([value]),
        errors="coerce"
    ).iloc[0]


def historical_percentile(
    value,
    historical_values,
    log_transform=False,
    positive_only=False
):

    values = pd.to_numeric(
        pd.Series(
            historical_values
        ),
        errors="coerce"
    ).dropna()


    if positive_only:

        values = values[
            values > 0
        ]

        if value <= 0:
            return 0.0


    if len(values) == 0:
        return np.nan


    if log_transform:

        values = np.log1p(
            np.clip(
                values,
                0,
                None
            )
        )

        value = np.log1p(
            max(
                float(value),
                0.0
            )
        )


    percentile = (
        (values <= value).mean()
        * 100
    )


    return float(
        np.clip(
            percentile,
            0,
            100
        )
    )


# =========================================================
# 3. CURRENT HAZARD FLAGS
# =========================================================

current_flood = safe_numeric(
    latest.get(
        "current_flood_flag",
        0
    )
)

current_landslide = safe_numeric(
    latest.get(
        "current_landslide_flag",
        0
    )
)

current_heavy_rain = safe_numeric(
    latest.get(
        "current_heavy_rain_flag",
        0
    )
)


current_flood = (
    0.0
    if pd.isna(current_flood)
    else float(current_flood)
)

current_landslide = (
    0.0
    if pd.isna(current_landslide)
    else float(current_landslide)
)

current_heavy_rain = (
    0.0
    if pd.isna(current_heavy_rain)
    else float(current_heavy_rain)
)


# =========================================================
# 4. WEATHER CONTEXT
# =========================================================

weather_level = safe_numeric(
    latest.get(
        "weather_max_warning_level_24h",
        np.nan
    )
)


weather_warning_score = (
    np.clip(
        weather_level / 3.0,
        0,
        1
    )
    * 100
    if pd.notna(weather_level)
    else 0.0
)


weather_context_score = float(
    max(
        50.0 * current_heavy_rain,
        weather_warning_score
    )
)


# =========================================================
# 5. LANDSLIDE CONTEXT
# =========================================================

landslide_level = safe_numeric(
    latest.get(
        "landslide_max_warning_level",
        np.nan
    )
)


landslide_warning_score = (
    np.clip(
        landslide_level / 3.0,
        0,
        1
    )
    * 100
    if pd.notna(landslide_level)
    else 0.0
)


landslide_context_score = float(
    max(
        50.0 * current_landslide,
        landslide_warning_score
    )
)


# =========================================================
# 6. FLOOD / RIVER CONTEXT
# =========================================================

river_warning = safe_numeric(
    latest.get(
        "river_active_warning_24h",
        np.nan
    )
)


river_warning_score = (
    np.clip(
        river_warning,
        0,
        1
    )
    * 100
    if pd.notna(river_warning)
    else 0.0
)


flood_context_score = float(
    max(
        50.0 * current_flood,
        river_warning_score
    )
)


# =========================================================
# 7. COMBINED HAZARD CONTEXT
# =========================================================

hazard_context_score = float(
    np.mean(
        [
            flood_context_score,
            weather_context_score,
            landslide_context_score
        ]
    )
)


# =========================================================
# 8. LATEST RAPID-GROWTH SCORE
# =========================================================

latest_refined_row = (
    refined_trend_df
    .sort_values(
        "availability_datetime"
    )
    .iloc[-1]
)


latest_rapid_growth_score = safe_numeric(
    latest_refined_row[
        "rapid_growth_score"
    ]
)


if pd.isna(
    latest_rapid_growth_score
):

    latest_rapid_growth_score = 0.0


latest_rapid_growth_score = float(
    np.clip(
        latest_rapid_growth_score,
        0,
        100
    )
)


# =========================================================
# 9. DRPI
# Disaster Response Priority Index
# =========================================================

current_impact_score = (
    historical_percentile(
        latest_current_affected,

        all_labelled_df[
            "affected_people"
        ],

        log_transform=True
    )
)


historical_impact_changes = (
    all_labelled_df[
        "target_next_affected_people"
    ]
    -
    all_labelled_df[
        "affected_people"
    ]
)


predicted_growth_score = (
    historical_percentile(
        latest_predicted_delta,
        historical_impact_changes,
        positive_only=True
    )
)


predicted_safety_score = (
    historical_percentile(
        latest_predicted_safety_population,

        all_labelled_df[
            "target_next_people_in_safety_centres"
        ],

        log_transform=True
    )
)


DRPI_COMPONENTS = {

    "Current Human Impact":
        current_impact_score,

    "Predicted Impact Growth":
        predicted_growth_score,

    "Predicted Safety Demand":
        predicted_safety_score,

    "Hazard Context":
        hazard_context_score,

    "Rapid Growth":
        latest_rapid_growth_score
}


valid_drpi_components = [
    float(value)
    for value
    in DRPI_COMPONENTS.values()
    if pd.notna(value)
]


DRPI = (
    float(
        np.mean(
            valid_drpi_components
        )
    )
    if valid_drpi_components
    else np.nan
)


if pd.notna(DRPI):

    DRPI = float(
        np.clip(
            DRPI,
            0,
            100
        )
    )


# =========================================================
# 10. SCPI
# Safety-Centre Pressure Index
# =========================================================

current_safety_score = (
    historical_percentile(
        latest_current_safety_population,

        all_labelled_df[
            "people_in_safety_centres"
        ],

        log_transform=True
    )
)


forecast_safety_score = (
    historical_percentile(
        latest_predicted_safety_population,

        all_labelled_df[
            "target_next_people_in_safety_centres"
        ],

        log_transform=True
    )
)


predicted_safety_increase = (
    latest_predicted_safety_population
    -
    latest_current_safety_population
)


historical_safety_changes = (
    all_labelled_df[
        "target_next_people_in_safety_centres"
    ]
    -
    all_labelled_df[
        "people_in_safety_centres"
    ]
)


safety_growth_score = (
    historical_percentile(
        predicted_safety_increase,
        historical_safety_changes,
        positive_only=True
    )
)


SCPI_COMPONENTS = {

    "Current Safety Population":
        current_safety_score,

    "Forecast Safety Population":
        forecast_safety_score,

    "Safety-Use Probability":
        latest_safety_use_probability * 100,

    "Predicted Safety Growth":
        safety_growth_score,

    "Hazard Context":
        hazard_context_score
}


valid_scpi_components = [
    float(value)
    for value
    in SCPI_COMPONENTS.values()
    if pd.notna(value)
]


SCPI = (
    float(
        np.mean(
            valid_scpi_components
        )
    )
    if valid_scpi_components
    else np.nan
)


if pd.notna(SCPI):

    SCPI = float(
        np.clip(
            SCPI,
            0,
            100
        )
    )


# =========================================================
# 11. RESEARCH INDEX BANDS
# =========================================================

def research_index_band(score):

    if pd.isna(score):
        return "Unavailable"

    if score < 25:
        return "Low"

    elif score < 50:
        return "Moderate"

    elif score < 75:
        return "High"

    return "Very High"


DRPI_BAND = (
    research_index_band(
        DRPI
    )
)


SCPI_BAND = (
    research_index_band(
        SCPI
    )
)


# =========================================================
# 12. DATA FRESHNESS
# =========================================================

river_age_hours = safe_numeric(
    latest.get(
        "latest_river_age_hours",
        np.nan
    )
)


landslide_age_hours = safe_numeric(
    latest.get(
        "latest_landslide_age_hours",
        np.nan
    )
)


def freshness_status(
    age_hours,
    threshold=24
):

    if pd.isna(age_hours):
        return "Unavailable"

    if age_hours <= threshold:
        return "Current"

    return "Stale"


river_freshness_status = (
    freshness_status(
        river_age_hours
    )
)


landslide_freshness_status = (
    freshness_status(
        landslide_age_hours
    )
)


# =========================================================
# 13. SAVE DASHBOARD RESEARCH CONTEXT
# =========================================================

research_context = {

    "snapshot_datetime":
        str(
            latest[
                "availability_datetime"
            ]
        ),

    "hazard_context": {

        "flood_context_score":
            flood_context_score,

        "weather_context_score":
            weather_context_score,

        "landslide_context_score":
            landslide_context_score,

        "combined_hazard_context_score":
            hazard_context_score
    },

    "trend": {

        "trend":
            str(
                latest_refined_row[
                    "impact_trend"
                ]
            ),

        "change_from_previous":
            float(
                latest_refined_row[
                    "impact_absolute_change"
                ]
            ),

        "rapid_growth_score":
            latest_rapid_growth_score,

        "monitoring_status":
            str(
                latest_refined_row[
                    "impact_monitoring_status"
                ]
            )
    },

    "DRPI": {

        "score":
            DRPI,

        "band":
            DRPI_BAND,

        "components":
            {
                key: (
                    float(value)
                    if pd.notna(value)
                    else None
                )
                for key, value
                in DRPI_COMPONENTS.items()
            },

        "note":
            (
                "Research-derived FloodImpact-LK "
                "priority indicator; not an official warning."
            )
    },

    "SCPI": {

        "score":
            SCPI,

        "band":
            SCPI_BAND,

        "components":
            {
                key: (
                    float(value)
                    if pd.notna(value)
                    else None
                )
                for key, value
                in SCPI_COMPONENTS.items()
            },

        "note":
            (
                "Research-derived relative safety-demand "
                "pressure indicator. It does not represent "
                "actual safety-centre capacity utilization."
            )
    },

    "freshness": {

        "river_age_hours":
            (
                float(river_age_hours)
                if pd.notna(river_age_hours)
                else None
            ),

        "river_status":
            river_freshness_status,

        "landslide_age_hours":
            (
                float(landslide_age_hours)
                if pd.notna(landslide_age_hours)
                else None
            ),

        "landslide_status":
            landslide_freshness_status
    }
}


research_context_path = (
    processed_dir
    / "latest_research_context.json"
)


with open(
    research_context_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        research_context,
        f,
        indent=4
    )


# =========================================================
# 14. QC OUTPUT
# =========================================================

print("=" * 72)
print("FINAL RESEARCH CONTEXT")
print("=" * 72)


print("\nHAZARD CONTEXT")

print(
    "Flood:",
    round(
        flood_context_score,
        2
    )
)

print(
    "Weather:",
    round(
        weather_context_score,
        2
    )
)

print(
    "Landslide:",
    round(
        landslide_context_score,
        2
    )
)

print(
    "Combined:",
    round(
        hazard_context_score,
        2
    )
)


print("\nDRPI COMPONENTS")

for key, value in (
    DRPI_COMPONENTS.items()
):

    print(
        key + ":",
        round(
            value,
            2
        )
        if pd.notna(value)
        else "Unavailable"
    )


print(
    "DRPI:",
    round(
        DRPI,
        2
    ),
    DRPI_BAND
)


print("\nSCPI COMPONENTS")

for key, value in (
    SCPI_COMPONENTS.items()
):

    print(
        key + ":",
        round(
            value,
            2
        )
        if pd.notna(value)
        else "Unavailable"
    )


print(
    "SCPI:",
    round(
        SCPI,
        2
    ),
    SCPI_BAND
)


print("\nDATA FRESHNESS")

print(
    "River age:",
    river_age_hours,
    "hours |",
    river_freshness_status
)

print(
    "Landslide age:",
    landslide_age_hours,
    "hours |",
    landslide_freshness_status
)


print(
    "\nTrend:",
    latest_refined_row[
        "impact_trend"
    ]
)

print(
    "Rapid-growth score:",
    round(
        latest_rapid_growth_score,
        4
    )
)

print(
    "Monitoring status:",
    latest_refined_row[
        "impact_monitoring_status"
    ]
)


print(
    "\nSaved:",
    research_context_path
)

print(
    "File exists:",
    research_context_path.exists()
)

print("=" * 72)

FINAL RESEARCH CONTEXT

HAZARD CONTEXT
Flood: 50.0
Weather: 50.0
Landslide: 50.0
Combined: 50.0

DRPI COMPONENTS
Current Human Impact: 75.72
Predicted Impact Growth: 0.0
Predicted Safety Demand: 66.74
Hazard Context: 50.0
Rapid Growth: 5.77
DRPI: 39.65 Moderate

SCPI COMPONENTS
Current Safety Population: 67.14
Forecast Safety Population: 66.74
Safety-Use Probability: 96.11
Predicted Safety Growth: 0.0
Hazard Context: 50.0
SCPI: 56.0 High

DATA FRESHNESS
River age: 21.5 hours | Current
Landslide age: 120.5 hours | Stale

Trend: Increasing
Rapid-growth score: 5.7692
Monitoring status: Normal Increase

Saved: data\processed\latest_research_context.json
File exists: True


In [65]:
# =========================================================
# FINAL REFINED SIMILAR-EVENT FINDER
# Recovery + dashboard export
#
# Reproduces Section 11.2 final method.
# =========================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances


processed_dir = Path("data/processed")


# =========================================================
# 1. Restore development reference if necessary
# =========================================================

if "development_df" not in globals():

    development_df = pd.concat(
        [
            train_df,
            validation_df
        ],
        axis=0
    ).copy()


# =========================================================
# 2. FINAL 19 SIMILARITY FEATURES
# =========================================================

SIMILARITY_COUNT_FEATURES = [

    "affected_people",
    "affected_families",
    "safety_centres",
    "people_in_safety_centres",

    "weather_reports_24h",
    "weather_max_rainfall_threshold_24h",
    "weather_max_wind_24h",

    "landslide_districts_at_risk",
    "landslide_level1_districts",
    "landslide_level2_districts",
    "landslide_level3_districts"
]


SIMILARITY_CHANGE_FEATURES = [

    "affected_people_change_from_prev",
    "affected_families_change_from_prev",
    "safety_population_change_from_prev"
]


SIMILARITY_STATE_FEATURES = [

    "weather_max_warning_level_24h",
    "weather_heavy_rain_24h",
    "current_heavy_rain_flag",
    "landslide_active_warning",
    "landslide_max_warning_level"
]


# =========================================================
# 3. BUILD SCALE-AWARE REPRESENTATION
# =========================================================

def build_similarity_matrix(df):

    transformed = pd.DataFrame(
        index=df.index
    )


    # Positive/count variables -> log1p
    for col in SIMILARITY_COUNT_FEATURES:

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        transformed[
            f"log_{col}"
        ] = np.log1p(
            values.clip(lower=0)
        )


    # Signed changes -> signed log1p
    for col in SIMILARITY_CHANGE_FEATURES:

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        transformed[
            f"signedlog_{col}"
        ] = (
            np.sign(values)
            *
            np.log1p(
                np.abs(values)
            )
        )


    # State / warning features unchanged
    for col in SIMILARITY_STATE_FEATURES:

        transformed[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


    return transformed


# =========================================================
# 4. HISTORICAL DEVELOPMENT LIBRARY
# =========================================================

history_similarity_refined = (
    build_similarity_matrix(
        development_df
    )
)


test_similarity_refined = (
    build_similarity_matrix(
        test_df
    )
)


print(
    "Refined similarity feature count:",
    history_similarity_refined.shape[1]
)


# =========================================================
# 5. FIT PREPROCESSING ON HISTORICAL DEVELOPMENT DATA ONLY
# =========================================================

refined_similarity_imputer = (
    SimpleImputer(
        strategy="median"
    )
)


refined_similarity_scaler = (
    StandardScaler()
)


history_refined_imputed = (
    refined_similarity_imputer
    .fit_transform(
        history_similarity_refined
    )
)


history_refined_scaled = (
    refined_similarity_scaler
    .fit_transform(
        history_refined_imputed
    )
)


test_refined_imputed = (
    refined_similarity_imputer
    .transform(
        test_similarity_refined
    )
)


test_refined_scaled = (
    refined_similarity_scaler
    .transform(
        test_refined_imputed
    )
)


# =========================================================
# 6. SIMILAR-EVENT FUNCTION
# =========================================================

def find_refined_similar_events(
    test_position,
    top_k=5
):

    query = (
        test_refined_scaled[
            test_position
        ]
        .reshape(1, -1)
    )


    distances = (
        euclidean_distances(
            query,
            history_refined_scaled
        )[0]
    )


    nearest_positions = (
        np.argsort(
            distances
        )[:top_k]
    )


    similarity_score = (
        1
        /
        (
            1
            +
            distances[
                nearest_positions
            ]
        )
    )


    historical = (
        development_df
        .iloc[
            nearest_positions
        ]
        .copy()
    )


    result = pd.DataFrame({

        "historical_datetime":
            historical[
                "availability_datetime"
            ].to_numpy(),

        "similarity_score":
            similarity_score,

        "historical_affected_people":
            historical[
                "affected_people"
            ].to_numpy(),

        "historical_people_in_safety_centres":
            historical[
                "people_in_safety_centres"
            ].to_numpy(),

        "historical_next_affected_people":
            historical[
                "target_next_affected_people"
            ].to_numpy(),

        "historical_next_safety_population":
            historical[
                "target_next_people_in_safety_centres"
            ].to_numpy(),

        "historical_escalation_risk":
            historical[
                "impact_escalation_risk"
            ].to_numpy()

    })


    return (
        result
        .sort_values(
            "similarity_score",
            ascending=False
        )
        .reset_index(drop=True)
    )


# =========================================================
# 7. ORIGINAL FINAL QUERY
#
# Latest held-out comparable TEST report.
# =========================================================

latest_test_position = (
    len(test_df) - 1
)


latest_refined_similar_events = (
    find_refined_similar_events(
        latest_test_position,
        top_k=5
    )
)


query_row = (
    test_df.iloc[
        latest_test_position
    ]
)


# =========================================================
# 8. SAVE DASHBOARD FILE
# =========================================================

similar_events_path = (
    processed_dir
    / "similar_events_dashboard.csv"
)


latest_refined_similar_events.to_csv(
    similar_events_path,
    index=False
)


# =========================================================
# 9. QC
# =========================================================

print("\n" + "=" * 76)
print("FINAL REFINED SIMILAR-EVENT FINDER")
print("=" * 76)


print(
    "Query date:",
    query_row[
        "availability_datetime"
    ]
)

print(
    "Query affected people:",
    query_row[
        "affected_people"
    ]
)

print(
    "Query safety population:",
    query_row[
        "people_in_safety_centres"
    ]
)


print(
    "\nTOP 5 SIMILAR HISTORICAL EVENTS"
)


print(
    latest_refined_similar_events
    .round(
        {
            "similarity_score": 4
        }
    )
    .to_string(
        index=False
    )
)


print(
    "\nSaved:",
    similar_events_path
)

print(
    "File exists:",
    similar_events_path.exists()
)

print(
    "\nNOTE:"
)

print(
    "Similarity score is distance-derived "
    "feature-space similarity."
)

print(
    "It is NOT probability or forecast confidence."
)

print("=" * 76)

Refined similarity feature count: 19

FINAL REFINED SIMILAR-EVENT FINDER
Query date: 2026-08-13 10:00:00
Query affected people: 16520.0
Query safety population: 263.0

TOP 5 SIMILAR HISTORICAL EVENTS
historical_datetime  similarity_score  historical_affected_people  historical_people_in_safety_centres  historical_next_affected_people  historical_next_safety_population historical_escalation_risk
2024-01-18 09:00:00            0.3743                      4244.0                               1309.0                           2202.0                             1228.0                        Low
2023-12-22 09:00:00            0.3677                     30377.0                                783.0                          29805.0                              737.0                        Low
2024-06-14 09:00:00            0.3462                    101852.0                                124.0                            948.0                              118.0                        Low
2024-12-

In [66]:
# =========================================================
# FIND ORIGINAL FINAL SHAP EXPLAINABILITY CELL
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(
    notebook_path,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)


matches = []


for i, cell in enumerate(nb["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    source_lower = source.lower()


    # Skip this diagnostic cell itself
    if (
        "find original final shap explainability cell"
        in source_lower
    ):
        continue


    # Strong SHAP-specific indicators.
    # Avoid using only "shap" because words such as
    # "reshape" would create false matches.
    indicators = [

        "import shap",
        "shap.explainer",
        "shap.treeexplainer",
        "shap.plots",
        "shap_values",
        "mean_abs_shap",
        "beeswarm",
        "waterfall",
        "explainable ai"
    ]


    matched_terms = [
        term
        for term in indicators
        if term in source_lower
    ]


    if matched_terms:

        matches.append(i)

        print(
            "\n" + "=" * 90
        )

        print(
            "MATCHED CELL:",
            i
        )

        print(
            "MATCHED TERMS:",
            matched_terms
        )

        print(
            "=" * 90
        )

        print(source)


print(
    "\nMatching cells:",
    matches
)


MATCHED CELL: 162
MATCHED TERMS: ['import shap', 'shap.treeexplainer', 'shap.plots', 'shap_values', 'mean_abs_shap', 'beeswarm', 'waterfall']
# =========================================================
# 10.1 SHAP EXPLANATION
#
# Explains the FINAL human-impact GradientBoosting model.
#
# IMPORTANT:
# The model predicts signed-log CHANGE in affected people.
# Therefore SHAP values explain the predicted CHANGE,
# not the absolute affected population directly.
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 1. Import SHAP
# =========================================================

try:
    import shap

except ImportError:
    raise ImportError(
        "SHAP is not installed. Run: pip install shap"
    )


# =========================================================
# 2. Extract fitted preprocessing + model
# ================================

In [59]:
# =========================================================
# FIND ORIGINAL REFINED SIMILAR-EVENT FINDER
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(
    notebook_path,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)


matches = []


for i, cell in enumerate(nb["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    source_lower = source.lower()


    # Skip this temporary diagnostic cell
    if (
        "find original refined similar-event finder"
        in source_lower
    ):
        continue


    indicators = [

        "latest_refined_similar_events",

        "similarity_score",

        "historical_datetime",

        "standardscaler",

        "similar event",

        "similar-event"
    ]


    matched_terms = [
        term
        for term in indicators
        if term in source_lower
    ]


    if matched_terms:

        matches.append(i)

        print(
            "\n" + "=" * 90
        )

        print(
            "MATCHED CELL:",
            i
        )

        print(
            "MATCHED TERMS:",
            matched_terms
        )

        print(
            "=" * 90
        )

        print(source)


print(
    "\nMatching cells:",
    matches
)


MATCHED CELL: 139
MATCHED TERMS: ['standardscaler']
# =========================================================
# 8.2 CORE Human-Impact Regression Models
#
# Compare:
# 1. Persistence baseline
# 2. Ridge
# 3. Random Forest
# 4. HistGradientBoosting
#
# Two strategies:
# A. Direct log-target prediction
# B. Residual log-change prediction
#
# MODEL SELECTION USES VALIDATION ONLY.
# TEST SET IS NOT USED HERE.
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

import numpy as np
import pandas as pd


# =========================================================
# 1. Prepare CORE feature matrices
# =========================================================

X_train_core = train_df[
    CORE_FEATURES
].copy()

X_val_core = val

In [50]:
# =========================================================
# SHOW ORIGINAL 9.1 UNCERTAINTY CELL
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

found = False

for i, cell in enumerate(nb["cells"]):

    source = "".join(
        cell.get("source", [])
    )

    if (
        "Human-Impact Prediction Uncertainty"
        in source
    ):

        print("=" * 90)
        print("FOUND SECTION AT CELL:", i)
        print("=" * 90)
        print(source)

        # Print the next code cell as well
        for j in range(i + 1, len(nb["cells"])):

            if nb["cells"][j].get("cell_type") == "code":

                print("\n" + "=" * 90)
                print("NEXT CODE CELL:", j)
                print("=" * 90)

                print(
                    "".join(
                        nb["cells"][j].get(
                            "source",
                            []
                        )
                    )
                )

                break

        found = True
        break


if not found:
    print("❌ 9.1 uncertainty section not found")

FOUND SECTION AT CELL: 127
# =========================================================
# SHOW ORIGINAL 9.1 UNCERTAINTY CELL
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

found = False

for i, cell in enumerate(nb["cells"]):

    source = "".join(
        cell.get("source", [])
    )

    if (
        "Human-Impact Prediction Uncertainty"
        in source
    ):

        print("=" * 90)
        print("FOUND SECTION AT CELL:", i)
        print("=" * 90)
        print(source)

        # Print the next code cell as well
        for j in range(i + 1, len(nb["cells"])):

            if nb["cells"][j].get("cell_type") == "code":

                print("\n" + "=" * 90)
                print("NEXT CODE CELL:", j)
                print("=" * 90)

                print(
                    "".join(
                  

In [51]:
# =========================================================
# FIND REAL ORIGINAL 9.1 UNCERTAINTY CODE
# Skip temporary search cells
# =========================================================

import json
from pathlib import Path

notebook_path = Path("FloodImpact_LK_Final.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

matches = []

for i, cell in enumerate(nb["cells"]):

    source = "".join(
        cell.get("source", [])
    )

    source_lower = source.lower()

    # Skip our temporary diagnostic/search cells
    if (
        "show original 9.1 uncertainty cell" in source_lower
        or
        "find real original 9.1 uncertainty code" in source_lower
    ):
        continue

    # Look for signatures of the REAL uncertainty section
    if (
        "human-impact prediction uncertainty" in source_lower
        or
        "fixed 90% prediction interval" in source_lower
        or
        (
            'loss="quantile"' in source_lower
            and "gradientboostingregressor" in source_lower
        )
    ):

        matches.append(i)

        print("\n" + "=" * 90)
        print("MATCHED CELL:", i)
        print("=" * 90)
        print(source)


print("\nMatching cells:", matches)

if not matches:
    print("❌ Original uncertainty cell not found")


MATCHED CELL: 150
# =========================================================
# 9.1 Human-Impact Prediction Uncertainty
#
# Fixed 90% prediction interval
# Lower = 5th percentile
# Upper = 95th percentile
#
# No test-based tuning.
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

import numpy as np
import pandas as pd


# =========================================================
# 1. Development target
# =========================================================

dev_impact_change = (
    development_df["target_next_affected_people"].astype(float)
    -
    development_df["affected_people"].astype(float)
)

dev_impact_change_log = signed_log1p(
    dev_impact_change
)


# =========================================================
# 2. Quantile-model factory
# =========================================================

def build_quantile_model(a

# 8. Baseline Models

## 8.1 Naive Forecasting Baselines

In [ ]:
# =========================================================
# 8.1 Naive Forecasting Baselines
# =========================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    mean_squared_log_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

import numpy as np
import pandas as pd


# =========================================================
# 1. Regression evaluation helper
# =========================================================

def regression_metrics(
    y_true,
    y_pred,
    model_name,
    target_name,
    split_name
):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # Predictions cannot be negative for population counts
    y_pred_nonnegative = np.clip(
        y_pred,
        0,
        None
    )

    mae = mean_absolute_error(
        y_true,
        y_pred_nonnegative
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred_nonnegative
        )
    )

    medae = median_absolute_error(
        y_true,
        y_pred_nonnegative
    )

    rmsle = np.sqrt(
        mean_squared_log_error(
            y_true,
            y_pred_nonnegative
        )
    )

    r2 = r2_score(
        y_true,
        y_pred_nonnegative
    )

    return {
        "model": model_name,
        "target": target_name,
        "split": split_name,
        "MAE": mae,
        "RMSE": rmse,
        "Median_AE": medae,
        "RMSLE": rmsle,
        "R2": r2
    }


baseline_results = []


# =========================================================
# 2. IMPACT BASELINE 1
#
# Persistence:
# predict that next affected population =
# current affected population.
# =========================================================

for df, split_name in [
    (validation_df, "validation"),
    (test_df, "test")
]:

    y_true = df[
        "target_next_affected_people"
    ]

    y_pred = df[
        "affected_people"
    ]

    baseline_results.append(
        regression_metrics(
            y_true,
            y_pred,
            "Impact Persistence",
            "Affected People",
            split_name
        )
    )


# =========================================================
# 3. IMPACT BASELINE 2
#
# Training-median constant baseline
# =========================================================

train_impact_median = (
    train_df[
        "target_next_affected_people"
    ].median()
)


for df, split_name in [
    (validation_df, "validation"),
    (test_df, "test")
]:

    y_true = df[
        "target_next_affected_people"
    ]

    y_pred = np.full(
        len(df),
        train_impact_median
    )

    baseline_results.append(
        regression_metrics(
            y_true,
            y_pred,
            "Impact Training Median",
            "Affected People",
            split_name
        )
    )


# =========================================================
# 4. SAFETY-CENTRE BASELINE 1
#
# Persistence:
# next sheltered population =
# current sheltered population.
# =========================================================

for df, split_name in [
    (validation_df, "validation"),
    (test_df, "test")
]:

    y_true = df[
        "target_next_people_in_safety_centres"
    ]

    y_pred = df[
        "people_in_safety_centres"
    ]

    baseline_results.append(
        regression_metrics(
            y_true,
            y_pred,
            "Safety Persistence",
            "Safety-Centre Population",
            split_name
        )
    )


# =========================================================
# 5. SAFETY-CENTRE BASELINE 2
#
# Zero-demand baseline
# Useful because the target is zero-heavy.
# =========================================================

for df, split_name in [
    (validation_df, "validation"),
    (test_df, "test")
]:

    y_true = df[
        "target_next_people_in_safety_centres"
    ]

    y_pred = np.zeros(
        len(df)
    )

    baseline_results.append(
        regression_metrics(
            y_true,
            y_pred,
            "Safety Zero Baseline",
            "Safety-Centre Population",
            split_name
        )
    )


# =========================================================
# 6. Convert regression results to table
# =========================================================

baseline_regression_df = pd.DataFrame(
    baseline_results
)


print(
    "REGRESSION BASELINE RESULTS"
)

print(
    baseline_regression_df
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 7. Escalation-risk classification baseline
#
# Predict the most common TRAINING class for every row.
# =========================================================

majority_risk_class = (
    train_df[
        "impact_escalation_risk"
    ]
    .mode()
    .iloc[0]
)


classification_results = []


for df, split_name in [
    (validation_df, "validation"),
    (test_df, "test")
]:

    y_true = df[
        "impact_escalation_risk"
    ]

    y_pred = np.full(
        len(df),
        majority_risk_class,
        dtype=object
    )

    classification_results.append(
        {
            "model":
                "Majority-Class Baseline",

            "split":
                split_name,

            "predicted_class":
                majority_risk_class,

            "accuracy":
                accuracy_score(
                    y_true,
                    y_pred
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    y_true,
                    y_pred
                ),

            "macro_f1":
                f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                ),

            "weighted_f1":
                f1_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
        }
    )


baseline_classification_df = pd.DataFrame(
    classification_results
)


print(
    "\nCLASSIFICATION BASELINE RESULTS"
)

print(
    baseline_classification_df
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 8. Baseline reference values
# =========================================================

print(
    "\nTraining target median used for impact baseline:",
    train_impact_median
)

print(
    "Training majority escalation class:",
    majority_risk_class
)

## 8.2 Core Human-Impact Regression Models

In [ ]:
# =========================================================
# 8.2 CORE Human-Impact Regression Models
#
# Compare:
# 1. Persistence baseline
# 2. Ridge
# 3. Random Forest
# 4. HistGradientBoosting
#
# Two strategies:
# A. Direct log-target prediction
# B. Residual log-change prediction
#
# MODEL SELECTION USES VALIDATION ONLY.
# TEST SET IS NOT USED HERE.
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

import numpy as np
import pandas as pd


# =========================================================
# 1. Prepare CORE feature matrices
# =========================================================

X_train_core = train_df[
    CORE_FEATURES
].copy()

X_val_core = validation_df[
    CORE_FEATURES
].copy()


# Ensure all selected features are numeric
for col in CORE_FEATURES:

    X_train_core[col] = pd.to_numeric(
        X_train_core[col],
        errors="coerce"
    )

    X_val_core[col] = pd.to_numeric(
        X_val_core[col],
        errors="coerce"
    )


y_train_impact = train_df[
    "target_next_affected_people"
].astype(float)

y_val_impact = validation_df[
    "target_next_affected_people"
].astype(float)


# =========================================================
# 2. Check for features completely missing in TRAIN
# =========================================================

all_missing_train_features = [
    col
    for col in CORE_FEATURES
    if X_train_core[col].isna().all()
]

print(
    "CORE features completely missing in training:",
    all_missing_train_features
)


# If any feature is completely missing in training,
# remove it safely from this experiment.

CORE_MODEL_FEATURES = [
    col
    for col in CORE_FEATURES
    if col not in all_missing_train_features
]

X_train_core = X_train_core[
    CORE_MODEL_FEATURES
]

X_val_core = X_val_core[
    CORE_MODEL_FEATURES
]


print(
    "CORE modelling feature count:",
    len(CORE_MODEL_FEATURES)
)


# =========================================================
# 3. Model factory
# =========================================================

def build_impact_models():

    return {

        "Ridge": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),

            (
                "scaler",
                StandardScaler()
            ),

            (
                "model",
                Ridge(
                    alpha=10.0
                )
            )
        ]),


        "Random Forest": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),

            (
                "model",
                RandomForestRegressor(
                    n_estimators=400,
                    max_depth=None,
                    min_samples_leaf=2,
                    max_features=0.7,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]),


        "HistGradientBoosting": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),

            (
                "model",
                HistGradientBoostingRegressor(
                    learning_rate=0.05,
                    max_iter=300,
                    max_leaf_nodes=31,
                    l2_regularization=1.0,
                    random_state=42
                )
            )
        ])
    }


# =========================================================
# 4. Persistence baseline on validation
# =========================================================

persistence_pred = validation_df[
    "affected_people"
].astype(float)


persistence_result = regression_metrics(
    y_val_impact,
    persistence_pred,
    "Persistence",
    "Affected People",
    "validation"
)

persistence_result["strategy"] = (
    "Naive Current = Next"
)


impact_core_results = [
    persistence_result
]


# =========================================================
# 5. DIRECT LOG-TARGET MODELS
#
# Predict:
# log1p(next affected people)
# =========================================================

direct_target_train = np.log1p(
    y_train_impact
)


impact_core_models = {}


direct_models = build_impact_models()


for model_name, model in direct_models.items():

    model.fit(
        X_train_core,
        direct_target_train
    )

    pred_log = model.predict(
        X_val_core
    )

    # Population cannot be negative
    pred_log = np.clip(
        pred_log,
        0,
        None
    )

    pred_original = np.expm1(
        pred_log
    )

    result = regression_metrics(
        y_val_impact,
        pred_original,
        model_name,
        "Affected People",
        "validation"
    )

    result["strategy"] = (
        "Direct Log Target"
    )

    impact_core_results.append(
        result
    )

    impact_core_models[
        f"{model_name}_direct"
    ] = model


# =========================================================
# 6. RESIDUAL LOG-CHANGE MODELS
#
# Predict:
#
# log1p(next affected)
# -
# log1p(current affected)
#
# Then:
#
# predicted next log impact
# =
# log1p(current impact)
# +
# predicted change
# =========================================================

train_log_change = (
    np.log1p(
        train_df[
            "target_next_affected_people"
        ].astype(float)
    )
    -
    np.log1p(
        train_df[
            "affected_people"
        ].astype(float)
    )
)


residual_models = build_impact_models()


for model_name, model in residual_models.items():

    model.fit(
        X_train_core,
        train_log_change
    )

    predicted_change = model.predict(
        X_val_core
    )


    current_val_log = np.log1p(
        validation_df[
            "affected_people"
        ].astype(float)
    )


    predicted_next_log = (
        current_val_log
        +
        predicted_change
    )


    predicted_next_log = np.clip(
        predicted_next_log,
        0,
        None
    )


    pred_original = np.expm1(
        predicted_next_log
    )


    result = regression_metrics(
        y_val_impact,
        pred_original,
        model_name,
        "Affected People",
        "validation"
    )

    result["strategy"] = (
        "Residual Log Change"
    )

    impact_core_results.append(
        result
    )

    impact_core_models[
        f"{model_name}_residual"
    ] = model


# =========================================================
# 7. Comparison table
# =========================================================

impact_core_results_df = pd.DataFrame(
    impact_core_results
)


# Reorder columns
impact_core_results_df = (
    impact_core_results_df[
        [
            "model",
            "strategy",
            "MAE",
            "RMSE",
            "Median_AE",
            "RMSLE",
            "R2"
        ]
    ]
)


# =========================================================
# 8. Improvement relative to Persistence
# =========================================================

baseline_mae = persistence_result[
    "MAE"
]

baseline_rmse = persistence_result[
    "RMSE"
]

baseline_rmsle = persistence_result[
    "RMSLE"
]


impact_core_results_df[
    "MAE_improvement_%"
] = (
    (
        baseline_mae
        -
        impact_core_results_df["MAE"]
    )
    /
    baseline_mae
    * 100
)


impact_core_results_df[
    "RMSE_improvement_%"
] = (
    (
        baseline_rmse
        -
        impact_core_results_df["RMSE"]
    )
    /
    baseline_rmse
    * 100
)


impact_core_results_df[
    "RMSLE_improvement_%"
] = (
    (
        baseline_rmsle
        -
        impact_core_results_df["RMSLE"]
    )
    /
    baseline_rmsle
    * 100
)


# =========================================================
# 9. Display results
# =========================================================

print(
    "\nCORE IMPACT MODEL - VALIDATION RESULTS"
)

print(
    impact_core_results_df
    .sort_values("RMSLE")
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 10. Best models under different metrics
# =========================================================

best_mae_row = (
    impact_core_results_df
    .loc[
        impact_core_results_df[
            "MAE"
        ].idxmin()
    ]
)

best_rmse_row = (
    impact_core_results_df
    .loc[
        impact_core_results_df[
            "RMSE"
        ].idxmin()
    ]
)

best_rmsle_row = (
    impact_core_results_df
    .loc[
        impact_core_results_df[
            "RMSLE"
        ].idxmin()
    ]
)


print(
    "\nBest by MAE:",
    best_mae_row["model"],
    "-",
    best_mae_row["strategy"]
)

print(
    "Best by RMSE:",
    best_rmse_row["model"],
    "-",
    best_rmse_row["strategy"]
)

print(
    "Best by RMSLE:",
    best_rmsle_row["model"],
    "-",
    best_rmsle_row["strategy"]
)

## 8.3 Additive Human-Impact Change Forecasting

In [ ]:
# =========================================================
# 8.3 Additive Human-Impact Change Forecasting
#
# Instead of predicting the total next population directly,
# predict the CHANGE from the current Situation Report.
# =========================================================

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor
)

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd


# =========================================================
# 1. Training target = absolute change
# =========================================================

train_delta = (
    train_df["target_next_affected_people"]
    -
    train_df["affected_people"]
).astype(float)


val_actual = (
    validation_df[
        "target_next_affected_people"
    ].astype(float)
)


val_current = (
    validation_df[
        "affected_people"
    ].astype(float)
)


# =========================================================
# 2. Signed-log transform of change
#
# Positive and negative changes are both preserved.
# =========================================================

def signed_log1p(x):

    x = np.asarray(
        x,
        dtype=float
    )

    return (
        np.sign(x)
        *
        np.log1p(
            np.abs(x)
        )
    )


def inverse_signed_log1p(x):

    x = np.asarray(
        x,
        dtype=float
    )

    return (
        np.sign(x)
        *
        np.expm1(
            np.abs(x)
        )
    )


train_delta_transformed = (
    signed_log1p(
        train_delta
    )
)


# =========================================================
# 3. Robust change-prediction models
# =========================================================

change_models = {

    "Ridge": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            Ridge(
                alpha=20.0
            )
        )
    ]),


    "Random Forest": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            RandomForestRegressor(
                n_estimators=400,
                max_depth=12,
                min_samples_leaf=5,
                max_features=0.7,
                random_state=42,
                n_jobs=-1
            )
        )
    ]),


    "HistGradientBoosting": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            HistGradientBoostingRegressor(
                loss="absolute_error",
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=15,
                min_samples_leaf=20,
                l2_regularization=5.0,
                random_state=42
            )
        )
    ]),


    "GradientBoosting Huber": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            GradientBoostingRegressor(
                loss="huber",
                n_estimators=250,
                learning_rate=0.03,
                max_depth=2,
                min_samples_leaf=10,
                random_state=42
            )
        )
    ])
}


# =========================================================
# 4. Persistence reference
# =========================================================

change_results = []


persistence_result = regression_metrics(
    val_actual,
    val_current,
    "Persistence",
    "Affected People",
    "validation"
)

persistence_result["strategy"] = (
    "Zero Change"
)

persistence_result["blend_alpha"] = 0.0

change_results.append(
    persistence_result
)


# =========================================================
# 5. Train models
#
# Then shrink predicted changes toward zero.
#
# alpha = 0  -> persistence
# alpha = 1  -> full ML change prediction
# =========================================================

ALPHA_GRID = np.arange(
    0.1,
    1.01,
    0.1
)


change_model_objects = {}


for model_name, model in change_models.items():

    model.fit(
        X_train_core,
        train_delta_transformed
    )


    transformed_delta_pred = (
        model.predict(
            X_val_core
        )
    )


    raw_delta_pred = (
        inverse_signed_log1p(
            transformed_delta_pred
        )
    )


    change_model_objects[
        model_name
    ] = model


    # -----------------------------------------------------
    # Conservative blending with persistence
    # -----------------------------------------------------

    for alpha in ALPHA_GRID:

        adjusted_delta = (
            alpha
            *
            raw_delta_pred
        )


        next_prediction = (
            val_current.to_numpy()
            +
            adjusted_delta
        )


        next_prediction = np.clip(
            next_prediction,
            0,
            None
        )


        result = regression_metrics(
            val_actual,
            next_prediction,
            model_name,
            "Affected People",
            "validation"
        )


        result["strategy"] = (
            "Additive Change"
        )

        result["blend_alpha"] = (
            round(
                float(alpha),
                2
            )
        )

        change_results.append(
            result
        )


# =========================================================
# 6. Results table
# =========================================================

change_results_df = pd.DataFrame(
    change_results
)


# Best alpha for each model using validation MAE
best_change_models = (
    change_results_df
    .sort_values("MAE")
    .groupby(
        "model",
        as_index=False
    )
    .first()
)


# =========================================================
# 7. Improvement relative to persistence
# =========================================================

PERSISTENCE_MAE = (
    persistence_result["MAE"]
)

PERSISTENCE_RMSE = (
    persistence_result["RMSE"]
)

PERSISTENCE_RMSLE = (
    persistence_result["RMSLE"]
)


best_change_models[
    "MAE_improvement_%"
] = (
    (
        PERSISTENCE_MAE
        -
        best_change_models["MAE"]
    )
    /
    PERSISTENCE_MAE
    * 100
)


best_change_models[
    "RMSE_improvement_%"
] = (
    (
        PERSISTENCE_RMSE
        -
        best_change_models["RMSE"]
    )
    /
    PERSISTENCE_RMSE
    * 100
)


best_change_models[
    "RMSLE_improvement_%"
] = (
    (
        PERSISTENCE_RMSLE
        -
        best_change_models["RMSLE"]
    )
    /
    PERSISTENCE_RMSLE
    * 100
)


# =========================================================
# 8. Display best result for each model
# =========================================================

print(
    "\nADDITIVE CHANGE MODEL - VALIDATION RESULTS"
)

print(
    best_change_models[
        [
            "model",
            "strategy",
            "blend_alpha",
            "MAE",
            "RMSE",
            "Median_AE",
            "RMSLE",
            "R2",
            "MAE_improvement_%",
            "RMSE_improvement_%",
            "RMSLE_improvement_%"
        ]
    ]
    .sort_values("MAE")
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 9. Overall best additive model
# =========================================================

best_additive_row = (
    best_change_models
    .loc[
        best_change_models[
            "MAE"
        ].idxmin()
    ]
)


print(
    "\nBest additive approach:"
)

print(
    "Model:",
    best_additive_row["model"]
)

print(
    "Blend alpha:",
    best_additive_row[
        "blend_alpha"
    ]
)

print(
    "MAE:",
    best_additive_row["MAE"]
)

print(
    "RMSE:",
    best_additive_row["RMSE"]
)

print(
    "RMSLE:",
    best_additive_row["RMSLE"]
)

print(
    "MAE improvement over persistence (%):",
    best_additive_row[
        "MAE_improvement_%"
    ]
)

In [ ]:
# =========================================================
# 8.4 Human-Impact Model Robustness by Forecast Horizon
# =========================================================

BEST_IMPACT_MODEL_NAME = "GradientBoosting Huber"
BEST_IMPACT_ALPHA = 1.0

best_impact_model = change_model_objects[
    BEST_IMPACT_MODEL_NAME
]


# =========================================================
# 1. Generate validation predictions
# =========================================================

val_change_transformed = (
    best_impact_model.predict(
        X_val_core
    )
)

val_change_pred = (
    inverse_signed_log1p(
        val_change_transformed
    )
)

val_best_pred = (
    validation_df[
        "affected_people"
    ].to_numpy(dtype=float)
    +
    BEST_IMPACT_ALPHA
    * val_change_pred
)

val_best_pred = np.clip(
    val_best_pred,
    0,
    None
)


validation_impact_results = (
    validation_df[
        [
            "availability_datetime",
            "hours_to_next_report",
            "affected_people",
            "target_next_affected_people"
        ]
    ]
    .copy()
)


validation_impact_results[
    "ml_prediction"
] = val_best_pred

validation_impact_results[
    "persistence_prediction"
] = validation_impact_results[
    "affected_people"
]


# =========================================================
# 2. Define evaluation-only horizon groups
#
# IMPORTANT:
# hours_to_next_report is NOT a model feature.
# It is used only to evaluate performance.
# =========================================================

def horizon_group(hours):

    if hours <= 12:
        return "<=12h"

    elif hours <= 24:
        return "12-24h"

    else:
        return "24-48h"


validation_impact_results[
    "horizon_group"
] = validation_impact_results[
    "hours_to_next_report"
].apply(horizon_group)


# =========================================================
# 3. Compare ML vs persistence by horizon
# =========================================================

horizon_results = []


for horizon, group in (
    validation_impact_results
    .groupby("horizon_group")
):

    y_true = group[
        "target_next_affected_people"
    ]


    ml_result = regression_metrics(
        y_true,
        group["ml_prediction"],
        "GradientBoosting Huber",
        "Affected People",
        horizon
    )

    persistence_result_h = regression_metrics(
        y_true,
        group["persistence_prediction"],
        "Persistence",
        "Affected People",
        horizon
    )


    horizon_results.append({
        "horizon": horizon,
        "reports": len(group),

        "ml_MAE":
            ml_result["MAE"],

        "persistence_MAE":
            persistence_result_h["MAE"],

        "MAE_improvement_%":
            (
                (
                    persistence_result_h["MAE"]
                    - ml_result["MAE"]
                )
                /
                persistence_result_h["MAE"]
                * 100
            ),

        "ml_RMSE":
            ml_result["RMSE"],

        "persistence_RMSE":
            persistence_result_h["RMSE"],

        "RMSE_improvement_%":
            (
                (
                    persistence_result_h["RMSE"]
                    - ml_result["RMSE"]
                )
                /
                persistence_result_h["RMSE"]
                * 100
            ),

        "ml_RMSLE":
            ml_result["RMSLE"],

        "persistence_RMSLE":
            persistence_result_h["RMSLE"]
    })


horizon_results_df = pd.DataFrame(
    horizon_results
)


print(
    "VALIDATION PERFORMANCE BY FORECAST HORIZON"
)

print(
    horizon_results_df
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 4. Error by disaster-impact magnitude
# =========================================================

def impact_band(value):

    if value < 1_000:
        return "<1K"

    elif value < 10_000:
        return "1K-10K"

    elif value < 100_000:
        return "10K-100K"

    else:
        return ">=100K"


validation_impact_results[
    "impact_band"
] = validation_impact_results[
    "target_next_affected_people"
].apply(impact_band)


magnitude_results = []


for band, group in (
    validation_impact_results
    .groupby("impact_band")
):

    y_true = group[
        "target_next_affected_people"
    ]

    ml_mae = mean_absolute_error(
        y_true,
        group["ml_prediction"]
    )

    persistence_mae = mean_absolute_error(
        y_true,
        group["persistence_prediction"]
    )


    magnitude_results.append({
        "impact_band": band,
        "reports": len(group),
        "ml_MAE": ml_mae,
        "persistence_MAE": persistence_mae,

        "MAE_improvement_%":
            (
                (
                    persistence_mae
                    - ml_mae
                )
                /
                persistence_mae
                * 100
            )
            if persistence_mae > 0
            else np.nan
    })


magnitude_results_df = pd.DataFrame(
    magnitude_results
)


print(
    "\nVALIDATION PERFORMANCE BY IMPACT MAGNITUDE"
)

print(
    magnitude_results_df
    .round(4)
    .to_string(index=False)
)

## 8.5 Safety-Centre Demand - Stage 1: Usage Classification

In [ ]:
# =========================================================
# 8.5 Safety-Centre Demand - Stage 1
# Predict whether next Situation Report will have
# positive safety-centre population
#
# VALIDATION ONLY for model selection.
# TEST SET remains untouched.
# =========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

import numpy as np
import pandas as pd


# =========================================================
# 1. Feature matrices
# =========================================================

X_train_safety = train_df[
    CORE_MODEL_FEATURES
].copy()

X_val_safety = validation_df[
    CORE_MODEL_FEATURES
].copy()


for col in CORE_MODEL_FEATURES:

    X_train_safety[col] = pd.to_numeric(
        X_train_safety[col],
        errors="coerce"
    )

    X_val_safety[col] = pd.to_numeric(
        X_val_safety[col],
        errors="coerce"
    )


y_train_safety_use = (
    train_df[
        "target_next_safety_centre_use"
    ].astype(int)
)

y_val_safety_use = (
    validation_df[
        "target_next_safety_centre_use"
    ].astype(int)
)


print("Training class distribution:")

print(
    y_train_safety_use
    .value_counts()
    .sort_index()
)

print("\nValidation class distribution:")

print(
    y_val_safety_use
    .value_counts()
    .sort_index()
)


# =========================================================
# 2. Evaluation helper
# =========================================================

def safety_classifier_metrics(
    y_true,
    y_pred,
    y_prob,
    model_name
):

    return {
        "model": model_name,

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                y_prob
            )
    }


safety_classifier_results = []


# =========================================================
# 3. Baseline 1 - Current-state persistence
#
# If people are currently in safety centres,
# predict that next report will also have usage.
# =========================================================

persistence_class_pred = (
    validation_df[
        "people_in_safety_centres"
    ] > 0
).astype(int)


# Binary probability-like baseline
persistence_class_prob = (
    persistence_class_pred
    .astype(float)
)


safety_classifier_results.append(
    safety_classifier_metrics(
        y_val_safety_use,
        persistence_class_pred,
        persistence_class_prob,
        "Usage Persistence"
    )
)


# =========================================================
# 4. Baseline 2 - Majority training class
# =========================================================

majority_safety_class = (
    y_train_safety_use
    .mode()
    .iloc[0]
)


majority_pred = np.full(
    len(y_val_safety_use),
    majority_safety_class
)

majority_prob = np.full(
    len(y_val_safety_use),
    y_train_safety_use.mean()
)


safety_classifier_results.append(
    safety_classifier_metrics(
        y_val_safety_use,
        majority_pred,
        majority_prob,
        "Majority Baseline"
    )
)


# =========================================================
# 5. Candidate classifiers
# =========================================================

safety_classifiers = {

    "Logistic Regression": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]),


    "Random Forest": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                max_depth=None,
                min_samples_leaf=3,
                max_features=0.7,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]),


    "HistGradientBoosting": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=15,
                min_samples_leaf=15,
                l2_regularization=2.0,
                random_state=42
            )
        )
    ])
}


safety_classifier_objects = {}


# =========================================================
# 6. Train and validate
# =========================================================

for model_name, model in safety_classifiers.items():

    model.fit(
        X_train_safety,
        y_train_safety_use
    )

    y_prob = model.predict_proba(
        X_val_safety
    )[:, 1]

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    result = safety_classifier_metrics(
        y_val_safety_use,
        y_pred,
        y_prob,
        model_name
    )

    safety_classifier_results.append(
        result
    )

    safety_classifier_objects[
        model_name
    ] = model


# =========================================================
# 7. Results
# =========================================================

safety_classifier_results_df = (
    pd.DataFrame(
        safety_classifier_results
    )
)


print(
    "\nSAFETY-CENTRE USAGE CLASSIFICATION - VALIDATION"
)

print(
    safety_classifier_results_df
    .sort_values(
        "f1",
        ascending=False
    )
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 8. Best ML classifier
# =========================================================

ml_only_results = (
    safety_classifier_results_df[
        ~safety_classifier_results_df[
            "model"
        ].isin(
            [
                "Usage Persistence",
                "Majority Baseline"
            ]
        )
    ]
)


best_safety_classifier_row = (
    ml_only_results
    .loc[
        ml_only_results[
            "f1"
        ].idxmax()
    ]
)


BEST_SAFETY_CLASSIFIER_NAME = (
    best_safety_classifier_row[
        "model"
    ]
)


print(
    "\nBest ML safety-usage classifier:",
    BEST_SAFETY_CLASSIFIER_NAME
)

print(
    "Validation F1:",
    best_safety_classifier_row[
        "f1"
    ]
)

print(
    "Balanced accuracy:",
    best_safety_classifier_row[
        "balanced_accuracy"
    ]
)

print(
    "ROC-AUC:",
    best_safety_classifier_row[
        "roc_auc"
    ]
)


# =========================================================
# 9. Confusion matrix
# =========================================================

best_safety_classifier = (
    safety_classifier_objects[
        BEST_SAFETY_CLASSIFIER_NAME
    ]
)

best_prob = (
    best_safety_classifier
    .predict_proba(
        X_val_safety
    )[:, 1]
)

best_pred = (
    best_prob >= 0.5
).astype(int)


print(
    "\nBest classifier confusion matrix:"
)

print(
    confusion_matrix(
        y_val_safety_use,
        best_pred
    )
)

## 8.6 Safety-Centre Demand - Stage 2: Positive Demand Regression

In [ ]:
# =========================================================
# 8.6 Safety-Centre Demand - Stage 2
#
# Stage 1:
# Predict whether next report has positive safety-centre use.
#
# Stage 2:
# Predict the amount of people in safety centres.
#
# MODEL SELECTION = VALIDATION ONLY.
# TEST SET remains untouched.
# =========================================================

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import numpy as np
import pandas as pd


# =========================================================
# 1. Positive-demand training subset
# =========================================================

train_positive_mask = (
    train_df[
        "target_next_people_in_safety_centres"
    ] > 0
)

val_positive_mask = (
    validation_df[
        "target_next_people_in_safety_centres"
    ] > 0
)


X_train_positive = (
    X_train_safety.loc[
        train_positive_mask
    ].copy()
)

X_val_positive = (
    X_val_safety.loc[
        val_positive_mask
    ].copy()
)


y_train_positive = (
    train_df.loc[
        train_positive_mask,
        "target_next_people_in_safety_centres"
    ].astype(float)
)

y_val_positive = (
    validation_df.loc[
        val_positive_mask,
        "target_next_people_in_safety_centres"
    ].astype(float)
)


current_train_positive = (
    train_df.loc[
        train_positive_mask,
        "people_in_safety_centres"
    ].astype(float)
)

current_val_positive = (
    validation_df.loc[
        val_positive_mask,
        "people_in_safety_centres"
    ].astype(float)
)


print(
    "Positive-demand training rows:",
    len(y_train_positive)
)

print(
    "Positive-demand validation rows:",
    len(y_val_positive)
)


# =========================================================
# 2. Signed-log change target
#
# next shelter population - current shelter population
# =========================================================

safety_train_delta = (
    y_train_positive
    -
    current_train_positive
)


safety_train_delta_transformed = (
    signed_log1p(
        safety_train_delta
    )
)


# =========================================================
# 3. Candidate positive-demand models
# =========================================================

safety_amount_models = {

    "Random Forest": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            RandomForestRegressor(
                n_estimators=400,
                max_depth=12,
                min_samples_leaf=4,
                max_features=0.7,
                random_state=42,
                n_jobs=-1
            )
        )
    ]),


    "HistGradientBoosting": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            HistGradientBoostingRegressor(
                loss="absolute_error",
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=15,
                min_samples_leaf=15,
                l2_regularization=3.0,
                random_state=42
            )
        )
    ]),


    "GradientBoosting Huber": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            GradientBoostingRegressor(
                loss="huber",
                n_estimators=250,
                learning_rate=0.03,
                max_depth=2,
                min_samples_leaf=8,
                random_state=42
            )
        )
    ])
}


# =========================================================
# 4. Positive-demand persistence baseline
# =========================================================

positive_amount_results = []


positive_persistence = regression_metrics(
    y_val_positive,
    current_val_positive,
    "Positive-Demand Persistence",
    "Safety-Centre Population",
    "validation-positive"
)

positive_amount_results.append(
    positive_persistence
)


# =========================================================
# 5. Train additive-change models
# =========================================================

safety_amount_model_objects = {}


for model_name, model in safety_amount_models.items():

    model.fit(
        X_train_positive,
        safety_train_delta_transformed
    )


    predicted_delta_transformed = (
        model.predict(
            X_val_positive
        )
    )


    predicted_delta = (
        inverse_signed_log1p(
            predicted_delta_transformed
        )
    )


    predicted_amount = (
        current_val_positive.to_numpy()
        +
        predicted_delta
    )


    predicted_amount = np.clip(
        predicted_amount,
        0,
        None
    )


    result = regression_metrics(
        y_val_positive,
        predicted_amount,
        model_name,
        "Safety-Centre Population",
        "validation-positive"
    )

    positive_amount_results.append(
        result
    )

    safety_amount_model_objects[
        model_name
    ] = model


# =========================================================
# 6. Positive-demand comparison
# =========================================================

positive_amount_results_df = pd.DataFrame(
    positive_amount_results
)


print(
    "\nPOSITIVE SAFETY-DEMAND REGRESSION - VALIDATION"
)

print(
    positive_amount_results_df[
        [
            "model",
            "MAE",
            "RMSE",
            "Median_AE",
            "RMSLE",
            "R2"
        ]
    ]
    .sort_values("MAE")
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 7. Select best ML amount model by validation MAE
# =========================================================

ml_positive_results = (
    positive_amount_results_df[
        positive_amount_results_df[
            "model"
        ] != "Positive-Demand Persistence"
    ]
)


best_amount_row = (
    ml_positive_results
    .loc[
        ml_positive_results[
            "MAE"
        ].idxmin()
    ]
)


BEST_SAFETY_AMOUNT_MODEL_NAME = (
    best_amount_row["model"]
)


best_safety_amount_model = (
    safety_amount_model_objects[
        BEST_SAFETY_AMOUNT_MODEL_NAME
    ]
)


print(
    "\nBest ML positive-demand model:",
    BEST_SAFETY_AMOUNT_MODEL_NAME
)


# =========================================================
# 8. Generate amount predictions for ALL validation rows
# =========================================================

all_val_delta_transformed = (
    best_safety_amount_model.predict(
        X_val_safety
    )
)

all_val_delta = (
    inverse_signed_log1p(
        all_val_delta_transformed
    )
)


all_val_amount_prediction = (
    validation_df[
        "people_in_safety_centres"
    ].to_numpy(dtype=float)
    +
    all_val_delta
)


all_val_amount_prediction = np.clip(
    all_val_amount_prediction,
    0,
    None
)


# =========================================================
# 9. Stage-1 Random Forest gating
# =========================================================

stage1_probability = (
    best_safety_classifier
    .predict_proba(
        X_val_safety
    )[:, 1]
)

stage1_prediction = (
    stage1_probability >= 0.5
).astype(int)


two_stage_prediction = np.where(
    stage1_prediction == 1,
    all_val_amount_prediction,
    0.0
)


# =========================================================
# 10. Overall two-stage validation performance
# =========================================================

y_val_safety_amount = (
    validation_df[
        "target_next_people_in_safety_centres"
    ].astype(float)
)


two_stage_result = regression_metrics(
    y_val_safety_amount,
    two_stage_prediction,
    "Two-Stage Safety Model",
    "Safety-Centre Population",
    "validation"
)


safety_persistence_result = regression_metrics(
    y_val_safety_amount,
    validation_df[
        "people_in_safety_centres"
    ].astype(float),
    "Safety Persistence",
    "Safety-Centre Population",
    "validation"
)


zero_result = regression_metrics(
    y_val_safety_amount,
    np.zeros(
        len(validation_df)
    ),
    "Zero Baseline",
    "Safety-Centre Population",
    "validation"
)


combined_safety_results_df = pd.DataFrame(
    [
        two_stage_result,
        safety_persistence_result,
        zero_result
    ]
)


# =========================================================
# 11. Improvement over persistence
# =========================================================

persistence_mae = (
    safety_persistence_result["MAE"]
)

persistence_rmse = (
    safety_persistence_result["RMSE"]
)

persistence_rmsle = (
    safety_persistence_result["RMSLE"]
)


combined_safety_results_df[
    "MAE_improvement_%"
] = (
    (
        persistence_mae
        -
        combined_safety_results_df["MAE"]
    )
    /
    persistence_mae
    * 100
)


combined_safety_results_df[
    "RMSE_improvement_%"
] = (
    (
        persistence_rmse
        -
        combined_safety_results_df["RMSE"]
    )
    /
    persistence_rmse
    * 100
)


combined_safety_results_df[
    "RMSLE_improvement_%"
] = (
    (
        persistence_rmsle
        -
        combined_safety_results_df["RMSLE"]
    )
    /
    persistence_rmsle
    * 100
)


print(
    "\nFINAL TWO-STAGE SAFETY MODEL - VALIDATION"
)

print(
    combined_safety_results_df[
        [
            "model",
            "MAE",
            "RMSE",
            "Median_AE",
            "RMSLE",
            "R2",
            "MAE_improvement_%",
            "RMSE_improvement_%",
            "RMSLE_improvement_%"
        ]
    ]
    .sort_values("MAE")
    .round(4)
    .to_string(index=False)
)

## 8.7 Impact Escalation Risk Classification

In [ ]:
# =========================================================
# 8.7 Impact Escalation Risk Classification
#
# Predict:
# Low / Moderate / High / Critical
#
# MODEL SELECTION USES VALIDATION ONLY.
# TEST SET remains untouched.
# =========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import numpy as np
import pandas as pd


# =========================================================
# 1. Prepare CORE features
# =========================================================

X_train_risk = train_df[
    CORE_MODEL_FEATURES
].copy()

X_val_risk = validation_df[
    CORE_MODEL_FEATURES
].copy()


for col in CORE_MODEL_FEATURES:

    X_train_risk[col] = pd.to_numeric(
        X_train_risk[col],
        errors="coerce"
    )

    X_val_risk[col] = pd.to_numeric(
        X_val_risk[col],
        errors="coerce"
    )


y_train_risk = (
    train_df[
        "impact_escalation_risk"
    ]
    .astype(str)
)

y_val_risk = (
    validation_df[
        "impact_escalation_risk"
    ]
    .astype(str)
)


RISK_ORDER = [
    "Low",
    "Moderate",
    "High",
    "Critical"
]


print(
    "Training risk distribution:"
)

print(
    y_train_risk.value_counts()
    .reindex(RISK_ORDER)
)


print(
    "\nValidation risk distribution:"
)

print(
    y_val_risk.value_counts()
    .reindex(RISK_ORDER)
)


# =========================================================
# 2. Evaluation helper
# =========================================================

def risk_metrics(
    y_true,
    y_pred,
    model_name
):

    return {
        "model": model_name,

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            )
    }


risk_results = []


# =========================================================
# 3. Majority-class baseline
# =========================================================

majority_risk_class = (
    y_train_risk
    .mode()
    .iloc[0]
)


majority_risk_pred = np.full(
    len(y_val_risk),
    majority_risk_class,
    dtype=object
)


risk_results.append(
    risk_metrics(
        y_val_risk,
        majority_risk_pred,
        "Majority Baseline"
    )
)


# =========================================================
# 4. Candidate classifiers
# =========================================================

risk_models = {

    "Logistic Regression": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]),


    "Random Forest": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=3,
                max_features=0.7,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]),


    "Extra Trees": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            ExtraTreesClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=3,
                max_features=0.7,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]),


    "HistGradientBoosting": Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=15,
                min_samples_leaf=15,
                l2_regularization=3.0,
                class_weight="balanced",
                random_state=42
            )
        )
    ])
}


risk_model_objects = {}


# =========================================================
# 5. Train + validate
# =========================================================

for model_name, model in risk_models.items():

    model.fit(
        X_train_risk,
        y_train_risk
    )

    y_pred = model.predict(
        X_val_risk
    )

    result = risk_metrics(
        y_val_risk,
        y_pred,
        model_name
    )

    risk_results.append(
        result
    )

    risk_model_objects[
        model_name
    ] = model


# =========================================================
# 6. Comparison
# =========================================================

risk_results_df = pd.DataFrame(
    risk_results
)


print(
    "\nESCALATION-RISK CLASSIFICATION - VALIDATION"
)

print(
    risk_results_df
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 7. Select best ML model
#
# Primary metric = Macro F1
# because Critical/High classes are less frequent.
# =========================================================

ml_risk_results = (
    risk_results_df[
        risk_results_df["model"]
        != "Majority Baseline"
    ]
)


best_risk_row = (
    ml_risk_results
    .loc[
        ml_risk_results[
            "macro_f1"
        ].idxmax()
    ]
)


BEST_RISK_MODEL_NAME = (
    best_risk_row["model"]
)

best_risk_model = (
    risk_model_objects[
        BEST_RISK_MODEL_NAME
    ]
)


print(
    "\nBest escalation-risk model:",
    BEST_RISK_MODEL_NAME
)

print(
    "Macro F1:",
    best_risk_row["macro_f1"]
)

print(
    "Balanced accuracy:",
    best_risk_row[
        "balanced_accuracy"
    ]
)

print(
    "Weighted F1:",
    best_risk_row[
        "weighted_f1"
    ]
)


# =========================================================
# 8. Detailed performance
# =========================================================

best_risk_pred = (
    best_risk_model.predict(
        X_val_risk
    )
)


print(
    "\nClassification report:"
)

print(
    classification_report(
        y_val_risk,
        best_risk_pred,
        labels=RISK_ORDER,
        zero_division=0
    )
)


print(
    "\nConfusion matrix:"
)

risk_confusion = confusion_matrix(
    y_val_risk,
    best_risk_pred,
    labels=RISK_ORDER
)

print(
    pd.DataFrame(
        risk_confusion,
        index=[
            f"Actual_{x}"
            for x in RISK_ORDER
        ],
        columns=[
            f"Pred_{x}"
            for x in RISK_ORDER
        ]
    )
)

## 8.8 Incremental Value of River Features

In [ ]:
# =========================================================
# 8.8 Controlled River-Feature Value Experiment
#
# Same algorithm + same rows + same target.
#
# CORE:
# Situation + Weather + Landslide + Temporal
#
# FULL:
# CORE + River
#
# TEST SET remains untouched.
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

import numpy as np
import pandas as pd


# =========================================================
# 1. Remove any feature completely missing in TRAIN
# =========================================================

full_all_missing_train = [
    col
    for col in FULL_MULTISOURCE_FEATURES
    if train_df[col].isna().all()
]

FULL_MODEL_FEATURES = [
    col
    for col in FULL_MULTISOURCE_FEATURES
    if col not in full_all_missing_train
]


print(
    "FULL features completely missing in training:",
    full_all_missing_train
)

print(
    "FULL modelling feature count:",
    len(FULL_MODEL_FEATURES)
)


# =========================================================
# 2. Prepare CORE and FULL matrices
# =========================================================

X_train_core_compare = train_df[
    CORE_MODEL_FEATURES
].copy()

X_val_core_compare = validation_df[
    CORE_MODEL_FEATURES
].copy()


X_train_full = train_df[
    FULL_MODEL_FEATURES
].copy()

X_val_full = validation_df[
    FULL_MODEL_FEATURES
].copy()


for col in CORE_MODEL_FEATURES:

    X_train_core_compare[col] = pd.to_numeric(
        X_train_core_compare[col],
        errors="coerce"
    )

    X_val_core_compare[col] = pd.to_numeric(
        X_val_core_compare[col],
        errors="coerce"
    )


for col in FULL_MODEL_FEATURES:

    X_train_full[col] = pd.to_numeric(
        X_train_full[col],
        errors="coerce"
    )

    X_val_full[col] = pd.to_numeric(
        X_val_full[col],
        errors="coerce"
    )


# =========================================================
# 3. Same additive-change target
# =========================================================

train_change = (
    train_df[
        "target_next_affected_people"
    ]
    -
    train_df[
        "affected_people"
    ]
).astype(float)


train_change_transformed = (
    signed_log1p(
        train_change
    )
)


y_val_actual = validation_df[
    "target_next_affected_people"
].astype(float)


val_current = validation_df[
    "affected_people"
].astype(float)


# =========================================================
# 4. Identical model architecture
# =========================================================

def build_huber_change_model():

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            GradientBoostingRegressor(
                loss="huber",
                n_estimators=250,
                learning_rate=0.03,
                max_depth=2,
                min_samples_leaf=10,
                random_state=42
            )
        )
    ])


core_compare_model = (
    build_huber_change_model()
)

full_compare_model = (
    build_huber_change_model()
)


# =========================================================
# 5. Train
# =========================================================

core_compare_model.fit(
    X_train_core_compare,
    train_change_transformed
)

full_compare_model.fit(
    X_train_full,
    train_change_transformed
)


# =========================================================
# 6. Validation predictions
# =========================================================

core_delta = inverse_signed_log1p(
    core_compare_model.predict(
        X_val_core_compare
    )
)

full_delta = inverse_signed_log1p(
    full_compare_model.predict(
        X_val_full
    )
)


core_prediction = np.clip(
    val_current.to_numpy()
    + core_delta,
    0,
    None
)

full_prediction = np.clip(
    val_current.to_numpy()
    + full_delta,
    0,
    None
)

persistence_prediction = (
    val_current.to_numpy()
)


# =========================================================
# 7. Overall validation comparison
# =========================================================

river_compare_results = []


for name, prediction in [
    (
        "Persistence",
        persistence_prediction
    ),
    (
        "CORE GradientBoosting Huber",
        core_prediction
    ),
    (
        "FULL + River GradientBoosting Huber",
        full_prediction
    )
]:

    result = regression_metrics(
        y_val_actual,
        prediction,
        name,
        "Affected People",
        "validation"
    )

    river_compare_results.append(
        result
    )


river_compare_df = pd.DataFrame(
    river_compare_results
)


print(
    "\nOVERALL VALIDATION - CORE vs FULL + RIVER"
)

print(
    river_compare_df[
        [
            "model",
            "MAE",
            "RMSE",
            "Median_AE",
            "RMSLE",
            "R2"
        ]
    ]
    .sort_values("MAE")
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 8. Evaluate only rows where an actual River report
# existed within the preceding 24 hours
# =========================================================

river_context_mask = (
    validation_df[
        "river_reports_24h"
    ] > 0
)


print(
    "\nValidation rows with actual River report in previous 24h:",
    int(
        river_context_mask.sum()
    )
)


if river_context_mask.sum() > 0:

    y_river_context = (
        y_val_actual[
            river_context_mask
        ]
    )


    context_results = []


    for name, prediction in [
        (
            "Persistence",
            persistence_prediction[
                river_context_mask.to_numpy()
            ]
        ),
        (
            "CORE GradientBoosting Huber",
            core_prediction[
                river_context_mask.to_numpy()
            ]
        ),
        (
            "FULL + River GradientBoosting Huber",
            full_prediction[
                river_context_mask.to_numpy()
            ]
        )
    ]:

        result = regression_metrics(
            y_river_context,
            prediction,
            name,
            "Affected People",
            "validation-river-context"
        )

        context_results.append(
            result
        )


    river_context_results_df = pd.DataFrame(
        context_results
    )


    print(
        "\nRIVER-CONTEXT VALIDATION RESULTS"
    )

    print(
        river_context_results_df[
            [
                "model",
                "MAE",
                "RMSE",
                "Median_AE",
                "RMSLE",
                "R2"
            ]
        ]
        .sort_values("MAE")
        .round(4)
        .to_string(index=False)
    )


# =========================================================
# 9. River incremental improvement
# =========================================================

core_mae = river_compare_df.loc[
    river_compare_df["model"]
    == "CORE GradientBoosting Huber",
    "MAE"
].iloc[0]


full_mae = river_compare_df.loc[
    river_compare_df["model"]
    == "FULL + River GradientBoosting Huber",
    "MAE"
].iloc[0]


print(
    "\nOverall MAE change from adding River features (%):",
    (
        (core_mae - full_mae)
        /
        core_mae
        * 100
    )
)

## 8.9 Time-Series Cross-Validation for Escalation-Risk Model

In [ ]:
# =========================================================
# 8.9 Time-Aware Tuning of Impact Escalation Risk Model
#
# Hyperparameters are selected using TRAINING DATA ONLY
# with TimeSeriesSplit.
#
# Validation set is used only after tuning.
# =========================================================

from sklearn.model_selection import (
    TimeSeriesSplit,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import (
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    accuracy_score
)

import pandas as pd


# =========================================================
# 1. Time-aware CV
# =========================================================

time_cv = TimeSeriesSplit(
    n_splits=4
)


# =========================================================
# 2. Pipeline
# =========================================================

risk_tuning_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        HistGradientBoostingClassifier(
            class_weight="balanced",
            random_state=42
        )
    )
])


# =========================================================
# 3. Small, controlled hyperparameter grid
# =========================================================

risk_param_grid = {

    "model__learning_rate": [
        0.03,
        0.05,
        0.08
    ],

    "model__max_leaf_nodes": [
        7,
        15
    ],

    "model__min_samples_leaf": [
        10,
        20
    ],

    "model__l2_regularization": [
        1.0,
        3.0
    ],

    "model__max_iter": [
        200,
        300
    ]
}


# =========================================================
# 4. Grid search using Macro-F1
# =========================================================

risk_grid_search = GridSearchCV(
    estimator=risk_tuning_pipeline,
    param_grid=risk_param_grid,
    scoring="f1_macro",
    cv=time_cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)


risk_grid_search.fit(
    X_train_risk,
    y_train_risk
)


print(
    "\nBest TimeSeries CV Macro-F1:",
    risk_grid_search.best_score_
)

print(
    "\nBest parameters:"
)

print(
    risk_grid_search.best_params_
)


# =========================================================
# 5. Evaluate tuned model on untouched validation set
# =========================================================

tuned_risk_model = (
    risk_grid_search.best_estimator_
)


tuned_risk_pred = (
    tuned_risk_model.predict(
        X_val_risk
    )
)


tuned_accuracy = accuracy_score(
    y_val_risk,
    tuned_risk_pred
)

tuned_balanced_accuracy = (
    balanced_accuracy_score(
        y_val_risk,
        tuned_risk_pred
    )
)

tuned_macro_f1 = f1_score(
    y_val_risk,
    tuned_risk_pred,
    average="macro",
    zero_division=0
)

tuned_weighted_f1 = f1_score(
    y_val_risk,
    tuned_risk_pred,
    average="weighted",
    zero_division=0
)


print(
    "\nTUNED RISK MODEL - VALIDATION"
)

print(
    "Accuracy:",
    tuned_accuracy
)

print(
    "Balanced Accuracy:",
    tuned_balanced_accuracy
)

print(
    "Macro F1:",
    tuned_macro_f1
)

print(
    "Weighted F1:",
    tuned_weighted_f1
)


# =========================================================
# 6. Compare against previous model
# =========================================================

print(
    "\nPrevious Macro F1:",
    best_risk_row["macro_f1"]
)

print(
    "Tuned Macro F1:",
    tuned_macro_f1
)

print(
    "Macro F1 improvement:",
    tuned_macro_f1
    - best_risk_row["macro_f1"]
)


# =========================================================
# 7. Detailed classification report
# =========================================================

print(
    "\nClassification report:"
)

print(
    classification_report(
        y_val_risk,
        tuned_risk_pred,
        labels=RISK_ORDER,
        zero_division=0
    )
)


# =========================================================
# 8. Confusion matrix
# =========================================================

tuned_confusion = confusion_matrix(
    y_val_risk,
    tuned_risk_pred,
    labels=RISK_ORDER
)


print(
    "\nConfusion matrix:"
)

print(
    pd.DataFrame(
        tuned_confusion,

        index=[
            f"Actual_{x}"
            for x in RISK_ORDER
        ],

        columns=[
            f"Pred_{x}"
            for x in RISK_ORDER
        ]
    )
)

## 8.10 Regression-Derived Impact Escalation Risk

In [ ]:
# =========================================================
# 8.10 Regression-Derived Impact Escalation Risk
#
# Use predicted affected-population CHANGE from the
# selected human-impact model and convert that prediction
# into the same research-derived escalation classes.
#
# VALIDATION ONLY.
# TEST SET remains untouched.
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import numpy as np
import pandas as pd


# =========================================================
# 1. Final TRAIN-derived thresholds from Section 7.5
# =========================================================

MODERATE_HIGH_THRESHOLD = 142.0
HIGH_CRITICAL_THRESHOLD = 5184.65


# =========================================================
# 2. Convert predicted change into risk category
# =========================================================

def predicted_change_to_risk(change):

    if change <= 0:
        return "Low"

    elif change <= MODERATE_HIGH_THRESHOLD:
        return "Moderate"

    elif change <= HIGH_CRITICAL_THRESHOLD:
        return "High"

    else:
        return "Critical"


# val_change_pred already comes from the selected
# GradientBoosting Huber additive-change model.

regression_derived_risk_pred = np.array(
    [
        predicted_change_to_risk(x)
        for x in val_change_pred
    ],
    dtype=object
)


# =========================================================
# 3. Evaluate regression-derived risk
# =========================================================

reg_risk_accuracy = accuracy_score(
    y_val_risk,
    regression_derived_risk_pred
)

reg_risk_balanced_accuracy = balanced_accuracy_score(
    y_val_risk,
    regression_derived_risk_pred
)

reg_risk_macro_f1 = f1_score(
    y_val_risk,
    regression_derived_risk_pred,
    average="macro",
    zero_division=0
)

reg_risk_weighted_f1 = f1_score(
    y_val_risk,
    regression_derived_risk_pred,
    average="weighted",
    zero_division=0
)


# =========================================================
# 4. Compare all three approaches
# =========================================================

risk_final_comparison = pd.DataFrame([
    {
        "model":
            "Original HistGradientBoosting",

        "accuracy":
            best_risk_row["accuracy"],

        "balanced_accuracy":
            best_risk_row["balanced_accuracy"],

        "macro_f1":
            best_risk_row["macro_f1"],

        "weighted_f1":
            best_risk_row["weighted_f1"]
    },

    {
        "model":
            "Tuned HistGradientBoosting",

        "accuracy":
            tuned_accuracy,

        "balanced_accuracy":
            tuned_balanced_accuracy,

        "macro_f1":
            tuned_macro_f1,

        "weighted_f1":
            tuned_weighted_f1
    },

    {
        "model":
            "Regression-Derived Risk",

        "accuracy":
            reg_risk_accuracy,

        "balanced_accuracy":
            reg_risk_balanced_accuracy,

        "macro_f1":
            reg_risk_macro_f1,

        "weighted_f1":
            reg_risk_weighted_f1
    }
])


print(
    "FINAL RISK APPROACH COMPARISON - VALIDATION"
)

print(
    risk_final_comparison
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 5. Detailed regression-derived classification report
# =========================================================

print(
    "\nREGRESSION-DERIVED RISK CLASSIFICATION REPORT"
)

print(
    classification_report(
        y_val_risk,
        regression_derived_risk_pred,
        labels=RISK_ORDER,
        zero_division=0
    )
)


# =========================================================
# 6. Confusion matrix
# =========================================================

regression_risk_confusion = confusion_matrix(
    y_val_risk,
    regression_derived_risk_pred,
    labels=RISK_ORDER
)


print(
    "\nREGRESSION-DERIVED RISK CONFUSION MATRIX"
)

print(
    pd.DataFrame(
        regression_risk_confusion,

        index=[
            f"Actual_{x}"
            for x in RISK_ORDER
        ],

        columns=[
            f"Pred_{x}"
            for x in RISK_ORDER
        ]
    )
)

## 8.11 Final Model Lock and Untouched Test Evaluation

In [ ]:
# =========================================================
# 8.11 FINAL MODEL LOCK + UNTOUCHED TEST EVALUATION
#
# Model selection is COMPLETE.
#
# Final selected approaches:
#
# Human Impact:
# GradientBoosting Huber additive-change model
#
# Safety Demand Stage 1:
# Random Forest classifier
#
# Safety Demand Stage 2:
# GradientBoosting Huber additive-change model
#
# Escalation Risk:
# Original HistGradientBoosting classifier
#
# Models are now refitted using TRAIN + VALIDATION,
# then evaluated ONCE on the untouched TEST set.
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import numpy as np
import pandas as pd


# =========================================================
# 1. Combine TRAIN + VALIDATION
# =========================================================

development_df = pd.concat(
    [
        train_df,
        validation_df
    ],
    axis=0
).copy()


print(
    "Development rows (Train + Validation):",
    len(development_df)
)

print(
    "Untouched Test rows:",
    len(test_df)
)


# =========================================================
# 2. Prepare CORE features
# =========================================================

X_dev = development_df[
    CORE_MODEL_FEATURES
].copy()

X_test = test_df[
    CORE_MODEL_FEATURES
].copy()


for col in CORE_MODEL_FEATURES:

    X_dev[col] = pd.to_numeric(
        X_dev[col],
        errors="coerce"
    )

    X_test[col] = pd.to_numeric(
        X_test[col],
        errors="coerce"
    )


# =========================================================
# PART A
# FINAL HUMAN-IMPACT FORECAST MODEL
# =========================================================

dev_impact_delta = (
    development_df[
        "target_next_affected_people"
    ].astype(float)
    -
    development_df[
        "affected_people"
    ].astype(float)
)


dev_impact_delta_transformed = (
    signed_log1p(
        dev_impact_delta
    )
)


final_impact_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            random_state=42
        )
    )
])


final_impact_model.fit(
    X_dev,
    dev_impact_delta_transformed
)


test_impact_delta_transformed = (
    final_impact_model.predict(
        X_test
    )
)


test_impact_delta = (
    inverse_signed_log1p(
        test_impact_delta_transformed
    )
)


test_current_impact = (
    test_df[
        "affected_people"
    ].to_numpy(dtype=float)
)


test_impact_prediction = np.clip(
    test_current_impact
    +
    test_impact_delta,
    0,
    None
)


test_impact_actual = (
    test_df[
        "target_next_affected_people"
    ].astype(float)
)


impact_test_result = regression_metrics(
    test_impact_actual,
    test_impact_prediction,
    "Final Impact Model",
    "Affected People",
    "test"
)


impact_persistence_test = regression_metrics(
    test_impact_actual,
    test_current_impact,
    "Impact Persistence",
    "Affected People",
    "test"
)


impact_test_comparison = pd.DataFrame(
    [
        impact_test_result,
        impact_persistence_test
    ]
)


impact_persistence_mae = (
    impact_persistence_test["MAE"]
)


impact_test_comparison[
    "MAE_improvement_over_persistence_%"
] = (
    (
        impact_persistence_mae
        -
        impact_test_comparison["MAE"]
    )
    /
    impact_persistence_mae
    * 100
)


print(
    "\n========================================"
)

print(
    "FINAL HUMAN-IMPACT TEST RESULTS"
)

print(
    "========================================"
)


print(
    impact_test_comparison[
        [
            "model",
            "MAE",
            "RMSE",
            "Median_AE",
            "RMSLE",
            "R2",
            "MAE_improvement_over_persistence_%"
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# =========================================================
# PART B
# FINAL SAFETY-CENTRE USAGE CLASSIFIER
# =========================================================

y_dev_safety_use = (
    development_df[
        "target_next_safety_centre_use"
    ].astype(int)
)


y_test_safety_use = (
    test_df[
        "target_next_safety_centre_use"
    ].astype(int)
)


final_safety_classifier = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=3,
            max_features=0.7,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])


final_safety_classifier.fit(
    X_dev,
    y_dev_safety_use
)


test_safety_probability = (
    final_safety_classifier
    .predict_proba(
        X_test
    )[:, 1]
)


test_safety_use_prediction = (
    test_safety_probability >= 0.5
).astype(int)


print(
    "\n========================================"
)

print(
    "FINAL SAFETY-USAGE TEST RESULTS"
)

print(
    "========================================"
)


print(
    "Accuracy:",
    accuracy_score(
        y_test_safety_use,
        test_safety_use_prediction
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_safety_use,
        test_safety_use_prediction
    )
)

print(
    "F1:",
    f1_score(
        y_test_safety_use,
        test_safety_use_prediction,
        zero_division=0
    )
)


print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_test_safety_use,
        test_safety_use_prediction
    )
)


# =========================================================
# PART C
# FINAL SAFETY-CENTRE AMOUNT MODEL
# =========================================================

dev_positive_mask = (
    development_df[
        "target_next_people_in_safety_centres"
    ] > 0
)


X_dev_positive = (
    X_dev.loc[
        dev_positive_mask
    ]
)


dev_safety_actual_positive = (
    development_df.loc[
        dev_positive_mask,
        "target_next_people_in_safety_centres"
    ].astype(float)
)


dev_safety_current_positive = (
    development_df.loc[
        dev_positive_mask,
        "people_in_safety_centres"
    ].astype(float)
)


dev_safety_delta = (
    dev_safety_actual_positive
    -
    dev_safety_current_positive
)


dev_safety_delta_transformed = (
    signed_log1p(
        dev_safety_delta
    )
)


final_safety_amount_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=8,
            random_state=42
        )
    )
])


final_safety_amount_model.fit(
    X_dev_positive,
    dev_safety_delta_transformed
)


test_safety_delta_transformed = (
    final_safety_amount_model.predict(
        X_test
    )
)


test_safety_delta = (
    inverse_signed_log1p(
        test_safety_delta_transformed
    )
)


test_current_safety_population = (
    test_df[
        "people_in_safety_centres"
    ].to_numpy(dtype=float)
)


test_safety_amount_prediction = np.clip(
    test_current_safety_population
    +
    test_safety_delta,
    0,
    None
)


# Stage-1 gating
final_two_stage_safety_prediction = (
    np.where(
        test_safety_use_prediction == 1,
        test_safety_amount_prediction,
        0.0
    )
)


test_safety_actual = (
    test_df[
        "target_next_people_in_safety_centres"
    ].astype(float)
)


final_safety_test_result = regression_metrics(
    test_safety_actual,
    final_two_stage_safety_prediction,
    "Final Two-Stage Safety Model",
    "Safety-Centre Population",
    "test"
)


safety_persistence_test = regression_metrics(
    test_safety_actual,
    test_current_safety_population,
    "Safety Persistence",
    "Safety-Centre Population",
    "test"
)


safety_test_comparison = pd.DataFrame(
    [
        final_safety_test_result,
        safety_persistence_test
    ]
)


safety_test_comparison[
    "MAE_improvement_over_persistence_%"
] = (
    (
        safety_persistence_test["MAE"]
        -
        safety_test_comparison["MAE"]
    )
    /
    safety_persistence_test["MAE"]
    * 100
)


print(
    "\n========================================"
)

print(
    "FINAL SAFETY-DEMAND TEST RESULTS"
)

print(
    "========================================"
)


print(
    safety_test_comparison[
        [
            "model",
            "MAE",
            "RMSE",
            "Median_AE",
            "RMSLE",
            "R2",
            "MAE_improvement_over_persistence_%"
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# =========================================================
# PART D
# FINAL IMPACT ESCALATION RISK MODEL
# =========================================================

y_dev_risk = (
    development_df[
        "impact_escalation_risk"
    ].astype(str)
)


y_test_risk = (
    test_df[
        "impact_escalation_risk"
    ].astype(str)
)


final_risk_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=15,
            min_samples_leaf=15,
            l2_regularization=3.0,
            class_weight="balanced",
            random_state=42
        )
    )
])


final_risk_model.fit(
    X_dev,
    y_dev_risk
)


test_risk_prediction = (
    final_risk_model.predict(
        X_test
    )
)


risk_test_accuracy = accuracy_score(
    y_test_risk,
    test_risk_prediction
)


risk_test_balanced_accuracy = (
    balanced_accuracy_score(
        y_test_risk,
        test_risk_prediction
    )
)


risk_test_macro_f1 = f1_score(
    y_test_risk,
    test_risk_prediction,
    average="macro",
    zero_division=0
)


risk_test_weighted_f1 = f1_score(
    y_test_risk,
    test_risk_prediction,
    average="weighted",
    zero_division=0
)


print(
    "\n========================================"
)

print(
    "FINAL ESCALATION-RISK TEST RESULTS"
)

print(
    "========================================"
)


print(
    "Accuracy:",
    risk_test_accuracy
)

print(
    "Balanced Accuracy:",
    risk_test_balanced_accuracy
)

print(
    "Macro F1:",
    risk_test_macro_f1
)

print(
    "Weighted F1:",
    risk_test_weighted_f1
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test_risk,
        test_risk_prediction,
        labels=RISK_ORDER,
        zero_division=0
    )
)


print(
    "\nConfusion Matrix:"
)

print(
    pd.DataFrame(
        confusion_matrix(
            y_test_risk,
            test_risk_prediction,
            labels=RISK_ORDER
        ),

        index=[
            f"Actual_{x}"
            for x in RISK_ORDER
        ],

        columns=[
            f"Pred_{x}"
            for x in RISK_ORDER
        ]
    )
)

In [ ]:
# =========================================================
# 9.1 Human-Impact Prediction Uncertainty
#
# Fixed 90% prediction interval
# Lower = 5th percentile
# Upper = 95th percentile
#
# No test-based tuning.
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

import numpy as np
import pandas as pd


# =========================================================
# 1. Development target
# =========================================================

dev_impact_change = (
    development_df["target_next_affected_people"].astype(float)
    -
    development_df["affected_people"].astype(float)
)

dev_impact_change_log = signed_log1p(
    dev_impact_change
)


# =========================================================
# 2. Quantile-model factory
# =========================================================

def build_quantile_model(alpha):

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            GradientBoostingRegressor(
                loss="quantile",
                alpha=alpha,
                n_estimators=250,
                learning_rate=0.03,
                max_depth=2,
                min_samples_leaf=10,
                random_state=42
            )
        )
    ])


# =========================================================
# 3. Fixed lower and upper models
# =========================================================

lower_90_model = build_quantile_model(
    0.05
)

upper_90_model = build_quantile_model(
    0.95
)


lower_90_model.fit(
    X_dev,
    dev_impact_change_log
)

upper_90_model.fit(
    X_dev,
    dev_impact_change_log
)


# =========================================================
# 4. Predict change limits
# =========================================================

lower_change_log = lower_90_model.predict(
    X_test
)

upper_change_log = upper_90_model.predict(
    X_test
)


lower_change = inverse_signed_log1p(
    lower_change_log
)

upper_change = inverse_signed_log1p(
    upper_change_log
)


current_test_impact = test_df[
    "affected_people"
].to_numpy(dtype=float)


lower_bound_raw = np.clip(
    current_test_impact + lower_change,
    0,
    None
)

upper_bound_raw = np.clip(
    current_test_impact + upper_change,
    0,
    None
)


# Guarantee correct ordering
prediction_lower_90 = np.minimum(
    lower_bound_raw,
    upper_bound_raw
)

prediction_upper_90 = np.maximum(
    lower_bound_raw,
    upper_bound_raw
)


# =========================================================
# 5. Actual test target
# =========================================================

actual_next_impact = test_df[
    "target_next_affected_people"
].to_numpy(dtype=float)


# Final locked point prediction from Section 8.11
point_prediction = test_impact_prediction


# =========================================================
# 6. Interval evaluation
# =========================================================

inside_interval = (
    (actual_next_impact >= prediction_lower_90)
    &
    (actual_next_impact <= prediction_upper_90)
)


empirical_coverage = inside_interval.mean()


interval_width = (
    prediction_upper_90
    -
    prediction_lower_90
)


print(
    "HUMAN-IMPACT 90% PREDICTION INTERVAL"
)

print(
    "Test observations:",
    len(test_df)
)

print(
    "Empirical coverage:",
    round(empirical_coverage, 4)
)

print(
    "Average interval width:",
    round(interval_width.mean(), 2)
)

print(
    "Median interval width:",
    round(
        np.median(interval_width),
        2
    )
)


# =========================================================
# 7. Forecast table
# =========================================================

impact_uncertainty_df = pd.DataFrame({

    "availability_datetime":
        test_df[
            "availability_datetime"
        ].to_numpy(),

    "current_affected_people":
        current_test_impact,

    "actual_next_affected_people":
        actual_next_impact,

    "predicted_next_affected_people":
        point_prediction,

    "prediction_lower_90":
        prediction_lower_90,

    "prediction_upper_90":
        prediction_upper_90,

    "interval_width":
        interval_width,

    "actual_inside_interval":
        inside_interval
})


print(
    "\nSAMPLE FORECASTS WITH UNCERTAINTY"
)

print(
    impact_uncertainty_df[
        [
            "availability_datetime",
            "current_affected_people",
            "actual_next_affected_people",
            "predicted_next_affected_people",
            "prediction_lower_90",
            "prediction_upper_90",
            "actual_inside_interval"
        ]
    ]
    .tail(10)
    .round(2)
    .to_string(index=False)
)

### Prediction Interval Finding

The fixed 90% model-based prediction interval achieved an empirical coverage of 80.67% on the chronologically held-out test period. This indicates under-coverage during unseen temporal conditions. Therefore, the interval is presented as a model-based uncertainty estimate rather than a guaranteed 90% confidence range. No interval parameters were retuned using the test set.

# 10. Explainable AI

## 10.1 SHAP Explanation for Human-Impact Forecasting

In [ ]:
# =========================================================
# 10.1 SHAP EXPLANATION
#
# Explains the FINAL human-impact GradientBoosting model.
#
# IMPORTANT:
# The model predicts signed-log CHANGE in affected people.
# Therefore SHAP values explain the predicted CHANGE,
# not the absolute affected population directly.
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 1. Import SHAP
# =========================================================

try:
    import shap

except ImportError:
    raise ImportError(
        "SHAP is not installed. Run: pip install shap"
    )


# =========================================================
# 2. Extract fitted preprocessing + model
# =========================================================

final_impact_imputer = (
    final_impact_model
    .named_steps["imputer"]
)

final_impact_gb_model = (
    final_impact_model
    .named_steps["model"]
)


# =========================================================
# 3. Transform TEST features exactly as model sees them
# =========================================================

X_test_shap = (
    final_impact_imputer
    .transform(
        X_test
    )
)


# =========================================================
# 4. Get transformed feature names
#
# Includes missing-value indicator features automatically.
# =========================================================

shap_feature_names = (
    final_impact_imputer
    .get_feature_names_out(
        CORE_MODEL_FEATURES
    )
)


print(
    "SHAP feature count:",
    len(shap_feature_names)
)


# =========================================================
# 5. Tree SHAP
# =========================================================

impact_explainer = shap.TreeExplainer(
    final_impact_gb_model
)


impact_shap_values = (
    impact_explainer
    .shap_values(
        X_test_shap
    )
)


impact_shap_values = np.asarray(
    impact_shap_values
)


# =========================================================
# 6. Global SHAP importance table
# =========================================================

mean_absolute_shap = np.mean(
    np.abs(
        impact_shap_values
    ),
    axis=0
)


shap_importance_df = pd.DataFrame({

    "feature":
        shap_feature_names,

    "mean_abs_shap":
        mean_absolute_shap
})


shap_importance_df = (
    shap_importance_df
    .sort_values(
        "mean_abs_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\nTOP 20 HUMAN-IMPACT SHAP FEATURES"
)

print(
    shap_importance_df
    .head(20)
    .round(6)
    .to_string(index=False)
)


# =========================================================
# 7. Create SHAP Explanation object
# =========================================================

expected_value = np.asarray(
    impact_explainer.expected_value
).reshape(-1)[0]


impact_shap_explanation = shap.Explanation(

    values=
        impact_shap_values,

    base_values=
        np.repeat(
            expected_value,
            len(X_test_shap)
        ),

    data=
        X_test_shap,

    feature_names=
        list(
            shap_feature_names
        )
)


# =========================================================
# 8. Global BAR plot
#
# Shows overall most influential features.
# =========================================================

print(
    "\nGenerating SHAP global importance plot..."
)

shap.plots.bar(
    impact_shap_explanation,
    max_display=15,
    show=True
)


# =========================================================
# 9. SHAP beeswarm
#
# Shows both importance and direction of contribution.
# =========================================================

print(
    "Generating SHAP beeswarm plot..."
)

shap.plots.beeswarm(
    impact_shap_explanation,
    max_display=15,
    show=True
)


# =========================================================
# 10. Explain latest TEST forecast individually
# =========================================================

latest_position = (
    len(X_test_shap) - 1
)


print(
    "\nLATEST TEST FORECAST"
)

print(
    "Date:",
    test_df.iloc[
        latest_position
    ]["availability_datetime"]
)

print(
    "Current affected people:",
    test_df.iloc[
        latest_position
    ]["affected_people"]
)

print(
    "Predicted next affected people:",
    round(
        float(
            test_impact_prediction[
                latest_position
            ]
        ),
        2
    )
)

print(
    "Actual next affected people:",
    test_df.iloc[
        latest_position
    ]["target_next_affected_people"]
)


print(
    "\nGenerating local SHAP waterfall..."
)

shap.plots.waterfall(
    impact_shap_explanation[
        latest_position
    ],
    max_display=15,
    show=True
)

### Explainability Finding

SHAP analysis showed that recent human-impact and safety-centre conditions were the strongest contributors to the model's predicted change in affected population. Previous safety-centre population, recent safety-population change, previous affected-population change, heavy-rain context, and landslide-warning indicators were among the most influential features.

Temporal reporting variables and missing-value indicators also contributed to predictions, indicating that part of the learned signal may reflect reporting and data-availability patterns. SHAP values describe model behaviour and should not be interpreted as causal effects.

# 11. Similar-Event Finder

## 11.1 Historical Event Similarity

In [ ]:
# =========================================================
# 11.1 HISTORICAL SIMILAR-EVENT FINDER
#
# Finds historical Situation Reports with conditions
# most similar to a selected current/test report.
#
# Only historical DEVELOPMENT data is used as the
# reference library.
# =========================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

import numpy as np
import pandas as pd


# =========================================================
# 1. Use an interpretable subset of state/context features
# =========================================================

SIMILARITY_FEATURES = [

    # Current human impact
    "affected_people",
    "affected_families",
    "safety_centres",
    "people_in_safety_centres",

    # Recent impact movement
    "affected_people_change_from_prev",
    "affected_families_change_from_prev",
    "safety_population_change_from_prev",

    # Weather context
    "weather_reports_24h",
    "weather_max_warning_level_24h",
    "weather_max_rainfall_threshold_24h",
    "weather_max_wind_24h",
    "weather_heavy_rain_24h",
    "current_heavy_rain_flag",

    # Landslide context
    "landslide_active_warning",
    "landslide_max_warning_level",
    "landslide_districts_at_risk",
    "landslide_level1_districts",
    "landslide_level2_districts",
    "landslide_level3_districts"
]


# =========================================================
# 2. Prepare historical reference data
# =========================================================

X_history_similarity = (
    development_df[
        SIMILARITY_FEATURES
    ]
    .copy()
)

X_test_similarity = (
    test_df[
        SIMILARITY_FEATURES
    ]
    .copy()
)


for col in SIMILARITY_FEATURES:

    X_history_similarity[col] = pd.to_numeric(
        X_history_similarity[col],
        errors="coerce"
    )

    X_test_similarity[col] = pd.to_numeric(
        X_test_similarity[col],
        errors="coerce"
    )


# =========================================================
# 3. Fit preprocessing using historical data only
# =========================================================

similarity_imputer = SimpleImputer(
    strategy="median"
)

similarity_scaler = StandardScaler()


history_imputed = similarity_imputer.fit_transform(
    X_history_similarity
)

history_scaled = similarity_scaler.fit_transform(
    history_imputed
)


test_imputed = similarity_imputer.transform(
    X_test_similarity
)

test_scaled = similarity_scaler.transform(
    test_imputed
)


# =========================================================
# 4. Similar-event function
# =========================================================

def find_similar_events(
    test_position,
    top_k=5
):

    query_vector = test_scaled[
        test_position
    ].reshape(1, -1)


    distances = euclidean_distances(
        query_vector,
        history_scaled
    )[0]


    nearest_positions = np.argsort(
        distances
    )[:top_k]


    # Convert distance into intuitive similarity score
    similarity_scores = (
        1 / (1 + distances[nearest_positions])
    )


    selected = (
        development_df
        .iloc[
            nearest_positions
        ]
        .copy()
    )


    result = pd.DataFrame({

        "historical_datetime":
            selected[
                "availability_datetime"
            ].to_numpy(),

        "similarity_score":
            similarity_scores,

        "historical_affected_people":
            selected[
                "affected_people"
            ].to_numpy(),

        "historical_people_in_safety_centres":
            selected[
                "people_in_safety_centres"
            ].to_numpy(),

        "historical_next_affected_people":
            selected[
                "target_next_affected_people"
            ].to_numpy(),

        "historical_next_safety_population":
            selected[
                "target_next_people_in_safety_centres"
            ].to_numpy(),

        "historical_escalation_risk":
            selected[
                "impact_escalation_risk"
            ].to_numpy()
    })


    return (
        result
        .sort_values(
            "similarity_score",
            ascending=False
        )
        .reset_index(drop=True)
    )


# =========================================================
# 5. Demonstrate using latest TEST report
# =========================================================

latest_test_position = (
    len(test_df) - 1
)


latest_similar_events = find_similar_events(
    latest_test_position,
    top_k=5
)


print(
    "CURRENT / QUERY EVENT"
)

print(
    "Date:",
    test_df.iloc[
        latest_test_position
    ]["availability_datetime"]
)

print(
    "Affected people:",
    test_df.iloc[
        latest_test_position
    ]["affected_people"]
)

print(
    "People in safety centres:",
    test_df.iloc[
        latest_test_position
    ]["people_in_safety_centres"]
)


print(
    "\nTOP 5 MOST SIMILAR HISTORICAL EVENTS"
)

print(
    latest_similar_events
    .round(
        {
            "similarity_score": 4
        }
    )
    .to_string(index=False)
)

## 11.2 Refined Similar-Event Finder with Scale-Aware Features

In [ ]:
# =========================================================
# 11.2 REFINED SIMILAR-EVENT FINDER
#
# Heavy-tailed population/count variables are transformed
# before distance calculation so that similarity is more
# meaningful across small, medium and extreme events.
# =========================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

import numpy as np
import pandas as pd


# =========================================================
# 1. Original variables used for similarity
# =========================================================

SIMILARITY_COUNT_FEATURES = [
    "affected_people",
    "affected_families",
    "safety_centres",
    "people_in_safety_centres",

    "weather_reports_24h",
    "weather_max_rainfall_threshold_24h",
    "weather_max_wind_24h",

    "landslide_districts_at_risk",
    "landslide_level1_districts",
    "landslide_level2_districts",
    "landslide_level3_districts"
]


SIMILARITY_CHANGE_FEATURES = [
    "affected_people_change_from_prev",
    "affected_families_change_from_prev",
    "safety_population_change_from_prev"
]


SIMILARITY_STATE_FEATURES = [
    "weather_max_warning_level_24h",
    "weather_heavy_rain_24h",
    "current_heavy_rain_flag",
    "landslide_active_warning",
    "landslide_max_warning_level"
]


# =========================================================
# 2. Build scale-aware feature representation
# =========================================================

def build_similarity_matrix(df):

    transformed = pd.DataFrame(
        index=df.index
    )

    # Positive count variables -> log1p
    for col in SIMILARITY_COUNT_FEATURES:

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        transformed[
            f"log_{col}"
        ] = np.log1p(
            values.clip(lower=0)
        )


    # Signed changes -> signed log1p
    for col in SIMILARITY_CHANGE_FEATURES:

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        transformed[
            f"signedlog_{col}"
        ] = (
            np.sign(values)
            *
            np.log1p(
                np.abs(values)
            )
        )


    # Warning/state features retained directly
    for col in SIMILARITY_STATE_FEATURES:

        transformed[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


    return transformed


history_similarity_refined = (
    build_similarity_matrix(
        development_df
    )
)

test_similarity_refined = (
    build_similarity_matrix(
        test_df
    )
)


print(
    "Refined similarity feature count:",
    history_similarity_refined.shape[1]
)


# =========================================================
# 3. Historical-data-only preprocessing
# =========================================================

refined_similarity_imputer = (
    SimpleImputer(
        strategy="median"
    )
)

refined_similarity_scaler = (
    StandardScaler()
)


history_refined_imputed = (
    refined_similarity_imputer
    .fit_transform(
        history_similarity_refined
    )
)

history_refined_scaled = (
    refined_similarity_scaler
    .fit_transform(
        history_refined_imputed
    )
)


test_refined_imputed = (
    refined_similarity_imputer
    .transform(
        test_similarity_refined
    )
)

test_refined_scaled = (
    refined_similarity_scaler
    .transform(
        test_refined_imputed
    )
)


# =========================================================
# 4. Refined similarity function
# =========================================================

def find_refined_similar_events(
    test_position,
    top_k=5
):

    query = test_refined_scaled[
        test_position
    ].reshape(1, -1)


    distances = euclidean_distances(
        query,
        history_refined_scaled
    )[0]


    nearest_positions = (
        np.argsort(distances)[:top_k]
    )


    similarity_score = (
        1 / (
            1 +
            distances[
                nearest_positions
            ]
        )
    )


    historical = (
        development_df
        .iloc[
            nearest_positions
        ]
        .copy()
    )


    result = pd.DataFrame({

        "historical_datetime":
            historical[
                "availability_datetime"
            ].to_numpy(),

        "similarity_score":
            similarity_score,

        "historical_affected_people":
            historical[
                "affected_people"
            ].to_numpy(),

        "historical_people_in_safety_centres":
            historical[
                "people_in_safety_centres"
            ].to_numpy(),

        "historical_next_affected_people":
            historical[
                "target_next_affected_people"
            ].to_numpy(),

        "historical_next_safety_population":
            historical[
                "target_next_people_in_safety_centres"
            ].to_numpy(),

        "historical_escalation_risk":
            historical[
                "impact_escalation_risk"
            ].to_numpy()
    })


    return (
        result
        .sort_values(
            "similarity_score",
            ascending=False
        )
        .reset_index(drop=True)
    )


# =========================================================
# 5. Latest test event
# =========================================================

latest_refined_similar_events = (
    find_refined_similar_events(
        len(test_df) - 1,
        top_k=5
    )
)


print(
    "\nREFINED TOP 5 SIMILAR HISTORICAL EVENTS"
)

print(
    latest_refined_similar_events
    .round(
        {
            "similarity_score": 4
        }
    )
    .to_string(index=False)
)

### Similar-Event Finding

The refined similarity analysis found no very close historical analogue for the latest evaluated event. The highest similarity score was 0.3743. This indicates that historical matches should be used as contextual references rather than direct forecasts. Similarity is calculated across human-impact state, recent impact changes, safety-centre conditions, weather indicators and landslide-warning context.

In [67]:
# =========================================================
# FINAL SHAP EXPLAINABILITY RECOVERY + SAVE
#
# Reproduces Section 10.1 using the locked
# held-out TEST evaluation human-impact model.
#
# IMPORTANT:
# SHAP explains predicted signed-log CHANGE,
# NOT absolute affected population directly.
# =========================================================

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 1. IMPORT SHAP
# =========================================================

try:
    import shap

except ImportError:
    raise ImportError(
        "SHAP is not installed. Run: pip install shap"
    )


# =========================================================
# 2. PATHS
# =========================================================

models_dir = Path("models")
processed_dir = Path("data/processed")
figures_dir = Path("outputs/figures")

figures_dir.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# 3. LOAD LOCKED EVALUATION MODEL
#
# This is the Train + Validation model that was used
# for final held-out TEST evaluation.
# =========================================================

final_impact_model = joblib.load(
    models_dir / "human_impact_model.joblib"
)


CORE_MODEL_FEATURES = joblib.load(
    models_dir / "core_model_features.joblib"
)


print(
    "Evaluation human-impact model loaded ✅"
)

print(
    "CORE feature count:",
    len(CORE_MODEL_FEATURES)
)


# =========================================================
# 4. REBUILD TEST FEATURE MATRIX
# =========================================================

X_test = (
    test_df[
        CORE_MODEL_FEATURES
    ]
    .copy()
)


for col in CORE_MODEL_FEATURES:

    X_test[col] = pd.to_numeric(
        X_test[col],
        errors="coerce"
    )


print(
    "TEST rows:",
    len(X_test)
)


# =========================================================
# 5. EXTRACT FITTED IMPUTER + GB MODEL
# =========================================================

final_impact_imputer = (
    final_impact_model
    .named_steps["imputer"]
)


final_impact_gb_model = (
    final_impact_model
    .named_steps["model"]
)


# =========================================================
# 6. TRANSFORM TEST FEATURES EXACTLY AS MODEL SEES THEM
# =========================================================

X_test_shap = (
    final_impact_imputer
    .transform(
        X_test
    )
)


shap_feature_names = (
    final_impact_imputer
    .get_feature_names_out(
        CORE_MODEL_FEATURES
    )
)


print(
    "SHAP transformed feature count:",
    len(shap_feature_names)
)


# =========================================================
# 7. TREE SHAP
# =========================================================

impact_explainer = shap.TreeExplainer(
    final_impact_gb_model
)


impact_shap_values = (
    impact_explainer
    .shap_values(
        X_test_shap
    )
)


impact_shap_values = np.asarray(
    impact_shap_values
)


print(
    "SHAP matrix shape:",
    impact_shap_values.shape
)


# =========================================================
# 8. GLOBAL FEATURE IMPORTANCE
# =========================================================

mean_absolute_shap = np.mean(
    np.abs(
        impact_shap_values
    ),
    axis=0
)


shap_importance_df = pd.DataFrame({

    "feature":
        shap_feature_names,

    "mean_abs_shap":
        mean_absolute_shap

})


shap_importance_df = (
    shap_importance_df
    .sort_values(
        "mean_abs_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


shap_importance_path = (
    processed_dir
    / "shap_feature_importance.csv"
)


shap_importance_df.to_csv(
    shap_importance_path,
    index=False
)


# =========================================================
# 9. CREATE SHAP EXPLANATION OBJECT
# =========================================================

expected_value = np.asarray(
    impact_explainer.expected_value
).reshape(-1)[0]


impact_shap_explanation = shap.Explanation(

    values=
        impact_shap_values,

    base_values=
        np.repeat(
            expected_value,
            len(X_test_shap)
        ),

    data=
        X_test_shap,

    feature_names=
        list(
            shap_feature_names
        )
)


# =========================================================
# 10. GLOBAL BAR PLOT
# =========================================================

plt.figure()

shap.plots.bar(
    impact_shap_explanation,
    max_display=15,
    show=False
)

plt.tight_layout()

bar_path = (
    figures_dir
    / "shap_global_bar.png"
)

plt.savefig(
    bar_path,
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# =========================================================
# 11. GLOBAL BEESWARM
# =========================================================

plt.figure()

shap.plots.beeswarm(
    impact_shap_explanation,
    max_display=15,
    show=False
)

plt.tight_layout()

beeswarm_path = (
    figures_dir
    / "shap_beeswarm.png"
)

plt.savefig(
    beeswarm_path,
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# =========================================================
# 12. LATEST TEST FORECAST
# =========================================================

def inverse_signed_log1p(x):

    x = np.asarray(
        x,
        dtype=float
    )

    return (
        np.sign(x)
        *
        np.expm1(
            np.abs(x)
        )
    )


latest_position = (
    len(X_test) - 1
)


test_delta_transformed = (
    final_impact_model
    .predict(
        X_test
    )
)


test_delta_prediction = (
    inverse_signed_log1p(
        test_delta_transformed
    )
)


test_current_affected = (
    test_df[
        "affected_people"
    ]
    .astype(float)
    .to_numpy()
)


test_impact_prediction = np.clip(
    test_current_affected
    +
    test_delta_prediction,
    0,
    None
)


# =========================================================
# 13. LATEST TEST WATERFALL
# =========================================================

plt.figure()

shap.plots.waterfall(
    impact_shap_explanation[
        latest_position
    ],
    max_display=15,
    show=False
)

plt.tight_layout()

waterfall_path = (
    figures_dir
    / "shap_latest_test_waterfall.png"
)

plt.savefig(
    waterfall_path,
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# =========================================================
# 14. SAVE LATEST LOCAL CONTRIBUTIONS
# =========================================================

latest_local_shap_df = pd.DataFrame({

    "feature":
        shap_feature_names,

    "feature_value":
        X_test_shap[
            latest_position
        ],

    "shap_value":
        impact_shap_values[
            latest_position
        ]

})


latest_local_shap_df[
    "abs_shap"
] = np.abs(
    latest_local_shap_df[
        "shap_value"
    ]
)


latest_local_shap_df = (
    latest_local_shap_df
    .sort_values(
        "abs_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


local_shap_path = (
    processed_dir
    / "shap_latest_test_contributions.csv"
)


latest_local_shap_df.to_csv(
    local_shap_path,
    index=False
)


# =========================================================
# 15. QC OUTPUT
# =========================================================

print("\n" + "=" * 78)
print("FINAL HUMAN-IMPACT SHAP EXPLAINABILITY")
print("=" * 78)


print(
    "\nTOP 20 SHAP FEATURES"
)

print(
    shap_importance_df
    .head(20)
    .round(6)
    .to_string(index=False)
)


print(
    "\nLATEST TEST FORECAST"
)

print(
    "Date:",
    test_df.iloc[
        latest_position
    ][
        "availability_datetime"
    ]
)

print(
    "Current affected:",
    test_df.iloc[
        latest_position
    ][
        "affected_people"
    ]
)

print(
    "Predicted next affected:",
    round(
        float(
            test_impact_prediction[
                latest_position
            ]
        ),
        2
    )
)

print(
    "Actual next affected:",
    test_df.iloc[
        latest_position
    ][
        "target_next_affected_people"
    ]
)


print(
    "\nSaved files:"
)

for path in [

    shap_importance_path,
    local_shap_path,
    bar_path,
    beeswarm_path,
    waterfall_path

]:

    print(
        path,
        "✅"
        if path.exists()
        else "❌"
    )


print(
    "\nIMPORTANT:"
)

print(
    "SHAP explains the model's predicted "
    "signed-log CHANGE in affected people."
)

print(
    "SHAP values are not causal effects."
)

print("=" * 78)

Evaluation human-impact model loaded ✅
CORE feature count: 46
TEST rows: 150
SHAP transformed feature count: 58
SHAP matrix shape: (150, 58)

FINAL HUMAN-IMPACT SHAP EXPLAINABILITY

TOP 20 SHAP FEATURES
                                            feature  mean_abs_shap
                      prev_people_in_safety_centres       0.493638
                 safety_population_change_from_prev       0.355895
                                   report_dayofweek       0.291383
               affected_people_log_change_from_prev       0.258756
                   affected_people_change_from_prev       0.242165
                            current_heavy_rain_flag       0.234586
                                      dayofyear_cos       0.169728
                        landslide_districts_at_risk       0.141761
                         landslide_level1_districts       0.137717
                 affected_families_change_from_prev       0.137006
                                weather_reports_24h       0.

In [68]:
# =========================================================
# CURRENT DEPLOYMENT SHAP EXPLANATION
#
# Explains the latest 2026-08-14 operational forecast.
#
# IMPORTANT:
# SHAP explains predicted signed-log CHANGE,
# not absolute affected population directly.
# =========================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap


# =========================================================
# 1. PATHS
# =========================================================

models_dir = Path("models")
processed_dir = Path("data/processed")
figures_dir = Path("outputs/figures")

figures_dir.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# 2. LOAD DEPLOYMENT HUMAN-IMPACT MODEL
# =========================================================

deployment_impact_model = joblib.load(
    models_dir
    / "deployment_human_impact_model.joblib"
)


CORE_MODEL_FEATURES = joblib.load(
    models_dir
    / "core_model_features.joblib"
)


print(
    "Deployment human-impact model loaded ✅"
)


# =========================================================
# 3. RESTORE LATEST FEATURE MATRIX IF NECESSARY
# =========================================================

if "latest_current_row" not in globals():

    latest_current_row = (
        feature_df
        .sort_values(
            "availability_datetime"
        )
        .iloc[[-1]]
        .copy()
    )


    latest_disaster_text = (
        latest_current_row[
            "disaster_types"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    latest_current_row[
        "current_flood_flag"
    ] = (
        latest_disaster_text
        .str.contains(
            "flood",
            regex=False
        )
        .astype(int)
    )


    latest_current_row[
        "current_landslide_flag"
    ] = (
        latest_disaster_text
        .str.contains(
            "landslide",
            regex=False
        )
        .astype(int)
    )


    latest_current_row[
        "current_heavy_rain_flag"
    ] = (
        latest_disaster_text
        .str.contains(
            "heavy rain",
            regex=False
        )
        .astype(int)
    )


X_latest = (
    latest_current_row[
        CORE_MODEL_FEATURES
    ]
    .copy()
)


for col in CORE_MODEL_FEATURES:

    X_latest[col] = pd.to_numeric(
        X_latest[col],
        errors="coerce"
    )


latest = (
    latest_current_row.iloc[0]
)


# =========================================================
# 4. EXTRACT DEPLOYMENT IMPUTER + GB MODEL
# =========================================================

deployment_imputer = (
    deployment_impact_model
    .named_steps["imputer"]
)


deployment_gb_model = (
    deployment_impact_model
    .named_steps["model"]
)


# =========================================================
# 5. TRANSFORM LATEST FEATURES
# =========================================================

X_latest_shap = (
    deployment_imputer
    .transform(
        X_latest
    )
)


deployment_shap_feature_names = (
    deployment_imputer
    .get_feature_names_out(
        CORE_MODEL_FEATURES
    )
)


print(
    "Deployment SHAP feature count:",
    len(
        deployment_shap_feature_names
    )
)


# =========================================================
# 6. TREE SHAP
# =========================================================

deployment_explainer = (
    shap.TreeExplainer(
        deployment_gb_model
    )
)


deployment_shap_values = (
    deployment_explainer
    .shap_values(
        X_latest_shap
    )
)


deployment_shap_values = np.asarray(
    deployment_shap_values
).reshape(1, -1)


expected_value = float(
    np.asarray(
        deployment_explainer.expected_value
    )
    .reshape(-1)[0]
)


# =========================================================
# 7. CREATE LOCAL SHAP TABLE
# =========================================================

deployment_local_shap_df = pd.DataFrame({

    "feature":
        deployment_shap_feature_names,

    "feature_value":
        X_latest_shap[0],

    "shap_value":
        deployment_shap_values[0]

})


deployment_local_shap_df[
    "abs_shap"
] = np.abs(
    deployment_local_shap_df[
        "shap_value"
    ]
)


deployment_local_shap_df[
    "direction"
] = np.where(

    deployment_local_shap_df[
        "shap_value"
    ] > 0,

    "Pushes predicted change upward",

    np.where(

        deployment_local_shap_df[
            "shap_value"
        ] < 0,

        "Pushes predicted change downward",

        "Neutral"
    )
)


deployment_local_shap_df = (
    deployment_local_shap_df
    .sort_values(
        "abs_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


# =========================================================
# 8. SHAP ADDITIVITY QC
# =========================================================

model_transformed_prediction = float(
    deployment_impact_model
    .predict(
        X_latest
    )[0]
)


shap_reconstructed_prediction = float(
    expected_value
    +
    deployment_shap_values[0].sum()
)


print(
    "Model transformed prediction:",
    round(
        model_transformed_prediction,
        6
    )
)

print(
    "SHAP reconstructed prediction:",
    round(
        shap_reconstructed_prediction,
        6
    )
)

print(
    "Difference:",
    round(
        abs(
            model_transformed_prediction
            -
            shap_reconstructed_prediction
        ),
        10
    )
)


# =========================================================
# 9. CREATE SHAP EXPLANATION
# =========================================================

latest_deployment_explanation = (
    shap.Explanation(

        values=
            deployment_shap_values[0],

        base_values=
            expected_value,

        data=
            X_latest_shap[0],

        feature_names=
            list(
                deployment_shap_feature_names
            )
    )
)


# =========================================================
# 10. SAVE WATERFALL
# =========================================================

plt.figure()


shap.plots.waterfall(
    latest_deployment_explanation,
    max_display=15,
    show=False
)


plt.tight_layout()


deployment_waterfall_path = (
    figures_dir
    / "shap_latest_deployment_waterfall.png"
)


plt.savefig(
    deployment_waterfall_path,
    dpi=200,
    bbox_inches="tight"
)


plt.close()


# =========================================================
# 11. SAVE CSV
# =========================================================

deployment_shap_path = (
    processed_dir
    / "shap_latest_deployment_contributions.csv"
)


deployment_local_shap_df.to_csv(
    deployment_shap_path,
    index=False
)


# =========================================================
# 12. SAVE TOP CONTRIBUTORS FOR DASHBOARD
# =========================================================

top_contributors = []


for _, row in (
    deployment_local_shap_df
    .head(10)
    .iterrows()
):

    top_contributors.append({

        "feature":
            str(
                row["feature"]
            ),

        "feature_value":
            (
                float(
                    row["feature_value"]
                )
                if pd.notna(
                    row["feature_value"]
                )
                else None
            ),

        "shap_value":
            float(
                row["shap_value"]
            ),

        "direction":
            str(
                row["direction"]
            )
    })


deployment_shap_json = {

    "snapshot_datetime":
        str(
            latest[
                "availability_datetime"
            ]
        ),

    "explanation_target":
        (
            "Predicted signed-log change "
            "in affected people"
        ),

    "base_value":
        expected_value,

    "model_transformed_prediction":
        model_transformed_prediction,

    "top_contributors":
        top_contributors,

    "note":
        (
            "SHAP values explain model contribution "
            "to the predicted change. "
            "They are not causal effects."
        )
}


deployment_shap_json_path = (
    processed_dir
    / "latest_deployment_shap.json"
)


with open(
    deployment_shap_json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        deployment_shap_json,
        f,
        indent=4
    )


# =========================================================
# 13. QC OUTPUT
# =========================================================

print("\n" + "=" * 78)
print("LATEST DEPLOYMENT SHAP EXPLANATION")
print("=" * 78)


print(
    "Snapshot:",
    latest[
        "availability_datetime"
    ]
)


print(
    "\nTOP 10 CURRENT CONTRIBUTORS"
)


print(
    deployment_local_shap_df[
        [
            "feature",
            "feature_value",
            "shap_value",
            "direction"
        ]
    ]
    .head(10)
    .round(6)
    .to_string(index=False)
)


print(
    "\nSaved files:"
)


for path in [

    deployment_shap_path,
    deployment_waterfall_path,
    deployment_shap_json_path

]:

    print(
        path,
        "✅"
        if path.exists()
        else "❌"
    )


print(
    "\nIMPORTANT:"
)

print(
    "This explains the CURRENT DEPLOYMENT forecast."
)

print(
    "SHAP contributions refer to predicted "
    "signed-log change."
)

print(
    "They are NOT causal effects."
)

print("=" * 78)

Deployment human-impact model loaded ✅
Deployment SHAP feature count: 58
Model transformed prediction: -0.352237
SHAP reconstructed prediction: -0.352237
Difference: 0.0

LATEST DEPLOYMENT SHAP EXPLANATION
Snapshot: 2026-08-14 10:00:00

TOP 10 CURRENT CONTRIBUTORS
                             feature  feature_value  shap_value                        direction
       prev_people_in_safety_centres     263.000000    0.184226   Pushes predicted change upward
             current_heavy_rain_flag       1.000000   -0.157871 Pushes predicted change downward
affected_people_log_change_from_prev       0.000968   -0.139125 Pushes predicted change downward
    affected_people_change_from_prev      16.000000   -0.126279 Pushes predicted change downward
         landslide_districts_at_risk       0.000000   -0.116911 Pushes predicted change downward
                   affected_families    4209.000000   -0.116283 Pushes predicted change downward
          latest_landslide_age_hours     120.500000   -0

# 12. Impact Trend and Rapid-Growth Anomaly Detection

## 12.1 Leakage-Safe Historical Trend Monitoring

In [ ]:
# =========================================================
# 12.1 IMPACT TREND + RAPID-GROWTH ANOMALY DETECTION
#
# Purpose:
# Detect unusually rapid increases in affected population.
#
# IMPORTANT:
# - Uses current and PAST reports only.
# - Rolling historical statistics are shifted by one row,
#   so the current observation is never used to define
#   its own anomaly threshold.
# - This is a research-derived monitoring indicator,
#   not an official DMC warning.
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 1. Use the full canonical feature timeline
# =========================================================

trend_df = (
    feature_df
    .copy()
    .sort_values(
        "availability_datetime"
    )
    .reset_index(drop=True)
)


print(
    "Trend-monitoring rows:",
    len(trend_df)
)


# =========================================================
# 2. Current impact movement
# =========================================================

trend_df[
    "impact_absolute_change"
] = pd.to_numeric(
    trend_df[
        "affected_people_change_from_prev"
    ],
    errors="coerce"
)


trend_df[
    "impact_log_change"
] = pd.to_numeric(
    trend_df[
        "affected_people_log_change_from_prev"
    ],
    errors="coerce"
)


# =========================================================
# 3. Simple trend label
#
# This describes current movement only.
# =========================================================

def classify_impact_trend(row):

    if not bool(
        row["previous_context_valid"]
    ):
        return "Insufficient Previous Context"

    change = row[
        "impact_absolute_change"
    ]

    if pd.isna(change):
        return "Insufficient Previous Context"

    if change > 0:
        return "Increasing"

    elif change < 0:
        return "Decreasing"

    else:
        return "Stable"


trend_df[
    "impact_trend"
] = trend_df.apply(
    classify_impact_trend,
    axis=1
)


# =========================================================
# 4. Leakage-safe rolling historical baseline
#
# Previous 60 eligible reports.
# Minimum 20 historical values required.
#
# shift(1) ensures CURRENT value is excluded.
# =========================================================

ROLLING_WINDOW = 60
MIN_HISTORY = 20


valid_change_series = (
    trend_df[
        "impact_log_change"
    ]
    .where(
        trend_df[
            "previous_context_valid"
        ] == True
    )
)


historical_change = (
    valid_change_series
    .shift(1)
)


rolling_median = (
    historical_change
    .rolling(
        window=ROLLING_WINDOW,
        min_periods=MIN_HISTORY
    )
    .median()
)


# =========================================================
# 5. Rolling Median Absolute Deviation (MAD)
# =========================================================

def rolling_mad(values):

    values = np.asarray(
        values,
        dtype=float
    )

    median = np.nanmedian(
        values
    )

    return np.nanmedian(
        np.abs(
            values - median
        )
    )


rolling_mad_values = (
    historical_change
    .rolling(
        window=ROLLING_WINDOW,
        min_periods=MIN_HISTORY
    )
    .apply(
        rolling_mad,
        raw=True
    )
)


trend_df[
    "historical_log_change_median"
] = rolling_median

trend_df[
    "historical_log_change_mad"
] = rolling_mad_values


# =========================================================
# 6. Robust anomaly score
#
# Robust Z = 0.6745 * (x - median) / MAD
# =========================================================

trend_df[
    "rapid_growth_score"
] = np.where(

    trend_df[
        "historical_log_change_mad"
    ] > 0,

    0.6745
    *
    (
        trend_df[
            "impact_log_change"
        ]
        -
        trend_df[
            "historical_log_change_median"
        ]
    )
    /
    trend_df[
        "historical_log_change_mad"
    ],

    np.nan
)


# =========================================================
# 7. Fixed anomaly rule
#
# Rapid-growth anomaly:
# - valid previous context
# - positive affected-person increase
# - robust score >= 3
#
# Threshold is fixed BEFORE examining final anomalies.
# =========================================================

ROBUST_Z_THRESHOLD = 3.0


trend_df[
    "rapid_growth_anomaly"
] = (

    (
        trend_df[
            "previous_context_valid"
        ] == True
    )

    &

    (
        trend_df[
            "impact_absolute_change"
        ] > 0
    )

    &

    (
        trend_df[
            "rapid_growth_score"
        ] >= ROBUST_Z_THRESHOLD
    )
)


# =========================================================
# 8. Monitoring status
# =========================================================

def anomaly_status(row):

    if not bool(
        row["previous_context_valid"]
    ):
        return "Gap / No Comparable Previous Report"

    if pd.isna(
        row["rapid_growth_score"]
    ):
        return "Insufficient Historical Baseline"

    if bool(
        row["rapid_growth_anomaly"]
    ):
        return "Rapid Growth Anomaly"

    if row[
        "impact_absolute_change"
    ] > 0:
        return "Normal Increase"

    if row[
        "impact_absolute_change"
    ] < 0:
        return "Decrease"

    return "Stable"


trend_df[
    "impact_monitoring_status"
] = trend_df.apply(
    anomaly_status,
    axis=1
)


# =========================================================
# 9. Overall summary
# =========================================================

print(
    "\nIMPACT TREND DISTRIBUTION"
)

print(
    trend_df[
        "impact_trend"
    ]
    .value_counts()
)


print(
    "\nRAPID-GROWTH ANOMALY SUMMARY"
)

print(
    trend_df[
        "impact_monitoring_status"
    ]
    .value_counts()
)


print(
    "\nNumber of rapid-growth anomalies:",
    int(
        trend_df[
            "rapid_growth_anomaly"
        ].sum()
    )
)


# =========================================================
# 10. Top anomaly events
# =========================================================

top_anomalies_df = (
    trend_df[
        trend_df[
            "rapid_growth_anomaly"
        ] == True
    ]
    [
        [
            "availability_datetime",
            "affected_people",
            "prev_affected_people",
            "impact_absolute_change",
            "impact_log_change",
            "rapid_growth_score",
            "weather_max_warning_level_24h",
            "landslide_max_warning_level",
            "landslide_districts_at_risk"
        ]
    ]
    .sort_values(
        "rapid_growth_score",
        ascending=False
    )
    .head(15)
)


print(
    "\nTOP RAPID-GROWTH ANOMALIES"
)

print(
    top_anomalies_df
    .round(3)
    .to_string(index=False)
)


# =========================================================
# 11. Latest monitoring status
# =========================================================

latest_trend_row = (
    trend_df.iloc[-1]
)


print(
    "\nLATEST IMPACT TREND STATUS"
)

print(
    "Date:",
    latest_trend_row[
        "availability_datetime"
    ]
)

print(
    "Affected people:",
    latest_trend_row[
        "affected_people"
    ]
)

print(
    "Change from previous:",
    latest_trend_row[
        "impact_absolute_change"
    ]
)

print(
    "Trend:",
    latest_trend_row[
        "impact_trend"
    ]
)

print(
    "Rapid-growth score:",
    latest_trend_row[
        "rapid_growth_score"
    ]
)

print(
    "Monitoring status:",
    latest_trend_row[
        "impact_monitoring_status"
    ]
)


# =========================================================
# 12. Visualization
#
# Latest 180 reports shown for readability.
# =========================================================

plot_df = (
    trend_df
    .tail(180)
    .copy()
)


plt.figure(
    figsize=(14, 6)
)


plt.plot(
    plot_df[
        "availability_datetime"
    ],
    np.log1p(
        plot_df[
            "affected_people"
        ]
    ),
    label="log1p(Affected People)"
)


anomaly_plot_df = (
    plot_df[
        plot_df[
            "rapid_growth_anomaly"
        ] == True
    ]
)


plt.scatter(
    anomaly_plot_df[
        "availability_datetime"
    ],
    np.log1p(
        anomaly_plot_df[
            "affected_people"
        ]
    ),
    marker="x",
    s=80,
    label="Rapid Growth Anomaly"
)


plt.title(
    "Recent Human-Impact Trend and Rapid-Growth Anomalies"
)

plt.xlabel(
    "Situation Report Availability Time"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.xticks(
    rotation=45
)

plt.legend()

plt.tight_layout()

plt.show()

## 12.2 Refined Percentile-Based Rapid-Growth Detection

In [ ]:
# =========================================================
# 12.2 REFINED RAPID-GROWTH ANOMALY DETECTION
#
# Replaces unstable MAD-based scores with a bounded,
# leakage-safe historical percentile approach.
#
# A report is flagged only when:
#
# 1. Previous report is temporally comparable
# 2. Increase is materially positive
# 3. Absolute increase is extreme relative to recent history
# 4. Relative/log increase is also extreme
#
# Current observation is NEVER included in its own
# historical reference window.
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


refined_trend_df = (
    trend_df
    .copy()
    .sort_values("availability_datetime")
    .reset_index(drop=True)
)


# =========================================================
# 1. Settings
# =========================================================

HISTORY_WINDOW = 120
MIN_POSITIVE_HISTORY = 20

# This threshold was already derived from TRAINING DATA
# in Section 7.5:
# median positive increase = 142 people.
#
# It prevents tiny increases such as 0 -> 4 from being
# labelled as operationally meaningful rapid growth.

MIN_MATERIAL_INCREASE = 142.0

EXTREME_PERCENTILE = 0.95


# =========================================================
# 2. Containers
# =========================================================

absolute_growth_percentile = []
relative_growth_percentile = []
historical_positive_count = []


# =========================================================
# 3. Leakage-safe rolling historical comparison
# =========================================================

for i in range(len(refined_trend_df)):

    current_row = refined_trend_df.iloc[i]

    # No comparable previous report
    if (
        current_row["previous_context_valid"] != True
        or pd.isna(current_row["impact_absolute_change"])
        or pd.isna(current_row["impact_log_change"])
    ):

        absolute_growth_percentile.append(np.nan)
        relative_growth_percentile.append(np.nan)
        historical_positive_count.append(0)

        continue


    # -----------------------------------------------------
    # Historical window uses ONLY reports before row i
    # -----------------------------------------------------

    start_position = max(
        0,
        i - HISTORY_WINDOW
    )

    historical_window = (
        refined_trend_df
        .iloc[start_position:i]
        .copy()
    )


    # Comparable historical positive increases only
    historical_positive = (
        historical_window[
            (
                historical_window[
                    "previous_context_valid"
                ] == True
            )
            &
            (
                historical_window[
                    "impact_absolute_change"
                ] > 0
            )
            &
            historical_window[
                "impact_absolute_change"
            ].notna()
            &
            historical_window[
                "impact_log_change"
            ].notna()
        ]
    )


    n_history = len(
        historical_positive
    )

    historical_positive_count.append(
        n_history
    )


    if n_history < MIN_POSITIVE_HISTORY:

        absolute_growth_percentile.append(
            np.nan
        )

        relative_growth_percentile.append(
            np.nan
        )

        continue


    current_absolute = float(
        current_row[
            "impact_absolute_change"
        ]
    )

    current_relative = float(
        current_row[
            "impact_log_change"
        ]
    )


    # -----------------------------------------------------
    # Empirical percentile:
    #
    # proportion of previous positive changes that
    # are <= the current change.
    #
    # Always bounded between 0 and 1.
    # -----------------------------------------------------

    abs_percentile = np.mean(
        historical_positive[
            "impact_absolute_change"
        ].to_numpy(dtype=float)
        <= current_absolute
    )


    rel_percentile = np.mean(
        historical_positive[
            "impact_log_change"
        ].to_numpy(dtype=float)
        <= current_relative
    )


    absolute_growth_percentile.append(
        abs_percentile
    )

    relative_growth_percentile.append(
        rel_percentile
    )


# =========================================================
# 4. Save percentile measures
# =========================================================

refined_trend_df[
    "absolute_growth_percentile"
] = absolute_growth_percentile


refined_trend_df[
    "relative_growth_percentile"
] = relative_growth_percentile


refined_trend_df[
    "historical_positive_count"
] = historical_positive_count


# =========================================================
# 5. Bounded rapid-growth score
#
# Both absolute AND relative growth must be unusual.
#
# Taking the minimum ensures that a report with:
# - huge relative increase but tiny absolute increase, or
# - huge absolute increase but normal relative movement
#
# does not automatically receive a high score.
# =========================================================

refined_trend_df[
    "rapid_growth_score"
] = (
    np.minimum(
        refined_trend_df[
            "absolute_growth_percentile"
        ],
        refined_trend_df[
            "relative_growth_percentile"
        ]
    )
    * 100
)


# =========================================================
# 6. Final refined anomaly rule
# =========================================================

refined_trend_df[
    "rapid_growth_anomaly"
] = (

    (
        refined_trend_df[
            "previous_context_valid"
        ] == True
    )

    &

    (
        refined_trend_df[
            "impact_absolute_change"
        ] >= MIN_MATERIAL_INCREASE
    )

    &

    (
        refined_trend_df[
            "absolute_growth_percentile"
        ] >= EXTREME_PERCENTILE
    )

    &

    (
        refined_trend_df[
            "relative_growth_percentile"
        ] >= EXTREME_PERCENTILE
    )
)


# =========================================================
# 7. Refined monitoring status
# =========================================================

def refined_monitoring_status(row):

    if row["previous_context_valid"] != True:

        return "Gap / No Comparable Previous Report"

    if pd.isna(
        row["rapid_growth_score"]
    ):

        return "Insufficient Historical Baseline"

    if row["rapid_growth_anomaly"]:

        return "Rapid Growth Anomaly"

    if row["impact_absolute_change"] > 0:

        return "Normal Increase"

    elif row["impact_absolute_change"] < 0:

        return "Decrease"

    else:

        return "Stable"


refined_trend_df[
    "impact_monitoring_status"
] = refined_trend_df.apply(
    refined_monitoring_status,
    axis=1
)


# =========================================================
# 8. Summary
# =========================================================

print(
    "REFINED RAPID-GROWTH MONITORING SUMMARY"
)

print(
    refined_trend_df[
        "impact_monitoring_status"
    ]
    .value_counts()
)


print(
    "\nNumber of refined rapid-growth anomalies:",
    int(
        refined_trend_df[
            "rapid_growth_anomaly"
        ].sum()
    )
)


print(
    "Percentage of timeline flagged (%):",
    round(
        refined_trend_df[
            "rapid_growth_anomaly"
        ].mean()
        * 100,
        2
    )
)


# =========================================================
# 9. Top refined anomalies
# =========================================================

refined_top_anomalies_df = (
    refined_trend_df[
        refined_trend_df[
            "rapid_growth_anomaly"
        ] == True
    ]
    [
        [
            "availability_datetime",
            "affected_people",
            "prev_affected_people",
            "impact_absolute_change",
            "impact_log_change",
            "absolute_growth_percentile",
            "relative_growth_percentile",
            "rapid_growth_score",
            "weather_max_warning_level_24h",
            "landslide_max_warning_level",
            "landslide_districts_at_risk"
        ]
    ]
    .sort_values(
        [
            "rapid_growth_score",
            "impact_absolute_change"
        ],
        ascending=[
            False,
            False
        ]
    )
    .head(15)
)


print(
    "\nTOP REFINED RAPID-GROWTH ANOMALIES"
)

print(
    refined_top_anomalies_df
    .round(3)
    .to_string(index=False)
)


# =========================================================
# 10. Latest monitoring status
# =========================================================

latest_refined_row = (
    refined_trend_df.iloc[-1]
)


print(
    "\nLATEST REFINED IMPACT STATUS"
)

print(
    "Date:",
    latest_refined_row[
        "availability_datetime"
    ]
)

print(
    "Affected people:",
    latest_refined_row[
        "affected_people"
    ]
)

print(
    "Change from previous:",
    latest_refined_row[
        "impact_absolute_change"
    ]
)

print(
    "Absolute-growth percentile:",
    latest_refined_row[
        "absolute_growth_percentile"
    ]
)

print(
    "Relative-growth percentile:",
    latest_refined_row[
        "relative_growth_percentile"
    ]
)

print(
    "Rapid-growth score:",
    latest_refined_row[
        "rapid_growth_score"
    ]
)

print(
    "Monitoring status:",
    latest_refined_row[
        "impact_monitoring_status"
    ]
)


# =========================================================
# 11. Visualization
# =========================================================

plot_df = (
    refined_trend_df
    .tail(180)
    .copy()
)


plt.figure(
    figsize=(14, 6)
)


plt.plot(
    plot_df[
        "availability_datetime"
    ],
    np.log1p(
        plot_df[
            "affected_people"
        ]
    ),
    label="log1p(Affected People)"
)


plot_anomalies = (
    plot_df[
        plot_df[
            "rapid_growth_anomaly"
        ] == True
    ]
)


plt.scatter(
    plot_anomalies[
        "availability_datetime"
    ],
    np.log1p(
        plot_anomalies[
            "affected_people"
        ]
    ),
    marker="x",
    s=80,
    label="Rapid Growth Anomaly"
)


plt.title(
    "Human-Impact Trend and Refined Rapid-Growth Anomalies"
)

plt.xlabel(
    "Situation Report Availability Time"
)

plt.ylabel(
    "log1p(Affected People)"
)

plt.xticks(
    rotation=45
)

plt.legend()

plt.tight_layout()

plt.show()

# 13. Research-Derived Priority Indices

## 13.1 DRPI and SCPI

The Disaster Response Priority Index (DRPI) and Safety-Centre Pressure Index
(SCPI) are research-derived composite indicators developed for FloodImpact-LK.
They are not official DMC, NBRO, Irrigation Department, or government warning
categories.

In [ ]:
# =========================================================
# 13.1 DRPI + SCPI
#
# DRPI = Disaster Response Priority Index
# SCPI = Safety-Centre Pressure Index
#
# Both:
# - Range: 0 to 100
# - Research-derived
# - NOT official government warning categories
#
# IMPORTANT:
# Final test evaluation has already been completed.
# Therefore, the selected models can now be refitted using
# ALL labelled historical data for deployment/current use.
# =========================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)


# =========================================================
# 0. Helper functions
# =========================================================

def signed_log1p(x):
    """
    Signed log transformation that preserves
    both positive and negative values.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    return (
        np.sign(x)
        *
        np.log1p(
            np.abs(x)
        )
    )


def inverse_signed_log1p(x):
    """
    Reverse signed-log transformation.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    return (
        np.sign(x)
        *
        np.expm1(
            np.abs(x)
        )
    )


def safe_numeric(value):
    """
    Convert a single value safely to numeric.
    """

    return pd.to_numeric(
        pd.Series([value]),
        errors="coerce"
    ).iloc[0]


# =========================================================
# 1. ALL LABELLED HISTORICAL DATA
#
# Train + Validation + Test are combined only AFTER
# final test evaluation has been completed.
# =========================================================

all_labelled_df = (
    pd.concat(
        [
            train_df,
            validation_df,
            test_df
        ],
        axis=0
    )
    .sort_values(
        "availability_datetime"
    )
    .reset_index(drop=True)
)


print(
    "Deployment training rows:",
    len(all_labelled_df)
)


# =========================================================
# 1.1 Prepare deployment feature matrix
# =========================================================

X_all = (
    all_labelled_df[
        CORE_MODEL_FEATURES
    ]
    .copy()
)


for col in CORE_MODEL_FEATURES:

    X_all[col] = pd.to_numeric(
        X_all[col],
        errors="coerce"
    )


# =========================================================
# 2. LATEST CURRENT REPORT
#
# feature_df contains the latest canonical report,
# including the latest report for which a future target
# may not yet exist.
# =========================================================

latest_current_row = (
    feature_df
    .sort_values(
        "availability_datetime"
    )
    .iloc[[-1]]
    .copy()
)


# =========================================================
# 2.1 Re-create Situation hazard-label features
#
# These features were originally created in modeling_df,
# but the latest row comes from feature_df.
# =========================================================

latest_disaster_text = (
    latest_current_row[
        "disaster_types"
    ]
    .fillna("")
    .astype(str)
    .str.lower()
)


latest_current_row[
    "current_flood_flag"
] = (
    latest_disaster_text
    .str.contains(
        "flood",
        regex=False
    )
    .astype(int)
)


latest_current_row[
    "current_landslide_flag"
] = (
    latest_disaster_text
    .str.contains(
        "landslide",
        regex=False
    )
    .astype(int)
)


latest_current_row[
    "current_heavy_rain_flag"
] = (
    latest_disaster_text
    .str.contains(
        "heavy rain",
        regex=False
    )
    .astype(int)
)


# =========================================================
# 2.2 Check latest row contains every CORE feature
# =========================================================

missing_latest_features = [
    col
    for col in CORE_MODEL_FEATURES
    if col not in latest_current_row.columns
]


print(
    "Missing latest CORE features:",
    missing_latest_features
)


if missing_latest_features:

    raise ValueError(
        "Latest row is missing required CORE features: "
        + str(missing_latest_features)
    )


# =========================================================
# 2.3 Build latest feature matrix
# =========================================================

X_latest = (
    latest_current_row[
        CORE_MODEL_FEATURES
    ]
    .copy()
)


for col in CORE_MODEL_FEATURES:

    X_latest[col] = pd.to_numeric(
        X_latest[col],
        errors="coerce"
    )


# Explicitly define latest report as a Series.
# This prevents dependence on stale notebook variables.

latest = latest_current_row.iloc[0]


print(
    "Latest report:",
    latest[
        "availability_datetime"
    ]
)


print(
    "Latest disaster_types:",
    latest[
        "disaster_types"
    ]
)


print(
    "Flood flag:",
    latest[
        "current_flood_flag"
    ]
)


print(
    "Landslide flag:",
    latest[
        "current_landslide_flag"
    ]
)


print(
    "Heavy-rain flag:",
    latest[
        "current_heavy_rain_flag"
    ]
)


# =========================================================
# PART A
# DEPLOYMENT HUMAN-IMPACT MODEL
#
# Selected model:
# GradientBoosting Huber additive-change forecasting
# =========================================================

all_impact_delta = (
    all_labelled_df[
        "target_next_affected_people"
    ].astype(float)
    -
    all_labelled_df[
        "affected_people"
    ].astype(float)
)


all_impact_delta_transformed = (
    signed_log1p(
        all_impact_delta
    )
)


deployment_impact_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            random_state=42
        )
    )
])


deployment_impact_model.fit(
    X_all,
    all_impact_delta_transformed
)


latest_predicted_delta_transformed = (
    deployment_impact_model
    .predict(
        X_latest
    )
)


latest_predicted_delta = float(
    inverse_signed_log1p(
        latest_predicted_delta_transformed
    )[0]
)


latest_current_affected = float(
    latest[
        "affected_people"
    ]
)


latest_predicted_affected = max(
    0.0,
    latest_current_affected
    +
    latest_predicted_delta
)


# =========================================================
# PART B
# DEPLOYMENT SAFETY-CENTRE USAGE CLASSIFIER
# =========================================================

deployment_safety_classifier = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=3,
            max_features=0.7,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])


deployment_safety_classifier.fit(
    X_all,
    all_labelled_df[
        "target_next_safety_centre_use"
    ].astype(int)
)


latest_safety_use_probability = float(
    deployment_safety_classifier
    .predict_proba(
        X_latest
    )[0, 1]
)


latest_safety_use_prediction = int(
    latest_safety_use_probability
    >= 0.5
)


# =========================================================
# PART C
# DEPLOYMENT SAFETY-CENTRE AMOUNT MODEL
#
# Trained only on positive safety-demand observations.
# =========================================================

positive_mask = (
    all_labelled_df[
        "target_next_people_in_safety_centres"
    ] > 0
)


X_all_positive = (
    X_all.loc[
        positive_mask
    ]
    .copy()
)


all_positive_safety_delta = (
    all_labelled_df.loc[
        positive_mask,
        "target_next_people_in_safety_centres"
    ].astype(float)
    -
    all_labelled_df.loc[
        positive_mask,
        "people_in_safety_centres"
    ].astype(float)
)


all_positive_safety_delta_log = (
    signed_log1p(
        all_positive_safety_delta
    )
)


deployment_safety_amount_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=8,
            random_state=42
        )
    )
])


deployment_safety_amount_model.fit(
    X_all_positive,
    all_positive_safety_delta_log
)


latest_safety_delta_transformed = (
    deployment_safety_amount_model
    .predict(
        X_latest
    )
)


latest_safety_delta = float(
    inverse_signed_log1p(
        latest_safety_delta_transformed
    )[0]
)


latest_current_safety_population = float(
    latest[
        "people_in_safety_centres"
    ]
)


latest_predicted_safety_amount_raw = max(
    0.0,
    latest_current_safety_population
    +
    latest_safety_delta
)


# Stage-1 classifier acts as the gate.

latest_predicted_safety_population = (
    latest_predicted_safety_amount_raw
    if latest_safety_use_prediction == 1
    else 0.0
)


# =========================================================
# PART D
# DEPLOYMENT ESCALATION-RISK MODEL
#
# IMPORTANT:
# This model had limited temporal generalization on the
# held-out test period.
#
# Therefore its output remains EXPERIMENTAL and must not
# be interpreted as an official warning.
# =========================================================

deployment_risk_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "model",
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=15,
            min_samples_leaf=15,
            l2_regularization=3.0,
            class_weight="balanced",
            random_state=42
        )
    )
])


deployment_risk_model.fit(
    X_all,
    all_labelled_df[
        "impact_escalation_risk"
    ].astype(str)
)


latest_risk_prediction = (
    deployment_risk_model
    .predict(
        X_latest
    )[0]
)


risk_probabilities = (
    deployment_risk_model
    .predict_proba(
        X_latest
    )[0]
)


risk_classes = (
    deployment_risk_model
    .named_steps["model"]
    .classes_
)


latest_risk_probability_dict = dict(
    zip(
        risk_classes,
        risk_probabilities
    )
)


# =========================================================
# PART E
# DEPLOYMENT HUMAN-IMPACT UNCERTAINTY
#
# Fixed 90% model-based interval
# = 5th to 95th conditional quantile.
# =========================================================

def build_deployment_quantile_model(alpha):

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),

        (
            "model",
            GradientBoostingRegressor(
                loss="quantile",
                alpha=alpha,
                n_estimators=250,
                learning_rate=0.03,
                max_depth=2,
                min_samples_leaf=10,
                random_state=42
            )
        )
    ])


deployment_lower_model = (
    build_deployment_quantile_model(
        0.05
    )
)


deployment_upper_model = (
    build_deployment_quantile_model(
        0.95
    )
)


deployment_lower_model.fit(
    X_all,
    all_impact_delta_transformed
)


deployment_upper_model.fit(
    X_all,
    all_impact_delta_transformed
)


latest_lower_delta_transformed = (
    deployment_lower_model
    .predict(
        X_latest
    )
)


latest_upper_delta_transformed = (
    deployment_upper_model
    .predict(
        X_latest
    )
)


latest_lower_delta = float(
    inverse_signed_log1p(
        latest_lower_delta_transformed
    )[0]
)


latest_upper_delta = float(
    inverse_signed_log1p(
        latest_upper_delta_transformed
    )[0]
)


latest_lower_90 = max(
    0.0,
    latest_current_affected
    +
    min(
        latest_lower_delta,
        latest_upper_delta
    )
)


latest_upper_90 = max(
    0.0,
    latest_current_affected
    +
    max(
        latest_lower_delta,
        latest_upper_delta
    )
)


# =========================================================
# 3. HISTORICAL PERCENTILE HELPER
#
# Converts current values into a historical
# relative-position score from 0 to 100.
# =========================================================

def historical_percentile(
    value,
    historical_values,
    log_transform=False,
    positive_only=False
):

    values = pd.to_numeric(
        pd.Series(
            historical_values
        ),
        errors="coerce"
    ).dropna()


    if positive_only:

        values = values[
            values > 0
        ]

        if value <= 0:
            return 0.0


    if len(values) == 0:

        return np.nan


    if log_transform:

        values = np.log1p(
            np.clip(
                values,
                0,
                None
            )
        )

        value = np.log1p(
            max(
                float(value),
                0.0
            )
        )


    percentile = (
        (
            values <= value
        ).mean()
        * 100
    )


    return float(
        np.clip(
            percentile,
            0,
            100
        )
    )


# =========================================================
# 4. REFINED HAZARD-CONTEXT SCORE
#
# Three research-context categories:
#
# 1. Flood / River
# 2. Weather / Heavy Rain
# 3. Landslide
#
# Situation Report hazard presence gives a baseline
# contextual score of 50.
#
# Official warning severity may increase the category
# context toward 100.
#
# This is NOT an official warning score.
# =========================================================


# ---------------------------------------------------------
# 4.1 Current Situation Report hazard presence
# ---------------------------------------------------------

current_flood = safe_numeric(
    latest.get(
        "current_flood_flag",
        0
    )
)


current_landslide = safe_numeric(
    latest.get(
        "current_landslide_flag",
        0
    )
)


current_heavy_rain = safe_numeric(
    latest.get(
        "current_heavy_rain_flag",
        0
    )
)


# Protect against missing flags.

current_flood = (
    0.0
    if pd.isna(current_flood)
    else float(current_flood)
)


current_landslide = (
    0.0
    if pd.isna(current_landslide)
    else float(current_landslide)
)


current_heavy_rain = (
    0.0
    if pd.isna(current_heavy_rain)
    else float(current_heavy_rain)
)


# ---------------------------------------------------------
# 4.2 Weather context
# ---------------------------------------------------------

weather_level = safe_numeric(
    latest.get(
        "weather_max_warning_level_24h",
        np.nan
    )
)


weather_warning_score = (
    np.clip(
        weather_level / 3.0,
        0,
        1
    )
    * 100
    if pd.notna(weather_level)
    else 0.0
)


weather_context_score = float(
    max(
        50.0
        * current_heavy_rain,

        weather_warning_score
    )
)


# ---------------------------------------------------------
# 4.3 Landslide context
#
# NBRO warning severity:
# Level 1 = Yellow
# Level 2 = Amber
# Level 3 = Red
# ---------------------------------------------------------

landslide_level = safe_numeric(
    latest.get(
        "landslide_max_warning_level",
        np.nan
    )
)


landslide_warning_score = (
    np.clip(
        landslide_level / 3.0,
        0,
        1
    )
    * 100
    if pd.notna(landslide_level)
    else 0.0
)


landslide_context_score = float(
    max(
        50.0
        * current_landslide,

        landslide_warning_score
    )
)


# ---------------------------------------------------------
# 4.4 Flood / River context
#
# Situation Report flood presence = baseline 50
# Active recent river warning = 100
#
# Missing river information is NOT interpreted as
# "no flood".
# ---------------------------------------------------------

river_warning = safe_numeric(
    latest.get(
        "river_active_warning_24h",
        np.nan
    )
)


river_warning_score = (
    np.clip(
        river_warning,
        0,
        1
    )
    * 100
    if pd.notna(river_warning)
    else 0.0
)


flood_context_score = float(
    max(
        50.0
        * current_flood,

        river_warning_score
    )
)


# ---------------------------------------------------------
# 4.5 Combined hazard context
# ---------------------------------------------------------

hazard_context_score = float(
    np.mean(
        [
            flood_context_score,
            weather_context_score,
            landslide_context_score
        ]
    )
)


print(
    "\nHAZARD CONTEXT"
)


print(
    "Flood context score:",
    round(
        flood_context_score,
        2
    )
)


print(
    "Weather context score:",
    round(
        weather_context_score,
        2
    )
)


print(
    "Landslide context score:",
    round(
        landslide_context_score,
        2
    )
)


print(
    "Combined hazard context score:",
    round(
        hazard_context_score,
        2
    )
)


# =========================================================
# 5. LATEST REFINED RAPID-GROWTH SCORE
#
# From Section 12.2.
# =========================================================

latest_rapid_growth_score = safe_numeric(
    refined_trend_df
    .iloc[-1][
        "rapid_growth_score"
    ]
)


if pd.isna(
    latest_rapid_growth_score
):

    latest_rapid_growth_score = 0.0


latest_rapid_growth_score = float(
    np.clip(
        latest_rapid_growth_score,
        0,
        100
    )
)


# =========================================================
# 6. DRPI
#
# Disaster Response Priority Index
#
# Components:
#
# 1. Current human impact
# 2. Predicted impact growth
# 3. Predicted safety demand
# 4. Hazard context
# 5. Rapid-growth behaviour
#
# Equal weighting is intentionally used because there is
# no labelled "true response priority" dataset available
# to learn empirical weights.
# =========================================================


# ---------------------------------------------------------
# 6.1 Current Human Impact
# ---------------------------------------------------------

current_impact_score = (
    historical_percentile(
        latest_current_affected,

        all_labelled_df[
            "affected_people"
        ],

        log_transform=True
    )
)


# ---------------------------------------------------------
# 6.2 Predicted Impact Growth
# ---------------------------------------------------------

historical_impact_changes = (
    all_labelled_df[
        "target_next_affected_people"
    ]
    -
    all_labelled_df[
        "affected_people"
    ]
)


predicted_growth_score = (
    historical_percentile(
        latest_predicted_delta,

        historical_impact_changes,

        positive_only=True
    )
)


# ---------------------------------------------------------
# 6.3 Predicted Safety Demand
# ---------------------------------------------------------

predicted_safety_score = (
    historical_percentile(
        latest_predicted_safety_population,

        all_labelled_df[
            "target_next_people_in_safety_centres"
        ],

        log_transform=True
    )
)


# ---------------------------------------------------------
# 6.4 DRPI component dictionary
# ---------------------------------------------------------

DRPI_COMPONENTS = {

    "Current Human Impact":
        current_impact_score,

    "Predicted Impact Growth":
        predicted_growth_score,

    "Predicted Safety Demand":
        predicted_safety_score,

    "Hazard Context":
        hazard_context_score,

    "Rapid Growth":
        latest_rapid_growth_score
}


valid_drpi_components = [
    float(x)
    for x in DRPI_COMPONENTS.values()
    if pd.notna(x)
]


if len(valid_drpi_components) == 0:

    DRPI = np.nan

else:

    DRPI = float(
        np.mean(
            valid_drpi_components
        )
    )


DRPI = (
    float(
        np.clip(
            DRPI,
            0,
            100
        )
    )
    if pd.notna(DRPI)
    else np.nan
)


# =========================================================
# 7. SCPI
#
# Safety-Centre Pressure Index
#
# IMPORTANT:
# This is NOT actual occupancy/capacity utilization.
#
# Actual safety-centre capacity data are not available.
# Therefore SCPI represents relative research-derived
# safety-demand pressure only.
# =========================================================


# ---------------------------------------------------------
# 7.1 Current safety population
# ---------------------------------------------------------

current_safety_score = (
    historical_percentile(
        latest_current_safety_population,

        all_labelled_df[
            "people_in_safety_centres"
        ],

        log_transform=True
    )
)


# ---------------------------------------------------------
# 7.2 Forecast safety population
# ---------------------------------------------------------

forecast_safety_score = (
    historical_percentile(
        latest_predicted_safety_population,

        all_labelled_df[
            "target_next_people_in_safety_centres"
        ],

        log_transform=True
    )
)


# ---------------------------------------------------------
# 7.3 Predicted safety-population growth
# ---------------------------------------------------------

predicted_safety_increase = (
    latest_predicted_safety_population
    -
    latest_current_safety_population
)


historical_safety_changes = (
    all_labelled_df[
        "target_next_people_in_safety_centres"
    ]
    -
    all_labelled_df[
        "people_in_safety_centres"
    ]
)


safety_growth_score = (
    historical_percentile(
        predicted_safety_increase,

        historical_safety_changes,

        positive_only=True
    )
)


# ---------------------------------------------------------
# 7.4 SCPI components
# ---------------------------------------------------------

SCPI_COMPONENTS = {

    "Current Safety Population":
        current_safety_score,

    "Forecast Safety Population":
        forecast_safety_score,

    "Safety-Use Probability":
        latest_safety_use_probability
        * 100,

    "Predicted Safety Growth":
        safety_growth_score,

    "Hazard Context":
        hazard_context_score
}


valid_scpi_components = [
    float(x)
    for x in SCPI_COMPONENTS.values()
    if pd.notna(x)
]


if len(valid_scpi_components) == 0:

    SCPI = np.nan

else:

    SCPI = float(
        np.mean(
            valid_scpi_components
        )
    )


SCPI = (
    float(
        np.clip(
            SCPI,
            0,
            100
        )
    )
    if pd.notna(SCPI)
    else np.nan
)


# =========================================================
# 8. RESEARCH-DERIVED INDEX BANDS
#
# These bands belong only to FloodImpact-LK.
# =========================================================

def research_index_band(score):

    if pd.isna(score):
        return "Unavailable"

    if score < 25:
        return "Low"

    elif score < 50:
        return "Moderate"

    elif score < 75:
        return "High"

    else:
        return "Very High"


DRPI_BAND = (
    research_index_band(
        DRPI
    )
)


SCPI_BAND = (
    research_index_band(
        SCPI
    )
)


# =========================================================
# 9. FINAL CURRENT RESEARCH SNAPSHOT
# =========================================================

print(
    "\n========================================"
)


print(
    "LATEST FLOODIMPACT-LK RESEARCH SNAPSHOT"
)


print(
    "========================================"
)


print(
    "Report time:",
    latest[
        "availability_datetime"
    ]
)


# =========================================================
# 9.1 HUMAN IMPACT
# =========================================================

print(
    "\nHUMAN IMPACT"
)


print(
    "Current affected people:",
    round(
        latest_current_affected,
        2
    )
)


print(
    "Predicted next affected people:",
    round(
        latest_predicted_affected,
        2
    )
)


print(
    "Predicted change:",
    round(
        latest_predicted_delta,
        2
    )
)


print(
    "Model-based 90% range:",
    round(
        latest_lower_90,
        2
    ),
    "-",
    round(
        latest_upper_90,
        2
    )
)


# =========================================================
# 9.2 SAFETY-CENTRE DEMAND
# =========================================================

print(
    "\nSAFETY-CENTRE DEMAND"
)


print(
    "Current safety population:",
    round(
        latest_current_safety_population,
        2
    )
)


print(
    "Predicted safety-use probability:",
    round(
        latest_safety_use_probability,
        4
    )
)


print(
    "Predicted next safety population:",
    round(
        latest_predicted_safety_population,
        2
    )
)


# =========================================================
# 9.3 EXPERIMENTAL ESCALATION RISK
# =========================================================

print(
    "\nEXPERIMENTAL ESCALATION RISK"
)


print(
    "Predicted class:",
    latest_risk_prediction
)


print(
    "Class probabilities:"
)


for key, value in (
    latest_risk_probability_dict.items()
):

    print(
        f"  {key}: {value:.4f}"
    )


# =========================================================
# 9.4 DRPI OUTPUT
# =========================================================

print(
    "\nDRPI COMPONENTS"
)


for key, value in (
    DRPI_COMPONENTS.items()
):

    if pd.isna(value):

        print(
            f"{key}: Unavailable"
        )

    else:

        print(
            f"{key}: {value:.2f}"
        )


print(
    "\nDRPI:",
    round(
        DRPI,
        2
    )
    if pd.notna(DRPI)
    else "Unavailable",
    "/ 100"
    if pd.notna(DRPI)
    else ""
)


print(
    "DRPI Band:",
    DRPI_BAND
)


# =========================================================
# 9.5 SCPI OUTPUT
# =========================================================

print(
    "\nSCPI COMPONENTS"
)


for key, value in (
    SCPI_COMPONENTS.items()
):

    if pd.isna(value):

        print(
            f"{key}: Unavailable"
        )

    else:

        print(
            f"{key}: {value:.2f}"
        )


print(
    "\nSCPI:",
    round(
        SCPI,
        2
    )
    if pd.notna(SCPI)
    else "Unavailable",
    "/ 100"
    if pd.notna(SCPI)
    else ""
)


print(
    "SCPI Band:",
    SCPI_BAND
)


# =========================================================
# 10. FINAL SAFETY DISCLAIMER
# =========================================================

print(
    "\nNOTE:"
)


print(
    "DRPI and SCPI are research-derived FloodImpact-LK indicators."
)


print(
    "They are not official DMC/NBRO/Irrigation Department warnings."
)


print(
    "SCPI does not represent actual safety-centre occupancy capacity."
)


print(
    "Experimental escalation-risk predictions should not be "
    "interpreted as official operational warnings."
)

# 14. District-Level Sri Lanka Hazard and Priority Map

## 14.1 Prepare Latest District Hazard Context

In [ ]:
# =========================================================
# 14.1 LOAD DISTRICT-LEVEL HAZARD DATA FROM PROCESSED FILES
# =========================================================

from pathlib import Path
import pandas as pd


# Project processed-data folder
processed_dir = Path("data/processed")


# =========================================================
# 1. Landslide district-warning observations
# =========================================================

landslide_path = (
    processed_dir
    / "landslide_warning_areas.csv"
)


if not landslide_path.exists():

    raise FileNotFoundError(
        f"Landslide file not found: {landslide_path.resolve()}"
    )


landslide_warning_areas_df = pd.read_csv(
    landslide_path
)


print("=" * 60)
print("LANDSLIDE WARNING AREAS")
print("=" * 60)

print(
    "Shape:",
    landslide_warning_areas_df.shape
)

print(
    "\nColumns:"
)

print(
    landslide_warning_areas_df.columns.tolist()
)


print(
    "\nPossible district/location columns:"
)

landslide_location_columns = [
    col
    for col in landslide_warning_areas_df.columns
    if any(
        word in col.lower()
        for word in [
            "district",
            "area",
            "division",
            "location",
            "dsd"
        ]
    )
]

print(
    landslide_location_columns
)


print(
    "\nPossible date/time columns:"
)

landslide_datetime_columns = [
    col
    for col in landslide_warning_areas_df.columns
    if (
        "date" in col.lower()
        or "time" in col.lower()
    )
]

print(
    landslide_datetime_columns
)


print(
    "\nPossible warning/severity columns:"
)

landslide_warning_columns = [
    col
    for col in landslide_warning_areas_df.columns
    if any(
        word in col.lower()
        for word in [
            "warning",
            "level",
            "severity"
        ]
    )
]

print(
    landslide_warning_columns
)


print(
    "\nLast 5 rows:"
)

display(
    landslide_warning_areas_df.tail(5)
)


# =========================================================
# 2. River station label inspection
#
# We need this later for a station -> district lookup.
# =========================================================

print("\n")
print("=" * 60)
print("RIVER STATION LABELS")
print("=" * 60)


river_station_labels = (
    river_station_df[
        "station_label_raw"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)


print(
    "Unique station labels:",
    river_station_labels.nunique()
)


print(
    "\nFirst 50 unique station labels:"
)

print(
    sorted(
        river_station_labels.unique()
    )[:50]
)

## 14.2 Latest District-Level Hazard State

In [ ]:
# =========================================================
# 14.2 LATEST DISTRICT-LEVEL HAZARD STATE
#
# Purpose:
# 1. Build a complete 25-district Sri Lanka table
# 2. Attach snapshot-aligned landslide context
# 3. Inspect snapshot-aligned River conditions
#
# IMPORTANT:
# - Human-impact forecasting remains NATIONAL.
# - No district-level human-impact prediction is claimed.
# - No hazard information AFTER the Situation Report
#   snapshot time is allowed.
# =========================================================

import numpy as np
import pandas as pd


# =========================================================
# 0. BASIC CHECKS
# =========================================================

required_objects = [
    "landslide_warning_areas_df",
    "river_station_df",
    "latest_current_row"
]


missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]


if missing_objects:

    raise NameError(
        "Required notebook objects missing: "
        + str(missing_objects)
    )


# =========================================================
# 1. OFFICIAL 25 SRI LANKAN DISTRICTS
# =========================================================

SRI_LANKA_DISTRICTS = [
    "Ampara",
    "Anuradhapura",
    "Badulla",
    "Batticaloa",
    "Colombo",
    "Galle",
    "Gampaha",
    "Hambantota",
    "Jaffna",
    "Kalutara",
    "Kandy",
    "Kegalle",
    "Kilinochchi",
    "Kurunegala",
    "Mannar",
    "Matale",
    "Matara",
    "Monaragala",
    "Mullaitivu",
    "Nuwara Eliya",
    "Polonnaruwa",
    "Puttalam",
    "Ratnapura",
    "Trincomalee",
    "Vavuniya"
]


district_hazard_df = pd.DataFrame({
    "district": SRI_LANKA_DISTRICTS
})


# =========================================================
# 2. SNAPSHOT DATETIME
#
# All current map information must be available
# at or before this time.
# =========================================================

snapshot_datetime = pd.to_datetime(
    latest_current_row[
        "availability_datetime"
    ].iloc[0],
    errors="coerce"
)


if pd.isna(snapshot_datetime):

    raise ValueError(
        "Latest Situation Report availability_datetime "
        "could not be parsed."
    )


print("=" * 65)

print(
    "DISTRICT HAZARD SNAPSHOT AUDIT"
)

print("=" * 65)

print(
    "Snapshot datetime:",
    snapshot_datetime
)


# =========================================================
# PART A
# LANDSLIDE DISTRICT CONTEXT
# =========================================================


# =========================================================
# 3. PREPARE LANDSLIDE DATA
# =========================================================

landslide_map_df = (
    landslide_warning_areas_df
    .copy()
)


required_landslide_columns = [
    "report_date",
    "issue_time",
    "district",
    "warning_level"
]


missing_landslide_columns = [
    col
    for col in required_landslide_columns
    if col not in landslide_map_df.columns
]


if missing_landslide_columns:

    raise KeyError(
        "Missing landslide columns: "
        + str(missing_landslide_columns)
    )


landslide_map_df[
    "report_date"
] = pd.to_datetime(
    landslide_map_df[
        "report_date"
    ],
    errors="coerce"
)


# ---------------------------------------------------------
# Build issue datetime
# ---------------------------------------------------------

landslide_map_df[
    "issue_datetime"
] = pd.to_datetime(
    (
        landslide_map_df[
            "report_date"
        ]
        .dt.strftime("%Y-%m-%d")
        +
        " "
        +
        landslide_map_df[
            "issue_time"
        ]
        .fillna("00:00:00")
        .astype(str)
        .str.strip()
    ),
    errors="coerce"
)


print(
    "\nLandslide rows with valid timestamp:",
    int(
        landslide_map_df[
            "issue_datetime"
        ].notna().sum()
    )
)


# =========================================================
# 4. NORMALIZE DISTRICT NAMES
# =========================================================

def normalize_district_name(value):

    if pd.isna(value):
        return np.nan


    text = (
        str(value)
        .strip()
        .lower()
        .replace(".", "")
    )


    aliases = {

        "rathnapura":
            "Ratnapura",

        "ratnapura":
            "Ratnapura",

        "nuwara eliya":
            "Nuwara Eliya",

        "nuwaraeliya":
            "Nuwara Eliya",

        "moneragala":
            "Monaragala",

        "monaragala":
            "Monaragala",

        "kilinochchi district":
            "Kilinochchi",

        "trincomalee district":
            "Trincomalee",

        "mulativu":
            "Mullaitivu",

        "mullaitivu":
            "Mullaitivu"
    }


    if text in aliases:

        return aliases[text]


    compact = text.replace(
        " ",
        ""
    )


    if compact in aliases:

        return aliases[compact]


    return text.title()


landslide_map_df[
    "district_clean"
] = (
    landslide_map_df[
        "district"
    ]
    .apply(
        normalize_district_name
    )
)


unmatched_landslide_districts = sorted(
    set(
        landslide_map_df[
            "district_clean"
        ]
        .dropna()
        .unique()
    )
    -
    set(
        SRI_LANKA_DISTRICTS
    )
)


print(
    "Unmatched landslide district names:",
    unmatched_landslide_districts
)


# =========================================================
# 5. SNAPSHOT-ALIGNED LANDSLIDE DATA
#
# Only reports issued at or before snapshot time.
# =========================================================

eligible_landslide_records = (
    landslide_map_df[
        (
            landslide_map_df[
                "issue_datetime"
            ].notna()
        )
        &
        (
            landslide_map_df[
                "issue_datetime"
            ]
            <= snapshot_datetime
        )
    ]
    .copy()
)


if len(
    eligible_landslide_records
) == 0:

    latest_landslide_datetime = pd.NaT

else:

    latest_landslide_datetime = (
        eligible_landslide_records[
            "issue_datetime"
        ].max()
    )


print(
    "\nLatest landslide bulletin available by snapshot:",
    latest_landslide_datetime
)


# =========================================================
# 6. LANDSLIDE FRESHNESS CHECK
#
# Because the district-warning CSV does not contain a
# reliable per-area validity end-time, a conservative
# 24-hour freshness window is used.
#
# Stale data != "No warning".
# =========================================================

LANDSLIDE_VALID_HOURS = 24


if pd.notna(
    latest_landslide_datetime
):

    landslide_age_hours = (
        (
            snapshot_datetime
            -
            latest_landslide_datetime
        )
        .total_seconds()
        / 3600
    )

else:

    landslide_age_hours = np.nan


landslide_context_current = bool(
    pd.notna(
        landslide_age_hours
    )
    and
    (
        landslide_age_hours
        <= LANDSLIDE_VALID_HOURS
    )
)


print(
    "Landslide bulletin age (hours):",
    round(
        landslide_age_hours,
        2
    )
    if pd.notna(
        landslide_age_hours
    )
    else "Unavailable"
)


print(
    "Landslide context current:",
    landslide_context_current
)


# =========================================================
# 7. EXTRACT CURRENT LANDSLIDE BULLETIN
# =========================================================

if landslide_context_current:

    latest_landslide_records = (
        eligible_landslide_records[
            eligible_landslide_records[
                "issue_datetime"
            ]
            == latest_landslide_datetime
        ]
        .copy()
    )


    latest_landslide_records[
        "warning_level"
    ] = pd.to_numeric(
        latest_landslide_records[
            "warning_level"
        ],
        errors="coerce"
    )


    latest_landslide_by_district = (
        latest_landslide_records
        .groupby(
            "district_clean",
            as_index=False
        )
        .agg(
            landslide_warning_level=(
                "warning_level",
                "max"
            )
        )
    )

else:

    latest_landslide_records = pd.DataFrame()


    latest_landslide_by_district = (
        pd.DataFrame(
            columns=[
                "district_clean",
                "landslide_warning_level"
            ]
        )
    )


print(
    "Rows in current landslide bulletin:",
    len(
        latest_landslide_records
    )
)


# =========================================================
# 8. LANDSLIDE WARNING LABEL FUNCTION
# =========================================================

def landslide_warning_label(level):

    if pd.isna(level):

        return "Unavailable"


    if level >= 3:

        return "Level 3 - Red"


    elif level >= 2:

        return "Level 2 - Amber"


    elif level >= 1:

        return "Level 1 - Yellow"


    else:

        return "No Current Warning"


# =========================================================
# 9. MERGE LANDSLIDE CONTEXT TO ALL 25 DISTRICTS
# =========================================================

district_hazard_df = (
    district_hazard_df
    .merge(
        latest_landslide_by_district,
        left_on="district",
        right_on="district_clean",
        how="left"
    )
)


district_hazard_df.drop(
    columns=[
        "district_clean"
    ],
    inplace=True,
    errors="ignore"
)


# =========================================================
# 10. LANDSLIDE STATUS SEMANTICS
# =========================================================

if landslide_context_current:

    district_hazard_df[
        "landslide_warning_level"
    ] = (
        district_hazard_df[
            "landslide_warning_level"
        ]
        .fillna(0)
        .astype(int)
    )


    district_hazard_df[
        "landslide_warning"
    ] = (
        district_hazard_df[
            "landslide_warning_level"
        ]
        .apply(
            landslide_warning_label
        )
    )


    district_hazard_df[
        "landslide_priority_score"
    ] = (
        district_hazard_df[
            "landslide_warning_level"
        ]
        / 3.0
        * 100
    )


    district_hazard_df[
        "landslide_data_status"
    ] = "Current"


else:

    # -----------------------------------------------------
    # IMPORTANT:
    # Old/stale bulletin must NOT become zero/no-warning.
    # -----------------------------------------------------

    district_hazard_df[
        "landslide_warning_level"
    ] = np.nan


    district_hazard_df[
        "landslide_warning"
    ] = (
        "Current bulletin unavailable / stale"
    )


    district_hazard_df[
        "landslide_priority_score"
    ] = np.nan


    district_hazard_df[
        "landslide_data_status"
    ] = "Stale / Unavailable"


district_hazard_df[
    "landslide_data_age_hours"
] = landslide_age_hours


# =========================================================
# 11. NATIONAL MODEL CONTEXT
#
# Repeated on each row only for later dashboard/map popup
# convenience.
#
# These values remain NATIONAL, not district predictions.
# =========================================================

if "latest_current_affected" in globals():

    district_hazard_df[
        "national_current_affected"
    ] = latest_current_affected


if "latest_predicted_affected" in globals():

    district_hazard_df[
        "national_predicted_next_affected"
    ] = latest_predicted_affected


if "latest_risk_prediction" in globals():

    district_hazard_df[
        "national_experimental_risk"
    ] = latest_risk_prediction


if "DRPI" in globals():

    district_hazard_df[
        "national_DRPI"
    ] = DRPI


if "SCPI" in globals():

    district_hazard_df[
        "national_SCPI"
    ] = SCPI


# =========================================================
# 12. DISPLAY DISTRICT LANDSLIDE TABLE
# =========================================================

print(
    "\nCURRENT 25-DISTRICT LANDSLIDE CONTEXT"
)


print(
    district_hazard_df[
        [
            "district",
            "landslide_warning_level",
            "landslide_warning",
            "landslide_priority_score",
            "landslide_data_status"
        ]
    ]
    .sort_values(
        [
            "landslide_warning_level",
            "district"
        ],
        ascending=[
            False,
            True
        ],
        na_position="last"
    )
    .round(2)
    .to_string(
        index=False
    )
)


# =========================================================
# PART B
# RIVER STATION CONTEXT
# =========================================================


# =========================================================
# 13. PREPARE RIVER DATA
# =========================================================

river_map_df = (
    river_station_df
    .copy()
)


required_river_columns = [
    "report_date",
    "issue_time",
    "station_label_raw",
    "water_level_current",
    "alert_level",
    "minor_flood_level",
    "major_flood_level"
]


missing_river_columns = [
    col
    for col in required_river_columns
    if col not in river_map_df.columns
]


if missing_river_columns:

    raise KeyError(
        "Missing River columns: "
        + str(
            missing_river_columns
        )
    )


river_map_df[
    "report_date"
] = pd.to_datetime(
    river_map_df[
        "report_date"
    ],
    errors="coerce"
)


river_map_df[
    "issue_datetime"
] = pd.to_datetime(
    (
        river_map_df[
            "report_date"
        ]
        .dt.strftime(
            "%Y-%m-%d"
        )
        +
        " "
        +
        river_map_df[
            "issue_time"
        ]
        .fillna(
            "00:00:00"
        )
        .astype(str)
        .str.strip()
    ),
    errors="coerce"
)


# =========================================================
# 14. SNAPSHOT-ALIGNED RIVER RECORDS
#
# Never use observations after the latest Situation
# Report availability time.
# =========================================================

eligible_river_records = (
    river_map_df[
        (
            river_map_df[
                "issue_datetime"
            ].notna()
        )
        &
        (
            river_map_df[
                "issue_datetime"
            ]
            <= snapshot_datetime
        )
    ]
    .copy()
)


if len(
    eligible_river_records
) == 0:

    latest_river_datetime = pd.NaT

    latest_river_records = (
        pd.DataFrame()
    )


else:

    latest_river_datetime = (
        eligible_river_records[
            "issue_datetime"
        ].max()
    )


    latest_river_records = (
        eligible_river_records[
            eligible_river_records[
                "issue_datetime"
            ]
            == latest_river_datetime
        ]
        .copy()
    )


print(
    "\nLatest River observation available by snapshot:",
    latest_river_datetime
)


# =========================================================
# 15. RIVER FRESHNESS CHECK
# =========================================================

RIVER_VALID_HOURS = 24


if pd.notna(
    latest_river_datetime
):

    river_age_hours = (
        (
            snapshot_datetime
            -
            latest_river_datetime
        )
        .total_seconds()
        / 3600
    )

else:

    river_age_hours = np.nan


river_context_current = bool(
    pd.notna(
        river_age_hours
    )
    and
    (
        river_age_hours
        <= RIVER_VALID_HOURS
    )
)


print(
    "River observation age (hours):",
    round(
        river_age_hours,
        2
    )
    if pd.notna(
        river_age_hours
    )
    else "Unavailable"
)


print(
    "River context current:",
    river_context_current
)


print(
    "Stations in snapshot-aligned River report:",
    len(
        latest_river_records
    )
)


# =========================================================
# 16. PREPARE NUMERIC RIVER FIELDS
# =========================================================

if len(
    latest_river_records
) > 0:

    numeric_river_columns = [
        "water_level_current",
        "alert_level",
        "minor_flood_level",
        "major_flood_level",
        "alert_ratio",
        "rainfall_24h_mm",
        "level_change"
    ]


    for col in numeric_river_columns:

        if col in latest_river_records.columns:

            latest_river_records[
                col
            ] = pd.to_numeric(
                latest_river_records[
                    col
                ],
                errors="coerce"
            )


# =========================================================
# 17. RIVER STATION STATE FUNCTION
# =========================================================

def river_station_state(row):

    current = row.get(
        "water_level_current",
        np.nan
    )

    alert = row.get(
        "alert_level",
        np.nan
    )

    minor = row.get(
        "minor_flood_level",
        np.nan
    )

    major = row.get(
        "major_flood_level",
        np.nan
    )


    if pd.isna(current):

        return "Unknown"


    if (
        pd.notna(major)
        and
        current >= major
    ):

        return "Major Flood"


    if (
        pd.notna(minor)
        and
        current >= minor
    ):

        return "Minor Flood"


    if (
        pd.notna(alert)
        and
        current >= alert
    ):

        return "Alert"


    return "Normal"


# =========================================================
# 18. CALCULATE CURRENT RIVER STATES
# =========================================================

river_state_rank = {
    "Major Flood": 4,
    "Minor Flood": 3,
    "Alert": 2,
    "Normal": 1,
    "Unknown": 0
}


if (
    river_context_current
    and
    len(
        latest_river_records
    ) > 0
):

    latest_river_records[
        "river_state"
    ] = (
        latest_river_records
        .apply(
            river_station_state,
            axis=1
        )
    )


    latest_river_records[
        "river_state_rank"
    ] = (
        latest_river_records[
            "river_state"
        ]
        .map(
            river_state_rank
        )
        .fillna(0)
        .astype(int)
    )


else:

    # No current River context available.
    if len(
        latest_river_records
    ) > 0:

        latest_river_records[
            "river_state"
        ] = (
            "Stale / Unavailable"
        )


        latest_river_records[
            "river_state_rank"
        ] = 0


# =========================================================
# 19. EXTRACT CURRENT ALERT/FLOOD STATIONS
# =========================================================

if (
    river_context_current
    and
    len(
        latest_river_records
    ) > 0
):

    latest_river_relevant_df = (
        latest_river_records[
            latest_river_records[
                "river_state"
            ]
            .isin(
                [
                    "Alert",
                    "Minor Flood",
                    "Major Flood"
                ]
            )
        ]
        .sort_values(
            [
                "river_state_rank",
                "alert_ratio"
            ],
            ascending=[
                False,
                False
            ],
            na_position="last"
        )
        [
            [
                col
                for col in [
                    "station_label_raw",
                    "water_level_current",
                    "alert_level",
                    "minor_flood_level",
                    "major_flood_level",
                    "alert_ratio",
                    "river_state",
                    "reported_trend",
                    "derived_trend"
                ]
                if col
                in latest_river_records.columns
            ]
        ]
        .copy()
    )


else:

    latest_river_relevant_df = pd.DataFrame()


# =========================================================
# 20. DISPLAY LATEST RIVER CONDITIONS
# =========================================================

print(
    "\nLATEST RIVER STATIONS AT ALERT / FLOOD LEVEL"
)


if not river_context_current:

    print(
        "Current River context unavailable or stale "
        "for this Situation Report snapshot."
    )


elif len(
    latest_river_relevant_df
) == 0:

    print(
        "No snapshot-aligned stations are at Alert, "
        "Minor Flood, or Major Flood level."
    )


else:

    print(
        latest_river_relevant_df
        .round(3)
        .to_string(
            index=False
        )
    )


# =========================================================
# 21. RIVER STATE SUMMARY
# =========================================================

if (
    river_context_current
    and
    len(
        latest_river_records
    ) > 0
):

    print(
        "\nRIVER STATE DISTRIBUTION"
    )


    print(
        latest_river_records[
            "river_state"
        ]
        .value_counts()
    )


# =========================================================
# 22. FINAL MAP-DATA AUDIT
# =========================================================

print(
    "\n"
    + "=" * 65
)


print(
    "FINAL MAP DATA AUDIT"
)


print(
    "=" * 65
)


print(
    "Situation snapshot:",
    snapshot_datetime
)


print(
    "District rows:",
    len(
        district_hazard_df
    )
)


print(
    "Landslide current:",
    landslide_context_current
)


print(
    "Landslide age hours:",
    round(
        landslide_age_hours,
        2
    )
    if pd.notna(
        landslide_age_hours
    )
    else "Unavailable"
)


print(
    "River current:",
    river_context_current
)


print(
    "River age hours:",
    round(
        river_age_hours,
        2
    )
    if pd.notna(
        river_age_hours
    )
    else "Unavailable"
)


print(
    "River stations at alert/flood:",
    len(
        latest_river_relevant_df
    )
)


print(
    "\nIMPORTANT:"
)


print(
    "District-level values represent hazard context only."
)


print(
    "Human-impact and safety-demand forecasts remain "
    "national-level outputs."
)

## 14.3 Interactive Sri Lanka District Hazard and Priority Map

The map visualizes district-level hazard context available at the Situation Report
snapshot time. National human-impact and safety-demand forecasts are shown only
as national context and are not interpreted as district-level predictions.

Stale or unavailable district hazard information is explicitly shown as unavailable
rather than being interpreted as the absence of a hazard.

In [ ]:
import sys

print("Notebook Python:")
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install folium

In [ ]:
import folium

print("Folium version:", folium.__version__)
print("Folium imported successfully")

In [ ]:
# =========================================================
# 14.3 INTERACTIVE SRI LANKA DISTRICT HAZARD MAP
#
# District level:
# - Landslide warning/context when current
# - Research-derived landslide priority
# - Data freshness/status
#
# National context:
# - Current affected population
# - Predicted next affected population
# - Model-based 90% range
# - Safety-centre demand
# - Experimental escalation risk
# - DRPI / SCPI
#
# IMPORTANT:
# Human-impact forecasts remain NATIONAL.
# No district-level human-impact forecast is claimed.
# =========================================================

import json
import numpy as np
import pandas as pd
import requests

try:
    import folium

    from folium.features import (
        GeoJson,
        GeoJsonTooltip
    )

except ImportError:

    raise ImportError(
        "Folium is not installed. "
        "Run: pip install folium"
    )


# =========================================================
# 1. DOWNLOAD SRI LANKA ADM2 DISTRICT BOUNDARIES
#
# geoBoundaries ADM2 contains 25 Sri Lankan districts.
# =========================================================

GEBOUNDARIES_API = (
    "https://www.geoboundaries.org/"
    "api/current/gbOpen/LKA/ADM2/"
)


response = requests.get(
    GEBOUNDARIES_API,
    timeout=30
)

response.raise_for_status()


boundary_metadata = (
    response.json()
)


geojson_url = (
    boundary_metadata[
        "simplifiedGeometryGeoJSON"
    ]
)


print(
    "Boundary source:",
    boundary_metadata[
        "boundarySource"
    ]
)

print(
    "Boundary administrative units:",
    boundary_metadata[
        "admUnitCount"
    ]
)

print(
    "Boundary year represented:",
    boundary_metadata[
        "boundaryYearRepresented"
    ]
)


# =========================================================
# 2. DOWNLOAD GEOJSON
# =========================================================

geo_response = requests.get(
    geojson_url,
    timeout=60
)

geo_response.raise_for_status()


sri_lanka_geojson = (
    geo_response.json()
)


print(
    "GeoJSON district features:",
    len(
        sri_lanka_geojson[
            "features"
        ]
    )
)


# =========================================================
# 3. INSPECT GEOJSON NAME FIELD
# =========================================================

first_properties = (
    sri_lanka_geojson[
        "features"
    ][0]["properties"]
)


print(
    "GeoJSON property fields:",
    list(
        first_properties.keys()
    )
)


# geoBoundaries normally uses shapeName.
if "shapeName" in first_properties:

    GEOJSON_NAME_FIELD = (
        "shapeName"
    )

elif "shape_name" in first_properties:

    GEOJSON_NAME_FIELD = (
        "shape_name"
    )

elif "NAME_2" in first_properties:

    GEOJSON_NAME_FIELD = (
        "NAME_2"
    )

elif "name" in first_properties:

    GEOJSON_NAME_FIELD = (
        "name"
    )

else:

    raise KeyError(
        "Unable to identify district-name "
        "field in GeoJSON."
    )


print(
    "GeoJSON district-name field:",
    GEOJSON_NAME_FIELD
)

# =========================================================
# 4. NORMALIZE GEOJSON DISTRICT NAMES
# =========================================================

def normalize_map_district(value):

    if pd.isna(value):
        return np.nan

    text = (
        str(value)
        .strip()
        .lower()
        .replace(".", "")
    )

    # -----------------------------------------------------
    # Remove GeoJSON suffix such as:
    # "Ampara District" -> "Ampara"
    # -----------------------------------------------------

    if text.endswith(" district"):
        text = text[:-9].strip()


    aliases = {

        "rathnapura":
            "Ratnapura",

        "ratnapura":
            "Ratnapura",

        "nuwaraeliya":
            "Nuwara Eliya",

        "nuwara eliya":
            "Nuwara Eliya",

        "moneragala":
            "Monaragala",

        "monaragala":
            "Monaragala",

        "mulativu":
            "Mullaitivu",

        "mullaitivu":
            "Mullaitivu",

        "kilinochchi":
            "Kilinochchi",

        "trincomalee":
            "Trincomalee"
    }


    if text in aliases:
        return aliases[text]


    compact = text.replace(
        " ",
        ""
    )


    compact_aliases = {

        "nuwaraeliya":
            "Nuwara Eliya",

        "rathnapura":
            "Ratnapura",

        "moneragala":
            "Monaragala",

        "mullaitivu":
            "Mullaitivu",

        "mulativu":
            "Mullaitivu",

        "kilinochchi":
            "Kilinochchi",

        "trincomalee":
            "Trincomalee"
    }


    if compact in compact_aliases:
        return compact_aliases[compact]


    return text.title()

# =========================================================
# APPLY NORMALIZATION TO GEOJSON DISTRICT FEATURES
# =========================================================

for feature in sri_lanka_geojson["features"]:

    original_name = (
        feature["properties"]
        .get(GEOJSON_NAME_FIELD)
    )

    feature["properties"]["district_clean"] = (
        normalize_map_district(
            original_name
        )
    )

# =========================================================
# 5. CHECK 25 DISTRICT MATCH
# =========================================================

geojson_districts = sorted(
    [
        feature[
            "properties"
        ][
            "district_clean"
        ]

        for feature in (
            sri_lanka_geojson[
                "features"
            ]
        )
    ]
)


dataset_districts = sorted(
    district_hazard_df[
        "district"
    ].tolist()
)


missing_from_geojson = sorted(
    set(
        dataset_districts
    )
    -
    set(
        geojson_districts
    )
)


extra_geojson_names = sorted(
    set(
        geojson_districts
    )
    -
    set(
        dataset_districts
    )
)


print(
    "\nDistricts missing from GeoJSON:",
    missing_from_geojson
)


print(
    "Unexpected GeoJSON district names:",
    extra_geojson_names
)


# =========================================================
# 6. PREPARE MAP TABLE
# =========================================================

district_map_df = (
    district_hazard_df
    .copy()
)


# =========================================================
# 6.1 Current district priority
#
# Current landslide warning:
# 0   -> no warning
# 33  -> Level 1
# 67  -> Level 2
# 100 -> Level 3
#
# If landslide bulletin is stale,
# priority remains NaN / unavailable.
# =========================================================

district_map_df[
    "district_priority_score"
] = district_map_df[
    "landslide_priority_score"
]


# =========================================================
# 6.2 Map status
# =========================================================

def district_map_status(row):

    if (
        row[
            "landslide_data_status"
        ]
        != "Current"
    ):

        return (
            "District landslide data "
            "stale / unavailable"
        )


    level = row[
        "landslide_warning_level"
    ]


    if pd.isna(level):

        return (
            "District hazard "
            "information unavailable"
        )


    if level >= 3:

        return (
            "Level 3 Landslide Warning"
        )


    elif level >= 2:

        return (
            "Level 2 Landslide Warning"
        )


    elif level >= 1:

        return (
            "Level 1 Landslide Warning"
        )


    else:

        return (
            "No current landslide warning"
        )


district_map_df[
    "district_map_status"
] = (
    district_map_df
    .apply(
        district_map_status,
        axis=1
    )
)


# =========================================================
# 7. ADD DISTRICT DATA TO GEOJSON PROPERTIES
# =========================================================

district_lookup = (
    district_map_df
    .set_index(
        "district"
    )
    .to_dict(
        orient="index"
    )
)


for feature in (
    sri_lanka_geojson[
        "features"
    ]
):

    district_name = (
        feature[
            "properties"
        ][
            "district_clean"
        ]
    )


    district_info = (
        district_lookup.get(
            district_name,
            {}
        )
    )


    warning_level = (
        district_info.get(
            "landslide_warning_level",
            np.nan
        )
    )


    priority_score = (
        district_info.get(
            "district_priority_score",
            np.nan
        )
    )


    feature[
        "properties"
    ][
        "landslide_warning"
    ] = (
        str(
            district_info.get(
                "landslide_warning",
                "Unavailable"
            )
        )
    )


    feature[
        "properties"
    ][
        "landslide_data_status"
    ] = (
        str(
            district_info.get(
                "landslide_data_status",
                "Unavailable"
            )
        )
    )


    feature[
        "properties"
    ][
        "map_status"
    ] = (
        str(
            district_info.get(
                "district_map_status",
                "Unavailable"
            )
        )
    )


    feature[
        "properties"
    ][
        "warning_level"
    ] = (
        None
        if pd.isna(
            warning_level
        )
        else int(
            warning_level
        )
    )


    feature[
        "properties"
    ][
        "priority_score"
    ] = (
        None
        if pd.isna(
            priority_score
        )
        else round(
            float(
                priority_score
            ),
            2
        )
    )


# =========================================================
# 8. DISTRICT STYLE FUNCTION
#
# IMPORTANT:
# Grey = data stale/unavailable,
# NOT "no hazard".
# =========================================================

def district_style_function(
    feature
):

    properties = (
        feature[
            "properties"
        ]
    )


    level = (
        properties.get(
            "warning_level"
        )
    )


    data_status = (
        properties.get(
            "landslide_data_status"
        )
    )


    # Stale / unavailable
    if (
        data_status != "Current"
    ):

        fill_color = "#BDBDBD"


    elif level is None:

        fill_color = "#BDBDBD"


    elif level >= 3:

        fill_color = "#D73027"


    elif level >= 2:

        fill_color = "#FDAE61"


    elif level >= 1:

        fill_color = "#FFFF66"


    else:

        fill_color = "#D9F0D3"


    return {

        "fillColor":
            fill_color,

        "color":
            "#555555",

        "weight":
            1.0,

        "fillOpacity":
            0.72
    }


# =========================================================
# 9. CREATE MAP
# =========================================================

sri_lanka_hazard_map = (
    folium.Map(
        location=[
            7.8731,
            80.7718
        ],

        zoom_start=7,

        tiles=(
            "CartoDB positron"
        ),

        control_scale=True
    )
)


# =========================================================
# 10. DISTRICT LAYER
# =========================================================

district_geojson_layer = (
    folium.GeoJson(

        sri_lanka_geojson,

        name=(
            "District Hazard Context"
        ),

        style_function=(
            district_style_function
        ),

        highlight_function=lambda x: {

            "weight": 3,

            "color": "#000000",

            "fillOpacity": 0.85
        },

        tooltip=(
            GeoJsonTooltip(

                fields=[
                    "district_clean",
                    "landslide_warning",
                    "landslide_data_status",
                    "priority_score"
                ],

                aliases=[
                    "District:",
                    "Landslide context:",
                    "Data status:",
                    "Research priority score:"
                ],

                localize=True,

                sticky=False
            )
        )
    )
)


district_geojson_layer.add_to(
    sri_lanka_hazard_map
)


# =========================================================
# 11. NATIONAL SNAPSHOT INFORMATION PANEL
# =========================================================

national_info_html = f"""
<div style="
    position: fixed;
    top: 10px;
    right: 10px;
    width: 340px;
    z-index: 9999;
    background-color: white;
    border: 2px solid #666;
    border-radius: 8px;
    padding: 12px;
    font-size: 13px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25);
">

<b style="font-size:16px;">
FloodImpact-LK
</b>

<br>

<b>Snapshot:</b>
{snapshot_datetime}

<hr style="margin:6px 0;">

<b>National Human Impact</b><br>

Current affected:
{latest_current_affected:,.0f}

<br>

Predicted next:
{latest_predicted_affected:,.0f}

<br>

90% model-based range:
{latest_lower_90:,.0f}
-
{latest_upper_90:,.0f}

<hr style="margin:6px 0;">

<b>Safety-Centre Demand</b><br>

Current population:
{latest_current_safety_population:,.0f}

<br>

Predicted next:
{latest_predicted_safety_population:,.0f}

<br>

Use probability:
{latest_safety_use_probability * 100:.1f}%

<hr style="margin:6px 0;">

<b>Experimental escalation risk:</b>
{latest_risk_prediction}

<br>

<b>DRPI:</b>
{DRPI:.2f}/100 ({DRPI_BAND})

<br>

<b>SCPI:</b>
{SCPI:.2f}/100 ({SCPI_BAND})

<hr style="margin:6px 0;">

<b>River context:</b><br>

Latest observation:
{latest_river_datetime}

<br>

Age:
{river_age_hours:.1f} h

<br>

Alert/Flood stations:
{len(latest_river_relevant_df)}

<hr style="margin:6px 0;">

<b>Landslide context:</b><br>

Latest bulletin:
{latest_landslide_datetime}

<br>

Age:
{landslide_age_hours:.1f} h

<br>

Status:
{
    "Current"
    if landslide_context_current
    else "Stale / unavailable"
}

<br><br>

<span style="
    font-size:11px;
    color:#555;
">
District colours represent available hazard context only.
National forecasts are not district-level predictions.
DRPI/SCPI and escalation risk are research-derived,
not official warnings.
</span>

</div>
"""
(

sri_lanka_hazard_map
.get_root()
.html
.add_child(
    folium.Element(
        national_info_html
    )
)
)

# =========================================================
# 12. LEGEND
# =========================================================

legend_html = """
<div style="
    position: fixed;
    bottom: 25px;
    left: 25px;
    width: 235px;
    z-index: 9999;
    background-color: white;
    border: 2px solid #777;
    border-radius: 6px;
    padding: 10px;
    font-size: 12px;
">

<b>District Landslide Context</b>

<br><br>

<span style="
display:inline-block;
width:14px;
height:14px;
background:#D9F0D3;
border:1px solid #555;">
</span>
No current warning

<br>

<span style="
display:inline-block;
width:14px;
height:14px;
background:#FFFF66;
border:1px solid #555;">
</span>
Level 1 - Yellow

<br>

<span style="
display:inline-block;
width:14px;
height:14px;
background:#FDAE61;
border:1px solid #555;">
</span>
Level 2 - Amber

<br>

<span style="
display:inline-block;
width:14px;
height:14px;
background:#D73027;
border:1px solid #555;">
</span>
Level 3 - Red

<br>

<span style="
display:inline-block;
width:14px;
height:14px;
background:#BDBDBD;
border:1px solid #555;">
</span>
Stale / unavailable data

</div>
"""

(
sri_lanka_hazard_map
.get_root()
.html
.add_child(
    folium.Element(
        legend_html
    )
)
)

# =========================================================
# 13. RIVER STATUS MARKER / MESSAGE
#
# At the current snapshot there are no Alert/Flood
# stations, so no misleading station markers are added.
#
# If active stations exist later, they can be added once
# verified station coordinates/district mapping is available.
# =========================================================

river_summary_html = f"""
<b>Snapshot-aligned River Monitoring</b><br>
Latest observation: {latest_river_datetime}<br>
Stations observed: {len(latest_river_records)}<br>
Alert/Flood stations: {len(latest_river_relevant_df)}
"""


folium.Marker(

    location=[
        7.8731,
        80.7718
    ],

    popup=folium.Popup(
        river_summary_html,
        max_width=300
    ),

    tooltip=(
        "National River Monitoring Summary"
    ),

    icon=folium.Icon(
        icon="info-sign"
    )

).add_to(
    sri_lanka_hazard_map
)


# =========================================================
# 14. LAYER CONTROL
# =========================================================

folium.LayerControl(
    collapsed=False
).add_to(
    sri_lanka_hazard_map
)


# =========================================================
# 15. SAVE MAP
# =========================================================

map_output_path = (
    "data/processed/"
    "floodimpact_lk_district_hazard_map.html"
)


sri_lanka_hazard_map.save(
    map_output_path
)


print(
    "\nInteractive map saved to:",
    map_output_path
)


# =========================================================
# 16. FINAL MAP QC
# =========================================================

print(
    "\nINTERACTIVE MAP QC"
)


print(
    "GeoJSON districts:",
    len(
        sri_lanka_geojson[
            "features"
        ]
    )
)


print(
    "Dataset districts:",
    len(
        district_map_df
    )
)


print(
    "Missing district matches:",
    missing_from_geojson
)


print(
    "Extra GeoJSON district names:",
    extra_geojson_names
)


print(
    "Landslide layer current:",
    landslide_context_current
)


print(
    "River layer current:",
    river_context_current
)


print(
    "Active River alert/flood stations:",
    len(
        latest_river_relevant_df
    )
)


# =========================================================
# 17. DISPLAY MAP IN NOTEBOOK
# =========================================================

sri_lanka_hazard_map

# 15. FloodImpact-LK Integrated Decision-Support Dashboard

The dashboard integrates the latest national human-impact forecast, safety-centre
demand estimate, model-based uncertainty, experimental escalation risk, research-derived
priority indices, trend monitoring, historical event similarity, and district-level
hazard context.

Human-impact and safety-demand forecasts are national-level outputs. District-level
visualization represents available hazard context only.

In [ ]:
# =========================================================
# 15.1 FLOODIMPACT-LK FINAL INTEGRATED DASHBOARD
#
# Creates a standalone HTML dashboard.
#
# Includes:
# - Latest situation snapshot
# - Human-impact forecast + uncertainty
# - Safety-centre demand
# - Experimental escalation risk
# - DRPI / SCPI
# - Trend / anomaly monitoring
# - Similar historical events
# - Data freshness
# - Interactive district hazard map
# =========================================================

from pathlib import Path
from IPython.display import display, HTML

import pandas as pd
import numpy as np


# =========================================================
# 1. OUTPUT PATHS
# =========================================================

processed_dir = Path(
    "data/processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)


dashboard_path = (
    processed_dir
    / "floodimpact_lk_dashboard.html"
)


map_filename = (
    "floodimpact_lk_district_hazard_map.html"
)


# =========================================================
# 2. LATEST TREND INFORMATION
# =========================================================

latest_dashboard_trend = (
    refined_trend_df
    .sort_values(
        "availability_datetime"
    )
    .iloc[-1]
)


latest_trend_label = (
    latest_dashboard_trend[
        "impact_trend"
    ]
)


latest_monitoring_status = (
    latest_dashboard_trend[
        "impact_monitoring_status"
    ]
)


latest_growth_score = (
    latest_dashboard_trend[
        "rapid_growth_score"
    ]
)


if pd.isna(
    latest_growth_score
):

    latest_growth_score_display = (
        "Unavailable"
    )

else:

    latest_growth_score_display = (
        f"{latest_growth_score:.2f}/100"
    )


# =========================================================
# 3. RISK PROBABILITY DISPLAY
# =========================================================

risk_probability_rows = ""


for risk_class in [
    "Low",
    "Moderate",
    "High",
    "Critical"
]:

    probability = (
        latest_risk_probability_dict
        .get(
            risk_class,
            0
        )
        * 100
    )


    risk_probability_rows += f"""
        <tr>
            <td>{risk_class}</td>
            <td>{probability:.1f}%</td>
        </tr>
    """


# =========================================================
# 4. SIMILAR EVENT TABLE
# =========================================================

similar_event_rows = ""


if (
    "latest_refined_similar_events"
    in globals()
    and
    len(
        latest_refined_similar_events
    ) > 0
):

    for _, row in (
        latest_refined_similar_events
        .head(5)
        .iterrows()
    ):

        historical_date = pd.to_datetime(
            row[
                "historical_datetime"
            ]
        ).strftime(
            "%Y-%m-%d %H:%M"
        )


        similar_event_rows += f"""
            <tr>
                <td>{historical_date}</td>
                <td>{row['similarity_score']:.3f}</td>
                <td>{row['historical_affected_people']:,.0f}</td>
                <td>{row['historical_people_in_safety_centres']:,.0f}</td>
                <td>{row['historical_escalation_risk']}</td>
            </tr>
        """

else:

    similar_event_rows = """
        <tr>
            <td colspan="5">
                Similar-event data unavailable
            </td>
        </tr>
    """


# =========================================================
# 5. DATA FRESHNESS DISPLAY
# =========================================================

if pd.notna(
    landslide_age_hours
):

    landslide_age_display = (
        f"{landslide_age_hours:.1f} h"
    )

else:

    landslide_age_display = (
        "Unavailable"
    )


if pd.notna(
    river_age_hours
):

    river_age_display = (
        f"{river_age_hours:.1f} h"
    )

else:

    river_age_display = (
        "Unavailable"
    )


landslide_status_display = (
    "Current"
    if landslide_context_current
    else "Stale / Unavailable"
)


river_status_display = (
    "Current"
    if river_context_current
    else "Stale / Unavailable"
)


# =========================================================
# 6. HELPER FOR INDEX CARD CLASS
# =========================================================

def dashboard_band_class(
    band
):

    mapping = {
        "Low":
            "band-low",

        "Moderate":
            "band-moderate",

        "High":
            "band-high",

        "Very High":
            "band-very-high",

        "Unavailable":
            "band-unavailable"
    }


    return mapping.get(
        band,
        "band-unavailable"
    )


drpi_css_class = (
    dashboard_band_class(
        DRPI_BAND
    )
)


scpi_css_class = (
    dashboard_band_class(
        SCPI_BAND
    )
)


# =========================================================
# 7. FORMAT CURRENT SNAPSHOT DATETIME
# =========================================================

dashboard_snapshot = (
    pd.to_datetime(
        snapshot_datetime
    )
    .strftime(
        "%Y-%m-%d %H:%M"
    )
)


# =========================================================
# 8. BUILD HTML DASHBOARD
# =========================================================

dashboard_html = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1">

<title>
FloodImpact-LK Dashboard
</title>


<style>

    * {{
        box-sizing: border-box;
    }}


    body {{
        margin: 0;
        font-family:
            Arial,
            Helvetica,
            sans-serif;

        background:
            #f4f6f8;

        color:
            #1f2937;
    }}


    .header {{
        background:
            #16324f;

        color:
            white;

        padding:
            22px 28px;
    }}


    .header h1 {{
        margin: 0;
        font-size: 28px;
    }}


    .header p {{
        margin:
            8px 0 0 0;

        opacity:
            0.90;
    }}


    .container {{
        width:
            min(
                1500px,
                96%
            );

        margin:
            20px auto 40px auto;
    }}


    .notice {{
        background:
            #fff8e1;

        border-left:
            5px solid #f0ad4e;

        padding:
            12px 16px;

        margin-bottom:
            18px;

        border-radius:
            5px;
    }}


    .grid {{
        display:
            grid;

        grid-template-columns:
            repeat(
                auto-fit,
                minmax(
                    220px,
                    1fr
                )
            );

        gap:
            14px;

        margin-bottom:
            20px;
    }}


    .card {{
        background:
            white;

        border-radius:
            10px;

        padding:
            18px;

        box-shadow:
            0 2px 8px
            rgba(
                0,
                0,
                0,
                0.08
            );
    }}


    .card-title {{
        font-size:
            13px;

        color:
            #6b7280;

        text-transform:
            uppercase;

        letter-spacing:
            0.5px;

        margin-bottom:
            8px;
    }}


    .card-value {{
        font-size:
            26px;

        font-weight:
            bold;

        color:
            #16324f;
    }}


    .card-sub {{
        margin-top:
            7px;

        font-size:
            13px;

        color:
            #6b7280;
    }}


    .band-low {{
        border-top:
            5px solid #4caf50;
    }}


    .band-moderate {{
        border-top:
            5px solid #f0ad4e;
    }}


    .band-high {{
        border-top:
            5px solid #ef6c00;
    }}


    .band-very-high {{
        border-top:
            5px solid #c62828;
    }}


    .band-unavailable {{
        border-top:
            5px solid #9e9e9e;
    }}


    .section {{
        background:
            white;

        border-radius:
            10px;

        padding:
            20px;

        margin-bottom:
            20px;

        box-shadow:
            0 2px 8px
            rgba(
                0,
                0,
                0,
                0.08
            );
    }}


    .section h2 {{
        margin-top:
            0;

        color:
            #16324f;
    }}


    .two-column {{
        display:
            grid;

        grid-template-columns:
            repeat(
                auto-fit,
                minmax(
                    350px,
                    1fr
                )
            );

        gap:
            20px;
    }}


    table {{
        width:
            100%;

        border-collapse:
            collapse;
    }}


    th,
    td {{
        padding:
            10px;

        border-bottom:
            1px solid #e5e7eb;

        text-align:
            left;

        font-size:
            14px;
    }}


    th {{
        background:
            #f8fafc;

        color:
            #374151;
    }}


    iframe {{
        width:
            100%;

        height:
            650px;

        border:
            none;

        border-radius:
            8px;
    }}


    .status-current {{
        color:
            #2e7d32;

        font-weight:
            bold;
    }}


    .status-stale {{
        color:
            #c62828;

        font-weight:
            bold;
    }}


    .footer-note {{
        color:
            #6b7280;

        font-size:
            12px;

        line-height:
            1.6;
    }}


    .metric-note {{
        font-size:
            12px;

        color:
            #6b7280;

        margin-top:
            4px;
    }}


</style>

</head>


<body>


<div class="header">

    <h1>
        FloodImpact-LK
    </h1>

    <p>
        Multi-Source Spatio-Temporal Flood and
        Rainfall-Induced Landslide Decision-Support Dashboard
    </p>

    <p>
        Snapshot:
        <b>{dashboard_snapshot}</b>
    </p>

</div>


<div class="container">


<div class="notice">

    <b>Research-use notice:</b>

    FloodImpact-LK is a research prototype.
    Human-impact and safety-demand forecasts are national-level.
    DRPI, SCPI and escalation-risk categories are research-derived
    indicators and are not official DMC, NBRO or Irrigation Department warnings.

</div>


<!-- ===================================================== -->
<!-- MAIN KPI CARDS -->
<!-- ===================================================== -->

<div class="grid">


    <div class="card">

        <div class="card-title">
            Current Affected People
        </div>

        <div class="card-value">
            {latest_current_affected:,.0f}
        </div>

        <div class="card-sub">
            Latest national Situation Report
        </div>

    </div>


    <div class="card">

        <div class="card-title">
            Predicted Next Affected
        </div>

        <div class="card-value">
            {latest_predicted_affected:,.0f}
        </div>

        <div class="card-sub">
            Predicted change:
            {latest_predicted_delta:+,.0f}
        </div>

    </div>


    <div class="card">

        <div class="card-title">
            Model-Based 90% Range
        </div>

        <div class="card-value"
             style="font-size:20px;">

            {latest_lower_90:,.0f}
            –
            {latest_upper_90:,.0f}

        </div>

        <div class="card-sub">
            Empirical test coverage was 80.67%
        </div>

    </div>


    <div class="card">

        <div class="card-title">
            Safety-Centre Use Probability
        </div>

        <div class="card-value">
            {latest_safety_use_probability * 100:.1f}%
        </div>

        <div class="card-sub">
            Predicted next population:
            {latest_predicted_safety_population:,.0f}
        </div>

    </div>


    <div class="card {drpi_css_class}">

        <div class="card-title">
            DRPI
        </div>

        <div class="card-value">
            {DRPI:.2f}/100
        </div>

        <div class="card-sub">
            {DRPI_BAND} research response priority
        </div>

    </div>


    <div class="card {scpi_css_class}">

        <div class="card-title">
            SCPI
        </div>

        <div class="card-value">
            {SCPI:.2f}/100
        </div>

        <div class="card-sub">
            {SCPI_BAND} relative safety-centre pressure
        </div>

    </div>


</div>


<!-- ===================================================== -->
<!-- RISK + TREND -->
<!-- ===================================================== -->

<div class="two-column">


<div class="section">

<h2>
Experimental Impact Escalation Risk
</h2>

<p>

Predicted class:

<b>
{latest_risk_prediction}
</b>

</p>


<table>

<thead>

<tr>
    <th>Risk Class</th>
    <th>Model Probability</th>
</tr>

</thead>

<tbody>

{risk_probability_rows}

</tbody>

</table>


<p class="metric-note">

Held-out test Macro-F1 = 0.3373.
This component should be interpreted cautiously.

</p>

</div>


<div class="section">

<h2>
Impact Trend Monitoring
</h2>


<table>

<tr>
    <th>Trend</th>
    <td>{latest_trend_label}</td>
</tr>

<tr>
    <th>Change from previous report</th>

    <td>
        {latest_dashboard_trend['impact_absolute_change']:+,.0f}
    </td>
</tr>

<tr>
    <th>Rapid-growth score</th>
    <td>{latest_growth_score_display}</td>
</tr>

<tr>
    <th>Monitoring status</th>
    <td>{latest_monitoring_status}</td>
</tr>

</table>


<p class="metric-note">

Rapid-growth anomalies are research-derived
historical percentile indicators and are not official warnings.

</p>

</div>


</div>


<!-- ===================================================== -->
<!-- SOURCE FRESHNESS -->
<!-- ===================================================== -->

<div class="section">

<h2>
Latest Hazard-Data Freshness
</h2>


<table>

<thead>

<tr>
    <th>Source</th>
    <th>Latest Data Available</th>
    <th>Age at Snapshot</th>
    <th>Status</th>
</tr>

</thead>


<tbody>

<tr>

    <td>
        Landslide warning context
    </td>

    <td>
        {latest_landslide_datetime}
    </td>

    <td>
        {landslide_age_display}
    </td>

    <td class="
        {'status-current'
         if landslide_context_current
         else 'status-stale'}
    ">
        {landslide_status_display}
    </td>

</tr>


<tr>

    <td>
        River station context
    </td>

    <td>
        {latest_river_datetime}
    </td>

    <td>
        {river_age_display}
    </td>

    <td class="
        {'status-current'
         if river_context_current
         else 'status-stale'}
    ">
        {river_status_display}
    </td>

</tr>

</tbody>

</table>


<p class="metric-note">

Missing or stale information is not interpreted as
absence of a hazard.

</p>

</div>


<!-- ===================================================== -->
<!-- SIMILAR EVENTS -->
<!-- ===================================================== -->

<div class="section">

<h2>
Most Similar Historical Events
</h2>


<table>

<thead>

<tr>

    <th>
        Historical Date
    </th>

    <th>
        Similarity Score
    </th>

    <th>
        Affected People
    </th>

    <th>
        Safety-Centre Population
    </th>

    <th>
        Historical Escalation Class
    </th>

</tr>

</thead>


<tbody>

{similar_event_rows}

</tbody>

</table>


<p class="metric-note">

Similarity scores represent feature-space similarity,
not probability or forecast confidence.

</p>

</div>


<!-- ===================================================== -->
<!-- DISTRICT MAP -->
<!-- ===================================================== -->

<div class="section">

<h2>
Sri Lanka District Hazard Context
</h2>


<iframe
    src="{map_filename}">
</iframe>


<p class="metric-note">

District colours represent available district-level
hazard context only.

The national affected-population and safety-demand
predictions are not district-level forecasts.

</p>

</div>


<!-- ===================================================== -->
<!-- MODEL PERFORMANCE SUMMARY -->
<!-- ===================================================== -->

<div class="section">

<h2>
Held-Out Test Performance Summary
</h2>


<table>

<thead>

<tr>
    <th>Output</th>
    <th>Evaluation</th>
</tr>

</thead>


<tbody>


<tr>

    <td>
        Human Impact Forecast
    </td>

    <td>
        MAE = 21,306.60;
        RMSE = 65,900.17;
        RMSLE = 3.6472;
        R² = 0.6481.
        Persistence MAE = 21,283.91.
    </td>

</tr>


<tr>

    <td>
        Safety-Centre Usage
    </td>

    <td>
        Accuracy = 88.67%;
        Balanced Accuracy = 88.68%;
        F1 = 88.44%.
    </td>

</tr>


<tr>

    <td>
        Safety-Centre Population
    </td>

    <td>
        MAE = 439.27;
        RMSE = 947.27;
        R² = 0.9724.
    </td>

</tr>


<tr>

    <td>
        Experimental Escalation Risk
    </td>

    <td>
        Accuracy = 48.00%;
        Balanced Accuracy = 33.84%;
        Macro-F1 = 33.73%.
    </td>

</tr>


</tbody>

</table>

</div>


<!-- ===================================================== -->
<!-- LIMITATION NOTE -->
<!-- ===================================================== -->

<div class="section footer-note">

<b>Interpretation and limitations</b>

<br><br>

The dashboard is designed as a research decision-support prototype.
Situation Reports contain strong temporal persistence and cumulative
human-impact values. The final human-impact model did not consistently
outperform the persistence benchmark on held-out MAE.

<br><br>

Safety-centre usage classification showed comparatively stronger
out-of-sample performance. The escalation-risk classifier showed weaker
generalization, particularly for rare Critical cases, and is therefore
explicitly labelled experimental.

<br><br>

The model-based 90% human-impact interval achieved 80.67% empirical
coverage on the held-out test period and should not be interpreted as
a guaranteed confidence interval.

<br><br>

District-level map values represent hazard context only because the
available human-impact forecasting target is national rather than
district-specific.

</div>


</div>

</body>

</html>
"""


# =========================================================
# 9. SAVE DASHBOARD
# =========================================================

dashboard_path.write_text(
    dashboard_html,
    encoding="utf-8"
)


print(
    "Dashboard saved to:",
    dashboard_path
)


print(
    "Dashboard exists:",
    dashboard_path.exists()
)


print(
    "District map exists:",
    (
        processed_dir
        / map_filename
    ).exists()
)


# =========================================================
# 10. DISPLAY DASHBOARD LINK IN NOTEBOOK
# =========================================================

display(
    HTML(
        f"""
        <div style="
            padding:15px;
            border:1px solid #ccc;
            border-radius:8px;
            background:#f8fafc;
        ">

        <b>
        FloodImpact-LK Dashboard Generated
        </b>

        <br><br>

        File:
        <code>
        {dashboard_path}
        </code>

        <br><br>

        Open the HTML file in a browser to use
        the full interactive dashboard.

        </div>
        """
    )
)

# 17. Disaster-Specific Impact Feasibility Check

This section investigates whether DMC Situation Reports provide separate human-impact
and safety-centre statistics for floods and rainfall-induced landslides.

Disaster-specific forecasting will only be developed if the source reports provide
sufficiently separable target values.

In [ ]:
# =========================================================
# 17.1 CHECK DMC REPORTS FOR DISASTER-SPECIFIC IMPACT DATA
# =========================================================

from pathlib import Path
import re
import fitz
import pandas as pd


pdf_folder = Path(
    "data/raw/situation_reports"
)

pdf_files = sorted(
    pdf_folder.glob("*.pdf")
)


print(
    "Situation Report PDFs found:",
    len(pdf_files)
)


# ---------------------------------------------------------
# Search phrases
# ---------------------------------------------------------

disaster_patterns = {
    "Flood": [
        r"\bflood\b",
        r"\bflooding\b"
    ],

    "Landslide": [
        r"\blandslide\b",
        r"\blandslides\b"
    ]
}


impact_patterns = [
    r"affected people",
    r"affected persons",
    r"people affected",
    r"affected families",
    r"safety centre",
    r"safety center",
    r"safe centre",
    r"safe center",
    r"people in safety",
    r"families in safety"
]


# ---------------------------------------------------------
# Function to extract text
# ---------------------------------------------------------

def extract_pdf_text(pdf_path):

    try:

        doc = fitz.open(pdf_path)

        text_parts = []

        for page in doc:

            text_parts.append(
                page.get_text("text")
            )

        doc.close()

        return "\n".join(
            text_parts
        )

    except Exception as e:

        return ""


# ---------------------------------------------------------
# Context extraction
# ---------------------------------------------------------

def find_contexts(
    text,
    keyword_regex,
    window=500
):

    contexts = []

    for match in re.finditer(
        keyword_regex,
        text,
        flags=re.IGNORECASE
    ):

        start = max(
            0,
            match.start() - window
        )

        end = min(
            len(text),
            match.end() + window
        )

        snippet = (
            text[start:end]
            .replace("\n", " ")
        )

        snippet = re.sub(
            r"\s+",
            " ",
            snippet
        )

        contexts.append(
            snippet.strip()
        )

    return contexts


# ---------------------------------------------------------
# Scan reports
# ---------------------------------------------------------

results = []


for i, pdf_path in enumerate(
    pdf_files,
    start=1
):

    text = extract_pdf_text(
        pdf_path
    )


    if not text.strip():
        continue


    lower_text = text.lower()


    has_impact_term = any(
        re.search(
            pattern,
            lower_text,
            flags=re.IGNORECASE
        )
        for pattern in impact_patterns
    )


    if not has_impact_term:
        continue


    for disaster_name, patterns in (
        disaster_patterns.items()
    ):

        disaster_found = False

        all_contexts = []


        for pattern in patterns:

            matches = find_contexts(
                text,
                pattern,
                window=600
            )

            if matches:

                disaster_found = True

                all_contexts.extend(
                    matches[:3]
                )


        if not disaster_found:
            continue


        for context in all_contexts[:3]:

            impact_nearby = any(
                re.search(
                    pattern,
                    context,
                    flags=re.IGNORECASE
                )
                for pattern in impact_patterns
            )


            number_count = len(
                re.findall(
                    r"\b\d[\d,]*\b",
                    context
                )
            )


            results.append({

                "file_name":
                    pdf_path.name,

                "disaster_type":
                    disaster_name,

                "impact_term_nearby":
                    impact_nearby,

                "numbers_in_context":
                    number_count,

                "context":
                    context
            })


    if i % 100 == 0:

        print(
            f"Processed {i}/{len(pdf_files)} PDFs"
        )


# ---------------------------------------------------------
# Save diagnostic results
# ---------------------------------------------------------

disaster_specific_audit_df = (
    pd.DataFrame(
        results
    )
)


output_path = (
    "data/processed/"
    "disaster_specific_feasibility_audit.csv"
)


disaster_specific_audit_df.to_csv(
    output_path,
    index=False
)


print(
    "\nAudit rows:",
    len(
        disaster_specific_audit_df
    )
)


if len(
    disaster_specific_audit_df
) > 0:

    print(
        "\nRows by disaster:"
    )

    print(
        disaster_specific_audit_df[
            "disaster_type"
        ].value_counts()
    )


    print(
        "\nImpact term nearby:"
    )

    print(
        disaster_specific_audit_df[
            "impact_term_nearby"
        ].value_counts()
    )


print(
    "\nSaved to:",
    output_path
)

In [ ]:
# =========================================================
# 17.2 INSPECT DISASTER-SPECIFIC IMPACT CANDIDATES
# =========================================================

import pandas as pd
import re


audit_df = pd.read_csv(
    "data/processed/disaster_specific_feasibility_audit.csv"
)


# Keep only contexts where impact-related terminology
# occurs near Flood / Landslide text.
candidate_df = (
    audit_df[
        audit_df["impact_term_nearby"] == True
    ]
    .copy()
)


print("=" * 80)
print("DISASTER-SPECIFIC IMPACT CANDIDATE AUDIT")
print("=" * 80)

print(
    "\nTotal candidate contexts:",
    len(candidate_df)
)

print(
    "\nBy disaster:"
)

print(
    candidate_df[
        "disaster_type"
    ].value_counts()
)


# =========================================================
# Helper: make snippets easier to read
# =========================================================

def clean_context(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


candidate_df[
    "context_clean"
] = candidate_df[
    "context"
].apply(
    clean_context
)


# =========================================================
# FLOOD SAMPLES
# =========================================================

flood_samples = (
    candidate_df[
        candidate_df[
            "disaster_type"
        ] == "Flood"
    ]
    .drop_duplicates(
        subset=[
            "file_name"
        ]
    )
    .head(10)
)


print(
    "\n\n"
    + "=" * 80
)

print(
    "FLOOD CANDIDATE SAMPLES"
)

print(
    "=" * 80
)


for number, (_, row) in enumerate(
    flood_samples.iterrows(),
    start=1
):

    print(
        f"\n--- FLOOD SAMPLE {number} ---"
    )

    print(
        "File:",
        row[
            "file_name"
        ]
    )

    print(
        "\nContext:"
    )

    print(
        row[
            "context_clean"
        ]
    )

    print(
        "\n"
        + "-" * 80
    )


# =========================================================
# LANDSLIDE SAMPLES
# =========================================================

landslide_samples = (
    candidate_df[
        candidate_df[
            "disaster_type"
        ] == "Landslide"
    ]
    .drop_duplicates(
        subset=[
            "file_name"
        ]
    )
    .head(10)
)


print(
    "\n\n"
    + "=" * 80
)

print(
    "LANDSLIDE CANDIDATE SAMPLES"
)

print(
    "=" * 80
)


for number, (_, row) in enumerate(
    landslide_samples.iterrows(),
    start=1
):

    print(
        f"\n--- LANDSLIDE SAMPLE {number} ---"
    )

    print(
        "File:",
        row[
            "file_name"
        ]
    )

    print(
        "\nContext:"
    )

    print(
        row[
            "context_clean"
        ]
    )

    print(
        "\n"
        + "-" * 80
    )


# =========================================================
# SAVE UNIQUE CANDIDATES FOR MANUAL REVIEW
# =========================================================

review_df = (
    candidate_df[
        [
            "file_name",
            "disaster_type",
            "numbers_in_context",
            "context_clean"
        ]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


review_path = (
    "data/processed/"
    "disaster_specific_manual_review.csv"
)


review_df.to_csv(
    review_path,
    index=False
)


print(
    "\nManual-review rows:",
    len(review_df)
)

print(
    "Saved to:",
    review_path
)

# 17. Final Disaster-Oriented Dashboard Export

In [ ]:
# =========================================================
# 17. FINAL DISASTER-ORIENTED DASHBOARD EXPORT
#
# Dashboard structure:
# 1. National Overview
# 2. Flood
# 3. Rainfall-Induced Landslide
#
# IMPORTANT:
# Human-impact and safety-centre forecasts are national.
# They are NOT split into Flood/Landslide unless source
# data provide valid disaster-specific targets.
# =========================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


processed_dir = Path(
    "data/processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# 1. LATEST SNAPSHOT
# =========================================================

latest_row = (
    latest_current_row.iloc[0]
    if isinstance(
        latest_current_row,
        pd.DataFrame
    )
    else latest_current_row
)


snapshot_dt = pd.to_datetime(
    latest_row[
        "availability_datetime"
    ]
)


# =========================================================
# 2. REPORTED HAZARDS
# =========================================================

reported_hazards = [
    item.strip()
    for item in str(
        latest_row.get(
            "disaster_types",
            ""
        )
    ).split(",")
    if item.strip()
]


# =========================================================
# 3. SAFE JSON CONVERTER
# =========================================================

def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        pd.Timestamp
    ):
        return value.isoformat()

    if isinstance(
        value,
        np.integer
    ):
        return int(value)

    if isinstance(
        value,
        np.floating
    ):

        if np.isnan(value):
            return None

        return float(value)

    if isinstance(
        value,
        np.bool_
    ):
        return bool(value)

    if isinstance(
        value,
        float
    ) and np.isnan(value):
        return None

    return value


def clean_json_object(obj):

    if isinstance(
        obj,
        dict
    ):

        return {
            str(key):
                clean_json_object(value)

            for key, value
            in obj.items()
        }

    if isinstance(
        obj,
        list
    ):

        return [
            clean_json_object(value)
            for value in obj
        ]

    return json_safe(
        obj
    )


# =========================================================
# 4. NATIONAL OVERVIEW
# =========================================================

national_overview = {

    "snapshot":
        snapshot_dt.strftime(
            "%Y-%m-%d %H:%M"
        ),

    "reported_hazards":
        reported_hazards,

    "current_affected_people":
        float(
            latest_current_affected
        ),

    "predicted_next_affected_people":
        float(
            latest_predicted_affected
        ),

    "predicted_change":
        float(
            latest_predicted_delta
        ),

    "prediction_lower_90":
        float(
            latest_lower_90
        ),

    "prediction_upper_90":
        float(
            latest_upper_90
        ),

    "current_safety_population":
        float(
            latest_current_safety_population
        ),

    "safety_use_probability":
        float(
            latest_safety_use_probability
        ),

    "predicted_safety_population":
        float(
            latest_predicted_safety_population
        ),

    "experimental_escalation_risk":
        str(
            latest_risk_prediction
        ),

    "risk_probabilities":
        {
            str(key):
                float(value)

            for key, value
            in latest_risk_probability_dict.items()
        },

    "drpi":
        float(
            DRPI
        ),

    "drpi_band":
        str(
            DRPI_BAND
        ),

    "scpi":
        float(
            SCPI
        ),

    "scpi_band":
        str(
            SCPI_BAND
        ),

    "important_note":
        (
            "Human-impact and safety-centre values are "
            "national Situation Report totals. "
            "They are not disaster-specific or district-specific."
        )
}


# =========================================================
# 5. FLOOD VIEW
# =========================================================

flood_view = {

    "disaster":
        "Flood",

    "snapshot":
        snapshot_dt.strftime(
            "%Y-%m-%d %H:%M"
        ),

    "flood_reported_in_latest_situation":
        bool(
            latest_row.get(
                "current_flood_flag",
                0
            )
        ),

    "national_affected_people_context":
        float(
            latest_current_affected
        ),

    "national_predicted_next_affected_context":
        float(
            latest_predicted_affected
        ),

    "national_prediction_lower_90":
        float(
            latest_lower_90
        ),

    "national_prediction_upper_90":
        float(
            latest_upper_90
        ),

    "weather_reports_24h":
        json_safe(
            latest_row.get(
                "weather_reports_24h",
                np.nan
            )
        ),

    "weather_max_warning_level_24h":
        json_safe(
            latest_row.get(
                "weather_max_warning_level_24h",
                np.nan
            )
        ),

    "weather_max_rainfall_threshold_24h":
        json_safe(
            latest_row.get(
                "weather_max_rainfall_threshold_24h",
                np.nan
            )
        ),

    "river_latest_datetime":
        (
            pd.to_datetime(
                latest_river_datetime
            ).isoformat()

            if pd.notna(
                latest_river_datetime
            )

            else None
        ),

    "river_age_hours":
        (
            float(
                river_age_hours
            )

            if pd.notna(
                river_age_hours
            )

            else None
        ),

    "river_status":
        (
            "Current"
            if river_context_current
            else "Stale / Unavailable"
        ),

    "river_station_count":
        int(
            len(
                latest_river_records
            )
        ),

    "river_alert_flood_station_count":
        int(
            len(
                latest_river_relevant_df
            )
        ),

    "flood_context_score":
        float(
            flood_context_score
        ),

    "combined_hazard_context_score":
        float(
            hazard_context_score
        ),

    "forecast_note":
        (
            "The displayed affected-population forecast is "
            "the national multi-hazard forecast and must not "
            "be interpreted as a Flood-only affected count."
        )
}


# =========================================================
# 6. LANDSLIDE VIEW
# =========================================================

landslide_view = {

    "disaster":
        "Rainfall-Induced Landslide",

    "snapshot":
        snapshot_dt.strftime(
            "%Y-%m-%d %H:%M"
        ),

    "landslide_reported_in_latest_situation":
        bool(
            latest_row.get(
                "current_landslide_flag",
                0
            )
        ),

    "national_affected_people_context":
        float(
            latest_current_affected
        ),

    "national_predicted_next_affected_context":
        float(
            latest_predicted_affected
        ),

    "national_prediction_lower_90":
        float(
            latest_lower_90
        ),

    "national_prediction_upper_90":
        float(
            latest_upper_90
        ),

    "heavy_rain_reported":
        bool(
            latest_row.get(
                "current_heavy_rain_flag",
                0
            )
        ),

    "weather_reports_24h":
        json_safe(
            latest_row.get(
                "weather_reports_24h",
                np.nan
            )
        ),

    "weather_max_warning_level_24h":
        json_safe(
            latest_row.get(
                "weather_max_warning_level_24h",
                np.nan
            )
        ),

    "weather_max_rainfall_threshold_24h":
        json_safe(
            latest_row.get(
                "weather_max_rainfall_threshold_24h",
                np.nan
            )
        ),

    "landslide_latest_datetime":
        (
            pd.to_datetime(
                latest_landslide_datetime
            ).isoformat()

            if pd.notna(
                latest_landslide_datetime
            )

            else None
        ),

    "landslide_age_hours":
        (
            float(
                landslide_age_hours
            )

            if pd.notna(
                landslide_age_hours
            )

            else None
        ),

    "landslide_status":
        (
            "Current"
            if landslide_context_current
            else "Stale / Unavailable"
        ),

    "landslide_context_score":
        float(
            landslide_context_score
        ),

    "combined_hazard_context_score":
        float(
            hazard_context_score
        ),

    "district_map_file":
        (
            "floodimpact_lk_district_hazard_map.html"
        ),

    "forecast_note":
        (
            "The displayed affected-population forecast is "
            "the national multi-hazard forecast and must not "
            "be interpreted as a Landslide-only affected count."
        )
}


# =========================================================
# 7. FINAL DASHBOARD OBJECT
# =========================================================

final_dashboard_data = {

    "project":
        "FloodImpact-LK",

    "dashboard_structure": [
        "National Overview",
        "Flood",
        "Rainfall-Induced Landslide"
    ],

    "national_overview":
        national_overview,

    "flood":
        flood_view,

    "landslide":
        landslide_view,

    "scope_warning":
        (
            "FloodImpact-LK is a research decision-support "
            "prototype and does not replace official DMC, "
            "NBRO, Meteorology or Irrigation Department warnings."
        )
}


final_dashboard_data = (
    clean_json_object(
        final_dashboard_data
    )
)


# =========================================================
# 8. SAVE
# =========================================================

output_path = (
    processed_dir
    / "final_dashboard_data.json"
)


output_path.write_text(
    json.dumps(
        final_dashboard_data,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# =========================================================
# 9. QC
# =========================================================

print(
    "=" * 70
)

print(
    "FINAL DISASTER-ORIENTED DASHBOARD EXPORT"
)

print(
    "=" * 70
)


print(
    "Saved to:",
    output_path
)


print(
    "File exists:",
    output_path.exists()
)


print(
    "\nDashboard views:"
)

print(
    "1. National Overview"
)

print(
    "2. Flood"
)

print(
    "3. Rainfall-Induced Landslide"
)


print(
    "\nCurrent reported hazards:",
    reported_hazards
)


print(
    "\nFlood context:"
)

print(
    "Flood mentioned:",
    flood_view[
        "flood_reported_in_latest_situation"
    ]
)

print(
    "River status:",
    flood_view[
        "river_status"
    ]
)

print(
    "River alert/flood stations:",
    flood_view[
        "river_alert_flood_station_count"
    ]
)


print(
    "\nLandslide context:"
)

print(
    "Landslide mentioned:",
    landslide_view[
        "landslide_reported_in_latest_situation"
    ]
)

print(
    "Landslide data status:",
    landslide_view[
        "landslide_status"
    ]
)


print(
    "\nIMPORTANT:"
)

print(
    "National affected/safety forecasts are not "
    "labelled as Flood-only or Landslide-only."
)

print(
    "=" * 70
)